In [1]:
# ================================================================
# T1.4b — GENERATOR-MATCHED SPATIAL NULL ATTRIBUTION SPECIFICITY
# CELL 1 — CONFIGURATION + ENVIRONMENT
# ================================================================

import os
import sys
import json
import time
import shutil
import hashlib
import inspect
import importlib.util
import platform
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms.functional as TF


# ================================================================
# T1.4b LOCKED EXPERIMENT CONFIGURATION
# ================================================================

EXPERIMENT_NAME = "T1_4B"

# ------------------------------------------------
# Production model roster
# ------------------------------------------------

RHO_LEVELS = [
    0.0,
    0.2,
    0.4,
    0.6,
    0.8,
    1.0,
]

REPLICATES = [1, 2, 3, 4, 5]

EXPECTED_MODEL_COUNT = (
    len(RHO_LEVELS) * len(REPLICATES)
)

assert EXPECTED_MODEL_COUNT == 30


# ------------------------------------------------
# Image sample
# ------------------------------------------------

N_IMAGES = 100
N_BENIGN = 50
N_MALIGNANT = 50

SAMPLE_SEED = 20260925_1


# ------------------------------------------------
# Null-mask design
# ------------------------------------------------

NULL_NAMESPACE = "t1_4b-null-v2"

N_NULL_A = 50
N_NULL_CPRIME = 50

NULL_ATTEMPT_CAP = 20_000


# ------------------------------------------------
# Frozen IG convention
# ------------------------------------------------

# These are EXPECTED values.
# Later cells will verify them directly against
# the frozen analysis_pipeline.py.

EXPECTED_IG_STEPS = 50
EXPECTED_IG_RETRY_STEPS = 100
EXPECTED_IG_SIGMA = 10.0


# ------------------------------------------------
# Primary statistical test
# ------------------------------------------------

ALPHA = 0.05

SIGN_FLIP_PERMUTATIONS = 100_000
SIGN_FLIP_SEED = 20260919


# ================================================================
# KAGGLE DIRECTORIES
# ================================================================

KAGGLE_ROOT = Path("/kaggle")

INPUT_ROOT = KAGGLE_ROOT / "input"
WORK_ROOT = KAGGLE_ROOT / "working"
TEMP_ROOT = KAGGLE_ROOT / "temp"


# ================================================================
# T1.4b OUTPUT DIRECTORY
# ================================================================

RESULTS_ROOT = (
    WORK_ROOT /
    "T1_4B_generator_matched_spatial_null"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# Important output subdirectories

LOCKS_DIR = RESULTS_ROOT / "locks"
MASKS_DIR = RESULTS_ROOT / "null_masks"
IG_DIR = RESULTS_ROOT / "ig_maps"
MODEL_RESULTS_DIR = RESULTS_ROOT / "model_level_results"
AUDIT_DIR = RESULTS_ROOT / "audits"
LOG_DIR = RESULTS_ROOT / "logs"

for d in [
    LOCKS_DIR,
    MASKS_DIR,
    IG_DIR,
    MODEL_RESULTS_DIR,
    AUDIT_DIR,
    LOG_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )


# ================================================================
# DEVICE
# ================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ================================================================
# ENVIRONMENT REPORT
# ================================================================

print("=" * 75)
print("T1.4b — GENERATOR-MATCHED SPATIAL NULL")
print("=" * 75)

print("\nEXPERIMENT CONFIGURATION")
print("-" * 75)

print("Experiment:", EXPERIMENT_NAME)

print("Rho levels:", RHO_LEVELS)
print("Replicates:", REPLICATES)
print("Expected models:", EXPECTED_MODEL_COUNT)

print("Images:", N_IMAGES)
print("Benign:", N_BENIGN)
print("Malignant:", N_MALIGNANT)
print("Sample seed:", SAMPLE_SEED)

print("Null namespace:", NULL_NAMESPACE)
print("Nulls / image / Design A:", N_NULL_A)
print("Nulls / image / Design C':", N_NULL_CPRIME)
print("Attempt cap:", NULL_ATTEMPT_CAP)

print("\nSTATISTICAL CONFIGURATION")
print("-" * 75)

print("Alpha:", ALPHA)
print("Sign-flip permutations:", SIGN_FLIP_PERMUTATIONS)
print("Sign-flip seed:", SIGN_FLIP_SEED)

print("\nEXPECTED IG CONFIGURATION")
print("-" * 75)

print("IG steps:", EXPECTED_IG_STEPS)
print("IG retry steps:", EXPECTED_IG_RETRY_STEPS)
print("IG sigma:", EXPECTED_IG_SIGMA)

print("\nENVIRONMENT")
print("-" * 75)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Platform:", platform.platform())

print("Device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU count:",
        torch.cuda.device_count()
    )

    for gpu_id in range(torch.cuda.device_count()):

        print(
            f"GPU {gpu_id}:",
            torch.cuda.get_device_name(gpu_id)
        )

        props = torch.cuda.get_device_properties(
            gpu_id
        )

        print(
            f"  VRAM:",
            round(
                props.total_memory / (1024 ** 3),
                2
            ),
            "GB"
        )

else:

    print("WARNING: CUDA is NOT available.")


print("\nDIRECTORIES")
print("-" * 75)

print("INPUT :", INPUT_ROOT)
print("WORK  :", WORK_ROOT)
print("TEMP  :", TEMP_ROOT)
print("RESULT:", RESULTS_ROOT)

assert INPUT_ROOT.exists(), (
    "Kaggle input directory does not exist."
)

assert WORK_ROOT.exists(), (
    "Kaggle working directory does not exist."
)

print("\n" + "=" * 75)
print("CELL 1 COMPLETE")
print("=" * 75)

T1.4b — GENERATOR-MATCHED SPATIAL NULL

EXPERIMENT CONFIGURATION
---------------------------------------------------------------------------
Experiment: T1_4B
Rho levels: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
Replicates: [1, 2, 3, 4, 5]
Expected models: 30
Images: 100
Benign: 50
Malignant: 50
Sample seed: 202609251
Null namespace: t1_4b-null-v2
Nulls / image / Design A: 50
Nulls / image / Design C': 50
Attempt cap: 20000

STATISTICAL CONFIGURATION
---------------------------------------------------------------------------
Alpha: 0.05
Sign-flip permutations: 100000
Sign-flip seed: 20260919

EXPECTED IG CONFIGURATION
---------------------------------------------------------------------------
IG steps: 50
IG retry steps: 100
IG sigma: 10.0

ENVIRONMENT
---------------------------------------------------------------------------
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Device: cuda
GPU 

In [7]:
# ================================================================
# CELL 1 — CONFIG (ANALYSIS NOTEBOOK)
# ================================================================

AUTO_PIN_VERSIONS = globals().get("AUTO_PIN_VERSIONS", True)
ALLOW_VERSION_DEVIATION = globals().get("ALLOW_VERSION_DEVIATION", False)
N_PROC_OVERRIDE = globals().get("N_PROC_OVERRIDE", 4)
MAX_MODELS_THIS_RUN = globals().get("MAX_MODELS_THIS_RUN", None)
DEADLINE_HOURS = globals().get("DEADLINE_HOURS", 11.0)
EST_MODEL_HOURS = 0.15

print("✓ analysis-notebook config loaded")

✓ analysis-notebook config loaded


In [8]:
# ================================================================
# CELL 2 — HELPERS
# ================================================================

import os, sys, json, time, shutil, hashlib, subprocess, platform
from pathlib import Path
from collections import deque
from datetime import datetime, timezone

ROOT = Path(os.environ.get("KAGGLE_TEST_ROOT", "/kaggle"))
INPUT, WORK = ROOT / "input", ROOT / "working"
TEMP = ROOT / "temp"

TEMP.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

PROJECT = WORK / "isic_final_project"
CACHE = TEMP / "isic_cache"
RECORDS = WORK / "analysis_records"
RESULTS = WORK / "results"
TRAINED_MODELS = WORK / "trained_models_readonly"
LOCAL_TMP = TEMP / "tmp_work"

for d in (RECORDS, RESULTS, LOCAL_TMP):
    d.mkdir(parents=True, exist_ok=True)

T0 = time.time()

def utc():
    return datetime.now(timezone.utc).isoformat()

def elapsed_h():
    return (time.time() - T0) / 3600

def log(msg):
    print(f"[{elapsed_h():5.2f} h] {msg}", flush=True)

def sha_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def J(rel):
    return json.load(open(PROJECT / rel))

def save_record(name, obj):
    with open(RECORDS / name, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)

def find_by_markers(root, required_subdirs, max_depth=6):
    root = Path(root)
    if not root.exists():
        return None

    required = set(required_subdirs)
    q = deque([(root, 0)])

    while q:
        d, depth = q.popleft()

        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue

        names = {e.name for e in entries if e.is_dir()}

        if required <= names:
            return d

        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))

    return None

def _has_min_files(d, exts, min_count, scan_cap=20000):
    n = 0

    try:
        with os.scandir(d) as it:
            for i, e in enumerate(it):
                if i >= scan_cap:
                    break

                if e.is_file() and e.name.lower().endswith(exts):
                    n += 1

                    if n >= min_count:
                        return True

    except (PermissionError, FileNotFoundError, NotADirectoryError):
        return False

    return False

def find_image_dir(root, exts=(".jpg", ".jpeg"), min_images=100, max_depth=6):
    root = Path(root)

    if not root.exists():
        return None

    q = deque([(root, 0)])
    found = []

    while q:
        d, depth = q.popleft()

        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue

        if _has_min_files(d, exts, min_images):
            found.append(d)
            continue

        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))

    if not found:
        return None

    found.sort(
        key=lambda p: (
            0 if "train" in str(p).lower() else 1,
            len(str(p))
        )
    )

    return found[0]

print("✓ helpers and paths loaded")
print("CACHE:", CACHE)

✓ helpers and paths loaded
CACHE: /kaggle/temp/isic_cache


In [9]:
# ================================================================
# STAGE 0 — BRING THE LOCKED PROJECT IN AND VERIFY THE WHOLE LOCK CHAIN
# ================================================================
PROJECT_MARKERS = {"step0I_locked_clean_population", "step1F_lesion_level_rho_assignments"}

if not PROJECT.exists():
    # Kaggle's mount layout is not fixed (seen both /kaggle/input/<slug>/... and /kaggle/input/datasets/<owner>/<slug>/...),
    # so this SEARCHES under /kaggle/input for the directory that actually has the two lock-chain marker folders,
    # rather than assuming a fixed depth. The dataset itself is never modified - only read and copied.
    src = find_by_markers(INPUT, PROJECT_MARKERS)
    assert src is not None, (
        "Project folder not found anywhere under /kaggle/input (searched up to 6 levels deep for a directory containing "
        f"{sorted(PROJECT_MARKERS)}). Check that your project dataset is attached under Add Data, and that the zip you "
        "uploaded actually has these folders at its top level (not nested inside an extra wrapper folder Kaggle can't see into).")
    log(f"found locked project at {src} - copying (read-only source, nothing there is modified) ...")
    shutil.copytree(src, PROJECT)
n_files = sum(1 for p in PROJECT.rglob("*") if p.is_file())
print(f"project working copy: {PROJECT}  ({n_files} files)")

# ---- verify every locked artifact against its recorded hash (environment and cache are re-recorded on this platform, so they are not checked here)
checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok))); print(("✓ " if ok else "✗ ") + name + (f"   {detail}" if detail else "")); assert ok, "LOCK CHAIN CHECK FAILED: " + name
sha = lambda rel: sha_file(PROJECT / rel)
F = J("step1U_final_protocol_lock/FINAL_PROTOCOL_HASH.json")
check("final protocol == recorded hash", sha("step1U_final_protocol_lock/FINAL_LOCKED_EXPERIMENT_PROTOCOL.json") == F["protocol_sha256"])
check("seed manifest == recorded hash", sha("step1P_seed_manifest/seed_manifest.json") == F["seed_manifest_sha256"])
_cand = {sha("step1I_protocol_lock/LOCKED_EXPERIMENT_PROTOCOL.json")} | {v for v in J("step1I_protocol_lock/protocol_hash.json").values() if isinstance(v, str)}
check("parent 1I protocol hash", F["parent_step1I_sha256"] in _cand)
check("A1 == recorded hash", sha("step2O_A1_final_lock/FINAL_LOCKED_AMENDMENT_A1.json") == J("step2O_A1_final_lock/A1_FINAL_HASH.json")["sha256"])
check("A1.1 == recorded hash", sha("step2P_A1_1_solver_addendum_lock/FINAL_LOCKED_A1_1_SOLVER_ADDENDUM.json") == J("step2P_A1_1_solver_addendum_lock/A1_1_FINAL_HASH.json")["sha256"])
GEN_SHA = J("step1J_generator_recovery_lock/generator_recovery_lock.json")["generator_sha256"]
check("generator == recorded hash", sha("step1J_generator_recovery_lock/recovered_original_hair_generator.py") == GEN_SHA)
check("solver == recorded hash", sha("step2Q_solver_implementation_v2/deterministic_evaluation_solver.py") == J("step2Q_solver_implementation_v2/solver_implementation_metadata.json")["solver_sha256"])
check("analysis code == frozen hash", sha("step2W_analysis_code_freeze/analysis_pipeline.py") == J("step2W_analysis_code_freeze/ANALYSIS_CODE_FINAL_HASH.json")["sha256"])
check("training code == frozen hash", sha("step2X_training_code_freeze/train_one_model.py") == J("step2X_training_code_freeze/TRAINING_CODE_FINAL_HASH.json")["sha256"])
E = J("step2S_evaluation_manifest_lock/EVALUATION_MANIFEST_FINAL_HASH.json")
check("3 evaluation manifests == locked hashes", all(sha(f"step2R_production_evaluation_manifests/evaluation_{c}.csv") == h for c, h in E["manifest_sha256"].items()))
check("XAI subset == locked hash", sha("step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv") == J("step2U_xai_subset_lock/XAI_SUBSET_FINAL_HASH.json")["sha256"])
RO = J("step2V_seed_table_and_run_order/RUN_ORDER_FINAL_HASH.json")
check("run-order manifest == locked hash", sha("step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json") == RO["sha256"])
check("seed table == locked hash", sha("step2V_seed_table_and_run_order/TRAINING_SEED_TABLE.csv") == RO["seed_table_sha256"])
asg = list((PROJECT / "step1F_lesion_level_rho_assignments").glob("rho_*_rep_*.csv"))
check("30 training assignment files present", len(asg) == 30, f"({len(asg)})")
RUN_ORDER = json.load(open(PROJECT / "step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json"))["run_order"]
check("run order lists 30 models", len(RUN_ORDER) == 30)
TRAIN = PROJECT / "step2X_training_code_freeze" / "train_one_model.py"
ANALYSIS = PROJECT / "step2W_analysis_code_freeze" / "analysis_pipeline.py"
save_record("stage0_lock_chain_verified.json", {"verified": dict(checks), "project_files": n_files, "utc": utc()})
print("\nSTAGE 0 PASSED: the locked project arrived intact.")


[ 0.00 h] found locked project at /kaggle/input/datasets/tejasseshadriiiii/isic-final-project - copying (read-only source, nothing there is modified) ...
project working copy: /kaggle/working/isic_final_project  (868 files)
✓ final protocol == recorded hash
✓ seed manifest == recorded hash
✓ parent 1I protocol hash
✓ A1 == recorded hash
✓ A1.1 == recorded hash
✓ generator == recorded hash
✓ solver == recorded hash
✓ analysis code == frozen hash
✓ training code == frozen hash
✓ 3 evaluation manifests == locked hashes
✓ XAI subset == locked hash
✓ run-order manifest == locked hash
✓ seed table == locked hash
✓ 30 training assignment files present   (30)
✓ run order lists 30 models

STAGE 0 PASSED: the locked project arrived intact.


In [10]:
# ================================================================
# STAGE 0b — COMPATIBILITY SYMLINK
# ================================================================

CONTENT_PROJECT = Path("/content/isic_final_project")

if CONTENT_PROJECT.exists():
    print(f"✓ {CONTENT_PROJECT} already exists - leaving it as is")
else:
    CONTENT_PROJECT.parent.mkdir(parents=True, exist_ok=True)

    try:
        CONTENT_PROJECT.symlink_to(
            PROJECT,
            target_is_directory=True
        )

        print(
            f"✓ compatibility symlink created: "
            f"{CONTENT_PROJECT} -> {PROJECT}"
        )

    except OSError as e:
        print(f"Symlink failed ({e}) - falling back to a full copy")
        shutil.copytree(PROJECT, CONTENT_PROJECT)

gen_sha = sha_file(
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

assert gen_sha == json.load(
    open(
        PROJECT
        / "step1J_generator_recovery_lock"
        / "generator_recovery_lock.json"
    )
)["generator_sha256"]

print("✓ generator file hash unchanged")

✓ compatibility symlink created: /content/isic_final_project -> /kaggle/working/isic_final_project
✓ generator file hash unchanged


In [11]:
# ================================================================
# STAGE 2 — BUILD THE 224x224 CACHE ON KAGGLE
#
# CPU / multiprocessing only — CUDA is NOT used here.
#
# IMPORTANT:
# - Scientific hair generator is NOT modified.
# - Only imports, literal constants, and function definitions
#   are loaded from the recovered generator.
# - Module-level validation/test code is deliberately not executed.
# - Locked population + medium severity + deterministic generator
#   are preserved.
# ================================================================

import textwrap
import subprocess
import time
import json
import hashlib
from pathlib import Path

# ----------------------------------------------------------------
# Locate ISIC image directory
# ----------------------------------------------------------------

IMAGE_DIR = (
    globals().get("IMAGE_DIR_OVERRIDE")
    or find_image_dir(INPUT, min_images=100)
)

assert IMAGE_DIR is not None, (
    "ISIC image folder not found anywhere under /kaggle/input. "
    "Check that the isic-2019-dataset-full dataset is attached "
    "through Add Data."
)

print("image folder:", IMAGE_DIR)

# ----------------------------------------------------------------
# Required locked paths
# ----------------------------------------------------------------

GEN_PATH = (
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

POP_PATH = (
    PROJECT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

CACHE_LOCK = RECORDS / "cache_final_hash.json"

CACHE.mkdir(
    parents=True,
    exist_ok=True
)

assert GEN_PATH.exists(), (
    f"Recovered hair generator not found:\n{GEN_PATH}"
)

assert POP_PATH.exists(), (
    f"Locked development population not found:\n{POP_PATH}"
)

# ----------------------------------------------------------------
# Required cache files
# ----------------------------------------------------------------

REQUIRED_CACHE_FILES = [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
]

cache_files_present = all(
    (CACHE / f).exists()
    for f in REQUIRED_CACHE_FILES
)

# ----------------------------------------------------------------
# Existing complete cache?
# ----------------------------------------------------------------

if (
    CACHE_LOCK.exists()
    and cache_files_present
    and not globals().get("ALLOW_RERUN_CACHE", False)
):

    print("=" * 80)
    print("STAGE 2 — EXISTING CACHE DETECTED")
    print("=" * 80)

    print("✓ cache_final_hash.json exists")

    for f in REQUIRED_CACHE_FILES:
        print("✓", CACHE / f)

    print("\n✓ Complete cache already exists.")
    print("Skipping rebuild.")

else:

    # ============================================================
    # Remove partial/stale cache
    # ============================================================

    print("=" * 80)
    print("STAGE 2 — PREPARING CLEAN CACHE BUILD")
    print("=" * 80)

    if CACHE_LOCK.exists():
        print("Removing stale/incomplete cache lock:", CACHE_LOCK)
        CACHE_LOCK.unlink()

    for name in [
        "clean_224.npy",
        "hair_224.npy",
        "mask_area.npy",
        "cache_index.csv",
        "_builder_record.json",
    ]:

        p = CACHE / name

        if p.exists():
            p.unlink()
            print("removed:", p)

    print("\n✓ Partial cache cleared.")

    # ============================================================
    # Standalone CPU cache-builder subprocess
    # ============================================================

    BUILDER = TEMP / "cache_builder.py"

    BUILDER.write_text(
        textwrap.dedent(
            r'''
            import argparse
            import ast
            import hashlib
            import json
            import random
            from pathlib import Path

            import numpy as np
            import pandas as pd
            from PIL import Image
            import multiprocessing as mp

            SEV = "medium"

            def sha_file(p, chunk=1 << 22):
                h = hashlib.sha256()

                with open(p, "rb") as f:
                    for b in iter(
                        lambda: f.read(chunk),
                        b""
                    ):
                        h.update(b)

                return h.hexdigest()

            def find_image(image_dir, image_id):

                for ext in (
                    ".jpg",
                    ".jpeg",
                    ".JPG",
                    ".JPEG"
                ):

                    p = (
                        image_dir
                        / f"{image_id}{ext}"
                    )

                    if p.exists():
                        return p

                raise FileNotFoundError(image_id)

            _G = {}

            # ====================================================
            # SAFE GENERATOR LOADER
            # ====================================================

            def load_generator(gen_path):

                source = Path(gen_path).read_text(
                    encoding="utf-8"
                )

                tree = ast.parse(source)

                safe_nodes = []

                for node in tree.body:

                    if isinstance(
                        node,
                        (
                            ast.Import,
                            ast.ImportFrom,
                        )
                    ):

                        safe_nodes.append(node)
                        continue

                    if isinstance(
                        node,
                        (
                            ast.FunctionDef,
                            ast.AsyncFunctionDef,
                        )
                    ):

                        safe_nodes.append(node)
                        continue

                    if isinstance(node, ast.Assign):

                        try:
                            ast.literal_eval(node.value)
                            safe_nodes.append(node)

                        except Exception:
                            pass

                    elif isinstance(node, ast.AnnAssign):

                        if node.value is not None:

                            try:
                                ast.literal_eval(node.value)
                                safe_nodes.append(node)

                            except Exception:
                                pass

                module = ast.Module(
                    body=safe_nodes,
                    type_ignores=[]
                )

                ast.fix_missing_locations(module)

                ns = {
                    "__file__": str(gen_path),
                    "__name__": "recovered_hair_generator",
                }

                exec(
                    compile(
                        module,
                        str(gen_path),
                        "exec"
                    ),
                    ns
                )

                assert "generate_hair_mask" in ns
                assert "apply_hair" in ns

                return ns

            # ====================================================
            # WORKER INITIALIZER
            # ====================================================

            def _init(
                gen_path,
                cp,
                hp,
                image_dir
            ):

                ns = load_generator(gen_path)

                _G.update(
                    gm=ns["generate_hair_mask"],
                    ah=ns["apply_hair"],

                    clean=np.load(
                        cp,
                        mmap_mode="r+"
                    ),

                    hair=np.load(
                        hp,
                        mmap_mode="r+"
                    ),

                    image_dir=Path(image_dir)
                )

            # ====================================================
            # WORKER
            # ====================================================

            def _work(job):

                lo, ids = job
                out = []

                for r, image_id in zip(
                    range(
                        lo,
                        lo + len(ids)
                    ),
                    ids
                ):

                    clean = np.asarray(
                        Image.open(
                            find_image(
                                _G["image_dir"],
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )

                    mask = _G["gm"](
                        image_id=image_id,
                        severity=SEV
                    )

                    hair = _G["ah"](
                        image=clean,
                        mask=mask,
                        severity=SEV,
                        image_id=image_id
                    )

                    m = mask > 0

                    outside = int(
                        (
                            np.any(
                                hair != clean,
                                axis=2
                            )
                            & ~m
                        ).sum()
                    )

                    _G["clean"][r] = clean
                    _G["hair"][r] = hair

                    out.append(
                        (
                            r,
                            int(m.sum()),
                            outside
                        )
                    )

                _G["clean"].flush()
                _G["hair"].flush()

                return out

            # ====================================================
            # MAIN
            # ====================================================

            def main():

                ap = argparse.ArgumentParser()

                ap.add_argument(
                    "--gen-path",
                    required=True
                )

                ap.add_argument(
                    "--pop-path",
                    required=True
                )

                ap.add_argument(
                    "--cache-dir",
                    required=True
                )

                ap.add_argument(
                    "--image-dir",
                    required=True
                )

                ap.add_argument(
                    "--n-proc",
                    type=int,
                    default=4
                )

                a = ap.parse_args()

                gen_path = Path(a.gen_path)
                pop_path = Path(a.pop_path)
                cache_dir = Path(a.cache_dir)
                image_dir = Path(a.image_dir)

                # ------------------------------------------------
                # Locked population
                # ------------------------------------------------

                pop = pd.read_csv(pop_path)

                assert list(pop.columns) == [
                    "image",
                    "lesion_id",
                    "class_name",
                    "malignant",
                    "split"
                ]

                pop = (
                    pop
                    .sort_values(
                        [
                            "split",
                            "image"
                        ]
                    )
                    .reset_index(drop=True)
                )

                N = len(pop)

                print(
                    f"locked population: {N} images",
                    flush=True
                )

                # ------------------------------------------------
                # Cache index
                # ------------------------------------------------

                idx = pop.copy()

                idx["row"] = np.arange(N)

                idx.to_csv(
                    cache_dir / "cache_index.csv",
                    index=False
                )

                # ------------------------------------------------
                # Allocate arrays
                # ------------------------------------------------

                clean_p = cache_dir / "clean_224.npy"
                hair_p = cache_dir / "hair_224.npy"
                area_p = cache_dir / "mask_area.npy"

                for p in (
                    clean_p,
                    hair_p
                ):

                    np.lib.format.open_memmap(
                        p,
                        mode="w+",
                        dtype=np.uint8,
                        shape=(
                            N,
                            224,
                            224,
                            3
                        )
                    ).flush()

                # ------------------------------------------------
                # Jobs
                # ------------------------------------------------

                ids = (
                    pop["image"]
                    .astype(str)
                    .tolist()
                )

                jobs = [
                    (
                        lo,
                        ids[
                            lo:
                            lo + 64
                        ]
                    )
                    for lo in range(
                        0,
                        N,
                        64
                    )
                ]

                area = np.zeros(
                    N,
                    dtype=np.int32
                )

                outside_total = 0
                done = 0

                print(
                    f"starting {a.n_proc} cache workers...",
                    flush=True
                )

                ctx = mp.get_context("fork")

                with ctx.Pool(
                    a.n_proc,
                    initializer=_init,
                    initargs=(
                        str(gen_path),
                        str(clean_p),
                        str(hair_p),
                        str(image_dir),
                    )
                ) as pool:

                    for res in pool.imap_unordered(
                        _work,
                        jobs
                    ):

                        for (
                            r,
                            ar,
                            o
                        ) in res:

                            area[r] = ar
                            outside_total += o

                        done += len(res)

                        print(
                            f"cache {done:6d}/{N}",
                            flush=True
                        )

                # ------------------------------------------------
                # Causal-integrity check
                # ------------------------------------------------

                if outside_total != 0:

                    print(
                        f"FATAL: {outside_total} pixels changed "
                        f"outside the hair mask",
                        flush=True
                    )

                    raise SystemExit(1)

                # ------------------------------------------------
                # Save mask areas
                # ------------------------------------------------

                np.save(
                    area_p,
                    area
                )

                # ------------------------------------------------
                # Deterministic spot check
                # ------------------------------------------------

                print(
                    "running deterministic 60-image spot check...",
                    flush=True
                )

                ns = load_generator(gen_path)

                cm = np.load(
                    clean_p,
                    mmap_mode="r"
                )

                hm = np.load(
                    hair_p,
                    mmap_mode="r"
                )

                sample_rows = random.Random(
                    20260919
                ).sample(
                    range(N),
                    min(60, N)
                )

                for r in sample_rows:

                    image_id = ids[r]

                    c = np.asarray(
                        Image.open(
                            find_image(
                                image_dir,
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )

                    h = ns["apply_hair"](
                        image=c,
                        mask=ns["generate_hair_mask"](
                            image_id=image_id,
                            severity=SEV
                        ),
                        severity=SEV,
                        image_id=image_id
                    )

                    assert np.array_equal(
                        cm[r],
                        c
                    ), (
                        f"clean cache mismatch at row {r}"
                    )

                    assert np.array_equal(
                        hm[r],
                        h
                    ), (
                        f"hair cache mismatch at row {r}"
                    )

                print(
                    "SPOT_CHECK_OK",
                    flush=True
                )

                # ------------------------------------------------
                # SHA256
                # ------------------------------------------------

                file_sha = {
                    p.name: sha_file(p)
                    for p in (
                        clean_p,
                        hair_p,
                        area_p,
                        cache_dir / "cache_index.csv"
                    )
                }

                record = {
                    "n_images": N,

                    "mask_area": {
                        "min": int(area.min()),
                        "mean": float(area.mean()),
                        "max": int(area.max()),
                    },

                    "file_sha256": file_sha,

                    "generator_sha256": sha_file(
                        gen_path
                    ),

                    "image_dir": str(image_dir),

                    "severity": SEV,
                }

                with open(
                    cache_dir / "_builder_record.json",
                    "w"
                ) as f:

                    json.dump(
                        record,
                        f,
                        indent=2,
                        sort_keys=True
                    )

                print(
                    "CACHE_BUILD_COMPLETE",
                    flush=True
                )

                print(
                    json.dumps(
                        record,
                        indent=2,
                        sort_keys=True
                    ),
                    flush=True
                )

            if __name__ == "__main__":
                main()
            '''
        )
    )

    # ============================================================
    # Run CPU builder
    # ============================================================

    n_proc = int(
        globals().get(
            "N_PROC_OVERRIDE",
            4
        )
    )

    print("=" * 80)
    print("LAUNCHING CPU CACHE BUILDER")
    print("=" * 80)

    print("CPU workers:", n_proc)
    print("GPU: NOT USED")
    print("Image directory:", IMAGE_DIR)
    print("Cache directory:", CACHE)

    t_cache = time.time()

    cmd = [
        sys.executable,
        str(BUILDER),

        "--gen-path",
        str(GEN_PATH),

        "--pop-path",
        str(POP_PATH),

        "--cache-dir",
        str(CACHE),

        "--image-dir",
        str(IMAGE_DIR),

        "--n-proc",
        str(n_proc),
    ]

    proc = subprocess.run(
        cmd,
        text=True
    )

    if proc.returncode != 0:
        raise RuntimeError(
            f"Cache builder failed with return code "
            f"{proc.returncode}"
        )

    print()
    print(
        f"Cache build wall time: "
        f"{(time.time() - t_cache) / 60:.2f} minutes"
    )

# ----------------------------------------------------------------
# Final cache verification
# ----------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL CACHE VERIFICATION")
print("=" * 80)

for f in REQUIRED_CACHE_FILES:
    p = CACHE / f
    assert p.exists(), f"Missing cache file: {p}"
    print("✓", p)

assert (CACHE / "_builder_record.json").exists()

print("\n✓ CACHE BUILD / VERIFICATION COMPLETE")
print("✓ GPU was not used.")

image folder: /kaggle/input/datasets/alifshahariar/isic-2019-dataset-full/isic-2019-dataset-full/train
STAGE 2 — PREPARING CLEAN CACHE BUILD

✓ Partial cache cleared.
LAUNCHING CPU CACHE BUILDER
CPU workers: 4
GPU: NOT USED
Image directory: /kaggle/input/datasets/alifshahariar/isic-2019-dataset-full/isic-2019-dataset-full/train
Cache directory: /kaggle/temp/isic_cache
locked population: 23227 images
starting 4 cache workers...
cache     64/23227
cache    128/23227
cache    192/23227
cache    256/23227
cache    320/23227
cache    384/23227
cache    448/23227
cache    512/23227
cache    576/23227
cache    640/23227
cache    704/23227
cache    768/23227
cache    832/23227
cache    896/23227
cache    960/23227
cache   1024/23227
cache   1088/23227
cache   1152/23227
cache   1216/23227
cache   1280/23227
cache   1344/23227
cache   1408/23227
cache   1472/23227
cache   1536/23227
cache   1600/23227
cache   1664/23227
cache   1728/23227
cache   1792/23227
cache   1856/23227
cache   1920/23227

In [12]:
# ================================================================
# T1.4b — CELL 2
# FROZEN INPUT + CACHE PREFLIGHT
#
# Purpose:
#   1. Locate the locked XAI subset
#   2. Locate the frozen generator
#   3. Locate the frozen analysis pipeline
#   4. Locate the newly rebuilt 224x224 cache
#   5. Verify all frozen cache SHA256 hashes
#
# NO MODEL LOADING
# NO IG
# NO NULL GENERATION
# ================================================================

import os
import json
import hashlib
from pathlib import Path

# ------------------------------------------------
# Restore paths in case notebook was restarted
# ------------------------------------------------

ROOT = Path("/kaggle")
INPUT = ROOT / "input"
WORK = ROOT / "working"
TEMP = ROOT / "temp"

RESULT = WORK / "T1_4B_generator_matched_spatial_null"

RESULT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Expected frozen hashes
# ------------------------------------------------

EXPECTED_CACHE_HASH = {
    "cache_index.csv":
        "d86bd82bb5294e11a542a344f7429e200dc0169baad9df9a9b8d96359987842c",

    "clean_224.npy":
        "455ddd37115fbac20af4448a792c7e96376196e798cb5bc5759107d839c5e912",

    "hair_224.npy":
        "5051802ed71028c38ca945992e33e979850919cea6f820c6c68156baa498cb49",

    "mask_area.npy":
        "bc6b6d5b65cc4d5714797f8170fe153db61e36736a6e7fda112f8edf9fad7e9c",
}

EXPECTED_GENERATOR_HASH = (
    "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
)

# ------------------------------------------------
# SHA256 helper
# ------------------------------------------------

def sha256_file(path, chunk_size=1 << 22):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------
# Locate frozen artifacts
# ------------------------------------------------

def find_unique(name):
    hits = list(INPUT.rglob(name))

    if len(hits) == 0:
        raise FileNotFoundError(
            f"Required artifact not found: {name}"
        )

    if len(hits) > 1:
        print(f"WARNING: multiple copies found for {name}:")
        for p in hits:
            print("  ", p)

    return hits[0]


LOCKED_XAI = find_unique("LOCKED_XAI_SUBSET.csv")
GEN_PATH = find_unique("recovered_original_hair_generator.py")
ANALYSIS_PATH = find_unique("analysis_pipeline.py")

# ------------------------------------------------
# Cache location
#
# IMPORTANT:
# The rebuilt cache was created here:
# /kaggle/temp/isic_cache
# ------------------------------------------------

CACHE = TEMP / "isic_cache"

assert CACHE.exists(), (
    f"Rebuilt cache directory does not exist:\n{CACHE}"
)

print("=" * 70)
print("T1.4b — FROZEN INPUT + CACHE PREFLIGHT")
print("=" * 70)

print("\nLOCKED XAI SUBSET")
print(" ", LOCKED_XAI)

print("\nFROZEN GENERATOR")
print(" ", GEN_PATH)

print("\nFROZEN ANALYSIS PIPELINE")
print(" ", ANALYSIS_PATH)

print("\nREBUILT CACHE")
print(" ", CACHE)


# ------------------------------------------------
# Verify cache files + hashes
# ------------------------------------------------

print("\n" + "=" * 70)
print("CACHE HASH VERIFICATION")
print("=" * 70)

CACHE_PATHS = {}

for filename, expected_hash in EXPECTED_CACHE_HASH.items():

    path = CACHE / filename

    assert path.exists(), (
        f"Missing cache artifact:\n{path}"
    )

    actual_hash = sha256_file(path)

    CACHE_PATHS[filename] = path

    print(f"\n{filename}")
    print("  path   :", path)
    print("  actual :", actual_hash)
    print("  expect :", expected_hash)

    assert actual_hash == expected_hash, (
        f"HASH MISMATCH for {filename}\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual_hash}"
    )

    print("  STATUS : PASS")


# ------------------------------------------------
# Verify generator hash
# ------------------------------------------------

print("\n" + "=" * 70)
print("GENERATOR HASH VERIFICATION")
print("=" * 70)

actual_generator_hash = sha256_file(GEN_PATH)

print("actual :", actual_generator_hash)
print("expect :", EXPECTED_GENERATOR_HASH)

assert actual_generator_hash == EXPECTED_GENERATOR_HASH, (
    "Frozen generator SHA256 mismatch!"
)

print("STATUS : PASS")


# ------------------------------------------------
# Basic cache metadata inspection
# ------------------------------------------------

print("\n" + "=" * 70)
print("CACHE METADATA")
print("=" * 70)

clean_path = CACHE / "clean_224.npy"
hair_path = CACHE / "hair_224.npy"
mask_area_path = CACHE / "mask_area.npy"
index_path = CACHE / "cache_index.csv"

print("clean_224.npy :", clean_path.stat().st_size / (1024**3), "GB")
print("hair_224.npy  :", hair_path.stat().st_size / (1024**3), "GB")
print("mask_area.npy :", mask_area_path.stat().st_size / (1024**2), "MB")
print("cache_index   :", index_path.stat().st_size / (1024**2), "MB")


# ------------------------------------------------
# Verify locked XAI subset
# ------------------------------------------------

print("\n" + "=" * 70)
print("LOCKED XAI SUBSET")
print("=" * 70)

xai_df = pd.read_csv(LOCKED_XAI)

print("Rows:", len(xai_df))
print("Columns:", list(xai_df.columns))

assert len(xai_df) == 100, (
    f"Expected exactly 100 XAI images, found {len(xai_df)}"
)

print("STATUS : PASS")


# ------------------------------------------------
# Final status
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 2 COMPLETE")
print("=" * 70)

print("✓ Locked XAI subset found")
print("✓ Frozen generator found")
print("✓ Frozen analysis pipeline found")
print("✓ 224x224 cache found")
print("✓ cache_index SHA256 PASS")
print("✓ clean_224 SHA256 PASS")
print("✓ hair_224 SHA256 PASS")
print("✓ mask_area SHA256 PASS")
print("✓ generator SHA256 PASS")
print("✓ 100-image XAI subset PASS")

print("\nNO MODEL LOADED")
print("NO IG COMPUTED")
print("NO NULL MASKS GENERATED")

print("\nREADY FOR NEXT PREFLIGHT CELL")

   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv
   /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv
   /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/isic_final_project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv
   /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
   /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
   /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/isic_final_project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
   /kaggle/

AssertionError: Expected exactly 100 XAI images, found 500

In [13]:
# ================================================================
# T1.4b — CELL 2A
# CREATE + FREEZE THE 100-IMAGE T1.4b SAMPLE
#
# Frozen source:
#   LOCKED_XAI_SUBSET.csv = 500 images
#
# T1.4b sample:
#   50 benign
#   50 malignant
#   SAMPLE_SEED = 20260925_1
#   without replacement
#   image IDs sorted lexicographically
#
# NO MODEL LOADING
# NO IG
# NO NULL GENERATION
# ================================================================

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

SAMPLE_SEED = 20260925_1

# ------------------------------------------------
# Verify frozen 500-image source
# ------------------------------------------------

assert len(xai_df) == 500, (
    f"Frozen LOCKED_XAI_SUBSET must contain 500 rows; found {len(xai_df)}"
)

assert "malignant" in xai_df.columns
assert "image" in xai_df.columns

# Normalize malignant to integer/bool safely
malignant_values = pd.to_numeric(
    xai_df["malignant"],
    errors="raise"
).astype(int)

assert set(malignant_values.unique()).issubset({0, 1}), (
    f"Unexpected malignant values: {malignant_values.unique()}"
)

xai_df = xai_df.copy()
xai_df["malignant"] = malignant_values

# ------------------------------------------------
# Check source class balance
# ------------------------------------------------

benign_df = xai_df[xai_df["malignant"] == 0].copy()
malignant_df = xai_df[xai_df["malignant"] == 1].copy()

print("=" * 70)
print("T1.4b — 500-IMAGE FROZEN SOURCE")
print("=" * 70)

print("Total    :", len(xai_df))
print("Benign   :", len(benign_df))
print("Malignant:", len(malignant_df))

assert len(benign_df) >= 50
assert len(malignant_df) >= 50

# ------------------------------------------------
# Deterministic stratified sampling
# ------------------------------------------------

rng = np.random.default_rng(SAMPLE_SEED)

benign_idx = rng.choice(
    len(benign_df),
    size=50,
    replace=False
)

malignant_idx = rng.choice(
    len(malignant_df),
    size=50,
    replace=False
)

sample_benign = benign_df.iloc[benign_idx].copy()
sample_malignant = malignant_df.iloc[malignant_idx].copy()

sample_df = pd.concat(
    [sample_benign, sample_malignant],
    axis=0,
    ignore_index=True
)

# ------------------------------------------------
# Sort image IDs lexicographically
# ------------------------------------------------

sample_df = sample_df.sort_values(
    "image",
    kind="mergesort"
).reset_index(drop=True)

# ------------------------------------------------
# Hard checks
# ------------------------------------------------

assert len(sample_df) == 100

assert (
    (sample_df["malignant"] == 0).sum() == 50
)

assert (
    (sample_df["malignant"] == 1).sum() == 50
)

assert sample_df["image"].is_unique

# ------------------------------------------------
# Hash the frozen image-ID list
# ------------------------------------------------

image_ids = sample_df["image"].astype(str).tolist()

image_list_bytes = "\n".join(image_ids).encode("utf-8")

sample_sha256 = hashlib.sha256(
    image_list_bytes
).hexdigest()

# ------------------------------------------------
# Write frozen sample lock
# ------------------------------------------------

SAMPLE_LOCK = RESULT / "T1_4B_IMAGE_SAMPLE_LOCK.json"

lock_payload = {
    "experiment": "T1_4B",
    "source": "LOCKED_XAI_SUBSET.csv",
    "source_rows": 500,
    "sample_seed": SAMPLE_SEED,
    "sample_size": 100,
    "benign": 50,
    "malignant": 50,
    "sampling": "stratified_without_replacement",
    "sort": "lexicographic_by_image",
    "image_list_sha256": sample_sha256,
    "images": image_ids,
}

with open(SAMPLE_LOCK, "w") as f:
    json.dump(
        lock_payload,
        f,
        indent=2
    )

# ------------------------------------------------
# Save sampled metadata
# ------------------------------------------------

SAMPLE_CSV = RESULT / "T1_4B_IMAGE_SAMPLE.csv"

sample_df.to_csv(
    SAMPLE_CSV,
    index=False
)

# ------------------------------------------------
# Final verification
# ------------------------------------------------

print("\n" + "=" * 70)
print("T1.4b — IMAGE SAMPLE LOCK")
print("=" * 70)

print("Sample seed       :", SAMPLE_SEED)
print("Sample size       :", len(sample_df))
print("Benign            :", (sample_df["malignant"] == 0).sum())
print("Malignant         :", (sample_df["malignant"] == 1).sum())
print("Unique image IDs  :", sample_df["image"].nunique())
print("Sorted             :", image_ids == sorted(image_ids))
print("Image-list SHA256 :", sample_sha256)

print("\nLOCK FILE")
print(SAMPLE_LOCK)

print("\nSAMPLE CSV")
print(SAMPLE_CSV)

print("\n" + "=" * 70)
print("CELL 2A COMPLETE")
print("=" * 70)

print("✓ 500-image frozen source preserved")
print("✓ 50 benign sampled")
print("✓ 50 malignant sampled")
print("✓ Sampling without replacement")
print("✓ IDs sorted lexicographically")
print("✓ Image-list SHA256 computed")
print("✓ T1_4B_IMAGE_SAMPLE_LOCK.json written")
print("✓ No model loaded")
print("✓ No IG computed")
print("✓ No null masks generated")

T1.4b — 500-IMAGE FROZEN SOURCE
Total    : 500
Benign   : 250
Malignant: 250

T1.4b — IMAGE SAMPLE LOCK
Sample seed       : 202609251
Sample size       : 100
Benign            : 50
Malignant         : 50
Unique image IDs  : 100
Sorted             : True
Image-list SHA256 : a1fca9e13e2d21f54e77ad594c1f55158ad6dc334ad269e59cd7f1616506a20c

LOCK FILE
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_IMAGE_SAMPLE_LOCK.json

SAMPLE CSV
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_IMAGE_SAMPLE.csv

CELL 2A COMPLETE
✓ 500-image frozen source preserved
✓ 50 benign sampled
✓ 50 malignant sampled
✓ Sampling without replacement
✓ IDs sorted lexicographically
✓ Image-list SHA256 computed
✓ T1_4B_IMAGE_SAMPLE_LOCK.json written
✓ No model loaded
✓ No IG computed
✓ No null masks generated


In [14]:
# ================================================================
# T1.4b — CELL 2B
# VERIFY 100-IMAGE SAMPLE ↔ FROZEN CACHE ALIGNMENT
#
# NO MODEL LOADING
# NO IG
# NO NULL GENERATION
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — CACHE ↔ SAMPLE ALIGNMENT")
print("=" * 70)

# ------------------------------------------------
# Paths
# ------------------------------------------------

SAMPLE_CSV = RESULT / "T1_4B_IMAGE_SAMPLE.csv"

CACHE_INDEX = CACHE / "cache_index.csv"
CLEAN_PATH = CACHE / "clean_224.npy"
HAIR_PATH = CACHE / "hair_224.npy"
MASK_AREA_PATH = CACHE / "mask_area.npy"

# ------------------------------------------------
# Load sample + cache index
# ------------------------------------------------

sample = pd.read_csv(SAMPLE_CSV)
cache_index = pd.read_csv(CACHE_INDEX)

print("Sample rows      :", len(sample))
print("Cache index rows :", len(cache_index))

assert len(sample) == 100
assert len(cache_index) == 23227

# ------------------------------------------------
# Inspect cache-index columns
# ------------------------------------------------

print("\nCache-index columns:")
print(list(cache_index.columns))

print("\nSample columns:")
print(list(sample.columns))

# ------------------------------------------------
# Identify image-ID column
# ------------------------------------------------

assert "image" in sample.columns, (
    "Sample CSV does not contain 'image'"
)

assert "image" in cache_index.columns, (
    "cache_index.csv does not contain 'image'"
)

# ------------------------------------------------
# Exact membership check
# ------------------------------------------------

sample_ids = sample["image"].astype(str).tolist()
cache_ids = cache_index["image"].astype(str)

cache_id_set = set(cache_ids)

missing_ids = [
    image_id
    for image_id in sample_ids
    if image_id not in cache_id_set
]

print("\nMissing sample IDs:", len(missing_ids))

if missing_ids:
    print("First missing IDs:")
    for x in missing_ids[:20]:
        print("  ", x)

assert len(missing_ids) == 0, (
    "Some locked T1.4b sample images are absent from cache_index.csv"
)

print("✓ All 100 sample images exist in cache index")

# ------------------------------------------------
# Check duplicate cache IDs
# ------------------------------------------------

duplicate_cache_ids = cache_ids[
    cache_ids.duplicated()
].unique()

print("Duplicate cache IDs:", len(duplicate_cache_ids))

assert len(duplicate_cache_ids) == 0, (
    "cache_index.csv contains duplicate image IDs"
)

# ------------------------------------------------
# Resolve cache row for every sample image
# ------------------------------------------------

cache_lookup = (
    cache_index
    .reset_index()
    .set_index(cache_index["image"].astype(str))
)

cache_rows = []

for image_id in sample_ids:

    row = cache_lookup.loc[image_id]

    cache_rows.append({
        "image": image_id,
        "cache_row": int(row["index"]),
    })

sample_cache = pd.DataFrame(cache_rows)

# ------------------------------------------------
# Check cache row uniqueness
# ------------------------------------------------

assert sample_cache["cache_row"].is_unique

print(
    "Unique cache rows:",
    sample_cache["cache_row"].nunique()
)

assert sample_cache["cache_row"].nunique() == 100

# ------------------------------------------------
# Load mask area arrays
# ------------------------------------------------

mask_area = np.load(
    MASK_AREA_PATH,
    mmap_mode="r"
)

clean = np.load(
    CLEAN_PATH,
    mmap_mode="r"
)

hair = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

print("\nCACHE ARRAY SHAPES")
print("clean :", clean.shape)
print("hair  :", hair.shape)
print("mask  :", mask_area.shape)

# ------------------------------------------------
# Hard shape checks
# ------------------------------------------------

assert clean.shape[0] == 23227
assert hair.shape[0] == 23227
assert mask_area.shape[0] == 23227

assert clean.shape[1:] == (224, 224, 3)
assert hair.shape[1:] == (224, 224, 3)

# ------------------------------------------------
# Verify sampled rows are valid
# ------------------------------------------------

rows = sample_cache["cache_row"].to_numpy()

assert rows.min() >= 0
assert rows.max() < 23227

print("\n✓ All sample cache rows are in bounds")

# ------------------------------------------------
# Verify mask-area metadata exists
# ------------------------------------------------

sample_mask_areas = np.asarray(
    mask_area[rows]
)

assert np.all(
    np.isfinite(sample_mask_areas)
)

assert np.all(
    sample_mask_areas > 0
)

print(
    "Mask areas:",
    "min =", int(sample_mask_areas.min()),
    "max =", int(sample_mask_areas.max()),
    "mean =", float(sample_mask_areas.mean())
)

# ------------------------------------------------
# Save resolved sample/cache mapping
# ------------------------------------------------

SAMPLE_CACHE_MAP = (
    RESULT /
    "T1_4B_IMAGE_SAMPLE_CACHE_MAP.csv"
)

sample_cache.to_csv(
    SAMPLE_CACHE_MAP,
    index=False
)

# ------------------------------------------------
# Final status
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 2B COMPLETE")
print("=" * 70)

print("✓ 100/100 sample IDs found in cache")
print("✓ Cache IDs unique")
print("✓ 100 unique cache rows resolved")
print("✓ clean_224 shape verified")
print("✓ hair_224 shape verified")
print("✓ mask_area shape verified")
print("✓ Sample cache rows in bounds")
print("✓ Sample mask areas finite and positive")

print("\nMapping written:")
print(SAMPLE_CACHE_MAP)

print("\nNO MODEL LOADED")
print("NO IG COMPUTED")
print("NO NULL MASKS GENERATED")

print("\nREADY FOR NEXT PREFLIGHT")

T1.4b — CACHE ↔ SAMPLE ALIGNMENT
Sample rows      : 100
Cache index rows : 23227

Cache-index columns:
['image', 'lesion_id', 'class_name', 'malignant', 'split', 'row']

Sample columns:
['image', 'lesion_id', 'class_name', 'malignant', 'row', 'mask_area', 'lesion_key', 'image_key']

Missing sample IDs: 0
✓ All 100 sample images exist in cache index
Duplicate cache IDs: 0
Unique cache rows: 100

CACHE ARRAY SHAPES
clean : (23227, 224, 224, 3)
hair  : (23227, 224, 224, 3)
mask  : (23227,)

✓ All sample cache rows are in bounds
Mask areas: min = 34 max = 1004 mean = 517.36

CELL 2B COMPLETE
✓ 100/100 sample IDs found in cache
✓ Cache IDs unique
✓ 100 unique cache rows resolved
✓ clean_224 shape verified
✓ hair_224 shape verified
✓ mask_area shape verified
✓ Sample cache rows in bounds
✓ Sample mask areas finite and positive

Mapping written:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_IMAGE_SAMPLE_CACHE_MAP.csv

NO MODEL LOADED
NO IG COMPUTED
NO NULL MASKS GENERATED

READY 

In [15]:
# ================================================================
# T1.4b — CELL 2C
# FROZEN TRUE-MASK INTEGRITY CHECK
#
# Verifies for all 100 locked T1.4b images:
#
#   generate_hair_mask(image_id, medium, 20260919, 224)
#       ↓
#   mask pixel count
#       ==
#   locked mask_area
#
# NO MODEL LOADING
# NO IG
# NO NULL GENERATION
# ================================================================

import ast
import types
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — FROZEN TRUE-MASK INTEGRITY")
print("=" * 70)

# ------------------------------------------------
# Frozen generator path
# ------------------------------------------------

GEN_PATH = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/"
    "isic-final-project/"
    "step1J_generator_recovery_lock/"
    "recovered_original_hair_generator.py"
)

assert GEN_PATH.exists(), (
    f"Frozen generator not found:\n{GEN_PATH}"
)

# ------------------------------------------------
# Verify generator SHA256 again
# ------------------------------------------------

EXPECTED_GEN_SHA256 = (
    "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
)

actual_gen_sha256 = hashlib.sha256(
    GEN_PATH.read_bytes()
).hexdigest()

assert actual_gen_sha256 == EXPECTED_GEN_SHA256, (
    "Generator SHA256 mismatch!"
)

print("✓ Generator SHA256 verified")

# ------------------------------------------------
# Safely load ONLY generator definitions
#
# The recovered generator contains top-level audit
# code, so we intentionally do not execute the entire
# source file.
# ------------------------------------------------

source = GEN_PATH.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

safe_nodes = []

for node in tree.body:

    # imports
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        safe_nodes.append(node)
        continue

    # functions
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    ):
        safe_nodes.append(node)
        continue

    # literal constants
    if isinstance(node, ast.Assign):
        try:
            ast.literal_eval(node.value)
            safe_nodes.append(node)
        except Exception:
            pass

    elif isinstance(node, ast.AnnAssign):
        if node.value is not None:
            try:
                ast.literal_eval(node.value)
                safe_nodes.append(node)
            except Exception:
                pass

module_ast = ast.Module(
    body=safe_nodes,
    type_ignores=[]
)

ast.fix_missing_locations(module_ast)

generator = types.ModuleType(
    "frozen_hair_generator"
)

generator.__file__ = str(GEN_PATH)

exec(
    compile(
        module_ast,
        str(GEN_PATH),
        "exec"
    ),
    generator.__dict__
)

# ------------------------------------------------
# Verify frozen API + constants
# ------------------------------------------------

assert hasattr(
    generator,
    "generate_hair_mask"
)

assert hasattr(
    generator,
    "apply_hair"
)

assert generator.GLOBAL_SEED == 20260919

assert generator.PRIMARY_SEVERITY == "medium"

print(
    "✓ GLOBAL_SEED =",
    generator.GLOBAL_SEED
)

print(
    "✓ PRIMARY_SEVERITY =",
    generator.PRIMARY_SEVERITY
)

print(
    "✓ generate_hair_mask() loaded"
)

# ------------------------------------------------
# Load frozen sample
# ------------------------------------------------

SAMPLE_CSV = (
    RESULT /
    "T1_4B_IMAGE_SAMPLE.csv"
)

sample = pd.read_csv(
    SAMPLE_CSV
)

assert len(sample) == 100

# ------------------------------------------------
# Required columns
# ------------------------------------------------

required = {
    "image",
    "malignant",
    "mask_area",
}

missing = required - set(sample.columns)

assert not missing, (
    f"Missing columns: {missing}"
)

# ------------------------------------------------
# Generate + verify every true mask
# ------------------------------------------------

records = []

mismatches = []

for idx, row in sample.iterrows():

    image_id = str(row["image"])

    locked_area = int(row["mask_area"])

    mask = generator.generate_hair_mask(
        image_id,
        generator.PRIMARY_SEVERITY,
        generator.GLOBAL_SEED,
        224
    )

    mask = np.asarray(mask)

    # ------------------------------------------------
    # Basic mask checks
    # ------------------------------------------------

    assert mask.shape == (224, 224), (
        f"Unexpected mask shape for {image_id}: "
        f"{mask.shape}"
    )

    # Convert safely to boolean for pixel counting
    mask_bool = mask.astype(bool)

    generated_area = int(
        mask_bool.sum()
    )

    # ------------------------------------------------
    # Hard frozen-area check
    # ------------------------------------------------

    passed = (
        generated_area ==
        locked_area
    )

    if not passed:
        mismatches.append({
            "image": image_id,
            "locked_area": locked_area,
            "generated_area": generated_area,
            "difference": (
                generated_area -
                locked_area
            ),
        })

    records.append({
        "image": image_id,
        "locked_mask_area": locked_area,
        "generated_mask_area": generated_area,
        "area_match": passed,
        "mask_pixels": generated_area,
    })

# ------------------------------------------------
# Results dataframe
# ------------------------------------------------

mask_audit = pd.DataFrame(
    records
)

# ------------------------------------------------
# Hard acceptance
# ------------------------------------------------

print("\n" + "=" * 70)
print("TRUE-MASK AUDIT")
print("=" * 70)

print(
    "Images checked     :",
    len(mask_audit)
)

print(
    "Exact area matches :",
    int(mask_audit["area_match"].sum())
)

print(
    "Mismatches         :",
    len(mismatches)
)

print(
    "Area min           :",
    int(mask_audit["generated_mask_area"].min())
)

print(
    "Area max           :",
    int(mask_audit["generated_mask_area"].max())
)

print(
    "Area mean          :",
    float(mask_audit["generated_mask_area"].mean())
)

assert len(mismatches) == 0, (
    "TRUE-MASK INTEGRITY FAILURE:\n"
    + str(mismatches[:20])
)

assert (
    mask_audit["area_match"].all()
)

# ------------------------------------------------
# Save audit
# ------------------------------------------------

MASK_AUDIT_PATH = (
    RESULT /
    "T1_4B_TRUE_MASK_INTEGRITY.csv"
)

mask_audit.to_csv(
    MASK_AUDIT_PATH,
    index=False
)

# ------------------------------------------------
# Final status
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 2C COMPLETE")
print("=" * 70)

print("✓ Frozen generator hash verified")
print("✓ GLOBAL_SEED = 20260919")
print("✓ PRIMARY_SEVERITY = medium")
print("✓ 100/100 true masks regenerated")
print("✓ 100/100 exact mask-area matches")
print("✓ No mask-area mismatches")
print("✓ Integrity audit saved")

print("\nAudit:")
print(MASK_AUDIT_PATH)

print("\nNO MODEL LOADED")
print("NO IG COMPUTED")
print("NO NULL MASKS GENERATED")

print("\nREADY FOR NULL-GEOMETRY PREFLIGHT")

T1.4b — FROZEN TRUE-MASK INTEGRITY
✓ Generator SHA256 verified
✓ GLOBAL_SEED = 20260919
✓ PRIMARY_SEVERITY = medium
✓ generate_hair_mask() loaded

TRUE-MASK AUDIT
Images checked     : 100
Exact area matches : 100
Mismatches         : 0
Area min           : 34
Area max           : 1004
Area mean          : 517.36

CELL 2C COMPLETE
✓ Frozen generator hash verified
✓ GLOBAL_SEED = 20260919
✓ PRIMARY_SEVERITY = medium
✓ 100/100 true masks regenerated
✓ 100/100 exact mask-area matches
✓ No mask-area mismatches
✓ Integrity audit saved

Audit:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_TRUE_MASK_INTEGRITY.csv

NO MODEL LOADED
NO IG COMPUTED
NO NULL MASKS GENERATED

READY FOR NULL-GEOMETRY PREFLIGHT


In [17]:
# ================================================================
# T1.4b — CELL 3R
# NULL-GEOMETRY PREFLIGHT — CORRECTED
#
# Purpose:
#   Validate proposal -> rejection logic.
#
# IMPORTANT:
#   Invalid random proposals are EXPECTED.
#   They must be rejected, not asserted as failures.
#
# NO PRODUCTION NULL GENERATION
# NO MODEL LOADING
# NO IG
# ================================================================

import hashlib
import numpy as np
from PIL import Image

print("=" * 70)
print("T1.4b — NULL-GEOMETRY PREFLIGHT — CORRECTED")
print("=" * 70)

NULL_NAMESPACE = "t1_4b-null-v2"
IMAGE_SIZE = 224
ATTEMPT_CAP = 20_000
NULLS_PER_DESIGN = 50

DESIGNS = {
    "A": "unconstrained",
    "Cprime": "centermatched",
}

# ------------------------------------------------
# Deterministic RNG
# ------------------------------------------------

def null_rng(image_id, design_tag, null_index):

    key = (
        f"{NULL_NAMESPACE}|"
        f"{image_id}|"
        f"{design_tag}|"
        f"{null_index}"
    ).encode("utf-8")

    digest = hashlib.sha256(key).digest()

    seed64 = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False,
    )

    return np.random.Generator(
        np.random.PCG64(seed64)
    )


# ------------------------------------------------
# Basic mask utilities
# ------------------------------------------------

def mask_area(mask):
    return int(
        np.asarray(mask, dtype=bool).sum()
    )


def mask_iou(a, b):

    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)

    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    if union == 0:
        return 0.0

    return float(intersection / union)


def mask_centroid(mask):

    yy, xx = np.nonzero(
        np.asarray(mask, dtype=bool)
    )

    if len(xx) == 0:
        return None

    return (
        float(xx.mean()),
        float(yy.mean()),
    )


def center_distance(mask):

    centroid = mask_centroid(mask)

    if centroid is None:
        return np.inf

    cx, cy = centroid

    center = (
        (IMAGE_SIZE - 1) / 2.0,
        (IMAGE_SIZE - 1) / 2.0,
    )

    return float(
        np.hypot(
            cx - center[0],
            cy - center[1],
        )
    )


def border_distance(mask):

    yy, xx = np.nonzero(
        np.asarray(mask, dtype=bool)
    )

    if len(xx) == 0:
        return np.inf

    return float(
        min(
            xx.min(),
            yy.min(),
            (IMAGE_SIZE - 1) - xx.max(),
            (IMAGE_SIZE - 1) - yy.max(),
        )
    )


# ------------------------------------------------
# STRICT TRANSLATION
#
# IMPORTANT:
# Do NOT clip.
#
# If ANY transformed pixel goes outside the canvas,
# reject the candidate.
# ------------------------------------------------

def translate_mask_strict(
    mask,
    dx,
    dy,
):

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    ys, xs = np.nonzero(mask)

    if len(xs) == 0:
        return None, False

    xs2 = xs + int(dx)
    ys2 = ys + int(dy)

    in_bounds = (
        (xs2 >= 0)
        & (xs2 < IMAGE_SIZE)
        & (ys2 >= 0)
        & (ys2 < IMAGE_SIZE)
    )

    # NO CLIPPING.
    # If even one pixel is outside, reject.
    if not bool(in_bounds.all()):
        return None, False

    out = np.zeros_like(mask)

    out[ys2, xs2] = True

    return out, True


# ------------------------------------------------
# Rotation
#
# Nearest-neighbor rotation around mask centroid.
#
# We retain PIL's nearest-neighbor operation here.
# The resulting mask is subsequently subjected to
# the acceptance rules.
# ------------------------------------------------

def rotate_about_centroid(
    mask,
    theta_deg,
):

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    centroid = mask_centroid(mask)

    if centroid is None:
        return None

    cx, cy = centroid

    img = Image.fromarray(
        (mask.astype(np.uint8) * 255)
    )

    rotated = img.rotate(
        angle=float(theta_deg),
        resample=Image.Resampling.NEAREST,
        expand=False,
        center=(cx, cy),
        fillcolor=0,
    )

    return (
        np.asarray(rotated) > 0
    )


# ------------------------------------------------
# Reflection
# ------------------------------------------------

def reflect_mask(
    mask,
    flip_h,
    flip_v,
):

    out = np.asarray(
        mask,
        dtype=bool,
    )

    if flip_h:
        out = np.fliplr(out)

    if flip_v:
        out = np.flipud(out)

    return out


# ------------------------------------------------
# Candidate transformation
# ------------------------------------------------

def transform_candidate(
    true_mask,
    theta,
    flip_h,
    flip_v,
    dx,
    dy,
):

    rotated = rotate_about_centroid(
        true_mask,
        theta,
    )

    if rotated is None:
        return None, {
            "reason": "empty_after_rotation"
        }

    reflected = reflect_mask(
        rotated,
        flip_h,
        flip_v,
    )

    translated, bounds_ok = translate_mask_strict(
        reflected,
        dx,
        dy,
    )

    if not bounds_ok:
        return None, {
            "reason": "out_of_bounds"
        }

    if translated is None:
        return None, {
            "reason": "empty_after_translation"
        }

    return translated, {
        "reason": "candidate_generated"
    }


# ------------------------------------------------
# Load first true mask
# ------------------------------------------------

test_image = str(
    sample.iloc[0]["image"]
)

true_mask = generator.generate_hair_mask(
    test_image,
    generator.PRIMARY_SEVERITY,
    generator.GLOBAL_SEED,
    IMAGE_SIZE,
)

true_mask = np.asarray(
    true_mask,
    dtype=bool,
)

true_area = mask_area(true_mask)

true_center_distance = center_distance(
    true_mask
)

print("\nTEST IMAGE")
print("Image:", test_image)
print("True area:", true_area)
print(
    "True center distance:",
    true_center_distance
)

# ------------------------------------------------
# Proposal smoke test
# ------------------------------------------------

print("\n" + "=" * 70)
print("PROPOSAL / REJECTION SMOKE TEST")
print("=" * 70)

for design_tag in DESIGNS:

    print("\nDESIGN:", design_tag)

    accepted = 0
    rejected = 0

    rejection_counts = {}

    for null_index in range(10):

        rng = null_rng(
            test_image,
            design_tag,
            null_index,
        )

        dx = int(
            rng.integers(
                -223,
                224,
            )
        )

        dy = int(
            rng.integers(
                -223,
                224,
            )
        )

        theta = float(
            rng.uniform(
                0.0,
                360.0,
            )
        )

        flip_h = bool(
            rng.integers(0, 2)
        )

        flip_v = bool(
            rng.integers(0, 2)
        )

        candidate, info = transform_candidate(
            true_mask,
            theta,
            flip_h,
            flip_v,
            dx,
            dy,
        )

        if candidate is None:

            rejected += 1

            reason = info["reason"]

            rejection_counts[reason] = (
                rejection_counts.get(reason, 0) + 1
            )

            print(
                f"  {null_index:02d}: "
                f"REJECTED — {reason}"
            )

            continue

        area = mask_area(candidate)
        iou = mask_iou(
            true_mask,
            candidate
        )

        candidate_center_distance = (
            center_distance(candidate)
        )

        # ------------------------------------------------
        # Acceptance predicates
        # ------------------------------------------------

        exact_area = (
            area == true_area
        )

        zero_overlap = (
            iou == 0.0
        )

        nonempty = (
            area > 0
        )

        center_ok = True

        if design_tag == "Cprime":

            lower = (
                true_center_distance * 0.85
            )

            upper = (
                true_center_distance * 1.15
            )

            center_ok = (
                lower
                <= candidate_center_distance
                <= upper
            )

        valid = (
            nonempty
            and exact_area
            and zero_overlap
            and center_ok
        )

        if valid:
            accepted += 1

            status = "ACCEPT"

        else:
            rejected += 1

            status = "REJECT"

            if not exact_area:
                reason = "area_mismatch"
            elif not zero_overlap:
                reason = "overlap"
            elif not center_ok:
                reason = "center_distance"
            elif not nonempty:
                reason = "empty"
            else:
                reason = "other"

            rejection_counts[reason] = (
                rejection_counts.get(reason, 0) + 1
            )

        print(
            f"  {null_index:02d}: "
            f"{status} | "
            f"area={area} "
            f"(target={true_area}) | "
            f"IoU={iou:.6f} | "
            f"center_dist={candidate_center_distance:.3f}"
        )

    print("\nSummary:")
    print("  accepted:", accepted)
    print("  rejected:", rejected)
    print("  rejection reasons:", rejection_counts)

# ------------------------------------------------
# Deterministic reproducibility check
# ------------------------------------------------

print("\n" + "=" * 70)
print("DETERMINISTIC REPRODUCIBILITY")
print("=" * 70)

for design_tag in DESIGNS:

    sequences = []

    for repeat in range(2):

        seq = []

        for null_index in range(5):

            rng = null_rng(
                test_image,
                design_tag,
                null_index,
            )

            seq.append((
                int(rng.integers(-223, 224)),
                int(rng.integers(-223, 224)),
                float(rng.uniform(0.0, 360.0)),
                bool(rng.integers(0, 2)),
                bool(rng.integers(0, 2)),
            ))

        sequences.append(seq)

    assert sequences[0] == sequences[1]

    print(
        f"✓ {design_tag}: deterministic"
    )

# ------------------------------------------------
# Final
# ------------------------------------------------

print("\n" + "=" * 70)
print("CELL 3R COMPLETE")
print("=" * 70)

print("✓ Invalid proposals are treated as rejections")
print("✓ Empty proposals are rejected")
print("✓ Out-of-bounds proposals are rejected")
print("✓ Translation performs NO clipping")
print("✓ Exact-area rule evaluated mechanically")
print("✓ IoU rule evaluated mechanically")
print("✓ C' center-distance rule evaluated mechanically")
print("✓ Deterministic seeds verified")
print("✓ No production null masks generated")
print("✓ No model loaded")
print("✓ No IG computed")

print("\nREADY FOR PRODUCTION NULL GENERATION")

T1.4b — NULL-GEOMETRY PREFLIGHT — CORRECTED

TEST IMAGE
Image: ISIC_0013637_downsampled
True area: 529
True center distance: 79.20288177568709

PROPOSAL / REJECTION SMOKE TEST

DESIGN: A
  00: REJECT | area=377 (target=529) | IoU=0.000000 | center_dist=31.988
  01: REJECTED — out_of_bounds
  02: REJECTED — out_of_bounds
  03: REJECTED — out_of_bounds
  04: REJECTED — out_of_bounds
  05: REJECTED — out_of_bounds
  06: REJECTED — out_of_bounds
  07: REJECTED — out_of_bounds
  08: REJECTED — out_of_bounds
  09: REJECTED — out_of_bounds

Summary:
  accepted: 0
  rejected: 10
  rejection reasons: {'area_mismatch': 1, 'out_of_bounds': 9}

DESIGN: Cprime
  00: REJECTED — out_of_bounds
  01: REJECTED — out_of_bounds
  02: REJECT | area=352 (target=529) | IoU=0.004561 | center_dist=67.153
  03: REJECTED — out_of_bounds
  04: REJECTED — out_of_bounds
  05: REJECTED — out_of_bounds
  06: REJECTED — out_of_bounds
  07: REJECTED — out_of_bounds
  08: REJECTED — out_of_bounds
  09: REJECT | area=354

In [18]:
# ================================================================
# T1.4b — CELL 4
# 100-IMAGE NULL FEASIBILITY AUDIT
#
# PURPOSE:
#   Test whether the frozen null-generation protocol can obtain
#   exactly 50 valid null masks per image/design within 20,000
#   proposal attempts.
#
# IMPORTANT:
#   - NO production null masks are saved.
#   - NO models are loaded.
#   - NO IG is computed.
#   - This is a feasibility audit only.
#   - A and C' are generated independently.
# ================================================================

import os
import json
import hashlib
import math
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


print("=" * 70)
print("T1.4b — CELL 4 — 100-IMAGE NULL FEASIBILITY AUDIT")
print("=" * 70)


# ----------------------------------------------------------------
# 1. FROZEN PARAMETERS
# ----------------------------------------------------------------

IMAGE_SIZE = 224

N_IMAGES = 100
N_NULLS_REQUIRED = 50
MAX_ATTEMPTS = 20_000

NULL_NAMESPACE = "t1_4b-null-v2"

CENTER_TOLERANCE = 0.15

DESIGNS = ["A", "Cprime"]

print("\nFrozen parameters:")
print(f"  image size          : {IMAGE_SIZE}")
print(f"  images              : {N_IMAGES}")
print(f"  valid nulls/design  : {N_NULLS_REQUIRED}")
print(f"  attempt cap         : {MAX_ATTEMPTS}")
print(f"  namespace            : {NULL_NAMESPACE}")
print(f"  C' tolerance        : ±{CENTER_TOLERANCE * 100:.1f}%")


# ----------------------------------------------------------------
# 2. LOCATE THE LOCKED SAMPLE
# ----------------------------------------------------------------

OUT_DIR = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null"
)

SAMPLE_CSV = OUT_DIR / "T1_4B_IMAGE_SAMPLE.csv"

assert SAMPLE_CSV.exists(), (
    f"Missing locked sample CSV:\n{SAMPLE_CSV}"
)

sample_df = pd.read_csv(SAMPLE_CSV)

assert len(sample_df) == 100, (
    f"Expected 100 locked images, found {len(sample_df)}"
)

assert sample_df["image"].is_unique, \
    "Locked sample contains duplicate image IDs."

IMAGE_IDS = sorted(sample_df["image"].astype(str).tolist())

print("\nLocked sample:")
print(f"  rows       : {len(sample_df)}")
print(f"  unique IDs : {len(set(IMAGE_IDS))}")
print(f"  first      : {IMAGE_IDS[0]}")
print(f"  last       : {IMAGE_IDS[-1]}")


# ----------------------------------------------------------------
# 3. LOCATE CACHE / GENERATOR
# ----------------------------------------------------------------

CACHE_DIR = Path("/kaggle/temp/isic_cache")

CACHE_INDEX_PATH = CACHE_DIR / "cache_index.csv"
MASK_AREA_PATH = CACHE_DIR / "mask_area.npy"

assert CACHE_INDEX_PATH.exists(), CACHE_INDEX_PATH
assert MASK_AREA_PATH.exists(), MASK_AREA_PATH

cache_index = pd.read_csv(CACHE_INDEX_PATH)
mask_area_array = np.load(MASK_AREA_PATH, mmap_mode="r")

cache_lookup = {
    str(row.image): int(row.row)
    for row in cache_index.itertuples(index=False)
}

missing = [
    image_id
    for image_id in IMAGE_IDS
    if image_id not in cache_lookup
]

assert not missing, (
    f"{len(missing)} locked images missing from cache. "
    f"First few: {missing[:5]}"
)

print("\nCache alignment:")
print(f"  cache rows       : {len(cache_index)}")
print(f"  locked IDs found : {len(IMAGE_IDS) - len(missing)}")


# ----------------------------------------------------------------
# 4. LOAD FROZEN GENERATOR
# ----------------------------------------------------------------

GENERATOR_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/isic-final-project/"
        "step1J_generator_recovery_lock/"
        "recovered_original_hair_generator.py"
    ),
    Path(
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/isic-final-project/"
        "step1A_artifact_generator/"
        "recovered_original_hair_generator.py"
    ),
]

GEN_PATH = next(
    (p for p in GENERATOR_CANDIDATES if p.exists()),
    None
)

assert GEN_PATH is not None, (
    "Frozen recovered_original_hair_generator.py not found."
)

print("\nFrozen generator:")
print(f"  {GEN_PATH}")


# ----------------------------------------------------------------
# Safe import of frozen generator
# ----------------------------------------------------------------

import importlib.util

spec = importlib.util.spec_from_file_location(
    "t1_4b_frozen_generator",
    GEN_PATH
)

generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)

assert hasattr(generator, "generate_hair_mask")
assert hasattr(generator, "GLOBAL_SEED")
assert hasattr(generator, "PRIMARY_SEVERITY")

print(f"  GLOBAL_SEED      : {generator.GLOBAL_SEED}")
print(f"  PRIMARY_SEVERITY : {generator.PRIMARY_SEVERITY}")

assert int(generator.GLOBAL_SEED) == 20260919
assert str(generator.PRIMARY_SEVERITY) == "medium"


# ----------------------------------------------------------------
# 5. REGENERATE TRUE MASK
# ----------------------------------------------------------------

def get_true_mask(image_id):
    """
    Regenerate the exact frozen production hair mask.
    """
    mask = generator.generate_hair_mask(
        image_id,
        generator.PRIMARY_SEVERITY,
        generator.GLOBAL_SEED,
        IMAGE_SIZE
    )

    mask = np.asarray(mask, dtype=bool)

    assert mask.shape == (IMAGE_SIZE, IMAGE_SIZE)

    return mask


# ----------------------------------------------------------------
# 6. DETERMINISTIC SEED
# ----------------------------------------------------------------

def null_seed(image_id, design_tag, null_index):
    """
    Frozen T1.4b-v2 seed namespace.

    SHA256(
        "t1_4b-null-v2" |
        image_id |
        design_tag |
        null_index
    ) -> first 64 bits -> PCG64
    """

    payload = (
        f"{NULL_NAMESPACE}|"
        f"{image_id}|"
        f"{design_tag}|"
        f"{null_index}"
    ).encode("utf-8")

    digest = hashlib.sha256(payload).digest()

    seed64 = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False
    )

    return seed64


# ----------------------------------------------------------------
# 7. STRICT MASK TRANSFORMATION
#
#    IMPORTANT:
#    We do NOT clip transformed pixels.
#
#    The transformation is performed directly on the true-mask
#    pixel coordinates:
#
#      rotation about mask centroid
#      reflection
#      integer translation
#
#    Candidate is rejected if any transformed pixel is outside
#    the 224x224 canvas.
# ----------------------------------------------------------------

def transform_mask_strict(
    true_mask,
    theta_deg,
    dx,
    dy,
    flip_h,
    flip_v
):
    """
    Strict nearest-neighbour coordinate rasterization.

    Returns:
        mask or None
        reason

    No clipping is performed.
    """

    ys, xs = np.where(true_mask)

    if len(xs) == 0:
        return None, "empty_source"

    # Pixel centroid of the true mask.
    cx = float(xs.mean())
    cy = float(ys.mean())

    theta = math.radians(float(theta_deg))

    cos_t = math.cos(theta)
    sin_t = math.sin(theta)

    # Rotate around the true-mask centroid.
    x0 = xs.astype(np.float64) - cx
    y0 = ys.astype(np.float64) - cy

    xr = (
        cx
        + cos_t * x0
        - sin_t * y0
    )

    yr = (
        cy
        + sin_t * x0
        + cos_t * y0
    )

    # Reflection around image axes.
    if flip_h:
        xr = (IMAGE_SIZE - 1) - xr

    if flip_v:
        yr = (IMAGE_SIZE - 1) - yr

    # Integer translation.
    xr = xr + int(dx)
    yr = yr + int(dy)

    # Nearest-neighbour rasterization.
    xr = np.rint(xr).astype(np.int64)
    yr = np.rint(yr).astype(np.int64)

    # STRICT in-bounds test.
    if (
        np.any(xr < 0)
        or np.any(xr >= IMAGE_SIZE)
        or np.any(yr < 0)
        or np.any(yr >= IMAGE_SIZE)
    ):
        return None, "out_of_bounds"

    # Rasterization can collapse multiple source pixels onto
    # the same destination pixel.
    linear = yr * IMAGE_SIZE + xr

    unique_linear = np.unique(linear)

    if len(unique_linear) == 0:
        return None, "empty"

    mask = np.zeros(
        IMAGE_SIZE * IMAGE_SIZE,
        dtype=bool
    )

    mask[unique_linear] = True

    return mask.reshape(
        IMAGE_SIZE,
        IMAGE_SIZE
    ), "ok"


# ----------------------------------------------------------------
# 8. C' CENTER-DISTANCE ACCEPTANCE
# ----------------------------------------------------------------

def center_distance(mask):
    ys, xs = np.where(mask)

    if len(xs) == 0:
        return np.nan

    cx = float(xs.mean())
    cy = float(ys.mean())

    image_cx = (IMAGE_SIZE - 1) / 2.0
    image_cy = (IMAGE_SIZE - 1) / 2.0

    return math.sqrt(
        (cx - image_cx) ** 2
        + (cy - image_cy) ** 2
    )


def cprime_accepts(
    candidate_mask,
    true_center_distance
):
    d = center_distance(candidate_mask)

    lower = true_center_distance * (
        1.0 - CENTER_TOLERANCE
    )

    upper = true_center_distance * (
        1.0 + CENTER_TOLERANCE
    )

    return (
        lower <= d <= upper,
        d,
        lower,
        upper
    )


# ----------------------------------------------------------------
# 9. IOU
# ----------------------------------------------------------------

def mask_iou(mask_a, mask_b):
    intersection = np.logical_and(
        mask_a,
        mask_b
    ).sum()

    union = np.logical_or(
        mask_a,
        mask_b
    ).sum()

    if union == 0:
        return 0.0

    return float(intersection / union)


# ----------------------------------------------------------------
# 10. ONE-DESIGN FEASIBILITY TEST
# ----------------------------------------------------------------

def feasibility_test(
    image_id,
    true_mask,
    design_tag,
    n_required=50,
    max_attempts=20_000
):
    """
    Generate proposals until either:
      - n_required valid nulls are found, or
      - max_attempts is exhausted.

    NO masks are persisted.
    """

    true_area = int(true_mask.sum())
    true_center = center_distance(true_mask)

    rng = np.random.Generator(
        np.random.PCG64(
            null_seed(
                image_id,
                design_tag,
                0
            )
        )
    )

    accepted = 0

    reasons = Counter()

    attempts_to_accept = []

    first_valid_parameters = []

    for attempt in range(1, max_attempts + 1):

        # --------------------------------------------------------
        # Proposal parameters
        # --------------------------------------------------------

        dx = int(
            rng.integers(
                -223,
                224
            )
        )

        dy = int(
            rng.integers(
                -223,
                224
            )
        )

        theta = int(
            rng.integers(
                0,
                360
            )
        )

        flip_h = bool(
            rng.integers(0, 2)
        )

        flip_v = bool(
            rng.integers(0, 2)
        )

        # --------------------------------------------------------
        # Transform
        # --------------------------------------------------------

        candidate, reason = transform_mask_strict(
            true_mask=true_mask,
            theta_deg=theta,
            dx=dx,
            dy=dy,
            flip_h=flip_h,
            flip_v=flip_v
        )

        if candidate is None:
            reasons[reason] += 1
            continue

        # --------------------------------------------------------
        # Exact area
        # --------------------------------------------------------

        candidate_area = int(candidate.sum())

        if candidate_area != true_area:
            reasons["area_mismatch"] += 1
            continue

        # --------------------------------------------------------
        # IoU = exactly zero
        # --------------------------------------------------------

        iou = mask_iou(
            candidate,
            true_mask
        )

        if iou != 0.0:
            reasons["overlap"] += 1
            continue

        # --------------------------------------------------------
        # C' center-distance criterion
        # --------------------------------------------------------

        if design_tag == "Cprime":

            accepted_center, d, lower, upper = (
                cprime_accepts(
                    candidate,
                    true_center
                )
            )

            if not accepted_center:
                reasons["center_distance"] += 1
                continue

        # --------------------------------------------------------
        # VALID
        # --------------------------------------------------------

        accepted += 1
        attempts_to_accept.append(attempt)

        if len(first_valid_parameters) < 3:
            first_valid_parameters.append({
                "attempt": attempt,
                "dx": dx,
                "dy": dy,
                "theta": theta,
                "flip_h": flip_h,
                "flip_v": flip_v,
                "area": candidate_area,
                "iou": iou,
                "center_distance": (
                    center_distance(candidate)
                )
            })

        if accepted >= n_required:
            return {
                "image": image_id,
                "design": design_tag,
                "success": True,
                "accepted": accepted,
                "attempts": attempt,
                "acceptance_rate": accepted / attempt,
                "reason": "",
                "true_area": true_area,
                "true_center_distance": true_center,
                "rejections": dict(reasons),
                "attempts_to_accept": attempts_to_accept,
                "first_valid_parameters":
                    first_valid_parameters
            }

    # ------------------------------------------------------------
    # CAP EXHAUSTED
    # ------------------------------------------------------------

    return {
        "image": image_id,
        "design": design_tag,
        "success": False,
        "accepted": accepted,
        "attempts": max_attempts,
        "acceptance_rate": (
            accepted / max_attempts
        ),
        "reason": "attempt_cap_exhausted",
        "true_area": true_area,
        "true_center_distance": true_center,
        "rejections": dict(reasons),
        "attempts_to_accept": attempts_to_accept,
        "first_valid_parameters":
            first_valid_parameters
    }


# ----------------------------------------------------------------
# 11. RUN 100-IMAGE FEASIBILITY AUDIT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNNING FEASIBILITY AUDIT")
print("=" * 70)

audit_records = []

start_time = time.time()

for image_number, image_id in enumerate(
    IMAGE_IDS,
    start=1
):

    true_mask = get_true_mask(image_id)

    # Frozen mask-area integrity.
    cache_row = cache_lookup[image_id]

    expected_area = int(
        mask_area_array[cache_row]
    )

    actual_area = int(
        true_mask.sum()
    )

    assert actual_area == expected_area, (
        f"TRUE MASK AREA MISMATCH: {image_id} | "
        f"actual={actual_area} expected={expected_area}"
    )

    image_results = {}

    for design_tag in DESIGNS:

        result = feasibility_test(
            image_id=image_id,
            true_mask=true_mask,
            design_tag=design_tag,
            n_required=N_NULLS_REQUIRED,
            max_attempts=MAX_ATTEMPTS
        )

        image_results[design_tag] = result

        audit_records.append(result)

    a = image_results["A"]
    c = image_results["Cprime"]

    print(
        f"[{image_number:03d}/100] "
        f"{image_id} | "
        f"A: {a['accepted']:02d}/50 "
        f"({a['attempts']:,} attempts) | "
        f"C': {c['accepted']:02d}/50 "
        f"({c['attempts']:,} attempts)"
    )


elapsed = time.time() - start_time


# ----------------------------------------------------------------
# 12. SUMMARY
# ----------------------------------------------------------------

audit_df = pd.DataFrame([
    {
        "image": r["image"],
        "design": r["design"],
        "success": r["success"],
        "accepted": r["accepted"],
        "attempts": r["attempts"],
        "acceptance_rate": r["acceptance_rate"],
        "reason": r["reason"],
        "true_area": r["true_area"],
        "true_center_distance":
            r["true_center_distance"],
        "rejections_json":
            json.dumps(
                r["rejections"],
                sort_keys=True
            )
    }
    for r in audit_records
])


print("\n" + "=" * 70)
print("FEASIBILITY AUDIT SUMMARY")
print("=" * 70)

for design in DESIGNS:

    sub = audit_df[
        audit_df["design"] == design
    ]

    print(f"\nDESIGN {design}")
    print("-" * 50)

    print(
        f"Images tested      : {len(sub)}"
    )

    print(
        f"Reached 50         : "
        f"{int(sub['success'].sum())}"
    )

    print(
        f"Failed             : "
        f"{int((~sub['success']).sum())}"
    )

    print(
        f"Median attempts    : "
        f"{sub['attempts'].median():.1f}"
    )

    print(
        f"Mean acceptance    : "
        f"{sub['acceptance_rate'].mean():.6f}"
    )

    print(
        f"Min acceptance     : "
        f"{sub['acceptance_rate'].min():.6f}"
    )

    print(
        f"Max acceptance     : "
        f"{sub['acceptance_rate'].max():.6f}"
    )


# ----------------------------------------------------------------
# 13. FAILURE LIST
# ----------------------------------------------------------------

failed = audit_df[
    ~audit_df["success"]
].copy()

print("\n" + "=" * 70)
print("FAILED IMAGE/DESIGN PAIRS")
print("=" * 70)

if len(failed) == 0:
    print("NONE — all 200 image/design feasibility tests reached 50.")
else:
    print(failed[
        [
            "image",
            "design",
            "accepted",
            "attempts",
            "reason"
        ]
    ].to_string(index=False))


# ----------------------------------------------------------------
# 14. SAVE AUDIT ONLY
# ----------------------------------------------------------------

AUDIT_PATH = (
    OUT_DIR /
    "T1_4B_NULL_FEASIBILITY_AUDIT.csv"
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False
)

SUMMARY = {
    "protocol": "T1.4b",
    "stage": "null_geometry_feasibility_audit",
    "namespace": NULL_NAMESPACE,
    "n_images": N_IMAGES,
    "n_nulls_required": N_NULLS_REQUIRED,
    "max_attempts": MAX_ATTEMPTS,
    "designs": DESIGNS,
    "elapsed_seconds": elapsed,
    "n_failed_image_design_pairs": int(len(failed)),
    "all_design_A_reached_50": bool(
        audit_df[
            audit_df["design"] == "A"
        ]["success"].all()
    ),
    "all_design_Cprime_reached_50": bool(
        audit_df[
            audit_df["design"] == "Cprime"
        ]["success"].all()
    )
}

SUMMARY_PATH = (
    OUT_DIR /
    "T1_4B_NULL_FEASIBILITY_AUDIT_SUMMARY.json"
)

with open(SUMMARY_PATH, "w") as f:
    json.dump(
        SUMMARY,
        f,
        indent=2
    )


# ----------------------------------------------------------------
# 15. FINAL GATE
# ----------------------------------------------------------------

print("\nSaved:")
print(f"  {AUDIT_PATH}")
print(f"  {SUMMARY_PATH}")

print("\n" + "=" * 70)
print("CELL 4 COMPLETE")
print("=" * 70)

if len(failed) == 0:

    print(
        "✓ ALL 100 IMAGES REACHED 50 VALID NULLS "
        "UNDER BOTH DESIGNS"
    )

    print(
        "✓ No production null masks were saved"
    )

    print(
        "✓ No model loaded"
    )

    print(
        "✓ No IG computed"
    )

    print(
        "\nREADY FOR CELL 5 — PRODUCTION NULL-MASK GENERATION"
    )

else:

    print(
        "⚠ FEASIBILITY GATE FAILED"
    )

    print(
        f"⚠ {len(failed)} image/design pair(s) "
        "did not reach 50 within 20,000 attempts."
    )

    print(
        "\nDO NOT START PRODUCTION GENERATION."
    )

    print(
        "Inspect the failure table before proceeding."
    )

T1.4b — CELL 4 — 100-IMAGE NULL FEASIBILITY AUDIT

Frozen parameters:
  image size          : 224
  images              : 100
  valid nulls/design  : 50
  attempt cap         : 20000
  namespace            : t1_4b-null-v2
  C' tolerance        : ±15.0%

Locked sample:
  rows       : 100
  unique IDs : 100
  first      : ISIC_0013637_downsampled
  last       : ISIC_0072895

Cache alignment:
  cache rows       : 23227
  locked IDs found : 100

Frozen generator:
  /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
STEP 1A — OTF ARTIFACT GENERATOR SPECIFICATION TEST


AssertionError: 

In [19]:
# ================================================================
# T1.4b — CELL 4R
# SAFE GENERATOR LOAD + 100-IMAGE NULL FEASIBILITY AUDIT
#
# REPLACES CELL 4
#
# IMPORTANT:
# The frozen generator contains an executable STEP 1A validation
# block that expects the original /root/.cache/kagglehub path.
# We must load only the frozen definitions, not execute that
# validation block.
# ================================================================

import os
import json
import hashlib
import math
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import importlib.util
import ast


print("=" * 70)
print("T1.4b — CELL 4R")
print("SAFE GENERATOR LOAD + 100-IMAGE NULL FEASIBILITY AUDIT")
print("=" * 70)


# ================================================================
# 1. FROZEN PARAMETERS
# ================================================================

IMAGE_SIZE = 224
N_IMAGES = 100
N_NULLS_REQUIRED = 50
MAX_ATTEMPTS = 20_000

NULL_NAMESPACE = "t1_4b-null-v2"
CENTER_TOLERANCE = 0.15

DESIGNS = ["A", "Cprime"]

print("\nFrozen parameters:")
print(f"  image size          : {IMAGE_SIZE}")
print(f"  images              : {N_IMAGES}")
print(f"  valid nulls/design  : {N_NULLS_REQUIRED}")
print(f"  attempt cap         : {MAX_ATTEMPTS}")
print(f"  namespace            : {NULL_NAMESPACE}")
print(f"  C' tolerance        : ±{CENTER_TOLERANCE * 100:.1f}%")


# ================================================================
# 2. OUTPUT / LOCKED SAMPLE
# ================================================================

OUT_DIR = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null"
)

SAMPLE_CSV = OUT_DIR / "T1_4B_IMAGE_SAMPLE.csv"

assert SAMPLE_CSV.exists(), (
    f"Missing locked sample:\n{SAMPLE_CSV}"
)

sample_df = pd.read_csv(SAMPLE_CSV)

assert len(sample_df) == 100, (
    f"Expected 100 locked images, found {len(sample_df)}"
)

IMAGE_IDS = sorted(
    sample_df["image"]
    .astype(str)
    .tolist()
)

assert len(set(IMAGE_IDS)) == 100

print("\nLocked sample:")
print(f"  rows       : {len(sample_df)}")
print(f"  unique IDs : {len(set(IMAGE_IDS))}")
print(f"  first      : {IMAGE_IDS[0]}")
print(f"  last       : {IMAGE_IDS[-1]}")


# ================================================================
# 3. CACHE ALIGNMENT
# ================================================================

CACHE_DIR = Path("/kaggle/temp/isic_cache")

CACHE_INDEX_PATH = CACHE_DIR / "cache_index.csv"
MASK_AREA_PATH = CACHE_DIR / "mask_area.npy"

assert CACHE_INDEX_PATH.exists()
assert MASK_AREA_PATH.exists()

cache_index = pd.read_csv(
    CACHE_INDEX_PATH
)

mask_area_array = np.load(
    MASK_AREA_PATH,
    mmap_mode="r"
)

cache_lookup = {
    str(row.image): int(row.row)
    for row in cache_index.itertuples(index=False)
}

missing = [
    image_id
    for image_id in IMAGE_IDS
    if image_id not in cache_lookup
]

assert not missing, (
    f"{len(missing)} locked images missing from cache. "
    f"First few: {missing[:5]}"
)

print("\nCache alignment:")
print(f"  cache rows       : {len(cache_index)}")
print(f"  locked IDs found : {len(IMAGE_IDS)}")


# ================================================================
# 4. LOCATE FROZEN GENERATOR
# ================================================================

GENERATOR_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/isic-final-project/"
        "step1J_generator_recovery_lock/"
        "recovered_original_hair_generator.py"
    ),
    Path(
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/isic-final-project/"
        "step1A_artifact_generator/"
        "recovered_original_hair_generator.py"
    ),
]

GEN_PATH = next(
    (
        p
        for p in GENERATOR_CANDIDATES
        if p.exists()
    ),
    None
)

assert GEN_PATH is not None, (
    "Frozen recovered_original_hair_generator.py "
    "was not found."
)

print("\nFrozen generator:")
print(f"  {GEN_PATH}")


# ================================================================
# 5. SAFE AST LOAD
#
# DO NOT use:
#
#     spec.loader.exec_module(generator)
#
# because the frozen source contains an executable STEP 1A
# validation section after the generator definitions.
#
# We execute only the source prefix before:
#
#     # Load locked population
#
# This preserves the frozen generator implementation while
# preventing its environment-specific validation code from
# executing.
# ================================================================

source_text = GEN_PATH.read_text(
    encoding="utf-8"
)

CUT_MARKER = "# Load locked population"

assert CUT_MARKER in source_text, (
    "Expected generator cut marker was not found. "
    "Do not continue — inspect the frozen source."
)

generator_definition_text = source_text.split(
    CUT_MARKER,
    1
)[0]

tree = ast.parse(
    generator_definition_text,
    filename=str(GEN_PATH)
)

generator_namespace = {
    "__name__": "t1_4b_frozen_generator_defs",
    "__file__": str(GEN_PATH),
}

compiled = compile(
    tree,
    filename=str(GEN_PATH),
    mode="exec"
)

exec(
    compiled,
    generator_namespace
)

assert (
    "generate_hair_mask"
    in generator_namespace
)

assert (
    "GLOBAL_SEED"
    in generator_namespace
)

assert (
    "PRIMARY_SEVERITY"
    in generator_namespace
)

generate_hair_mask = generator_namespace[
    "generate_hair_mask"
]

GLOBAL_SEED = generator_namespace[
    "GLOBAL_SEED"
]

PRIMARY_SEVERITY = generator_namespace[
    "PRIMARY_SEVERITY"
]

print("\nSafe generator load:")
print(f"  generate_hair_mask : {generate_hair_mask}")
print(f"  GLOBAL_SEED        : {GLOBAL_SEED}")
print(f"  PRIMARY_SEVERITY   : {PRIMARY_SEVERITY}")

assert int(GLOBAL_SEED) == 20260919
assert str(PRIMARY_SEVERITY) == "medium"

print("  ✓ Frozen generator constants verified")
print("  ✓ STEP 1A external-image validation NOT executed")


# ================================================================
# 6. TRUE MASK REGENERATION
# ================================================================

def get_true_mask(image_id):

    mask = generate_hair_mask(
        image_id,
        PRIMARY_SEVERITY,
        GLOBAL_SEED,
        IMAGE_SIZE
    )

    mask = np.asarray(
        mask,
        dtype=bool
    )

    assert mask.shape == (
        IMAGE_SIZE,
        IMAGE_SIZE
    )

    return mask


# ================================================================
# 7. DETERMINISTIC NULL SEED
# ================================================================

def null_seed(
    image_id,
    design_tag,
    null_index
):

    payload = (
        f"{NULL_NAMESPACE}|"
        f"{image_id}|"
        f"{design_tag}|"
        f"{null_index}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        payload
    ).digest()

    seed64 = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False
    )

    return seed64


# ================================================================
# 8. STRICT TRANSFORMATION
#
# No clipping.
# Any transformed pixel outside [0,223] is rejected.
# ================================================================

def transform_mask_strict(
    true_mask,
    theta_deg,
    dx,
    dy,
    flip_h,
    flip_v
):

    ys, xs = np.where(true_mask)

    if len(xs) == 0:
        return None, "empty_source"

    cx = float(xs.mean())
    cy = float(ys.mean())

    theta = math.radians(
        float(theta_deg)
    )

    cos_t = math.cos(theta)
    sin_t = math.sin(theta)

    x0 = (
        xs.astype(np.float64)
        - cx
    )

    y0 = (
        ys.astype(np.float64)
        - cy
    )

    xr = (
        cx
        + cos_t * x0
        - sin_t * y0
    )

    yr = (
        cy
        + sin_t * x0
        + cos_t * y0
    )

    if flip_h:
        xr = (
            IMAGE_SIZE - 1
        ) - xr

    if flip_v:
        yr = (
            IMAGE_SIZE - 1
        ) - yr

    xr = xr + int(dx)
    yr = yr + int(dy)

    xr = np.rint(
        xr
    ).astype(np.int64)

    yr = np.rint(
        yr
    ).astype(np.int64)

    # STRICT in-bounds.
    if (
        np.any(xr < 0)
        or np.any(xr >= IMAGE_SIZE)
        or np.any(yr < 0)
        or np.any(yr >= IMAGE_SIZE)
    ):
        return None, "out_of_bounds"

    linear = (
        yr * IMAGE_SIZE
        + xr
    )

    unique_linear = np.unique(
        linear
    )

    if len(unique_linear) == 0:
        return None, "empty"

    mask = np.zeros(
        IMAGE_SIZE * IMAGE_SIZE,
        dtype=bool
    )

    mask[unique_linear] = True

    return (
        mask.reshape(
            IMAGE_SIZE,
            IMAGE_SIZE
        ),
        "ok"
    )


# ================================================================
# 9. CENTER DISTANCE
# ================================================================

def center_distance(mask):

    ys, xs = np.where(mask)

    if len(xs) == 0:
        return np.nan

    cx = float(xs.mean())
    cy = float(ys.mean())

    image_cx = (
        IMAGE_SIZE - 1
    ) / 2.0

    image_cy = (
        IMAGE_SIZE - 1
    ) / 2.0

    return math.sqrt(
        (cx - image_cx) ** 2
        + (cy - image_cy) ** 2
    )


def cprime_accepts(
    candidate_mask,
    true_center_distance
):

    d = center_distance(
        candidate_mask
    )

    lower = (
        true_center_distance
        * (1.0 - CENTER_TOLERANCE)
    )

    upper = (
        true_center_distance
        * (1.0 + CENTER_TOLERANCE)
    )

    return (
        lower <= d <= upper,
        d,
        lower,
        upper
    )


# ================================================================
# 10. IOU
# ================================================================

def mask_iou(
    mask_a,
    mask_b
):

    intersection = np.logical_and(
        mask_a,
        mask_b
    ).sum()

    union = np.logical_or(
        mask_a,
        mask_b
    ).sum()

    if union == 0:
        return 0.0

    return float(
        intersection / union
    )


# ================================================================
# 11. FEASIBILITY TEST
# ================================================================

def feasibility_test(
    image_id,
    true_mask,
    design_tag,
    n_required=50,
    max_attempts=20_000
):

    true_area = int(
        true_mask.sum()
    )

    true_center = center_distance(
        true_mask
    )

    # Independent deterministic sequence
    # for each image/design.
    rng = np.random.Generator(
        np.random.PCG64(
            null_seed(
                image_id,
                design_tag,
                0
            )
        )
    )

    accepted = 0

    reasons = Counter()

    attempts_to_accept = []

    first_valid_parameters = []

    for attempt in range(
        1,
        max_attempts + 1
    ):

        dx = int(
            rng.integers(
                -223,
                224
            )
        )

        dy = int(
            rng.integers(
                -223,
                224
            )
        )

        theta = int(
            rng.integers(
                0,
                360
            )
        )

        flip_h = bool(
            rng.integers(
                0,
                2
            )
        )

        flip_v = bool(
            rng.integers(
                0,
                2
            )
        )

        candidate, reason = (
            transform_mask_strict(
                true_mask=true_mask,
                theta_deg=theta,
                dx=dx,
                dy=dy,
                flip_h=flip_h,
                flip_v=flip_v
            )
        )

        if candidate is None:
            reasons[reason] += 1
            continue

        candidate_area = int(
            candidate.sum()
        )

        if candidate_area != true_area:
            reasons[
                "area_mismatch"
            ] += 1
            continue

        iou = mask_iou(
            candidate,
            true_mask
        )

        if iou != 0.0:
            reasons[
                "overlap"
            ] += 1
            continue

        if design_tag == "Cprime":

            (
                accepted_center,
                d,
                lower,
                upper
            ) = cprime_accepts(
                candidate,
                true_center
            )

            if not accepted_center:
                reasons[
                    "center_distance"
                ] += 1
                continue

        accepted += 1

        attempts_to_accept.append(
            attempt
        )

        if len(
            first_valid_parameters
        ) < 3:

            first_valid_parameters.append({
                "attempt": attempt,
                "dx": dx,
                "dy": dy,
                "theta": theta,
                "flip_h": flip_h,
                "flip_v": flip_v,
                "area": candidate_area,
                "iou": iou,
                "center_distance":
                    center_distance(
                        candidate
                    )
            })

        if accepted >= n_required:

            return {
                "image": image_id,
                "design": design_tag,
                "success": True,
                "accepted": accepted,
                "attempts": attempt,
                "acceptance_rate":
                    accepted / attempt,
                "reason": "",
                "true_area": true_area,
                "true_center_distance":
                    true_center,
                "rejections":
                    dict(reasons),
                "attempts_to_accept":
                    attempts_to_accept,
                "first_valid_parameters":
                    first_valid_parameters
            }

    return {
        "image": image_id,
        "design": design_tag,
        "success": False,
        "accepted": accepted,
        "attempts": max_attempts,
        "acceptance_rate":
            accepted / max_attempts,
        "reason":
            "attempt_cap_exhausted",
        "true_area": true_area,
        "true_center_distance":
            true_center,
        "rejections":
            dict(reasons),
        "attempts_to_accept":
            attempts_to_accept,
        "first_valid_parameters":
            first_valid_parameters
    }


# ================================================================
# 12. RUN 100 × 2 FEASIBILITY TESTS
# ================================================================

print("\n" + "=" * 70)
print("RUNNING 100-IMAGE FEASIBILITY AUDIT")
print("=" * 70)

audit_records = []

start_time = time.time()

for image_number, image_id in enumerate(
    IMAGE_IDS,
    start=1
):

    true_mask = get_true_mask(
        image_id
    )

    cache_row = cache_lookup[
        image_id
    ]

    expected_area = int(
        mask_area_array[
            cache_row
        ]
    )

    actual_area = int(
        true_mask.sum()
    )

    assert actual_area == expected_area, (
        f"TRUE MASK AREA MISMATCH: "
        f"{image_id} | "
        f"actual={actual_area} | "
        f"expected={expected_area}"
    )

    row_results = {}

    for design_tag in DESIGNS:

        result = feasibility_test(
            image_id=image_id,
            true_mask=true_mask,
            design_tag=design_tag,
            n_required=N_NULLS_REQUIRED,
            max_attempts=MAX_ATTEMPTS
        )

        row_results[
            design_tag
        ] = result

        audit_records.append(
            result
        )

    a = row_results["A"]
    c = row_results["Cprime"]

    print(
        f"[{image_number:03d}/100] "
        f"{image_id} | "
        f"A: {a['accepted']:02d}/50 "
        f"({a['attempts']:,}) | "
        f"C': {c['accepted']:02d}/50 "
        f"({c['attempts']:,})"
    )


elapsed = (
    time.time()
    - start_time
)


# ================================================================
# 13. AUDIT DATAFRAME
# ================================================================

audit_df = pd.DataFrame([
    {
        "image":
            r["image"],

        "design":
            r["design"],

        "success":
            r["success"],

        "accepted":
            r["accepted"],

        "attempts":
            r["attempts"],

        "acceptance_rate":
            r["acceptance_rate"],

        "reason":
            r["reason"],

        "true_area":
            r["true_area"],

        "true_center_distance":
            r["true_center_distance"],

        "rejections_json":
            json.dumps(
                r["rejections"],
                sort_keys=True
            )
    }
    for r in audit_records
])


# ================================================================
# 14. SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("FEASIBILITY AUDIT SUMMARY")
print("=" * 70)

for design in DESIGNS:

    sub = audit_df[
        audit_df["design"] == design
    ]

    print(f"\nDESIGN {design}")
    print("-" * 50)

    print(
        f"Images tested      : "
        f"{len(sub)}"
    )

    print(
        f"Reached 50         : "
        f"{int(sub['success'].sum())}"
    )

    print(
        f"Failed             : "
        f"{int((~sub['success']).sum())}"
    )

    print(
        f"Median attempts    : "
        f"{sub['attempts'].median():.1f}"
    )

    print(
        f"Mean acceptance    : "
        f"{sub['acceptance_rate'].mean():.6f}"
    )

    print(
        f"Min acceptance     : "
        f"{sub['acceptance_rate'].min():.6f}"
    )

    print(
        f"Max acceptance     : "
        f"{sub['acceptance_rate'].max():.6f}"
    )


# ================================================================
# 15. FAILURES
# ================================================================

failed = audit_df[
    ~audit_df["success"]
].copy()

print("\n" + "=" * 70)
print("FAILED IMAGE/DESIGN PAIRS")
print("=" * 70)

if len(failed) == 0:

    print(
        "NONE — all 200 image/design "
        "tests reached 50."
    )

else:

    print(
        failed[
            [
                "image",
                "design",
                "accepted",
                "attempts",
                "reason"
            ]
        ].to_string(
            index=False
        )
    )


# ================================================================
# 16. SAVE FEASIBILITY AUDIT
# ================================================================

AUDIT_PATH = (
    OUT_DIR /
    "T1_4B_NULL_FEASIBILITY_AUDIT.csv"
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False
)

SUMMARY = {
    "protocol": "T1.4b",
    "stage":
        "null_geometry_feasibility_audit",
    "namespace":
        NULL_NAMESPACE,
    "n_images":
        N_IMAGES,
    "n_nulls_required":
        N_NULLS_REQUIRED,
    "max_attempts":
        MAX_ATTEMPTS,
    "designs":
        DESIGNS,
    "elapsed_seconds":
        elapsed,
    "n_failed_image_design_pairs":
        int(len(failed)),
    "all_design_A_reached_50":
        bool(
            audit_df[
                audit_df["design"] == "A"
            ]["success"].all()
        ),
    "all_design_Cprime_reached_50":
        bool(
            audit_df[
                audit_df["design"] == "Cprime"
            ]["success"].all()
        )
}

SUMMARY_PATH = (
    OUT_DIR /
    "T1_4B_NULL_FEASIBILITY_AUDIT_SUMMARY.json"
)

with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        SUMMARY,
        f,
        indent=2
    )


# ================================================================
# 17. FINAL GATE
# ================================================================

print("\nSaved:")
print(f"  {AUDIT_PATH}")
print(f"  {SUMMARY_PATH}")

print("\n" + "=" * 70)
print("CELL 4R COMPLETE")
print("=" * 70)

if len(failed) == 0:

    print(
        "✓ ALL 100 IMAGES REACHED 50 VALID NULLS "
        "UNDER BOTH DESIGNS"
    )

    print(
        "✓ No production null masks were saved"
    )

    print(
        "✓ No model loaded"
    )

    print(
        "✓ No IG computed"
    )

    print(
        "\nREADY FOR PRODUCTION NULL GENERATION"
    )

else:

    print(
        "⚠ FEASIBILITY GATE FAILED"
    )

    print(
        f"⚠ {len(failed)} image/design pair(s) "
        "did not reach 50 within 20,000 attempts."
    )

    print(
        "\nDO NOT START PRODUCTION GENERATION."
    )

T1.4b — CELL 4R
SAFE GENERATOR LOAD + 100-IMAGE NULL FEASIBILITY AUDIT

Frozen parameters:
  image size          : 224
  images              : 100
  valid nulls/design  : 50
  attempt cap         : 20000
  namespace            : t1_4b-null-v2
  C' tolerance        : ±15.0%

Locked sample:
  rows       : 100
  unique IDs : 100
  first      : ISIC_0013637_downsampled
  last       : ISIC_0072895

Cache alignment:
  cache rows       : 23227
  locked IDs found : 100

Frozen generator:
  /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py

Safe generator load:
  generate_hair_mask : <function generate_hair_mask at 0x7df9307ba200>
  GLOBAL_SEED        : 20260919
  PRIMARY_SEVERITY   : medium
  ✓ Frozen generator constants verified
  ✓ STEP 1A external-image validation NOT executed

RUNNING 100-IMAGE FEASIBILITY AUDIT
[001/100] ISIC_0013637_downsampled | A: 03/50 (20,000) | C': 00/50 (20,000)
[002/100] ISIC_0013835_dow

In [20]:
# ============================================================
# CELL 4S-OPT — VECTORISED NULL FEASIBILITY DIAGNOSTIC
# T1.4b — NO PRODUCTION NULLS
# ============================================================

import os
import json
import time
import hashlib
import ast
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# FROZEN T1.4b CONSTANTS
# ------------------------------------------------------------

IMAGE_SIZE = 224
N_IMAGES = 100
N_REQUIRED = 50
MAX_ATTEMPTS = 20_000

NULL_NAMESPACE = "t1_4b-null-v2"

CENTER_TOLERANCE = 0.15

IMAGE_HALF_DIAGONAL = (
    np.sqrt(IMAGE_SIZE**2 + IMAGE_SIZE**2) / 2.0
)

CPRIME_TOLERANCE_PX = (
    CENTER_TOLERANCE * IMAGE_HALF_DIAGONAL
)

RESULT_DIR = Path(
    "/kaggle/working/T1_4B_generator_matched_spatial_null"
)

SAMPLE_CSV = RESULT_DIR / "T1_4B_IMAGE_SAMPLE.csv"
CACHE_MAP_CSV = RESULT_DIR / "T1_4B_IMAGE_SAMPLE_CACHE_MAP.csv"

OUT_CSV = (
    RESULT_DIR /
    "T1_4B_CORRECTED_CPRIME_FEASIBILITY_DIAGNOSTIC_OPT.csv"
)

print("=" * 70)
print("T1.4b — VECTORISED NULL-GEOMETRY FEASIBILITY")
print("=" * 70)

print(f"Images                  : {N_IMAGES}")
print(f"Required nulls/design   : {N_REQUIRED}")
print(f"Attempt cap             : {MAX_ATTEMPTS}")
print(f"Image half-diagonal     : {IMAGE_HALF_DIAGONAL:.8f}")
print(f"C' tolerance            : ±{CPRIME_TOLERANCE_PX:.8f} px")
print(f"Namespace               : {NULL_NAMESPACE}")
print()
print("CPU vectorised geometry")
print("NO model loading")
print("NO IG")
print("NO production null masks")
print()


# ============================================================
# 1. SAFE-LOAD FROZEN GENERATOR DEFINITIONS
# ============================================================

PROJECT_ROOT = Path("/kaggle/working/isic_final_project")

GENERATOR_CANDIDATES = [
    PROJECT_ROOT /
    "step1J_generator_recovery_lock" /
    "recovered_original_hair_generator.py",

    Path(
        "/kaggle/input/datasets/tejasseshadriiiii/"
        "isic-final-project/"
        "step1J_generator_recovery_lock/"
        "recovered_original_hair_generator.py"
    ),
]

GENERATOR_PATH = None

for p in GENERATOR_CANDIDATES:
    if p.exists():
        GENERATOR_PATH = p
        break

if GENERATOR_PATH is None:
    raise FileNotFoundError(
        "Frozen recovered_original_hair_generator.py not found."
    )

print("Generator:")
print(GENERATOR_PATH)


# The frozen generator contains an executable STEP 1A
# environment-specific self-test after the definitions.
# Load only the definition section.

source = GENERATOR_PATH.read_text()

CUT_MARKERS = [
    "# Load locked population",
    "# LOAD LOCKED POPULATION",
]

cut_pos = None

for marker in CUT_MARKERS:
    pos = source.find(marker)
    if pos >= 0:
        cut_pos = pos
        break

if cut_pos is None:
    raise RuntimeError(
        "Could not find safe generator cut marker."
    )

definition_source = source[:cut_pos]

tree = ast.parse(definition_source)

generator_ns = {
    "__name__": "frozen_generator_t1_4b",
    "__file__": str(GENERATOR_PATH),
}

exec(
    compile(
        tree,
        str(GENERATOR_PATH),
        "exec"
    ),
    generator_ns
)

generate_hair_mask = generator_ns["generate_hair_mask"]

GLOBAL_SEED = generator_ns["GLOBAL_SEED"]
PRIMARY_SEVERITY = generator_ns["PRIMARY_SEVERITY"]

print(f"GLOBAL_SEED           : {GLOBAL_SEED}")
print(f"PRIMARY_SEVERITY      : {PRIMARY_SEVERITY}")

assert GLOBAL_SEED == 20260919
assert PRIMARY_SEVERITY == "medium"


# ============================================================
# 2. LOAD SAMPLE + CACHE MAP
# ============================================================

sample_df = pd.read_csv(SAMPLE_CSV)
map_df = pd.read_csv(CACHE_MAP_CSV)

assert len(sample_df) == 100
assert len(map_df) == 100

print()
print("Sample rows            :", len(sample_df))
print("Cache-map rows         :", len(map_df))


# ============================================================
# 3. DETERMINISTIC NULL SEED
# ============================================================

def null_seed(
    image_id: str,
    design_tag: str,
    null_index: int
):
    payload = (
        f"{NULL_NAMESPACE}|"
        f"{image_id}|"
        f"{design_tag}|"
        f"{null_index}"
    )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).digest()

    seed_int = int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False
    )

    return seed_int


# ============================================================
# 4. TRUE MASK
# ============================================================

def get_true_mask(image_id):
    """
    Regenerate the exact frozen true mask.
    """

    mask = generate_hair_mask(
        image_id=image_id,
        severity=PRIMARY_SEVERITY,
    )

    mask = np.asarray(mask, dtype=bool)

    assert mask.shape == (
        IMAGE_SIZE,
        IMAGE_SIZE
    )

    return mask


# ============================================================
# 5. PRECOMPUTED TRUE-MASK GEOMETRY
# ============================================================

def mask_coordinates(mask):
    """
    Return foreground pixel coordinates as float64.
    Coordinate convention:
        x = column
        y = row
    """

    y, x = np.nonzero(mask)

    coords = np.column_stack(
        [x.astype(np.float64),
         y.astype(np.float64)]
    )

    return coords


def centroid(coords):
    return coords.mean(axis=0)


def center_distance(c):
    image_center = np.array(
        [
            (IMAGE_SIZE - 1) / 2.0,
            (IMAGE_SIZE - 1) / 2.0
        ],
        dtype=np.float64
    )

    return float(
        np.linalg.norm(c - image_center)
    )


# ============================================================
# 6. VECTORISED TRANSFORM
# ============================================================

def transform_coordinates(
    coords,
    angle_deg,
    reflect_x,
    reflect_y,
    dx,
    dy,
    cx,
    cy
):
    """
    Transform foreground coordinates without constructing
    a 224x224 image.

    Order:
        1. rotate about true-mask centroid
        2. reflect about image axes if requested
        3. integer translation

    Nearest-neighbour rasterisation is represented by np.rint.
    """

    theta = np.deg2rad(angle_deg)

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    # Shift to mask centroid
    x = coords[:, 0] - cx
    y = coords[:, 1] - cy

    # Rotate
    xr = (
        cos_t * x
        - sin_t * y
        + cx
    )

    yr = (
        sin_t * x
        + cos_t * y
        + cy
    )

    # Horizontal reflection
    if reflect_x:
        xr = (IMAGE_SIZE - 1) - xr

    # Vertical reflection
    if reflect_y:
        yr = (IMAGE_SIZE - 1) - yr

    # Integer translation
    xr = xr + dx
    yr = yr + dy

    # Nearest-neighbour rasterisation
    xi = np.rint(xr).astype(np.int32)
    yi = np.rint(yr).astype(np.int32)

    return xi, yi


# ============================================================
# 7. FAST ACCEPTANCE CHECK
# ============================================================

def evaluate_candidate(
    coords,
    true_mask,
    true_flat_indices,
    angle_deg,
    reflect_x,
    reflect_y,
    dx,
    dy,
    true_area,
    true_center_distance,
    design_tag,
):
    """
    Returns:
        accepted, reason, area, center_distance
    """

    cx = coords[:, 0].mean()
    cy = coords[:, 1].mean()

    xi, yi = transform_coordinates(
        coords=coords,
        angle_deg=angle_deg,
        reflect_x=reflect_x,
        reflect_y=reflect_y,
        dx=dx,
        dy=dy,
        cx=cx,
        cy=cy,
    )

    # --------------------------------------------------------
    # IN-BOUNDS
    # --------------------------------------------------------

    if (
        np.any(xi < 0)
        or np.any(xi >= IMAGE_SIZE)
        or np.any(yi < 0)
        or np.any(yi >= IMAGE_SIZE)
    ):
        return False, "out_of_bounds", 0, np.nan

    # --------------------------------------------------------
    # RASTERIZED PIXEL INDICES
    # --------------------------------------------------------

    flat = (
        yi.astype(np.int64) * IMAGE_SIZE
        + xi.astype(np.int64)
    )

    # --------------------------------------------------------
    # EXACT AREA
    #
    # Important:
    # np.rint can theoretically map multiple source pixels
    # to one raster pixel after arbitrary rotation.
    # Therefore unique() is required.
    # --------------------------------------------------------

    unique_flat = np.unique(flat)

    area = len(unique_flat)

    if area != true_area:
        return False, "area_mismatch", area, np.nan

    # --------------------------------------------------------
    # ZERO OVERLAP / IoU = 0
    # --------------------------------------------------------

    overlap = np.intersect1d(
        unique_flat,
        true_flat_indices,
        assume_unique=True
    )

    if len(overlap) != 0:
        return False, "overlap", area, np.nan

    # --------------------------------------------------------
    # PIXEL CENTROID AFTER RASTERISATION
    # --------------------------------------------------------

    ys = unique_flat // IMAGE_SIZE
    xs = unique_flat % IMAGE_SIZE

    null_center = np.array(
        [
            xs.mean(),
            ys.mean()
        ],
        dtype=np.float64
    )

    null_center_distance = float(
        np.linalg.norm(
            null_center
            - np.array(
                [
                    (IMAGE_SIZE - 1) / 2.0,
                    (IMAGE_SIZE - 1) / 2.0
                ],
                dtype=np.float64
            )
        )
    )

    # --------------------------------------------------------
    # DESIGN C'
    #
    # Frozen definition:
    # ±15% of IMAGE HALF-DIAGONAL,
    # NOT ±15% of true distance.
    # --------------------------------------------------------

    if design_tag == "Cprime":

        if (
            abs(
                null_center_distance
                - true_center_distance
            )
            > CPRIME_TOLERANCE_PX
        ):
            return (
                False,
                "center_distance",
                area,
                null_center_distance
            )

    return (
        True,
        "accepted",
        area,
        null_center_distance
    )


# ============================================================
# 8. DETERMINISTIC PROPOSAL GENERATION
# ============================================================

def draw_proposal(rng):
    """
    One deterministic proposal.

    Translation is deliberately sampled over the full canvas
    range. Invalid placements are rejected rather than clipped.
    """

    angle_deg = rng.uniform(
        0.0,
        360.0
    )

    reflect_x = bool(
        rng.integers(0, 2)
    )

    reflect_y = bool(
        rng.integers(0, 2)
    )

    dx = int(
        rng.integers(
            -(IMAGE_SIZE - 1),
            IMAGE_SIZE
        )
    )

    dy = int(
        rng.integers(
            -(IMAGE_SIZE - 1),
            IMAGE_SIZE
        )
    )

    return (
        angle_deg,
        reflect_x,
        reflect_y,
        dx,
        dy
    )


# ============================================================
# 9. ONE IMAGE / DESIGN
# ============================================================

def run_feasibility(
    image_id,
    design_tag
):

    true_mask = get_true_mask(image_id)

    coords = mask_coordinates(true_mask)

    true_area = int(true_mask.sum())

    true_flat_indices = np.flatnonzero(
        true_mask.ravel()
    ).astype(np.int64)

    true_center = centroid(coords)

    true_center_distance = center_distance(
        true_center
    )

    seed = null_seed(
        image_id,
        design_tag,
        0
    )

    # Important:
    # use one deterministic RNG stream per image/design.
    rng = np.random.Generator(
        np.random.PCG64(seed)
    )

    accepted = 0

    rejection_counts = {
        "out_of_bounds": 0,
        "area_mismatch": 0,
        "overlap": 0,
        "center_distance": 0,
    }

    start = time.time()

    for attempt in range(
        MAX_ATTEMPTS
    ):

        (
            angle_deg,
            reflect_x,
            reflect_y,
            dx,
            dy
        ) = draw_proposal(rng)

        (
            ok,
            reason,
            area,
            cdist
        ) = evaluate_candidate(
            coords=coords,
            true_mask=true_mask,
            true_flat_indices=true_flat_indices,
            angle_deg=angle_deg,
            reflect_x=reflect_x,
            reflect_y=reflect_y,
            dx=dx,
            dy=dy,
            true_area=true_area,
            true_center_distance=true_center_distance,
            design_tag=design_tag,
        )

        if ok:
            accepted += 1

        else:
            rejection_counts[reason] += 1

    elapsed = time.time() - start

    return {
        "image": image_id,
        "design": design_tag,
        "true_area": true_area,
        "true_center_distance": true_center_distance,
        "attempts": MAX_ATTEMPTS,
        "accepted": accepted,
        "acceptance_rate": accepted / MAX_ATTEMPTS,
        "reached_50": accepted >= N_REQUIRED,
        "elapsed_sec": elapsed,
        **rejection_counts,
    }


# ============================================================
# 10. RUN ALL 100 × 2
# ============================================================

records = []

overall_start = time.time()

for i, image_id in enumerate(
    sample_df["image"].tolist(),
    start=1
):

    for design_tag in [
        "A",
        "Cprime"
    ]:

        result = run_feasibility(
            image_id=image_id,
            design_tag=design_tag
        )

        records.append(result)

        print(
            f"[{i:03d}/100] "
            f"{design_tag:7s} | "
            f"valid={result['accepted']:5d} | "
            f"rate={result['acceptance_rate']:.6f} | "
            f"{result['elapsed_sec']:.2f}s"
        )


# ============================================================
# 11. SAVE DIAGNOSTIC
# ============================================================

diag_df = pd.DataFrame(records)

diag_df.to_csv(
    OUT_CSV,
    index=False
)

total_elapsed = (
    time.time() - overall_start
)


# ============================================================
# 12. SUMMARY
# ============================================================

print()
print("=" * 70)
print("FEASIBILITY SUMMARY")
print("=" * 70)

for design in [
    "A",
    "Cprime"
]:

    sub = diag_df[
        diag_df["design"] == design
    ]

    print()
    print(f"DESIGN {design}")

    print(
        "  valid median       :",
        float(sub["accepted"].median())
    )

    print(
        "  valid mean         :",
        float(sub["accepted"].mean())
    )

    print(
        "  valid min          :",
        int(sub["accepted"].min())
    )

    print(
        "  valid max          :",
        int(sub["accepted"].max())
    )

    print(
        "  images >= 50       :",
        int(sub["reached_50"].sum()),
        "/ 100"
    )

    print(
        "  mean acceptance    :",
        float(sub["acceptance_rate"].mean())
    )

    print(
        "  mean OOB rejects    :",
        float(sub["out_of_bounds"].mean())
    )

    print(
        "  mean area rejects   :",
        float(sub["area_mismatch"].mean())
    )

    print(
        "  mean overlap rejects:",
        float(sub["overlap"].mean())
    )

    if design == "Cprime":
        print(
            "  mean center rejects :",
            float(sub["center_distance"].mean())
        )

print()
print(
    f"TOTAL WALL TIME: "
    f"{total_elapsed / 60:.2f} min"
)

print()
print("Saved:")
print(OUT_CSV)

print()
print("=" * 70)
print("CELL 4S-OPT COMPLETE")
print("=" * 70)

T1.4b — VECTORISED NULL-GEOMETRY FEASIBILITY
Images                  : 100
Required nulls/design   : 50
Attempt cap             : 20000
Image half-diagonal     : 158.39191899
C' tolerance            : ±23.75878785 px
Namespace               : t1_4b-null-v2

CPU vectorised geometry
NO model loading
NO IG
NO production null masks

Generator:
/kaggle/working/isic_final_project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
GLOBAL_SEED           : 20260919
PRIMARY_SEVERITY      : medium

Sample rows            : 100
Cache-map rows         : 100
[001/100] A       | valid=    9 | rate=0.000450 | 1.62s
[001/100] Cprime  | valid=    4 | rate=0.000200 | 1.59s
[002/100] A       | valid=    2 | rate=0.000100 | 1.62s
[002/100] Cprime  | valid=    0 | rate=0.000000 | 1.59s
[003/100] A       | valid=    0 | rate=0.000000 | 1.67s
[003/100] Cprime  | valid=    0 | rate=0.000000 | 1.70s
[004/100] A       | valid=  213 | rate=0.010650 | 1.66s
[004/100] Cprime  | valid=   17 | rate=0

In [21]:
# ============================================================
# CELL 4T — CONDITIONED-TRANSLATION FEASIBILITY DIAGNOSTIC
#
# PURPOSE:
#   Determine whether the previous 20k-attempt failure was
#   primarily caused by inefficient OOB translation proposals.
#
# IMPORTANT:
#   DIAGNOSTIC ONLY.
#   NO production null masks.
#   NO models.
#   NO IG.
#
# We sample angle/reflections first, then choose dx/dy only
# from translations that can geometrically fit the transformed
# mask inside the 224x224 canvas.
# ============================================================

import numpy as np
import pandas as pd
import hashlib
import time
from pathlib import Path

print("=" * 70)
print("T1.4b — CONDITIONED-TRANSLATION FEASIBILITY DIAGNOSTIC")
print("=" * 70)

IMAGE_SIZE = 224
N_REQUIRED = 50
MAX_ATTEMPTS = 20_000

NULL_NAMESPACE = "t1_4b-null-v2"

CENTER_TOLERANCE = 0.15
IMAGE_HALF_DIAGONAL = (
    np.sqrt(IMAGE_SIZE**2 + IMAGE_SIZE**2) / 2.0
)
CPRIME_TOLERANCE_PX = (
    CENTER_TOLERANCE * IMAGE_HALF_DIAGONAL
)

RESULT_DIR = Path(
    "/kaggle/working/T1_4B_generator_matched_spatial_null"
)

SAMPLE_CSV = RESULT_DIR / "T1_4B_IMAGE_SAMPLE.csv"

print(f"Images                : 100")
print(f"Required valid nulls  : {N_REQUIRED}")
print(f"Attempt cap           : {MAX_ATTEMPTS}")
print(
    f"C' tolerance          : ±{CPRIME_TOLERANCE_PX:.6f} px"
)
print()


# ============================================================
# 1. SAFE LOAD FROZEN GENERATOR
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/working/isic_final_project"
)

GENERATOR_CANDIDATES = [
    PROJECT_ROOT /
    "step1J_generator_recovery_lock" /
    "recovered_original_hair_generator.py",

    Path(
        "/kaggle/input/datasets/tejasseshadriiiii/"
        "isic-final-project/"
        "step1J_generator_recovery_lock/"
        "recovered_original_hair_generator.py"
    ),
]

GENERATOR_PATH = next(
    (
        p for p in GENERATOR_CANDIDATES
        if p.exists()
    ),
    None
)

if GENERATOR_PATH is None:
    raise FileNotFoundError(
        "Frozen generator not found."
    )

source = GENERATOR_PATH.read_text()

CUT_MARKERS = [
    "# Load locked population",
    "# LOAD LOCKED POPULATION",
]

cut_pos = None

for marker in CUT_MARKERS:
    pos = source.find(marker)
    if pos >= 0:
        cut_pos = pos
        break

if cut_pos is None:
    raise RuntimeError(
        "Could not locate safe generator cut marker."
    )

definition_source = source[:cut_pos]

generator_ns = {
    "__name__": "frozen_generator_t1_4b",
    "__file__": str(GENERATOR_PATH),
}

exec(
    compile(
        definition_source,
        str(GENERATOR_PATH),
        "exec"
    ),
    generator_ns
)

generate_hair_mask = (
    generator_ns["generate_hair_mask"]
)

GLOBAL_SEED = generator_ns["GLOBAL_SEED"]
PRIMARY_SEVERITY = generator_ns["PRIMARY_SEVERITY"]

assert GLOBAL_SEED == 20260919
assert PRIMARY_SEVERITY == "medium"

print("Generator PASS")
print("GLOBAL_SEED      :", GLOBAL_SEED)
print("PRIMARY_SEVERITY :", PRIMARY_SEVERITY)


# ============================================================
# 2. LOAD SAMPLE
# ============================================================

sample_df = pd.read_csv(
    SAMPLE_CSV
)

assert len(sample_df) == 100

image_ids = sample_df[
    "image"
].tolist()

print(
    "Sample images:",
    len(image_ids)
)


# ============================================================
# 3. DETERMINISTIC SEED
# ============================================================

def null_seed(
    image_id,
    design_tag
):
    payload = (
        f"{NULL_NAMESPACE}|"
        f"{image_id}|"
        f"{design_tag}|"
        f"conditioned-translation"
    )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False
    )


# ============================================================
# 4. TRUE MASK COORDINATES
# ============================================================

def get_true_mask(image_id):

    mask = generate_hair_mask(
        image_id=image_id,
        severity=PRIMARY_SEVERITY
    )

    mask = np.asarray(
        mask,
        dtype=bool
    )

    assert mask.shape == (
        IMAGE_SIZE,
        IMAGE_SIZE
    )

    return mask


def get_coords(mask):

    y, x = np.nonzero(mask)

    return np.column_stack(
        [
            x.astype(np.float64),
            y.astype(np.float64)
        ]
    )


# ============================================================
# 5. IMAGE-CENTER DISTANCE
# ============================================================

IMAGE_CENTER = np.array(
    [
        (IMAGE_SIZE - 1) / 2.0,
        (IMAGE_SIZE - 1) / 2.0
    ],
    dtype=np.float64
)


def center_distance(coords):

    c = coords.mean(axis=0)

    return float(
        np.linalg.norm(
            c - IMAGE_CENTER
        )
    )


# ============================================================
# 6. TRANSFORM BEFORE TRANSLATION
# ============================================================

def rotate_reflect(
    coords,
    angle_deg,
    reflect_x,
    reflect_y
):

    cx = coords[:, 0].mean()
    cy = coords[:, 1].mean()

    x = coords[:, 0] - cx
    y = coords[:, 1] - cy

    theta = np.deg2rad(
        angle_deg
    )

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    xr = (
        cos_t * x
        - sin_t * y
        + cx
    )

    yr = (
        sin_t * x
        + cos_t * y
        + cy
    )

    if reflect_x:
        xr = (
            IMAGE_SIZE - 1
        ) - xr

    if reflect_y:
        yr = (
            IMAGE_SIZE - 1
        ) - yr

    return xr, yr


# ============================================================
# 7. FEASIBLE INTEGER TRANSLATION RANGE
# ============================================================

def feasible_translation_range(
    x,
    y
):

    # Need:
    #
    # 0 <= x + dx <= 223
    #
    # Therefore:
    #
    # -min(x) <= dx <= 223-max(x)
    #
    # Same for y.

    dx_min = int(
        np.ceil(
            -np.min(x)
        )
    )

    dx_max = int(
        np.floor(
            (IMAGE_SIZE - 1)
            - np.max(x)
        )
    )

    dy_min = int(
        np.ceil(
            -np.min(y)
        )
    )

    dy_max = int(
        np.floor(
            (IMAGE_SIZE - 1)
            - np.max(y)
        )
    )

    if (
        dx_min > dx_max
        or dy_min > dy_max
    ):
        return None

    return (
        dx_min,
        dx_max,
        dy_min,
        dy_max
    )


# ============================================================
# 8. CANDIDATE EVALUATION
# ============================================================

def evaluate_candidate(
    x,
    y,
    dx,
    dy,
    true_flat,
    true_area,
    true_center_distance,
    design
):

    xi = np.rint(
        x + dx
    ).astype(np.int32)

    yi = np.rint(
        y + dy
    ).astype(np.int32)

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if (
        np.any(xi < 0)
        or np.any(xi >= IMAGE_SIZE)
        or np.any(yi < 0)
        or np.any(yi >= IMAGE_SIZE)
    ):
        return False, "out_of_bounds"

    # --------------------------------------------------------
    # Rasterised pixels
    # --------------------------------------------------------

    flat = (
        yi.astype(np.int64)
        * IMAGE_SIZE
        + xi.astype(np.int64)
    )

    unique_flat = np.unique(
        flat
    )

    # --------------------------------------------------------
    # Exact area
    # --------------------------------------------------------

    if len(unique_flat) != true_area:
        return False, "area_mismatch"

    # --------------------------------------------------------
    # Zero overlap
    # --------------------------------------------------------

    overlap = np.intersect1d(
        unique_flat,
        true_flat,
        assume_unique=True
    )

    if len(overlap) != 0:
        return False, "overlap"

    # --------------------------------------------------------
    # Pixel centroid
    # --------------------------------------------------------

    ys = (
        unique_flat
        // IMAGE_SIZE
    )

    xs = (
        unique_flat
        % IMAGE_SIZE
    )

    null_center = np.array(
        [
            xs.mean(),
            ys.mean()
        ],
        dtype=np.float64
    )

    null_dist = float(
        np.linalg.norm(
            null_center
            - IMAGE_CENTER
        )
    )

    # --------------------------------------------------------
    # C'
    # --------------------------------------------------------

    if design == "Cprime":

        if (
            abs(
                null_dist
                - true_center_distance
            )
            > CPRIME_TOLERANCE_PX
        ):
            return False, "center_distance"

    return True, "accepted"


# ============================================================
# 9. FEASIBILITY RUN
# ============================================================

def run_image(
    image_id,
    design
):

    true_mask = get_true_mask(
        image_id
    )

    coords = get_coords(
        true_mask
    )

    true_area = int(
        true_mask.sum()
    )

    true_flat = np.flatnonzero(
        true_mask.ravel()
    ).astype(np.int64)

    true_cd = center_distance(
        coords
    )

    rng = np.random.Generator(
        np.random.PCG64(
            null_seed(
                image_id,
                design
            )
        )
    )

    accepted = 0

    rejection_counts = {
        "area_mismatch": 0,
        "overlap": 0,
        "center_distance": 0,
        "out_of_bounds": 0,
        "no_feasible_translation": 0,
    }

    feasible_translation_samples = []

    start = time.time()

    for attempt in range(
        MAX_ATTEMPTS
    ):

        # ----------------------------------------------------
        # Draw angle/reflections exactly first
        # ----------------------------------------------------

        angle = rng.uniform(
            0.0,
            360.0
        )

        reflect_x = bool(
            rng.integers(0, 2)
        )

        reflect_y = bool(
            rng.integers(0, 2)
        )

        x, y = rotate_reflect(
            coords,
            angle,
            reflect_x,
            reflect_y
        )

        ranges = feasible_translation_range(
            x,
            y
        )

        if ranges is None:

            rejection_counts[
                "no_feasible_translation"
            ] += 1

            continue

        (
            dx_min,
            dx_max,
            dy_min,
            dy_max
        ) = ranges

        # ----------------------------------------------------
        # Sample ONLY feasible translations
        # ----------------------------------------------------

        dx = int(
            rng.integers(
                dx_min,
                dx_max + 1
            )
        )

        dy = int(
            rng.integers(
                dy_min,
                dy_max + 1
            )
        )

        feasible_translation_samples.append(
            (
                dx_max - dx_min + 1,
                dy_max - dy_min + 1
            )
        )

        ok, reason = evaluate_candidate(
            x=x,
            y=y,
            dx=dx,
            dy=dy,
            true_flat=true_flat,
            true_area=true_area,
            true_center_distance=true_cd,
            design=design
        )

        if ok:

            accepted += 1

        else:

            rejection_counts[
                reason
            ] += 1

    elapsed = time.time() - start

    if feasible_translation_samples:

        widths = np.array(
            feasible_translation_samples,
            dtype=np.int64
        )

        mean_dx_options = float(
            widths[:, 0].mean()
        )

        mean_dy_options = float(
            widths[:, 1].mean()
        )

    else:

        mean_dx_options = 0.0
        mean_dy_options = 0.0

    return {
        "image": image_id,
        "design": design,
        "true_area": true_area,
        "true_center_distance": true_cd,
        "attempts": MAX_ATTEMPTS,
        "accepted": accepted,
        "acceptance_rate": (
            accepted / MAX_ATTEMPTS
        ),
        "reached_50": (
            accepted >= N_REQUIRED
        ),
        "mean_dx_options": mean_dx_options,
        "mean_dy_options": mean_dy_options,
        "elapsed_sec": elapsed,
        **rejection_counts
    }


# ============================================================
# 10. RUN
# ============================================================

records = []

overall_start = time.time()

for i, image_id in enumerate(
    image_ids,
    start=1
):

    for design in [
        "A",
        "Cprime"
    ]:

        result = run_image(
            image_id,
            design
        )

        records.append(
            result
        )

        print(
            f"[{i:03d}/100] "
            f"{design:7s} | "
            f"valid="
            f"{result['accepted']:5d} | "
            f"rate="
            f"{result['acceptance_rate']:.4f}"
        )


# ============================================================
# 11. SAVE
# ============================================================

diag_df = pd.DataFrame(
    records
)

OUT_CSV = (
    RESULT_DIR /
    "T1_4B_CONDITIONED_TRANSLATION_FEASIBILITY.csv"
)

diag_df.to_csv(
    OUT_CSV,
    index=False
)


# ============================================================
# 12. SUMMARY
# ============================================================

print()
print("=" * 70)
print("CONDITIONED-TRANSLATION SUMMARY")
print("=" * 70)

for design in [
    "A",
    "Cprime"
]:

    sub = diag_df[
        diag_df["design"] == design
    ]

    print()
    print(f"DESIGN {design}")

    print(
        "  valid median        :",
        float(sub["accepted"].median())
    )

    print(
        "  valid mean          :",
        float(sub["accepted"].mean())
    )

    print(
        "  valid min           :",
        int(sub["accepted"].min())
    )

    print(
        "  valid max           :",
        int(sub["accepted"].max())
    )

    print(
        "  images >= 50        :",
        int(sub["reached_50"].sum()),
        "/ 100"
    )

    print(
        "  mean acceptance     :",
        float(sub["acceptance_rate"].mean())
    )

    print(
        "  mean area rejects   :",
        float(sub["area_mismatch"].mean())
    )

    print(
        "  mean overlap rejects:",
        float(sub["overlap"].mean())
    )

    if design == "Cprime":
        print(
            "  mean center rejects :",
            float(sub["center_distance"].mean())
        )

    print(
        "  mean feasible dx    :",
        float(sub["mean_dx_options"].mean())
    )

    print(
        "  mean feasible dy    :",
        float(sub["mean_dy_options"].mean())
    )

print()
print(
    f"TOTAL WALL TIME: "
    f"{(time.time() - overall_start)/60:.2f} min"
)

print()
print("Saved:")
print(OUT_CSV)

print()
print("=" * 70)
print("CELL 4T COMPLETE")
print("=" * 70)

T1.4b — CONDITIONED-TRANSLATION FEASIBILITY DIAGNOSTIC
Images                : 100
Required valid nulls  : 50
Attempt cap           : 20000
C' tolerance          : ±23.758788 px

Generator PASS
GLOBAL_SEED      : 20260919
PRIMARY_SEVERITY : medium
Sample images: 100
[001/100] A       | valid=  545 | rate=0.0272
[001/100] Cprime  | valid=  150 | rate=0.0075
[002/100] A       | valid=  239 | rate=0.0120
[002/100] Cprime  | valid=   42 | rate=0.0021
[003/100] A       | valid=   53 | rate=0.0027
[003/100] Cprime  | valid=   41 | rate=0.0021
[004/100] A       | valid= 1712 | rate=0.0856
[004/100] Cprime  | valid=   84 | rate=0.0042
[005/100] A       | valid= 1009 | rate=0.0505
[005/100] Cprime  | valid=  352 | rate=0.0176
[006/100] A       | valid= 1681 | rate=0.0840
[006/100] Cprime  | valid=  188 | rate=0.0094
[007/100] A       | valid=  103 | rate=0.0052
[007/100] Cprime  | valid=   41 | rate=0.0021
[008/100] A       | valid=   58 | rate=0.0029
[008/100] Cprime  | valid=   31 | rate=0.00

In [22]:
# ============================================================
# CELL 4U — PRODUCTION NULL-MASK GENERATION
# T1.4b v2
#
# 100 candidate images
# 50 Design-A nulls
# 50 Design-C' nulls
# 20,000 attempt cap / image / design
#
# ALL-OR-NOTHING:
# If either design cannot reach 50, the image is dropped.
#
# NO MODEL
# NO IG
# CPU ONLY
# ============================================================

import numpy as np
import pandas as pd
import hashlib
import json
import time
import ast
from pathlib import Path

print("=" * 70)
print("T1.4b — PRODUCTION NULL-MASK GENERATION")
print("=" * 70)

# ------------------------------------------------------------
# FROZEN CONFIGURATION
# ------------------------------------------------------------

IMAGE_SIZE = 224

N_REQUIRED = 50
MAX_ATTEMPTS = 20_000

NULL_NAMESPACE = "t1_4b-null-v2"

CENTER_TOLERANCE = 0.15

IMAGE_HALF_DIAGONAL = (
    np.sqrt(IMAGE_SIZE**2 + IMAGE_SIZE**2) / 2.0
)

CPRIME_TOLERANCE_PX = (
    CENTER_TOLERANCE
    * IMAGE_HALF_DIAGONAL
)

RESULT_DIR = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null"
)

SAMPLE_CSV = (
    RESULT_DIR /
    "T1_4B_IMAGE_SAMPLE.csv"
)

OUT_MASK_DIR = (
    RESULT_DIR /
    "T1_4B_NULL_MASKS"
)

OUT_MASK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

DROPPED_PATH = (
    RESULT_DIR /
    "T1_4B_DROPPED_IMAGES.csv"
)

PROD_SUMMARY_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_GENERATION_SUMMARY.json"
)

print("Required nulls        :", N_REQUIRED)
print("Attempt cap           :", MAX_ATTEMPTS)
print("C' tolerance          :", CPRIME_TOLERANCE_PX)
print("Output directory      :", OUT_MASK_DIR)
print()


# ============================================================
# 1. SAFE LOAD FROZEN GENERATOR
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/working/isic_final_project"
)

GENERATOR_CANDIDATES = [
    PROJECT_ROOT /
    "step1J_generator_recovery_lock" /
    "recovered_original_hair_generator.py",

    Path(
        "/kaggle/input/datasets/tejasseshadriiiii/"
        "isic-final-project/"
        "step1J_generator_recovery_lock/"
        "recovered_original_hair_generator.py"
    ),
]

GENERATOR_PATH = next(
    (
        p for p in GENERATOR_CANDIDATES
        if p.exists()
    ),
    None
)

if GENERATOR_PATH is None:
    raise FileNotFoundError(
        "Frozen generator not found."
    )

source = GENERATOR_PATH.read_text()

CUT_MARKERS = [
    "# Load locked population",
    "# LOAD LOCKED POPULATION",
]

cut_pos = None

for marker in CUT_MARKERS:
    pos = source.find(marker)
    if pos >= 0:
        cut_pos = pos
        break

if cut_pos is None:
    raise RuntimeError(
        "Could not locate frozen generator cut marker."
    )

definition_source = source[:cut_pos]

generator_ns = {
    "__name__": "frozen_generator_t1_4b",
    "__file__": str(GENERATOR_PATH),
}

exec(
    compile(
        definition_source,
        str(GENERATOR_PATH),
        "exec"
    ),
    generator_ns
)

generate_hair_mask = (
    generator_ns["generate_hair_mask"]
)

GLOBAL_SEED = generator_ns["GLOBAL_SEED"]
PRIMARY_SEVERITY = generator_ns["PRIMARY_SEVERITY"]

assert GLOBAL_SEED == 20260919
assert PRIMARY_SEVERITY == "medium"

print("Generator PASS")
print("GLOBAL_SEED      :", GLOBAL_SEED)
print("PRIMARY_SEVERITY :", PRIMARY_SEVERITY)


# ============================================================
# 2. LOAD LOCKED SAMPLE
# ============================================================

sample_df = pd.read_csv(
    SAMPLE_CSV
)

assert len(sample_df) == 100

image_ids = (
    sample_df["image"]
    .astype(str)
    .tolist()
)

print()
print("Candidate images:", len(image_ids))


# ============================================================
# 3. DETERMINISTIC SEED
# ============================================================

def null_seed(
    image_id,
    design_tag,
    null_index
):

    payload = (
        f"{NULL_NAMESPACE}|"
        f"{image_id}|"
        f"{design_tag}|"
        f"{null_index}"
    )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False
    )


# ============================================================
# 4. TRUE MASK / COORDINATES
# ============================================================

def get_true_mask(image_id):

    mask = generate_hair_mask(
        image_id=image_id,
        severity=PRIMARY_SEVERITY
    )

    mask = np.asarray(
        mask,
        dtype=bool
    )

    assert mask.shape == (
        IMAGE_SIZE,
        IMAGE_SIZE
    )

    return mask


def get_coords(mask):

    y, x = np.nonzero(mask)

    return np.column_stack(
        [
            x.astype(np.float64),
            y.astype(np.float64)
        ]
    )


IMAGE_CENTER = np.array(
    [
        (IMAGE_SIZE - 1) / 2.0,
        (IMAGE_SIZE - 1) / 2.0
    ],
    dtype=np.float64
)


def center_distance(coords):

    c = coords.mean(axis=0)

    return float(
        np.linalg.norm(
            c - IMAGE_CENTER
        )
    )


# ============================================================
# 5. ROTATE + REFLECT
# ============================================================

def rotate_reflect(
    coords,
    angle_deg,
    reflect_x,
    reflect_y
):

    cx = coords[:, 0].mean()
    cy = coords[:, 1].mean()

    x = coords[:, 0] - cx
    y = coords[:, 1] - cy

    theta = np.deg2rad(
        angle_deg
    )

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    xr = (
        cos_t * x
        - sin_t * y
        + cx
    )

    yr = (
        sin_t * x
        + cos_t * y
        + cy
    )

    if reflect_x:
        xr = (
            IMAGE_SIZE - 1
        ) - xr

    if reflect_y:
        yr = (
            IMAGE_SIZE - 1
        ) - yr

    return xr, yr


# ============================================================
# 6. FEASIBLE TRANSLATION RANGE
# ============================================================

def feasible_translation_range(
    x,
    y
):

    dx_min = int(
        np.ceil(
            -np.min(x)
        )
    )

    dx_max = int(
        np.floor(
            (IMAGE_SIZE - 1)
            - np.max(x)
        )
    )

    dy_min = int(
        np.ceil(
            -np.min(y)
        )
    )

    dy_max = int(
        np.floor(
            (IMAGE_SIZE - 1)
            - np.max(y)
        )
    )

    if (
        dx_min > dx_max
        or dy_min > dy_max
    ):
        return None

    return (
        dx_min,
        dx_max,
        dy_min,
        dy_max
    )


# ============================================================
# 7. CANDIDATE EVALUATION
# ============================================================

def evaluate_candidate(
    x,
    y,
    dx,
    dy,
    true_flat,
    true_area,
    true_center_distance,
    design
):

    xi = np.rint(
        x + dx
    ).astype(np.int32)

    yi = np.rint(
        y + dy
    ).astype(np.int32)

    # --------------------------------------------------------
    # IN-BOUNDS
    # --------------------------------------------------------

    if (
        np.any(xi < 0)
        or np.any(xi >= IMAGE_SIZE)
        or np.any(yi < 0)
        or np.any(yi >= IMAGE_SIZE)
    ):
        return (
            False,
            "out_of_bounds",
            None,
            None,
            None
        )

    # --------------------------------------------------------
    # RASTERISED PIXELS
    # --------------------------------------------------------

    flat = (
        yi.astype(np.int64)
        * IMAGE_SIZE
        + xi.astype(np.int64)
    )

    unique_flat = np.unique(
        flat
    )

    area = len(
        unique_flat
    )

    # --------------------------------------------------------
    # EXACT AREA
    # --------------------------------------------------------

    if area != true_area:

        return (
            False,
            "area_mismatch",
            None,
            None,
            None
        )

    # --------------------------------------------------------
    # ZERO OVERLAP
    # --------------------------------------------------------

    overlap = np.intersect1d(
        unique_flat,
        true_flat,
        assume_unique=True
    )

    if len(overlap) != 0:

        return (
            False,
            "overlap",
            None,
            None,
            None
        )

    # --------------------------------------------------------
    # PIXEL CENTROID
    # --------------------------------------------------------

    ys = (
        unique_flat
        // IMAGE_SIZE
    )

    xs = (
        unique_flat
        % IMAGE_SIZE
    )

    null_center = np.array(
        [
            xs.mean(),
            ys.mean()
        ],
        dtype=np.float64
    )

    null_center_distance = float(
        np.linalg.norm(
            null_center
            - IMAGE_CENTER
        )
    )

    # --------------------------------------------------------
    # C'
    # --------------------------------------------------------

    if design == "Cprime":

        if (
            abs(
                null_center_distance
                - true_center_distance
            )
            > CPRIME_TOLERANCE_PX
        ):

            return (
                False,
                "center_distance",
                None,
                None,
                None
            )

    # --------------------------------------------------------
    # ACCEPTED
    # --------------------------------------------------------

    candidate_mask = np.zeros(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        ),
        dtype=np.uint8
    )

    candidate_mask[
        yi,
        xi
    ] = 1

    # Sanity check after construction
    assert int(
        candidate_mask.sum()
    ) == true_area

    assert (
        np.sum(
            candidate_mask
            & true_flat.reshape(
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        )
        == 0
    )

    return (
        True,
        "accepted",
        candidate_mask,
        null_center_distance,
        (
            float(angle_deg_placeholder)
            if False else None
        )
    )


# ============================================================
# 8. GENERATE ONE DESIGN
# ============================================================

def generate_design(
    image_id,
    design
):

    true_mask = get_true_mask(
        image_id
    )

    coords = get_coords(
        true_mask
    )

    true_area = int(
        true_mask.sum()
    )

    true_flat = np.flatnonzero(
        true_mask.ravel()
    ).astype(np.int64)

    true_cd = center_distance(
        coords
    )

    accepted_masks = []
    accepted_meta = []

    rejection_counts = {
        "out_of_bounds": 0,
        "area_mismatch": 0,
        "overlap": 0,
        "center_distance": 0,
        "no_feasible_translation": 0,
    }

    start = time.time()

    # --------------------------------------------------------
    # IMPORTANT:
    # Each null slot gets its own deterministic RNG.
    # This follows the frozen namespace/image/design/null_index
    # seed convention.
    # --------------------------------------------------------

    for null_index in range(
        N_REQUIRED
    ):

        rng = np.random.Generator(
            np.random.PCG64(
                null_seed(
                    image_id,
                    design,
                    null_index
                )
            )
        )

        accepted = False

        for attempt in range(
            1,
            MAX_ATTEMPTS + 1
        ):

            # ------------------------------------------------
            # Angle + reflections
            # ------------------------------------------------

            angle_deg = rng.uniform(
                0.0,
                360.0
            )

            reflect_x = bool(
                rng.integers(0, 2)
            )

            reflect_y = bool(
                rng.integers(0, 2)
            )

            x, y = rotate_reflect(
                coords,
                angle_deg,
                reflect_x,
                reflect_y
            )

            ranges = (
                feasible_translation_range(
                    x,
                    y
                )
            )

            if ranges is None:

                rejection_counts[
                    "no_feasible_translation"
                ] += 1

                continue

            (
                dx_min,
                dx_max,
                dy_min,
                dy_max
            ) = ranges

            # ------------------------------------------------
            # ONLY SAMPLE FEASIBLE TRANSLATIONS
            # ------------------------------------------------

            dx = int(
                rng.integers(
                    dx_min,
                    dx_max + 1
                )
            )

            dy = int(
                rng.integers(
                    dy_min,
                    dy_max + 1
                )
            )

            xi = np.rint(
                x + dx
            ).astype(np.int32)

            yi = np.rint(
                y + dy
            ).astype(np.int32)

            # ------------------------------------------------
            # IN-BOUNDS
            # ------------------------------------------------

            if (
                np.any(xi < 0)
                or np.any(xi >= IMAGE_SIZE)
                or np.any(yi < 0)
                or np.any(yi >= IMAGE_SIZE)
            ):

                rejection_counts[
                    "out_of_bounds"
                ] += 1

                continue

            # ------------------------------------------------
            # RASTERISED AREA
            # ------------------------------------------------

            flat = (
                yi.astype(np.int64)
                * IMAGE_SIZE
                + xi.astype(np.int64)
            )

            unique_flat = np.unique(
                flat
            )

            if (
                len(unique_flat)
                != true_area
            ):

                rejection_counts[
                    "area_mismatch"
                ] += 1

                continue

            # ------------------------------------------------
            # ZERO OVERLAP
            # ------------------------------------------------

            overlap = np.intersect1d(
                unique_flat,
                true_flat,
                assume_unique=True
            )

            if len(overlap) != 0:

                rejection_counts[
                    "overlap"
                ] += 1

                continue

            # ------------------------------------------------
            # CENTER DISTANCE
            # ------------------------------------------------

            ys = (
                unique_flat
                // IMAGE_SIZE
            )

            xs = (
                unique_flat
                % IMAGE_SIZE
            )

            null_center = np.array(
                [
                    xs.mean(),
                    ys.mean()
                ],
                dtype=np.float64
            )

            null_cd = float(
                np.linalg.norm(
                    null_center
                    - IMAGE_CENTER
                )
            )

            if design == "Cprime":

                if (
                    abs(
                        null_cd
                        - true_cd
                    )
                    > CPRIME_TOLERANCE_PX
                ):

                    rejection_counts[
                        "center_distance"
                    ] += 1

                    continue

            # ------------------------------------------------
            # ACCEPT
            # ------------------------------------------------

            candidate_mask = np.zeros(
                (
                    IMAGE_SIZE,
                    IMAGE_SIZE
                ),
                dtype=np.uint8
            )

            candidate_mask[
                yi,
                xi
            ] = 1

            assert (
                int(
                    candidate_mask.sum()
                )
                == true_area
            )

            assert (
                np.sum(
                    candidate_mask.astype(bool)
                    &
                    true_mask
                )
                == 0
            )

            accepted_masks.append(
                candidate_mask
            )

            accepted_meta.append(
                {
                    "null_index": null_index,
                    "accepted_attempt": attempt,
                    "angle_deg": angle_deg,
                    "reflect_x": reflect_x,
                    "reflect_y": reflect_y,
                    "dx": dx,
                    "dy": dy,
                    "true_area": true_area,
                    "null_area": int(
                        candidate_mask.sum()
                    ),
                    "true_center_distance":
                        true_cd,
                    "null_center_distance":
                        null_cd,
                    "center_distance_abs_diff":
                        abs(
                            null_cd - true_cd
                        ),
                }
            )

            accepted = True
            break

        # ----------------------------------------------------
        # Failure of one null slot
        # ----------------------------------------------------

        if not accepted:

            return {
                "success": False,
                "image": image_id,
                "design": design,
                "accepted": len(
                    accepted_masks
                ),
                "rejection_counts":
                    rejection_counts,
                "elapsed_sec":
                    time.time() - start,
            }

    return {
        "success": True,
        "image": image_id,
        "design": design,
        "accepted_masks":
            accepted_masks,
        "accepted_meta":
            accepted_meta,
        "rejection_counts":
            rejection_counts,
        "elapsed_sec":
            time.time() - start,
    }


# ============================================================
# 9. GENERATE BOTH DESIGNS FOR ALL 100 IMAGES
# ============================================================

all_results = []
successful = []
failed = []

overall_start = time.time()

for idx, image_id in enumerate(
    image_ids,
    start=1
):

    print()
    print(
        f"[{idx:03d}/100] {image_id}"
    )

    result_A = generate_design(
        image_id,
        "A"
    )

    result_C = generate_design(
        image_id,
        "Cprime"
    )

    ok_A = result_A["success"]
    ok_C = result_C["success"]

    print(
        f"  A      : "
        f"{result_A['accepted']}/50 "
        f"{'PASS' if ok_A else 'FAIL'}"
    )

    print(
        f"  Cprime : "
        f"{result_C['accepted']}/50 "
        f"{'PASS' if ok_C else 'FAIL'}"
    )

    if ok_A and ok_C:

        successful.append(
            image_id
        )

        # ----------------------------------------------------
        # SAVE MASK ARRAYS
        # ----------------------------------------------------

        image_key = hashlib.sha256(
            image_id.encode("utf-8")
        ).hexdigest()[:16]

        np.savez_compressed(
            OUT_MASK_DIR /
            f"{image_key}_A.npz",
            masks=np.stack(
                result_A[
                    "accepted_masks"
                ],
                axis=0
            ).astype(np.uint8)
        )

        np.savez_compressed(
            OUT_MASK_DIR /
            f"{image_key}_Cprime.npz",
            masks=np.stack(
                result_C[
                    "accepted_masks"
                ],
                axis=0
            ).astype(np.uint8)
        )

        for design_result in [
            result_A,
            result_C
        ]:

            for meta in design_result[
                "accepted_meta"
            ]:

                all_results.append(
                    {
                        "image":
                            image_id,
                        "design":
                            design_result[
                                "design"
                            ],
                        **meta
                    }
                )

    else:

        failed.append(
            {
                "image": image_id,
                "A_accepted":
                    result_A["accepted"],
                "Cprime_accepted":
                    result_C["accepted"],
                "A_success":
                    ok_A,
                "Cprime_success":
                    ok_C,
                "A_rejections":
                    json.dumps(
                        result_A[
                            "rejection_counts"
                        ],
                        sort_keys=True
                    ),
                "Cprime_rejections":
                    json.dumps(
                        result_C[
                            "rejection_counts"
                        ],
                        sort_keys=True
                    ),
            }
        )


# ============================================================
# 10. SAVE MANIFEST + DROPPED IMAGES
# ============================================================

manifest_df = pd.DataFrame(
    all_results
)

dropped_df = pd.DataFrame(
    failed
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False
)

dropped_df.to_csv(
    DROPPED_PATH,
    index=False
)


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

total_time = (
    time.time()
    - overall_start
)

summary = {
    "candidate_images": len(image_ids),
    "retained_images": len(successful),
    "dropped_images": len(failed),
    "nulls_per_design_per_image":
        N_REQUIRED,
    "designs": [
        "A",
        "Cprime"
    ],
    "attempt_cap":
        MAX_ATTEMPTS,
    "cprime_tolerance_px":
        CPRIME_TOLERANCE_PX,
    "namespace":
        NULL_NAMESPACE,
    "global_seed":
        GLOBAL_SEED,
    "primary_severity":
        PRIMARY_SEVERITY,
    "wall_time_sec":
        total_time,
}

PROD_SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        indent=2
    )
)

print()
print("=" * 70)
print("PRODUCTION NULL GENERATION SUMMARY")
print("=" * 70)

print(
    "Candidate images :", len(image_ids)
)

print(
    "Retained images   :", len(successful)
)

print(
    "Dropped images    :", len(failed)
)

print(
    "A masks generated :",
    len(successful) * 50
)

print(
    "C' masks generated:",
    len(successful) * 50
)

print(
    "Total masks       :",
    len(successful) * 100
)

print(
    f"Wall time         : "
    f"{total_time/60:.2f} min"
)

print()
print("Manifest:")
print(MANIFEST_PATH)

print()
print("Dropped:")
print(DROPPED_PATH)

print()
print("Summary:")
print(PROD_SUMMARY_PATH)

print()
print("=" * 70)
print("CELL 4U COMPLETE")
print("=" * 70)

T1.4b — PRODUCTION NULL-MASK GENERATION
Required nulls        : 50
Attempt cap           : 20000
C' tolerance          : 23.758787847867996
Output directory      : /kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_MASKS

Generator PASS
GLOBAL_SEED      : 20260919
PRIMARY_SEVERITY : medium

Candidate images: 100

[001/100] ISIC_0013637_downsampled


KeyError: 'accepted'

In [23]:
# ============================================================
# CELL 4U-R — RESUME/RE-RUN PRODUCTION NULL GENERATION
#
# Fixes only the bookkeeping bug in Cell 4U:
# success results use len(accepted_masks), not ['accepted'].
#
# Existing generate_design() is reused unchanged.
# Deterministic seeds => same masks as intended.
# ============================================================

print("=" * 70)
print("T1.4b — PRODUCTION NULL GENERATION — RECOVERY")
print("=" * 70)

# ------------------------------------------------------------
# Reconfirm required objects from Cell 4U
# ------------------------------------------------------------

required_objects = [
    "image_ids",
    "generate_design",
    "OUT_MASK_DIR",
    "MANIFEST_PATH",
    "DROPPED_PATH",
    "PROD_SUMMARY_PATH",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects from Cell 4U: {missing}. "
        "Rerun Cell 4U after fixing the print line."
    )

print("Required Cell 4U objects: PASS")


# ============================================================
# 1. HELPER — ACCEPTED COUNT
# ============================================================

def accepted_count(result):

    if result["success"]:
        return len(
            result["accepted_masks"]
        )

    return int(
        result["accepted"]
    )


# ============================================================
# 2. FRESH OUTPUT RECORDS
# ============================================================

all_results = []
successful = []
failed = []

overall_start = time.time()


# ============================================================
# 3. RUN ALL 100 IMAGES
# ============================================================

for idx, image_id in enumerate(
    image_ids,
    start=1
):

    print()
    print(
        f"[{idx:03d}/100] {image_id}"
    )

    # --------------------------------------------------------
    # Design A
    # --------------------------------------------------------

    result_A = generate_design(
        image_id,
        "A"
    )

    # --------------------------------------------------------
    # Design C'
    # --------------------------------------------------------

    result_C = generate_design(
        image_id,
        "Cprime"
    )

    ok_A = result_A["success"]
    ok_C = result_C["success"]

    n_A = accepted_count(
        result_A
    )

    n_C = accepted_count(
        result_C
    )

    print(
        f"  A      : "
        f"{n_A}/50 "
        f"{'PASS' if ok_A else 'FAIL'}"
    )

    print(
        f"  Cprime : "
        f"{n_C}/50 "
        f"{'PASS' if ok_C else 'FAIL'}"
    )

    # ========================================================
    # 4. ALL-OR-NOTHING IMAGE RETENTION
    # ========================================================

    if ok_A and ok_C:

        successful.append(
            image_id
        )

        image_key = hashlib.sha256(
            image_id.encode("utf-8")
        ).hexdigest()[:16]

        # ----------------------------------------------------
        # Save A masks
        # ----------------------------------------------------

        A_masks = np.stack(
            result_A["accepted_masks"],
            axis=0
        ).astype(np.uint8)

        assert A_masks.shape == (
            50,
            IMAGE_SIZE,
            IMAGE_SIZE
        )

        np.savez_compressed(
            OUT_MASK_DIR /
            f"{image_key}_A.npz",
            masks=A_masks
        )

        # ----------------------------------------------------
        # Save C' masks
        # ----------------------------------------------------

        C_masks = np.stack(
            result_C["accepted_masks"],
            axis=0
        ).astype(np.uint8)

        assert C_masks.shape == (
            50,
            IMAGE_SIZE,
            IMAGE_SIZE
        )

        np.savez_compressed(
            OUT_MASK_DIR /
            f"{image_key}_Cprime.npz",
            masks=C_masks
        )

        # ----------------------------------------------------
        # Manifest rows
        # ----------------------------------------------------

        for design_result in [
            result_A,
            result_C
        ]:

            for meta in design_result[
                "accepted_meta"
            ]:

                all_results.append(
                    {
                        "image":
                            image_id,

                        "design":
                            design_result[
                                "design"
                            ],

                        **meta
                    }
                )

    else:

        failed.append(
            {
                "image":
                    image_id,

                "A_accepted":
                    n_A,

                "Cprime_accepted":
                    n_C,

                "A_success":
                    ok_A,

                "Cprime_success":
                    ok_C,

                "A_rejections":
                    json.dumps(
                        result_A[
                            "rejection_counts"
                        ],
                        sort_keys=True
                    ),

                "Cprime_rejections":
                    json.dumps(
                        result_C[
                            "rejection_counts"
                        ],
                        sort_keys=True
                    ),
            }
        )


# ============================================================
# 5. SAVE MANIFEST
# ============================================================

manifest_df = pd.DataFrame(
    all_results
)

dropped_df = pd.DataFrame(
    failed
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False
)

dropped_df.to_csv(
    DROPPED_PATH,
    index=False
)


# ============================================================
# 6. FINAL SUMMARY
# ============================================================

total_time = (
    time.time()
    - overall_start
)

summary = {
    "candidate_images":
        len(image_ids),

    "retained_images":
        len(successful),

    "dropped_images":
        len(failed),

    "nulls_per_design_per_image":
        N_REQUIRED,

    "designs":
        [
            "A",
            "Cprime"
        ],

    "attempt_cap":
        MAX_ATTEMPTS,

    "cprime_tolerance_px":
        CPRIME_TOLERANCE_PX,

    "namespace":
        NULL_NAMESPACE,

    "global_seed":
        GLOBAL_SEED,

    "primary_severity":
        PRIMARY_SEVERITY,

    "wall_time_sec":
        total_time,
}

PROD_SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        indent=2
    )
)


# ============================================================
# 7. FINAL OUTPUT
# ============================================================

print()
print("=" * 70)
print("PRODUCTION NULL GENERATION SUMMARY")
print("=" * 70)

print(
    "Candidate images :", len(image_ids)
)

print(
    "Retained images   :", len(successful)
)

print(
    "Dropped images    :", len(failed)
)

print(
    "A masks generated :",
    len(successful) * 50
)

print(
    "C' masks generated:",
    len(successful) * 50
)

print(
    "Total masks       :",
    len(successful) * 100
)

print(
    f"Wall time         : "
    f"{total_time / 60:.2f} min"
)

print()
print("Manifest:")
print(MANIFEST_PATH)

print()
print("Dropped:")
print(DROPPED_PATH)

print()
print("Summary:")
print(PROD_SUMMARY_PATH)

print()
print("=" * 70)
print("CELL 4U-R COMPLETE")
print("=" * 70)

T1.4b — PRODUCTION NULL GENERATION — RECOVERY
Required Cell 4U objects: PASS

[001/100] ISIC_0013637_downsampled
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[002/100] ISIC_0013835_downsampled
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[003/100] ISIC_0014212_downsampled
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[004/100] ISIC_0024482
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[005/100] ISIC_0024496
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[006/100] ISIC_0024978
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[007/100] ISIC_0025005
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[008/100] ISIC_0025100
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[009/100] ISIC_0025346
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[010/100] ISIC_0025395
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[011/100] ISIC_0025825
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[012/100] ISIC_0026121
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[013/100] ISIC_0026132
  A      : 50/50 PASS
  Cprime : 50/50 PASS

[0

In [24]:
# ================================================================
# CELL 4V — PRODUCTION NULL-MASK INTEGRITY AUDIT
# ================================================================
import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — PRODUCTION NULL-MASK INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------
# Paths
# ------------------------------------------------
assert OUT_MASK_DIR.exists(), f"Missing null-mask directory: {OUT_MASK_DIR}"
assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert DROPPED_PATH.exists(), f"Missing dropped log: {DROPPED_PATH}"
assert PROD_SUMMARY_PATH.exists(), f"Missing summary: {PROD_SUMMARY_PATH}"

manifest = pd.read_csv(MANIFEST_PATH)
dropped = pd.read_csv(DROPPED_PATH)

with open(PROD_SUMMARY_PATH, "r") as f:
    summary = json.load(f)

print(f"Manifest rows : {len(manifest)}")
print(f"Dropped rows  : {len(dropped)}")

# ------------------------------------------------
# Expected retained population
# ------------------------------------------------
retained_ids = sorted(manifest["image"].unique())

assert len(retained_ids) == 99, (
    f"Expected 99 retained images, found {len(retained_ids)}"
)

assert len(dropped) == 1, (
    f"Expected 1 dropped image, found {len(dropped)}"
)

# ------------------------------------------------
# Required manifest structure
# ------------------------------------------------
required_cols = {
    "image",
    "design",
    "null_index",
    "attempts",
}

missing_cols = required_cols - set(manifest.columns)
assert not missing_cols, f"Manifest missing columns: {missing_cols}"

# ------------------------------------------------
# Check exactly 50 masks per image/design
# ------------------------------------------------
counts = (
    manifest
    .groupby(["image", "design"])
    .size()
    .reset_index(name="n")
)

assert set(counts["n"].unique()) == {50}, (
    f"Non-50 null counts detected:\n{counts[counts['n'] != 50]}"
)

design_counts = manifest.groupby("design").size().to_dict()

print("\nManifest counts:")
print(design_counts)

assert design_counts.get("A", 0) == 4950
assert design_counts.get("Cprime", 0) == 4950

# ------------------------------------------------
# Check null indices are exactly 0..49
# ------------------------------------------------
for (image_id, design), g in manifest.groupby(["image", "design"]):
    idx = sorted(g["null_index"].astype(int).tolist())
    assert idx == list(range(50)), (
        f"Bad null indices for {image_id} / {design}: {idx}"
    )

print("Null-index coverage       : PASS")

# ------------------------------------------------
# Load true-mask lookup
# ------------------------------------------------
sample_df = pd.read_csv(SAMPLE_CSV)

true_mask_lookup = {}

for _, row in sample_df.iterrows():
    image_id = row["image"]

    cache_row = int(row["row"])

    true_mask_lookup[image_id] = np.asarray(
        hair_masks[cache_row],
        dtype=np.uint8
    )

assert len(true_mask_lookup) == 100

# ------------------------------------------------
# Geometry constants
# ------------------------------------------------
H = 224
W = 224

half_diag = np.sqrt(H**2 + W**2) / 2.0
C_TOL = 0.15 * half_diag

print(f"C' centroid tolerance : {C_TOL:.8f} px")

# ------------------------------------------------
# Audit every saved NPZ
# ------------------------------------------------
geometry_rows = []

for image_id in retained_ids:

    true_mask = true_mask_lookup[image_id].astype(bool)

    true_area = int(true_mask.sum())

    true_y, true_x = np.where(true_mask)

    true_cx = float(true_x.mean())
    true_cy = float(true_y.mean())

    center_x = (W - 1) / 2.0
    center_y = (H - 1) / 2.0

    true_center_distance = np.sqrt(
        (true_cx - center_x)**2 +
        (true_cy - center_y)**2
    )

    # -----------------------------
    # Design A
    # -----------------------------
    path_A = OUT_MASK_DIR / f"{image_id}_A.npz"

    assert path_A.exists(), f"Missing A file: {path_A}"

    arr_A = np.load(path_A)["A_masks"]

    assert arr_A.shape == (50, H, W), (
        f"{image_id} A shape = {arr_A.shape}"
    )

    # -----------------------------
    # Design C'
    # -----------------------------
    path_C = OUT_MASK_DIR / f"{image_id}_Cprime.npz"

    assert path_C.exists(), f"Missing C' file: {path_C}"

    arr_C = np.load(path_C)["C_masks"]

    assert arr_C.shape == (50, H, W), (
        f"{image_id} C' shape = {arr_C.shape}"
    )

    for design, arr in [("A", arr_A), ("Cprime", arr_C)]:

        arr_bool = arr.astype(bool)

        for k in range(50):

            m = arr_bool[k]

            # ----------------------------------------
            # Binary mask
            # ----------------------------------------
            unique_vals = np.unique(arr[k])

            assert set(unique_vals).issubset({0, 1}), (
                f"Non-binary mask: {image_id} {design} {k}"
            )

            # ----------------------------------------
            # Exact area equality
            # ----------------------------------------
            null_area = int(m.sum())

            assert null_area == true_area, (
                f"Area mismatch: {image_id} {design} {k}: "
                f"true={true_area}, null={null_area}"
            )

            # ----------------------------------------
            # In-bounds
            # ----------------------------------------
            assert m.shape == (H, W)

            # ----------------------------------------
            # Zero pixel overlap / IoU
            # ----------------------------------------
            intersection = int(np.logical_and(m, true_mask).sum())

            assert intersection == 0, (
                f"Overlap detected: {image_id} {design} {k}"
            )

            union = int(np.logical_or(m, true_mask).sum())

            assert union == 2 * true_area, (
                f"Unexpected union: {image_id} {design} {k}"
            )

            iou = intersection / union

            assert iou == 0.0

            # ----------------------------------------
            # C' center-distance criterion
            # ----------------------------------------
            null_y, null_x = np.where(m)

            null_cx = float(null_x.mean())
            null_cy = float(null_y.mean())

            null_center_distance = np.sqrt(
                (null_cx - center_x)**2 +
                (null_cy - center_y)**2
            )

            center_distance_error = abs(
                null_center_distance -
                true_center_distance
            )

            if design == "Cprime":
                assert center_distance_error <= C_TOL + 1e-9, (
                    f"C' centroid-distance violation: "
                    f"{image_id} null={k} "
                    f"error={center_distance_error:.6f}"
                )

            geometry_rows.append({
                "image": image_id,
                "design": design,
                "null_index": k,
                "true_area": true_area,
                "null_area": null_area,
                "intersection": intersection,
                "iou": iou,
                "true_center_distance": true_center_distance,
                "null_center_distance": null_center_distance,
                "center_distance_error": center_distance_error,
            })

geometry_audit = pd.DataFrame(geometry_rows)

GEOMETRY_AUDIT_PATH = (
    RESULT_DIR / "T1_4B_NULL_GEOMETRY_AUDIT.csv"
)

geometry_audit.to_csv(GEOMETRY_AUDIT_PATH, index=False)

# ------------------------------------------------
# Summary
# ------------------------------------------------
print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print(f"Retained images              : {len(retained_ids)}")
print(f"Dropped images               : {len(dropped)}")
print(f"Design A masks audited       : {(geometry_audit.design == 'A').sum()}")
print(f"Design C' masks audited      : {(geometry_audit.design == 'Cprime').sum()}")
print(f"Total masks audited          : {len(geometry_audit)}")

print(
    f"Exact area equality          : "
    f"{(geometry_audit.null_area == geometry_audit.true_area).all()}"
)

print(
    f"Zero overlap                 : "
    f"{(geometry_audit.intersection == 0).all()}"
)

print(
    f"Zero IoU                     : "
    f"{(geometry_audit.iou == 0).all()}"
)

print(
    f"C' criterion                 : "
    f"{(geometry_audit.loc[geometry_audit.design == 'Cprime', 'center_distance_error'] "
    f"<= C_TOL + 1e-9).all()}"
)

print(f"\nAudit written to:")
print(GEOMETRY_AUDIT_PATH)

print("\n" + "=" * 70)
print("CELL 4V COMPLETE")
print("=" * 70)

SyntaxError: f-string: expecting '}' (2092708685.py, line 297)

In [25]:
# ================================================================
# CELL 4V — PRODUCTION NULL-MASK INTEGRITY AUDIT
# ================================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — PRODUCTION NULL-MASK INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------
# Paths
# ------------------------------------------------
assert OUT_MASK_DIR.exists(), f"Missing null-mask directory: {OUT_MASK_DIR}"
assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert DROPPED_PATH.exists(), f"Missing dropped log: {DROPPED_PATH}"
assert PROD_SUMMARY_PATH.exists(), f"Missing summary: {PROD_SUMMARY_PATH}"

manifest = pd.read_csv(MANIFEST_PATH)
dropped = pd.read_csv(DROPPED_PATH)

with open(PROD_SUMMARY_PATH, "r") as f:
    summary = json.load(f)

print(f"Manifest rows : {len(manifest)}")
print(f"Dropped rows  : {len(dropped)}")

# ------------------------------------------------
# Expected retained population
# ------------------------------------------------
retained_ids = sorted(manifest["image"].unique())

assert len(retained_ids) == 99, (
    f"Expected 99 retained images, found {len(retained_ids)}"
)

assert len(dropped) == 1, (
    f"Expected 1 dropped image, found {len(dropped)}"
)

# ------------------------------------------------
# Required manifest structure
# ------------------------------------------------
required_cols = {
    "image",
    "design",
    "null_index",
    "attempts",
}

missing_cols = required_cols - set(manifest.columns)

assert not missing_cols, (
    f"Manifest missing columns: {missing_cols}"
)

# ------------------------------------------------
# Check exactly 50 masks per image/design
# ------------------------------------------------
counts = (
    manifest
    .groupby(["image", "design"])
    .size()
    .reset_index(name="n")
)

bad_counts = counts[counts["n"] != 50]

assert len(bad_counts) == 0, (
    f"Non-50 null counts detected:\n{bad_counts}"
)

design_counts = manifest.groupby("design").size().to_dict()

print("\nManifest counts:")
print(design_counts)

assert design_counts.get("A", 0) == 4950
assert design_counts.get("Cprime", 0) == 4950

# ------------------------------------------------
# Check null indices are exactly 0..49
# ------------------------------------------------
for (image_id, design), g in manifest.groupby(
    ["image", "design"]
):
    idx = sorted(
        g["null_index"].astype(int).tolist()
    )

    assert idx == list(range(50)), (
        f"Bad null indices for {image_id} / {design}: {idx}"
    )

print("Null-index coverage       : PASS")

# ------------------------------------------------
# Load true-mask lookup
# ------------------------------------------------
sample_df = pd.read_csv(SAMPLE_CSV)

true_mask_lookup = {}

for _, row in sample_df.iterrows():

    image_id = row["image"]
    cache_row = int(row["row"])

    true_mask_lookup[image_id] = np.asarray(
        hair_masks[cache_row],
        dtype=np.uint8
    )

assert len(true_mask_lookup) == 100

# ------------------------------------------------
# Geometry constants
# ------------------------------------------------
H = 224
W = 224

half_diag = np.sqrt(H**2 + W**2) / 2.0
C_TOL = 0.15 * half_diag

print(f"C' centroid tolerance : {C_TOL:.8f} px")

# ------------------------------------------------
# Audit every saved NPZ
# ------------------------------------------------
geometry_rows = []

for image_num, image_id in enumerate(retained_ids, start=1):

    true_mask = true_mask_lookup[image_id].astype(bool)

    true_area = int(true_mask.sum())

    true_y, true_x = np.where(true_mask)

    true_cx = float(true_x.mean())
    true_cy = float(true_y.mean())

    center_x = (W - 1) / 2.0
    center_y = (H - 1) / 2.0

    true_center_distance = np.sqrt(
        (true_cx - center_x)**2 +
        (true_cy - center_y)**2
    )

    # -----------------------------
    # Design A
    # -----------------------------
    path_A = OUT_MASK_DIR / f"{image_id}_A.npz"

    assert path_A.exists(), (
        f"Missing A file: {path_A}"
    )

    arr_A = np.load(path_A)["A_masks"]

    assert arr_A.shape == (50, H, W), (
        f"{image_id} A shape = {arr_A.shape}"
    )

    # -----------------------------
    # Design C'
    # -----------------------------
    path_C = OUT_MASK_DIR / f"{image_id}_Cprime.npz"

    assert path_C.exists(), (
        f"Missing C' file: {path_C}"
    )

    arr_C = np.load(path_C)["C_masks"]

    assert arr_C.shape == (50, H, W), (
        f"{image_id} C' shape = {arr_C.shape}"
    )

    # -----------------------------
    # Audit both designs
    # -----------------------------
    for design, arr in [
        ("A", arr_A),
        ("Cprime", arr_C)
    ]:

        arr_bool = arr.astype(bool)

        for k in range(50):

            m = arr_bool[k]

            # ----------------------------------------
            # Binary mask
            # ----------------------------------------
            unique_vals = np.unique(arr[k])

            assert set(unique_vals).issubset({0, 1}), (
                f"Non-binary mask: "
                f"{image_id} {design} {k}"
            )

            # ----------------------------------------
            # Exact area equality
            # ----------------------------------------
            null_area = int(m.sum())

            assert null_area == true_area, (
                f"Area mismatch: "
                f"{image_id} {design} {k}: "
                f"true={true_area}, null={null_area}"
            )

            # ----------------------------------------
            # Zero pixel overlap / IoU
            # ----------------------------------------
            intersection = int(
                np.logical_and(m, true_mask).sum()
            )

            assert intersection == 0, (
                f"Overlap detected: "
                f"{image_id} {design} {k}"
            )

            union = int(
                np.logical_or(m, true_mask).sum()
            )

            assert union == 2 * true_area, (
                f"Unexpected union: "
                f"{image_id} {design} {k}"
            )

            iou = intersection / union

            assert iou == 0.0

            # ----------------------------------------
            # C' centroid-distance criterion
            # ----------------------------------------
            null_y, null_x = np.where(m)

            null_cx = float(null_x.mean())
            null_cy = float(null_y.mean())

            null_center_distance = np.sqrt(
                (null_cx - center_x)**2 +
                (null_cy - center_y)**2
            )

            center_distance_error = abs(
                null_center_distance -
                true_center_distance
            )

            if design == "Cprime":

                assert center_distance_error <= C_TOL + 1e-9, (
                    f"C' centroid-distance violation: "
                    f"{image_id} null={k} "
                    f"error={center_distance_error:.6f}"
                )

            geometry_rows.append({
                "image": image_id,
                "design": design,
                "null_index": k,
                "true_area": true_area,
                "null_area": null_area,
                "intersection": intersection,
                "iou": iou,
                "true_center_distance": true_center_distance,
                "null_center_distance": null_center_distance,
                "center_distance_error": center_distance_error,
            })

    # Progress every 10 images
    if image_num % 10 == 0 or image_num == len(retained_ids):
        print(
            f"Audited {image_num}/{len(retained_ids)} images"
        )

# ------------------------------------------------
# Build audit dataframe
# ------------------------------------------------
geometry_audit = pd.DataFrame(geometry_rows)

GEOMETRY_AUDIT_PATH = (
    RESULT_DIR / "T1_4B_NULL_GEOMETRY_AUDIT.csv"
)

geometry_audit.to_csv(
    GEOMETRY_AUDIT_PATH,
    index=False
)

# ------------------------------------------------
# Final checks
# ------------------------------------------------
area_ok = (
    geometry_audit["null_area"]
    == geometry_audit["true_area"]
).all()

overlap_ok = (
    geometry_audit["intersection"] == 0
).all()

iou_ok = (
    geometry_audit["iou"] == 0
).all()

cprime_rows = geometry_audit[
    geometry_audit["design"] == "Cprime"
]

cprime_ok = (
    cprime_rows["center_distance_error"]
    <= C_TOL + 1e-9
).all()

# ------------------------------------------------
# Summary
# ------------------------------------------------
print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print(f"Retained images              : {len(retained_ids)}")
print(f"Dropped images               : {len(dropped)}")
print(
    f"Design A masks audited       : "
    f"{(geometry_audit.design == 'A').sum()}"
)
print(
    f"Design C' masks audited      : "
    f"{(geometry_audit.design == 'Cprime').sum()}"
)
print(
    f"Total masks audited          : "
    f"{len(geometry_audit)}"
)

print(
    f"Exact area equality          : "
    f"{area_ok}"
)

print(
    f"Zero overlap                 : "
    f"{overlap_ok}"
)

print(
    f"Zero IoU                     : "
    f"{iou_ok}"
)

print(
    f"C' criterion                 : "
    f"{cprime_ok}"
)

# ------------------------------------------------
# Hard assertions
# ------------------------------------------------
assert len(geometry_audit) == 9900
assert area_ok
assert overlap_ok
assert iou_ok
assert cprime_ok

print("\nAll production null-mask integrity checks: PASS")

print("\nAudit written to:")
print(GEOMETRY_AUDIT_PATH)

print("\n" + "=" * 70)
print("CELL 4V COMPLETE")
print("=" * 70)

T1.4b — PRODUCTION NULL-MASK INTEGRITY AUDIT
Manifest rows : 9900
Dropped rows  : 1


AssertionError: Manifest missing columns: {'attempts'}

In [26]:
# ================================================================
# CELL 4V — PRODUCTION NULL-MASK INTEGRITY AUDIT
# ================================================================

import json
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — PRODUCTION NULL-MASK INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------
# Paths
# ------------------------------------------------
assert OUT_MASK_DIR.exists(), f"Missing null-mask directory: {OUT_MASK_DIR}"
assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert DROPPED_PATH.exists(), f"Missing dropped log: {DROPPED_PATH}"
assert PROD_SUMMARY_PATH.exists(), f"Missing summary: {PROD_SUMMARY_PATH}"

manifest = pd.read_csv(MANIFEST_PATH)
dropped = pd.read_csv(DROPPED_PATH)

with open(PROD_SUMMARY_PATH, "r") as f:
    summary = json.load(f)

print(f"Manifest rows : {len(manifest)}")
print(f"Dropped rows  : {len(dropped)}")

print("\nManifest columns:")
print(list(manifest.columns))

# ------------------------------------------------
# Expected retained population
# ------------------------------------------------
retained_ids = sorted(manifest["image"].unique())

assert len(retained_ids) == 99, (
    f"Expected 99 retained images, found {len(retained_ids)}"
)

assert len(dropped) == 1, (
    f"Expected 1 dropped image, found {len(dropped)}"
)

# ------------------------------------------------
# Required manifest structure
# NOTE:
# The production writer does not save an "attempts"
# column, so we only require fields actually written.
# ------------------------------------------------
required_cols = {
    "image",
    "design",
    "null_index",
}

missing_cols = required_cols - set(manifest.columns)

assert not missing_cols, (
    f"Manifest missing required columns: {missing_cols}"
)

print("Manifest schema            : PASS")

# ------------------------------------------------
# Check exactly 50 masks per image/design
# ------------------------------------------------
counts = (
    manifest
    .groupby(["image", "design"])
    .size()
    .reset_index(name="n")
)

bad_counts = counts[counts["n"] != 50]

assert len(bad_counts) == 0, (
    f"Non-50 null counts detected:\n{bad_counts}"
)

design_counts = manifest.groupby("design").size().to_dict()

print("\nManifest counts:")
print(design_counts)

assert design_counts.get("A", 0) == 4950
assert design_counts.get("Cprime", 0) == 4950

print("50 nulls/image/design      : PASS")

# ------------------------------------------------
# Check null indices exactly 0..49
# ------------------------------------------------
for (image_id, design), g in manifest.groupby(
    ["image", "design"]
):

    idx = sorted(
        g["null_index"].astype(int).tolist()
    )

    assert idx == list(range(50)), (
        f"Bad null indices for "
        f"{image_id} / {design}: {idx}"
    )

print("Null-index coverage         : PASS")

# ------------------------------------------------
# Load true-mask lookup
# ------------------------------------------------
sample_df = pd.read_csv(SAMPLE_CSV)

true_mask_lookup = {}

for _, row in sample_df.iterrows():

    image_id = row["image"]
    cache_row = int(row["row"])

    true_mask_lookup[image_id] = np.asarray(
        hair_masks[cache_row],
        dtype=np.uint8
    )

assert len(true_mask_lookup) == 100

# ------------------------------------------------
# Geometry constants
# ------------------------------------------------
H = 224
W = 224

half_diag = np.sqrt(H**2 + W**2) / 2.0
C_TOL = 0.15 * half_diag

print(
    f"C' centroid tolerance     : "
    f"{C_TOL:.8f} px"
)

# ------------------------------------------------
# Audit every saved NPZ
# ------------------------------------------------
geometry_rows = []

for image_num, image_id in enumerate(
    retained_ids,
    start=1
):

    true_mask = (
        true_mask_lookup[image_id]
        .astype(bool)
    )

    true_area = int(true_mask.sum())

    true_y, true_x = np.where(true_mask)

    true_cx = float(true_x.mean())
    true_cy = float(true_y.mean())

    center_x = (W - 1) / 2.0
    center_y = (H - 1) / 2.0

    true_center_distance = np.sqrt(
        (true_cx - center_x)**2 +
        (true_cy - center_y)**2
    )

    # ============================================================
    # DESIGN A
    # ============================================================
    path_A = (
        OUT_MASK_DIR /
        f"{image_id}_A.npz"
    )

    assert path_A.exists(), (
        f"Missing Design A file: {path_A}"
    )

    with np.load(path_A) as data_A:
        assert "A_masks" in data_A.files, (
            f"A_masks missing from {path_A}"
        )
        arr_A = data_A["A_masks"]

    assert arr_A.shape == (50, H, W), (
        f"{image_id} A shape = {arr_A.shape}"
    )

    # ============================================================
    # DESIGN C'
    # ============================================================
    path_C = (
        OUT_MASK_DIR /
        f"{image_id}_Cprime.npz"
    )

    assert path_C.exists(), (
        f"Missing Design C' file: {path_C}"
    )

    with np.load(path_C) as data_C:
        assert "C_masks" in data_C.files, (
            f"C_masks missing from {path_C}"
        )
        arr_C = data_C["C_masks"]

    assert arr_C.shape == (50, H, W), (
        f"{image_id} C' shape = {arr_C.shape}"
    )

    # ============================================================
    # AUDIT BOTH DESIGNS
    # ============================================================
    for design, arr in [
        ("A", arr_A),
        ("Cprime", arr_C)
    ]:

        arr_bool = arr.astype(bool)

        for k in range(50):

            m = arr_bool[k]

            # ----------------------------------------------------
            # Binary mask
            # ----------------------------------------------------
            unique_vals = np.unique(arr[k])

            assert set(unique_vals).issubset({0, 1}), (
                f"Non-binary mask: "
                f"{image_id} {design} {k}"
            )

            # ----------------------------------------------------
            # Exact area equality
            # ----------------------------------------------------
            null_area = int(m.sum())

            assert null_area == true_area, (
                f"Area mismatch: "
                f"{image_id} {design} {k}: "
                f"true={true_area}, "
                f"null={null_area}"
            )

            # ----------------------------------------------------
            # Zero overlap / IoU
            # ----------------------------------------------------
            intersection = int(
                np.logical_and(
                    m,
                    true_mask
                ).sum()
            )

            assert intersection == 0, (
                f"Overlap detected: "
                f"{image_id} {design} {k}"
            )

            union = int(
                np.logical_or(
                    m,
                    true_mask
                ).sum()
            )

            assert union == 2 * true_area, (
                f"Unexpected union: "
                f"{image_id} {design} {k}"
            )

            iou = intersection / union

            assert iou == 0.0

            # ----------------------------------------------------
            # Centroid-distance calculation
            # ----------------------------------------------------
            null_y, null_x = np.where(m)

            null_cx = float(null_x.mean())
            null_cy = float(null_y.mean())

            null_center_distance = np.sqrt(
                (null_cx - center_x)**2 +
                (null_cy - center_y)**2
            )

            center_distance_error = abs(
                null_center_distance -
                true_center_distance
            )

            # ----------------------------------------------------
            # C' criterion
            # ----------------------------------------------------
            if design == "Cprime":

                assert (
                    center_distance_error
                    <= C_TOL + 1e-9
                ), (
                    f"C' centroid-distance violation: "
                    f"{image_id} null={k} "
                    f"error={center_distance_error:.6f}"
                )

            geometry_rows.append({
                "image": image_id,
                "design": design,
                "null_index": k,
                "true_area": true_area,
                "null_area": null_area,
                "intersection": intersection,
                "iou": iou,
                "true_center_distance": (
                    true_center_distance
                ),
                "null_center_distance": (
                    null_center_distance
                ),
                "center_distance_error": (
                    center_distance_error
                ),
            })

    if (
        image_num % 10 == 0
        or image_num == len(retained_ids)
    ):
        print(
            f"Audited "
            f"{image_num}/{len(retained_ids)} images"
        )

# ------------------------------------------------
# Build audit dataframe
# ------------------------------------------------
geometry_audit = pd.DataFrame(
    geometry_rows
)

GEOMETRY_AUDIT_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_GEOMETRY_AUDIT.csv"
)

geometry_audit.to_csv(
    GEOMETRY_AUDIT_PATH,
    index=False
)

# ------------------------------------------------
# Final checks
# ------------------------------------------------
area_ok = (
    geometry_audit["null_area"]
    ==
    geometry_audit["true_area"]
).all()

overlap_ok = (
    geometry_audit["intersection"]
    == 0
).all()

iou_ok = (
    geometry_audit["iou"]
    == 0
).all()

cprime_rows = geometry_audit[
    geometry_audit["design"] == "Cprime"
]

cprime_ok = (
    cprime_rows["center_distance_error"]
    <= C_TOL + 1e-9
).all()

# ------------------------------------------------
# Final summary
# ------------------------------------------------
print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print(
    f"Retained images              : "
    f"{len(retained_ids)}"
)

print(
    f"Dropped images               : "
    f"{len(dropped)}"
)

print(
    f"Design A masks audited       : "
    f"{(geometry_audit.design == 'A').sum()}"
)

print(
    f"Design C' masks audited      : "
    f"{(geometry_audit.design == 'Cprime').sum()}"
)

print(
    f"Total masks audited          : "
    f"{len(geometry_audit)}"
)

print(
    f"Exact area equality          : "
    f"{area_ok}"
)

print(
    f"Zero overlap                 : "
    f"{overlap_ok}"
)

print(
    f"Zero IoU                     : "
    f"{iou_ok}"
)

print(
    f"C' criterion                 : "
    f"{cprime_ok}"
)

# ------------------------------------------------
# Hard gates
# ------------------------------------------------
assert len(geometry_audit) == 9900
assert area_ok
assert overlap_ok
assert iou_ok
assert cprime_ok

print("\nAll production null-mask integrity checks: PASS")

print("\nAudit written to:")
print(GEOMETRY_AUDIT_PATH)

print("\n" + "=" * 70)
print("CELL 4V COMPLETE")
print("=" * 70)

T1.4b — PRODUCTION NULL-MASK INTEGRITY AUDIT
Manifest rows : 9900
Dropped rows  : 1

Manifest columns:
['image', 'design', 'null_index', 'accepted_attempt', 'angle_deg', 'reflect_x', 'reflect_y', 'dx', 'dy', 'true_area', 'null_area', 'true_center_distance', 'null_center_distance', 'center_distance_abs_diff']
Manifest schema            : PASS

Manifest counts:
{'A': 4950, 'Cprime': 4950}
50 nulls/image/design      : PASS
Null-index coverage         : PASS


NameError: name 'hair_masks' is not defined

In [27]:
# ================================================================
# CELL 4V-R — RESTORE TRUE-MASK CACHE HANDLE
# ================================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — RESTORING TRUE-MASK CACHE HANDLE")
print("=" * 70)

# ------------------------------------------------
# Locate the already-validated rebuilt cache
# ------------------------------------------------
CACHE_DIR = Path("/kaggle/temp/isic_cache")

MASK_AREA_PATH = CACHE_DIR / "mask_area.npy"
CACHE_INDEX_PATH = CACHE_DIR / "cache_index.csv"

assert MASK_AREA_PATH.exists(), (
    f"Missing mask_area.npy: {MASK_AREA_PATH}"
)

assert CACHE_INDEX_PATH.exists(), (
    f"Missing cache_index.csv: {CACHE_INDEX_PATH}"
)

# Find the frozen/rebuilt mask array
mask_candidates = [
    CACHE_DIR / "mask.npy",
    CACHE_DIR / "masks.npy",
    CACHE_DIR / "hair_mask.npy",
    CACHE_DIR / "hair_masks.npy",
]

MASK_PATH = next(
    (p for p in mask_candidates if p.exists()),
    None
)

assert MASK_PATH is not None, (
    "Could not locate the cached mask array. "
    f"Checked: {mask_candidates}"
)

print(f"Mask array : {MASK_PATH}")

# ------------------------------------------------
# Load true masks
# ------------------------------------------------
hair_masks = np.load(
    MASK_PATH,
    mmap_mode="r"
)

cache_index = pd.read_csv(
    CACHE_INDEX_PATH
)

print(f"hair_masks shape : {hair_masks.shape}")
print(f"cache index rows : {len(cache_index)}")

# ------------------------------------------------
# Basic cache integrity
# ------------------------------------------------
assert hair_masks.ndim == 3
assert hair_masks.shape[1:] == (224, 224)
assert len(cache_index) == hair_masks.shape[0]

print("Mask/cache alignment : PASS")

# ------------------------------------------------
# Confirm sample mapping
# ------------------------------------------------
sample_df = pd.read_csv(SAMPLE_CSV)

assert len(sample_df) == 100

missing = set(sample_df["image"]) - set(cache_index["image"])

assert len(missing) == 0, (
    f"Sample images missing from cache: {missing}"
)

print("100-image sample/cache mapping : PASS")

print("\n" + "=" * 70)
print("CELL 4V-R COMPLETE")
print("=" * 70)

T1.4b — RESTORING TRUE-MASK CACHE HANDLE


AssertionError: Could not locate the cached mask array. Checked: [PosixPath('/kaggle/temp/isic_cache/mask.npy'), PosixPath('/kaggle/temp/isic_cache/masks.npy'), PosixPath('/kaggle/temp/isic_cache/hair_mask.npy'), PosixPath('/kaggle/temp/isic_cache/hair_masks.npy')]

In [28]:
# ================================================================
# CELL 4V-S — INSPECT VALIDATED CACHE FILES
# ================================================================

from pathlib import Path

CACHE_DIR = Path("/kaggle/temp/isic_cache")

print("=" * 70)
print("T1.4b — CACHE FILE INVENTORY")
print("=" * 70)

assert CACHE_DIR.exists(), (
    f"Cache directory missing: {CACHE_DIR}"
)

files = sorted(CACHE_DIR.iterdir())

print(f"\nCache directory: {CACHE_DIR}")
print(f"Files found    : {len(files)}\n")

for p in files:
    if p.is_file():
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"{p.name:<45} {size_mb:>10.2f} MB")
    else:
        print(f"{p.name:<45} <DIR>")

print("\n" + "=" * 70)
print("CELL 4V-S COMPLETE")
print("=" * 70)

T1.4b — CACHE FILE INVENTORY

Cache directory: /kaggle/temp/isic_cache
Files found    : 5

_builder_record.json                                0.00 MB
cache_index.csv                                     0.93 MB
clean_224.npy                                    3334.34 MB
hair_224.npy                                     3334.34 MB
mask_area.npy                                       0.09 MB

CELL 4V-S COMPLETE


In [29]:
# ================================================================
# CELL 4V-R2 — RESTORE TRUE-MASK AUDIT INPUTS
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — RESTORING TRUE-MASK AUDIT INPUTS")
print("=" * 70)

CACHE_DIR = Path("/kaggle/temp/isic_cache")

CLEAN_PATH = CACHE_DIR / "clean_224.npy"
HAIR_PATH = CACHE_DIR / "hair_224.npy"
CACHE_INDEX_PATH = CACHE_DIR / "cache_index.csv"

assert CLEAN_PATH.exists()
assert HAIR_PATH.exists()
assert CACHE_INDEX_PATH.exists()

# ------------------------------------------------
# Load only what the audit needs
# ------------------------------------------------
hair_224 = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

cache_index = pd.read_csv(
    CACHE_INDEX_PATH
)

print(f"hair_224 shape : {hair_224.shape}")
print(f"cache rows     : {len(cache_index)}")

# ------------------------------------------------
# Validate cache structure
# ------------------------------------------------
assert hair_224.shape == (23227, 224, 224, 3)
assert len(cache_index) == 23227

print("Hair cache shape       : PASS")
print("Cache index alignment  : PASS")

# ------------------------------------------------
# Load locked 100-image sample
# ------------------------------------------------
sample_df = pd.read_csv(SAMPLE_CSV)

assert len(sample_df) == 100

# ------------------------------------------------
# Build image -> cache-row lookup
# ------------------------------------------------
cache_row_lookup = dict(
    zip(
        cache_index["image"],
        cache_index.index
    )
)

missing = [
    image_id
    for image_id in sample_df["image"]
    if image_id not in cache_row_lookup
]

assert len(missing) == 0, (
    f"Sample images missing from cache: {missing}"
)

# ------------------------------------------------
# IMPORTANT:
# Reconstruct the true binary hair mask from
# clean_224 vs hair_224.
#
# The frozen cache stores the clean and hair images,
# while mask_area.npy stores the corresponding area.
# ------------------------------------------------
mask_area = np.load(
    CACHE_DIR / "mask_area.npy",
    mmap_mode="r"
)

assert mask_area.shape == (23227,)

print("Mask-area cache shape  : PASS")

# ------------------------------------------------
# Build true-mask lookup
# ------------------------------------------------
true_mask_lookup = {}

for _, row in sample_df.iterrows():

    image_id = row["image"]

    cache_row = cache_row_lookup[image_id]

    clean_img = np.asarray(
        np.load(CLEAN_PATH, mmap_mode="r")[cache_row]
    )

    hair_img = np.asarray(
        hair_224[cache_row]
    )

    # Changed pixels define the frozen hair mask.
    true_mask = np.any(
        clean_img != hair_img,
        axis=2
    )

    # Verify against frozen mask_area.
    expected_area = int(mask_area[cache_row])
    actual_area = int(true_mask.sum())

    assert actual_area == expected_area, (
        f"Mask-area mismatch for {image_id}: "
        f"actual={actual_area}, "
        f"expected={expected_area}"
    )

    true_mask_lookup[image_id] = (
        true_mask.astype(np.uint8)
    )

print(
    f"True masks reconstructed : "
    f"{len(true_mask_lookup)}/100"
)

print(
    f"Mask areas verified      : "
    f"{len(true_mask_lookup)}/100"
)

assert len(true_mask_lookup) == 100

print("\nTrue-mask reconstruction : PASS")

print("\n" + "=" * 70)
print("CELL 4V-R2 COMPLETE")
print("=" * 70)

T1.4b — RESTORING TRUE-MASK AUDIT INPUTS
hair_224 shape : (23227, 224, 224, 3)
cache rows     : 23227
Hair cache shape       : PASS
Cache index alignment  : PASS
Mask-area cache shape  : PASS
True masks reconstructed : 100/100
Mask areas verified      : 100/100

True-mask reconstruction : PASS

CELL 4V-R2 COMPLETE


In [30]:
# ================================================================
# CELL 4V-CONT — AUDIT ALL SAVED PRODUCTION NULL MASKS
# ================================================================

print("=" * 70)
print("T1.4b — AUDITING ALL 9,900 PRODUCTION NULL MASKS")
print("=" * 70)

# ------------------------------------------------
# Geometry constants
# ------------------------------------------------
H = 224
W = 224

half_diag = np.sqrt(H**2 + W**2) / 2.0
C_TOL = 0.15 * half_diag

center_x = (W - 1) / 2.0
center_y = (H - 1) / 2.0

print(f"C' centroid tolerance : {C_TOL:.8f} px")

# ------------------------------------------------
# Re-load production manifest
# ------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

retained_ids = sorted(
    manifest["image"].unique()
)

assert len(retained_ids) == 99

# ------------------------------------------------
# Audit records
# ------------------------------------------------
geometry_rows = []

for image_num, image_id in enumerate(
    retained_ids,
    start=1
):

    # ============================================================
    # TRUE MASK
    # ============================================================
    true_mask = (
        true_mask_lookup[image_id]
        .astype(bool)
    )

    true_area = int(true_mask.sum())

    true_y, true_x = np.where(true_mask)

    true_cx = float(true_x.mean())
    true_cy = float(true_y.mean())

    true_center_distance = np.sqrt(
        (true_cx - center_x)**2 +
        (true_cy - center_y)**2
    )

    # ============================================================
    # LOAD DESIGN A
    # ============================================================
    path_A = (
        OUT_MASK_DIR /
        f"{image_id}_A.npz"
    )

    assert path_A.exists(), (
        f"Missing Design A file: {path_A}"
    )

    with np.load(path_A) as data_A:

        assert "A_masks" in data_A.files, (
            f"A_masks missing from {path_A}"
        )

        arr_A = data_A["A_masks"]

    assert arr_A.shape == (50, H, W), (
        f"{image_id} A shape = {arr_A.shape}"
    )

    # ============================================================
    # LOAD DESIGN C'
    # ============================================================
    path_C = (
        OUT_MASK_DIR /
        f"{image_id}_Cprime.npz"
    )

    assert path_C.exists(), (
        f"Missing Design C' file: {path_C}"
    )

    with np.load(path_C) as data_C:

        assert "C_masks" in data_C.files, (
            f"C_masks missing from {path_C}"
        )

        arr_C = data_C["C_masks"]

    assert arr_C.shape == (50, H, W), (
        f"{image_id} C' shape = {arr_C.shape}"
    )

    # ============================================================
    # AUDIT BOTH DESIGNS
    # ============================================================
    for design, arr in [
        ("A", arr_A),
        ("Cprime", arr_C)
    ]:

        # Binary-value check for whole array
        unique_values = np.unique(arr)

        assert set(unique_values).issubset({0, 1}), (
            f"Non-binary values in "
            f"{image_id} / {design}: "
            f"{unique_values}"
        )

        arr_bool = arr.astype(bool)

        for k in range(50):

            m = arr_bool[k]

            # ----------------------------------------------------
            # Exact area equality
            # ----------------------------------------------------
            null_area = int(m.sum())

            assert null_area == true_area, (
                f"AREA FAILURE: "
                f"{image_id} / {design} / null={k}: "
                f"true={true_area}, "
                f"null={null_area}"
            )

            # ----------------------------------------------------
            # Zero pixel overlap
            # ----------------------------------------------------
            intersection = int(
                np.logical_and(
                    m,
                    true_mask
                ).sum()
            )

            assert intersection == 0, (
                f"OVERLAP FAILURE: "
                f"{image_id} / {design} / null={k}: "
                f"intersection={intersection}"
            )

            # ----------------------------------------------------
            # IoU
            # ----------------------------------------------------
            union = int(
                np.logical_or(
                    m,
                    true_mask
                ).sum()
            )

            assert union == 2 * true_area, (
                f"UNION FAILURE: "
                f"{image_id} / {design} / null={k}"
            )

            iou = intersection / union

            assert iou == 0.0

            # ----------------------------------------------------
            # Null centroid
            # ----------------------------------------------------
            null_y, null_x = np.where(m)

            null_cx = float(null_x.mean())
            null_cy = float(null_y.mean())

            null_center_distance = np.sqrt(
                (null_cx - center_x)**2 +
                (null_cy - center_y)**2
            )

            center_distance_error = abs(
                null_center_distance -
                true_center_distance
            )

            # ----------------------------------------------------
            # Design C' criterion
            # ----------------------------------------------------
            if design == "Cprime":

                assert (
                    center_distance_error
                    <= C_TOL + 1e-9
                ), (
                    f"C' CENTER FAILURE: "
                    f"{image_id} / null={k}: "
                    f"error={center_distance_error:.8f}, "
                    f"tolerance={C_TOL:.8f}"
                )

            geometry_rows.append({
                "image": image_id,
                "design": design,
                "null_index": k,
                "true_area": true_area,
                "null_area": null_area,
                "intersection": intersection,
                "iou": iou,
                "true_center_distance": (
                    true_center_distance
                ),
                "null_center_distance": (
                    null_center_distance
                ),
                "center_distance_error": (
                    center_distance_error
                ),
            })

    if (
        image_num % 10 == 0
        or image_num == len(retained_ids)
    ):
        print(
            f"Audited "
            f"{image_num}/{len(retained_ids)} images"
        )

# ================================================================
# BUILD AUDIT TABLE
# ================================================================
geometry_audit = pd.DataFrame(
    geometry_rows
)

GEOMETRY_AUDIT_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_GEOMETRY_AUDIT.csv"
)

geometry_audit.to_csv(
    GEOMETRY_AUDIT_PATH,
    index=False
)

# ================================================================
# FINAL GATES
# ================================================================
assert len(geometry_audit) == 9900

area_ok = (
    geometry_audit["null_area"]
    ==
    geometry_audit["true_area"]
).all()

overlap_ok = (
    geometry_audit["intersection"]
    == 0
).all()

iou_ok = (
    geometry_audit["iou"]
    == 0
).all()

cprime_rows = geometry_audit[
    geometry_audit["design"] == "Cprime"
]

cprime_ok = (
    cprime_rows["center_distance_error"]
    <= C_TOL + 1e-9
).all()

# ================================================================
# FINAL REPORT
# ================================================================
print("\n" + "=" * 70)
print("PRODUCTION NULL-MASK AUDIT SUMMARY")
print("=" * 70)

print(
    f"Retained images              : "
    f"{len(retained_ids)}"
)

print(
    f"Design A masks audited       : "
    f"{(geometry_audit['design'] == 'A').sum()}"
)

print(
    f"Design C' masks audited      : "
    f"{(geometry_audit['design'] == 'Cprime').sum()}"
)

print(
    f"Total masks audited          : "
    f"{len(geometry_audit)}"
)

print(
    f"Exact area equality          : "
    f"{area_ok}"
)

print(
    f"Zero pixel overlap           : "
    f"{overlap_ok}"
)

print(
    f"Zero IoU                     : "
    f"{iou_ok}"
)

print(
    f"C' centroid criterion       : "
    f"{cprime_ok}"
)

# ================================================================
# HARD PASS
# ================================================================
assert area_ok
assert overlap_ok
assert iou_ok
assert cprime_ok

print("\nALL 9,900 PRODUCTION NULL MASKS: PASS")

print("\nAudit file:")
print(GEOMETRY_AUDIT_PATH)

print("\n" + "=" * 70)
print("CELL 4V-CONT COMPLETE")
print("=" * 70)

T1.4b — AUDITING ALL 9,900 PRODUCTION NULL MASKS
C' centroid tolerance : 23.75878785 px


AssertionError: Missing Design A file: /kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_MASKS/ISIC_0013637_downsampled_A.npz

In [31]:
# ================================================================
# CELL 4V-DIAG — VERIFY ACTUAL NULL-MASK FILENAMES
# ================================================================

print("=" * 70)
print("T1.4b — NULL-MASK FILENAME DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------
# Load manifest
# ------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

print("\nManifest image examples:")
print(manifest["image"].drop_duplicates().head(10).tolist())

# ------------------------------------------------
# Inspect actual files
# ------------------------------------------------
mask_files = sorted(OUT_MASK_DIR.glob("*.npz"))

print(f"\nNPZ files found : {len(mask_files)}")

print("\nFirst 20 actual filenames:")
for p in mask_files[:20]:
    print(p.name)

# ------------------------------------------------
# Compare expected vs actual
# ------------------------------------------------
expected_A = [
    OUT_MASK_DIR / f"{image_id}_A.npz"
    for image_id in sorted(manifest["image"].unique())
]

expected_C = [
    OUT_MASK_DIR / f"{image_id}_Cprime.npz"
    for image_id in sorted(manifest["image"].unique())
]

missing_A = [
    str(p)
    for p in expected_A
    if not p.exists()
]

missing_C = [
    str(p)
    for p in expected_C
    if not p.exists()
]

print("\nExpected A files missing :", len(missing_A))
print("Expected C' files missing:", len(missing_C))

if missing_A:
    print("\nFirst 10 missing A paths:")
    for p in missing_A[:10]:
        print(p)

if missing_C:
    print("\nFirst 10 missing C' paths:")
    for p in missing_C[:10]:
        print(p)

# ------------------------------------------------
# Identify filename pattern
# ------------------------------------------------
print("\nActual filename suffix counts:")

suffix_counts = {}

for p in mask_files:
    name = p.name

    if "_A" in name:
        suffix = "_A"
    elif "_Cprime" in name:
        suffix = "_Cprime"
    else:
        suffix = "OTHER"

    suffix_counts[suffix] = (
        suffix_counts.get(suffix, 0) + 1
    )

print(suffix_counts)

print("\n" + "=" * 70)
print("CELL 4V-DIAG COMPLETE")
print("=" * 70)

T1.4b — NULL-MASK FILENAME DIAGNOSTIC

Manifest image examples:
['ISIC_0013637_downsampled', 'ISIC_0013835_downsampled', 'ISIC_0014212_downsampled', 'ISIC_0024482', 'ISIC_0024496', 'ISIC_0024978', 'ISIC_0025005', 'ISIC_0025100', 'ISIC_0025346', 'ISIC_0025395']

NPZ files found : 198

First 20 actual filenames:
0072e43c1a71ee6b_A.npz
0072e43c1a71ee6b_Cprime.npz
019abb6fb5367c2b_A.npz
019abb6fb5367c2b_Cprime.npz
029ddc30446f3f55_A.npz
029ddc30446f3f55_Cprime.npz
041e48a0b9e9e12c_A.npz
041e48a0b9e9e12c_Cprime.npz
05a782ba8b6baf35_A.npz
05a782ba8b6baf35_Cprime.npz
05d18d9c48b90a67_A.npz
05d18d9c48b90a67_Cprime.npz
08423b6e44a8b48a_A.npz
08423b6e44a8b48a_Cprime.npz
0c3a8c6f723d48e6_A.npz
0c3a8c6f723d48e6_Cprime.npz
0e4df2382e273005_A.npz
0e4df2382e273005_Cprime.npz
111ed3696632b901_A.npz
111ed3696632b901_Cprime.npz

Expected A files missing : 99
Expected C' files missing: 99

First 10 missing A paths:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_MASKS/ISIC_0013637_downsam

In [32]:
# ================================================================
# CELL 4V-CONT — AUDIT SAVED NULL MASKS USING IMAGE-KEY FILENAMES
# ================================================================

print("=" * 70)
print("T1.4b — AUDITING ALL 9,900 PRODUCTION NULL MASKS")
print("=" * 70)

# ------------------------------------------------
# Geometry constants
# ------------------------------------------------
H = 224
W = 224

half_diag = np.sqrt(H**2 + W**2) / 2.0
C_TOL = 0.15 * half_diag

center_x = (W - 1) / 2.0
center_y = (H - 1) / 2.0

print(f"C' centroid tolerance : {C_TOL:.8f} px")

# ------------------------------------------------
# Load manifest + sample lock
# ------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)
sample_df = pd.read_csv(SAMPLE_CSV)

# ------------------------------------------------
# Verify image -> image_key mapping exists
# ------------------------------------------------
required_sample_cols = {
    "image",
    "image_key",
    "row"
}

missing_sample_cols = (
    required_sample_cols
    - set(sample_df.columns)
)

assert not missing_sample_cols, (
    f"SAMPLE_CSV missing columns: "
    f"{missing_sample_cols}"
)

# ------------------------------------------------
# Build mappings
# ------------------------------------------------
sample_map = (
    sample_df[
        ["image", "image_key", "row"]
    ]
    .drop_duplicates("image")
)

assert len(sample_map) == 100

image_to_key = dict(
    zip(
        sample_map["image"],
        sample_map["image_key"]
    )
)

image_to_row = dict(
    zip(
        sample_map["image"],
        sample_map["row"]
    )
)

# ------------------------------------------------
# Retained population
# ------------------------------------------------
retained_ids = sorted(
    manifest["image"].unique()
)

assert len(retained_ids) == 99

missing_keys = [
    image_id
    for image_id in retained_ids
    if image_id not in image_to_key
]

assert len(missing_keys) == 0, (
    f"Missing image_key for retained images: "
    f"{missing_keys}"
)

print(
    f"Retained images           : "
    f"{len(retained_ids)}"
)

# ------------------------------------------------
# Verify expected NPZ files from image_key
# ------------------------------------------------
missing_A = []
missing_C = []

for image_id in retained_ids:

    image_key = image_to_key[image_id]

    path_A = (
        OUT_MASK_DIR /
        f"{image_key}_A.npz"
    )

    path_C = (
        OUT_MASK_DIR /
        f"{image_key}_Cprime.npz"
    )

    if not path_A.exists():
        missing_A.append(
            (image_id, image_key)
        )

    if not path_C.exists():
        missing_C.append(
            (image_id, image_key)
        )

assert len(missing_A) == 0, (
    f"Missing Design A files: {missing_A[:10]}"
)

assert len(missing_C) == 0, (
    f"Missing Design C' files: {missing_C[:10]}"
)

print("Design A file mapping      : PASS")
print("Design C' file mapping     : PASS")

# ------------------------------------------------
# Audit records
# ------------------------------------------------
geometry_rows = []

for image_num, image_id in enumerate(
    retained_ids,
    start=1
):

    image_key = image_to_key[image_id]

    # ============================================================
    # TRUE MASK
    # ============================================================
    true_mask = (
        true_mask_lookup[image_id]
        .astype(bool)
    )

    true_area = int(true_mask.sum())

    true_y, true_x = np.where(true_mask)

    true_cx = float(true_x.mean())
    true_cy = float(true_y.mean())

    true_center_distance = np.sqrt(
        (true_cx - center_x)**2 +
        (true_cy - center_y)**2
    )

    # ============================================================
    # LOAD DESIGN A
    # ============================================================
    path_A = (
        OUT_MASK_DIR /
        f"{image_key}_A.npz"
    )

    with np.load(path_A) as data_A:

        assert "A_masks" in data_A.files, (
            f"A_masks missing from {path_A}"
        )

        arr_A = data_A["A_masks"]

    assert arr_A.shape == (50, H, W), (
        f"{image_id} / A shape = {arr_A.shape}"
    )

    # ============================================================
    # LOAD DESIGN C'
    # ============================================================
    path_C = (
        OUT_MASK_DIR /
        f"{image_key}_Cprime.npz"
    )

    with np.load(path_C) as data_C:

        assert "C_masks" in data_C.files, (
            f"C_masks missing from {path_C}"
        )

        arr_C = data_C["C_masks"]

    assert arr_C.shape == (50, H, W), (
        f"{image_id} / C' shape = {arr_C.shape}"
    )

    # ============================================================
    # AUDIT BOTH DESIGNS
    # ============================================================
    for design, arr in [
        ("A", arr_A),
        ("Cprime", arr_C)
    ]:

        # --------------------------------------------------------
        # Binary check
        # --------------------------------------------------------
        unique_values = np.unique(arr)

        assert set(unique_values).issubset({0, 1}), (
            f"Non-binary values: "
            f"{image_id} / {design}: "
            f"{unique_values}"
        )

        arr_bool = arr.astype(bool)

        for k in range(50):

            m = arr_bool[k]

            # ----------------------------------------------------
            # Exact area equality
            # ----------------------------------------------------
            null_area = int(m.sum())

            assert null_area == true_area, (
                f"AREA FAILURE: "
                f"{image_id} / {design} / null={k}: "
                f"true={true_area}, "
                f"null={null_area}"
            )

            # ----------------------------------------------------
            # Zero overlap
            # ----------------------------------------------------
            intersection = int(
                np.logical_and(
                    m,
                    true_mask
                ).sum()
            )

            assert intersection == 0, (
                f"OVERLAP FAILURE: "
                f"{image_id} / {design} / null={k}: "
                f"intersection={intersection}"
            )

            # ----------------------------------------------------
            # IoU
            # ----------------------------------------------------
            union = int(
                np.logical_or(
                    m,
                    true_mask
                ).sum()
            )

            assert union == 2 * true_area, (
                f"UNION FAILURE: "
                f"{image_id} / {design} / null={k}"
            )

            iou = intersection / union

            assert iou == 0.0

            # ----------------------------------------------------
            # Null centroid
            # ----------------------------------------------------
            null_y, null_x = np.where(m)

            null_cx = float(null_x.mean())
            null_cy = float(null_y.mean())

            null_center_distance = np.sqrt(
                (null_cx - center_x)**2 +
                (null_cy - center_y)**2
            )

            center_distance_error = abs(
                null_center_distance -
                true_center_distance
            )

            # ----------------------------------------------------
            # C' centroid criterion
            # ----------------------------------------------------
            if design == "Cprime":

                assert (
                    center_distance_error
                    <= C_TOL + 1e-9
                ), (
                    f"C' CENTER FAILURE: "
                    f"{image_id} / null={k}: "
                    f"error={center_distance_error:.8f}, "
                    f"tolerance={C_TOL:.8f}"
                )

            geometry_rows.append({
                "image": image_id,
                "image_key": image_key,
                "design": design,
                "null_index": k,
                "true_area": true_area,
                "null_area": null_area,
                "intersection": intersection,
                "iou": iou,
                "true_center_distance": (
                    true_center_distance
                ),
                "null_center_distance": (
                    null_center_distance
                ),
                "center_distance_error": (
                    center_distance_error
                ),
            })

    if (
        image_num % 10 == 0
        or image_num == len(retained_ids)
    ):
        print(
            f"Audited "
            f"{image_num}/{len(retained_ids)} images"
        )

# ================================================================
# BUILD AUDIT TABLE
# ================================================================
geometry_audit = pd.DataFrame(
    geometry_rows
)

GEOMETRY_AUDIT_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_GEOMETRY_AUDIT.csv"
)

geometry_audit.to_csv(
    GEOMETRY_AUDIT_PATH,
    index=False
)

# ================================================================
# FINAL GATES
# ================================================================
assert len(geometry_audit) == 9900

area_ok = (
    geometry_audit["null_area"]
    ==
    geometry_audit["true_area"]
).all()

overlap_ok = (
    geometry_audit["intersection"]
    == 0
).all()

iou_ok = (
    geometry_audit["iou"]
    == 0
).all()

cprime_rows = geometry_audit[
    geometry_audit["design"] == "Cprime"
]

cprime_ok = (
    cprime_rows["center_distance_error"]
    <= C_TOL + 1e-9
).all()

# ================================================================
# FINAL REPORT
# ================================================================
print("\n" + "=" * 70)
print("PRODUCTION NULL-MASK AUDIT SUMMARY")
print("=" * 70)

print(
    f"Retained images              : "
    f"{len(retained_ids)}"
)

print(
    f"Design A masks audited       : "
    f"{(geometry_audit['design'] == 'A').sum()}"
)

print(
    f"Design C' masks audited      : "
    f"{(geometry_audit['design'] == 'Cprime').sum()}"
)

print(
    f"Total masks audited          : "
    f"{len(geometry_audit)}"
)

print(
    f"Exact area equality          : "
    f"{area_ok}"
)

print(
    f"Zero pixel overlap           : "
    f"{overlap_ok}"
)

print(
    f"Zero IoU                     : "
    f"{iou_ok}"
)

print(
    f"C' centroid criterion       : "
    f"{cprime_ok}"
)

# ================================================================
# HARD PASS
# ================================================================
assert area_ok
assert overlap_ok
assert iou_ok
assert cprime_ok

print("\nALL 9,900 PRODUCTION NULL MASKS: PASS")

print("\nAudit file:")
print(GEOMETRY_AUDIT_PATH)

print("\n" + "=" * 70)
print("CELL 4V-CONT COMPLETE")
print("=" * 70)

T1.4b — AUDITING ALL 9,900 PRODUCTION NULL MASKS
C' centroid tolerance : 23.75878785 px
Retained images           : 99


AssertionError: Missing Design A files: [('ISIC_0013637_downsampled', '340db516a73b627d3ffdbf56db21791ee744da058dbc854bfb6f06823c741f34'), ('ISIC_0013835_downsampled', '10d52d1f399664095917204b8b97097e9dc46fa38e9313c6361f3dd9ea0318ab'), ('ISIC_0014212_downsampled', '84157ba91b5848b73bddc6a7871d664098bad3eb371e040333a8efba44b9b54f'), ('ISIC_0024482', '0706ceeeb06fab1823700076ea345911f85131163b622860410a6be69d5fe7e0'), ('ISIC_0024496', 'a781b2b9d0e6307c3a27258a542ad85ccde75c12bf0f6848534ae03dcd9c4847'), ('ISIC_0024978', 'aa4af0e20c230189379e875bdd50c80d0c92c7f23037d308800601518c56a06e'), ('ISIC_0025005', '065a0aec3d5c65e16597d6cc4dcd35e4f862bacf110837ce533023eb612991a3'), ('ISIC_0025100', '06ac012658b1b345b7ed0bed789e6659967a3b67d54862744542a08f130b0018'), ('ISIC_0025346', 'd261c592fc9d3ff59d454004d7048d775886005938149ccb0d5aad915e54ada5'), ('ISIC_0025395', '855ec202869b2645d213a383c63c7fb65c964deaf7e266730c392e8c88ca2e29')]

In [33]:
# ================================================================
# CELL 4V-MAPCHECK — RESOLVE MANIFEST IMAGE_KEY -> NPZ STEM
# ================================================================

print("=" * 70)
print("T1.4b — RESOLVING NULL-MASK FILE IDENTIFIERS")
print("=" * 70)

manifest = pd.read_csv(MANIFEST_PATH)
sample_df = pd.read_csv(SAMPLE_CSV)

retained_ids = sorted(
    manifest["image"].unique()
)

# ------------------------------------------------
# Actual NPZ stems
# ------------------------------------------------
actual_npz = sorted(
    p.stem
    for p in OUT_MASK_DIR.glob("*.npz")
)

actual_A_stems = sorted(
    p.stem[:-2]          # remove "_A"
    for p in OUT_MASK_DIR.glob("*_A.npz")
)

actual_C_stems = sorted(
    p.stem[:-7]          # remove "_Cprime"
    for p in OUT_MASK_DIR.glob("*_Cprime.npz")
)

print(f"Actual A files      : {len(actual_A_stems)}")
print(f"Actual C' files     : {len(actual_C_stems)}")

assert len(actual_A_stems) == 99
assert len(actual_C_stems) == 99

# ------------------------------------------------
# Manifest image_key lookup
# ------------------------------------------------
sample_map = (
    sample_df[
        ["image", "image_key"]
    ]
    .drop_duplicates("image")
)

assert len(sample_map) == 100

image_to_key = dict(
    zip(
        sample_map["image"],
        sample_map["image_key"]
    )
)

# ------------------------------------------------
# Print first 20 mappings
# ------------------------------------------------
print("\nFirst 20 manifest -> actual comparisons:")

for image_id in retained_ids[:20]:

    key = str(image_to_key[image_id])

    matches = [
        stem
        for stem in actual_A_stems
        if key.startswith(stem)
        or stem.startswith(key)
    ]

    print(
        f"{image_id:<32} "
        f"key={key[:24]}... "
        f"matches={matches[:3]}"
    )

# ------------------------------------------------
# Determine exact relationship
# ------------------------------------------------
print("\nChecking prefix relationship...")

prefix_matches = []

for image_id in retained_ids:

    key = str(image_to_key[image_id])

    matches = [
        stem
        for stem in actual_A_stems
        if key.startswith(stem)
    ]

    prefix_matches.append(
        (image_id, key, matches)
    )

n_prefix = sum(
    len(matches) == 1
    for _, _, matches in prefix_matches
)

n_zero = sum(
    len(matches) == 0
    for _, _, matches in prefix_matches
)

n_multi = sum(
    len(matches) > 1
    for _, _, matches in prefix_matches
)

print(f"\nUnique prefix matches : {n_prefix}")
print(f"No prefix matches     : {n_zero}")
print(f"Multiple matches      : {n_multi}")

# ------------------------------------------------
# Show unresolved examples
# ------------------------------------------------
unresolved = [
    (image_id, key, matches)
    for image_id, key, matches in prefix_matches
    if len(matches) != 1
]

if unresolved:

    print("\nUnresolved examples:")

    for image_id, key, matches in unresolved[:20]:

        print(
            f"\nimage    : {image_id}"
            f"\nimage_key: {key}"
            f"\nmatches  : {matches}"
        )

# ------------------------------------------------
# Also test reverse relationship:
# actual 16-char stem inside manifest image_key
# ------------------------------------------------
print("\nReverse lookup test...")

reverse_matches = []

for stem in actual_A_stems:

    matches = [
        image_id
        for image_id in retained_ids
        if stem in str(image_to_key[image_id])
    ]

    reverse_matches.append(
        (stem, matches)
    )

reverse_unique = sum(
    len(matches) == 1
    for _, matches in reverse_matches
)

reverse_zero = sum(
    len(matches) == 0
    for _, matches in reverse_matches
)

reverse_multi = sum(
    len(matches) > 1
    for _, matches in reverse_matches
)

print(f"Unique reverse matches : {reverse_unique}")
print(f"No reverse matches     : {reverse_zero}")
print(f"Multiple reverse matches: {reverse_multi}")

print("\n" + "=" * 70)
print("CELL 4V-MAPCHECK COMPLETE")
print("=" * 70)

T1.4b — RESOLVING NULL-MASK FILE IDENTIFIERS
Actual A files      : 99
Actual C' files     : 99

First 20 manifest -> actual comparisons:
ISIC_0013637_downsampled         key=340db516a73b627d3ffdbf56... matches=[]
ISIC_0013835_downsampled         key=10d52d1f399664095917204b... matches=[]
ISIC_0014212_downsampled         key=84157ba91b5848b73bddc6a7... matches=[]
ISIC_0024482                     key=0706ceeeb06fab1823700076... matches=[]
ISIC_0024496                     key=a781b2b9d0e6307c3a27258a... matches=[]
ISIC_0024978                     key=aa4af0e20c230189379e875b... matches=[]
ISIC_0025005                     key=065a0aec3d5c65e16597d6cc... matches=[]
ISIC_0025100                     key=06ac012658b1b345b7ed0bed... matches=[]
ISIC_0025346                     key=d261c592fc9d3ff59d454004... matches=[]
ISIC_0025395                     key=855ec202869b2645d213a383... matches=[]
ISIC_0025825                     key=bec33d18644df4e52a5d2ac8... matches=[]
ISIC_0026121               

In [34]:
# ================================================================
# CELL 4V-MAPCHECK2 — IDENTIFY EXACT NPZ FILE KEY
# ================================================================

print("=" * 70)
print("T1.4b — IDENTIFYING EXACT NULL-MASK FILE KEY")
print("=" * 70)

manifest = pd.read_csv(MANIFEST_PATH)
sample_df = pd.read_csv(SAMPLE_CSV)

retained_ids = sorted(
    manifest["image"].unique()
)

# ------------------------------------------------
# Actual saved NPZ stems
# ------------------------------------------------
actual_A_stems = sorted(
    p.name[:-len("_A.npz")]
    for p in OUT_MASK_DIR.glob("*_A.npz")
)

actual_C_stems = sorted(
    p.name[:-len("_Cprime.npz")]
    for p in OUT_MASK_DIR.glob("*_Cprime.npz")
)

assert len(actual_A_stems) == 99
assert len(actual_C_stems) == 99

actual_stem_set = set(actual_A_stems)

# ------------------------------------------------
# Inspect sample columns
# ------------------------------------------------
print("\nSample columns:")
print(list(sample_df.columns))

# ------------------------------------------------
# Candidate identifier columns
# ------------------------------------------------
candidate_cols = [
    c for c in [
        "image",
        "image_key",
        "lesion_key",
        "row",
        "class_name",
        "lesion_id"
    ]
    if c in sample_df.columns
]

print("\nCandidate identifier columns:")
print(candidate_cols)

# ------------------------------------------------
# Test direct string relationships
# ------------------------------------------------
for col in candidate_cols:

    if col == "row":
        continue

    values = (
        sample_df
        .set_index("image")
        .loc[retained_ids, col]
        .astype(str)
    )

    exact = sum(
        v in actual_stem_set
        for v in values
    )

    prefix = sum(
        any(
            v.startswith(stem)
            for stem in actual_A_stems
        )
        for v in values
    )

    suffix = sum(
        any(
            stem.startswith(v)
            for stem in actual_A_stems
        )
        for v in values
    )

    first16 = sum(
        v[:16] in actual_stem_set
        for v in values
    )

    last16 = sum(
        v[-16:] in actual_stem_set
        for v in values
    )

    print(
        f"\n{col}:"
        f"\n  exact       = {exact}/99"
        f"\n  value-prefix= {prefix}/99"
        f"\n  stem-prefix = {suffix}/99"
        f"\n  first16     = {first16}/99"
        f"\n  last16      = {last16}/99"
    )

# ------------------------------------------------
# Print first 10 retained rows with all identifiers
# ------------------------------------------------
print("\n" + "-" * 70)
print("FIRST 10 RETAINED IDENTIFIER ROWS")
print("-" * 70)

display_cols = [
    c for c in [
        "image",
        "lesion_id",
        "lesion_key",
        "image_key",
        "row"
    ]
    if c in sample_df.columns
]

print(
    sample_df[
        sample_df["image"].isin(retained_ids)
    ][display_cols]
    .head(10)
    .to_string(index=False)
)

# ------------------------------------------------
# Print actual stems
# ------------------------------------------------
print("\n" + "-" * 70)
print("FIRST 20 ACTUAL NPZ STEMS")
print("-" * 70)

for stem in actual_A_stems[:20]:
    print(stem)

print("\n" + "=" * 70)
print("CELL 4V-MAPCHECK2 COMPLETE")
print("=" * 70)

T1.4b — IDENTIFYING EXACT NULL-MASK FILE KEY

Sample columns:
['image', 'lesion_id', 'class_name', 'malignant', 'row', 'mask_area', 'lesion_key', 'image_key']

Candidate identifier columns:
['image', 'image_key', 'lesion_key', 'row', 'class_name', 'lesion_id']


KeyError: 'image'

In [35]:
# ================================================================
# CELL 4V-MAPCHECK3 — INSPECT PRODUCTION WRITER NAMING
# ================================================================

import inspect

print("=" * 70)
print("T1.4b — INSPECTING PRODUCTION NULL WRITER")
print("=" * 70)

# ------------------------------------------------
# Check whether generate_design is still available
# ------------------------------------------------
print(
    "generate_design available:",
    "generate_design" in globals()
)

# ------------------------------------------------
# Inspect production output variables
# ------------------------------------------------
for name in [
    "OUT_MASK_DIR",
    "MANIFEST_PATH",
    "DROPPED_PATH",
    "PROD_SUMMARY_PATH"
]:
    print(
        f"{name}:",
        globals().get(name, "<NOT DEFINED>")
    )

# ------------------------------------------------
# Inspect generate_design source
# ------------------------------------------------
if "generate_design" in globals():

    print("\n" + "-" * 70)
    print("generate_design SOURCE")
    print("-" * 70)

    try:
        src = inspect.getsource(generate_design)
        print(src)
    except Exception as e:
        print(
            "Could not inspect generate_design:",
            repr(e)
        )

# ------------------------------------------------
# Inspect variables that may contain the production
# save loop / accepted metadata
# ------------------------------------------------
interesting = [
    name for name in globals()
    if any(
        token in name.lower()
        for token in [
            "result_a",
            "result_c",
            "mask_dir",
            "manifest",
            "image_key",
            "save"
        ]
    )
]

print("\n" + "-" * 70)
print("RELEVANT LIVE VARIABLES")
print("-" * 70)

for name in sorted(interesting):
    obj = globals()[name]

    if isinstance(obj, (str, int, float, Path)):
        print(
            f"{name}: {obj}"
        )
    elif isinstance(obj, pd.DataFrame):
        print(
            f"{name}: DataFrame shape={obj.shape}, "
            f"columns={list(obj.columns)}"
        )
    else:
        print(
            f"{name}: {type(obj).__name__}"
        )

print("\n" + "=" * 70)
print("CELL 4V-MAPCHECK3 COMPLETE")
print("=" * 70)

T1.4b — INSPECTING PRODUCTION NULL WRITER
generate_design available: True
OUT_MASK_DIR: /kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_MASKS
MANIFEST_PATH: /kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_MASK_MANIFEST.csv
DROPPED_PATH: /kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_DROPPED_IMAGES.csv
PROD_SUMMARY_PATH: /kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_GENERATION_SUMMARY.json

----------------------------------------------------------------------
generate_design SOURCE
----------------------------------------------------------------------
def generate_design(
    image_id,
    design
):

    true_mask = get_true_mask(
        image_id
    )

    coords = get_coords(
        true_mask
    )

    true_area = int(
        true_mask.sum()
    )

    true_flat = np.flatnonzero(
        true_mask.ravel()
    ).astype(np.int64)

    true_cd = center_distance(
        coords
    )

    accepted_masks = []
    accepted_m

In [36]:
# ================================================================
# CELL 4V-MAPCHECK4 — INSPECT ACTUAL NPZ SAVE FUNCTION
# ================================================================

import inspect

print("=" * 70)
print("T1.4b — INSPECTING ACTUAL NPZ SAVE FUNCTION")
print("=" * 70)

assert "save_record" in globals(), (
    "save_record is not defined in the current kernel"
)

print("\n" + "-" * 70)
print("save_record SOURCE")
print("-" * 70)

try:
    src = inspect.getsource(save_record)
    print(src)
except Exception as e:
    print(
        "Could not inspect save_record:",
        repr(e)
    )

print("\n" + "=" * 70)
print("CELL 4V-MAPCHECK4 COMPLETE")
print("=" * 70)

T1.4b — INSPECTING ACTUAL NPZ SAVE FUNCTION

----------------------------------------------------------------------
save_record SOURCE
----------------------------------------------------------------------
def save_record(name, obj):
    with open(RECORDS / name, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)


CELL 4V-MAPCHECK4 COMPLETE


In [37]:
# ================================================================
# CELL 4V-MAPCHECK5 — RECOVER PRODUCTION FILE ORDER
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("T1.4b — RECOVERING PRODUCTION NULL-FILE ORDER")
print("=" * 70)

# ------------------------------------------------
# Load manifest
# ------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

# The production loop generated images in this order
manifest_image_order = list(
    dict.fromkeys(manifest["image"].tolist())
)

print(
    f"Manifest retained-image order : "
    f"{len(manifest_image_order)}"
)

assert len(manifest_image_order) == 99

# ------------------------------------------------
# Actual NPZ files
# ------------------------------------------------
A_files = list(
    OUT_MASK_DIR.glob("*_A.npz")
)

C_files = list(
    OUT_MASK_DIR.glob("*_Cprime.npz")
)

assert len(A_files) == 99
assert len(C_files) == 99

# ------------------------------------------------
# File timestamps
# ------------------------------------------------
A_by_time = sorted(
    A_files,
    key=lambda p: p.stat().st_mtime_ns
)

C_by_time = sorted(
    C_files,
    key=lambda p: p.stat().st_mtime_ns
)

# ------------------------------------------------
# Check timestamp uniqueness
# ------------------------------------------------
A_times = [
    p.stat().st_mtime_ns
    for p in A_by_time
]

C_times = [
    p.stat().st_mtime_ns
    for p in C_by_time
]

print(
    f"Unique A timestamps : "
    f"{len(set(A_times))}/99"
)

print(
    f"Unique C' timestamps: "
    f"{len(set(C_times))}/99"
)

# ------------------------------------------------
# Print first 20 in filesystem creation order
# ------------------------------------------------
print("\nFirst 20 A files by timestamp:")

for i, p in enumerate(A_by_time[:20]):
    print(
        f"{i:03d}  "
        f"{p.name}  "
        f"{p.stat().st_mtime_ns}"
    )

print("\nFirst 20 C' files by timestamp:")

for i, p in enumerate(C_by_time[:20]):
    print(
        f"{i:03d}  "
        f"{p.name}  "
        f"{p.stat().st_mtime_ns}"
    )

# ------------------------------------------------
# Pair A/C by timestamp rank
# ------------------------------------------------
paired = []

for i in range(99):

    a_stem = A_by_time[i].name[
        :-len("_A.npz")
    ]

    c_stem = C_by_time[i].name[
        :-len("_Cprime.npz")
    ]

    paired.append({
        "production_index": i,
        "image": manifest_image_order[i],
        "A_stem": a_stem,
        "Cprime_stem": c_stem,
        "same_stem": a_stem == c_stem,
        "A_time": A_times[i],
        "C_time": C_times[i],
    })

pair_df = pd.DataFrame(paired)

print("\nA/C same-stem pairs:")
print(
    pair_df["same_stem"]
    .value_counts()
    .to_string()
)

assert pair_df["same_stem"].all()

# ------------------------------------------------
# Save provisional mapping for inspection only
# ------------------------------------------------
MAPCHECK_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_FILENAME_ORDER_DIAGNOSTIC.csv"
)

pair_df.to_csv(
    MAPCHECK_PATH,
    index=False
)

# ------------------------------------------------
# Show mapping
# ------------------------------------------------
print("\nFirst 20 recovered mappings:")

print(
    pair_df[
        [
            "production_index",
            "image",
            "A_stem",
            "Cprime_stem"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nDiagnostic written to:")
print(MAPCHECK_PATH)

print("\n" + "=" * 70)
print("CELL 4V-MAPCHECK5 COMPLETE")
print("=" * 70)

T1.4b — RECOVERING PRODUCTION NULL-FILE ORDER
Manifest retained-image order : 99
Unique A timestamps : 99/99
Unique C' timestamps: 99/99

First 20 A files by timestamp:
000  cf16923d30fedadf_A.npz  1790365131386297458
001  8503feeeb355a7c2_A.npz  1790365135021552131
002  13f608f44e585625_A.npz  1790365140082906712
003  671f2c5c10464648_A.npz  1790365141540008792
004  7dab21b90f21d7cb_A.npz  1790365142396068764
005  f17a7aef2cb8acc0_A.npz  1790365143584151997
006  a072efa0354b103f_A.npz  1790365146881382989
007  549f5d9309ee78c1_A.npz  1790365155732003033
008  98d52fea09f8ba79_A.npz  1790365156433052146
009  7fa1d6dbe7a6b66b_A.npz  1790365157468124660
010  05d18d9c48b90a67_A.npz  1790365159383258828
011  ffbf20c3a80408df_A.npz  1790365160337325666
012  1df7ba3bd8778bfd_A.npz  1790365161273391244
013  327b1a0e8e859ec5_A.npz  1790365161765425714
014  0c3a8c6f723d48e6_A.npz  1790365164860642554
015  2a7b6805c2c4b9d8_A.npz  1790365167516828637
016  111ed3696632b901_A.npz  179036517051103840

In [38]:
# ================================================================
# CELL 4V-MAPCHECK6 — INSPECT NPZ CONTENTS
# ================================================================

import numpy as np
from pathlib import Path

print("=" * 70)
print("T1.4b — INSPECTING SAVED NULL NPZ CONTENTS")
print("=" * 70)

# ------------------------------------------------
# Use the first recovered A/C pair
# ------------------------------------------------
a_path = A_by_time[0]
c_path = C_by_time[0]

print("\nA file:")
print(a_path)

print("\nC' file:")
print(c_path)

# ------------------------------------------------
# Inspect A
# ------------------------------------------------
print("\n" + "-" * 70)
print("A NPZ")
print("-" * 70)

with np.load(a_path, allow_pickle=True) as z:

    print("Keys:")
    print(list(z.keys()))

    for key in z.keys():
        arr = z[key]

        print(
            f"\n{key}:"
            f"\n  type  = {type(arr)}"
            f"\n  shape = {getattr(arr, 'shape', None)}"
            f"\n  dtype = {getattr(arr, 'dtype', None)}"
        )

        # Scalars / small metadata objects
        if np.ndim(arr) == 0:
            try:
                print("  value =", arr.item())
            except Exception:
                print("  value =", arr)

        elif np.size(arr) <= 20:
            try:
                print("  value =", arr)
            except Exception:
                pass

# ------------------------------------------------
# Inspect C'
# ------------------------------------------------
print("\n" + "-" * 70)
print("C' NPZ")
print("-" * 70)

with np.load(c_path, allow_pickle=True) as z:

    print("Keys:")
    print(list(z.keys()))

    for key in z.keys():
        arr = z[key]

        print(
            f"\n{key}:"
            f"\n  type  = {type(arr)}"
            f"\n  shape = {getattr(arr, 'shape', None)}"
            f"\n  dtype = {getattr(arr, 'dtype', None)}"
        )

        if np.ndim(arr) == 0:
            try:
                print("  value =", arr.item())
            except Exception:
                print("  value =", arr)

        elif np.size(arr) <= 20:
            try:
                print("  value =", arr)
            except Exception:
                pass

# ------------------------------------------------
# Compare file-level metadata if available
# ------------------------------------------------
print("\n" + "=" * 70)
print("EXPECTED FIRST MAPPING")
print("=" * 70)

print("Production index :", pair_df.iloc[0]["production_index"])
print("Manifest image   :", pair_df.iloc[0]["image"])
print("A stem           :", pair_df.iloc[0]["A_stem"])
print("C' stem          :", pair_df.iloc[0]["Cprime_stem"])

print("\n" + "=" * 70)
print("CELL 4V-MAPCHECK6 COMPLETE")
print("=" * 70)

T1.4b — INSPECTING SAVED NULL NPZ CONTENTS

A file:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_MASKS/cf16923d30fedadf_A.npz

C' file:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_NULL_MASKS/cf16923d30fedadf_Cprime.npz

----------------------------------------------------------------------
A NPZ
----------------------------------------------------------------------
Keys:
['masks']

masks:
  type  = <class 'numpy.ndarray'>
  shape = (50, 224, 224)
  dtype = uint8

----------------------------------------------------------------------
C' NPZ
----------------------------------------------------------------------
Keys:
['masks']

masks:
  type  = <class 'numpy.ndarray'>
  shape = (50, 224, 224)
  dtype = uint8

EXPECTED FIRST MAPPING
Production index : 0
Manifest image   : ISIC_0013637_downsampled
A stem           : cf16923d30fedadf
C' stem          : cf16923d30fedadf

CELL 4V-MAPCHECK6 COMPLETE


In [39]:
# ================================================================
# CELL 4V-FINALMAPAUDIT — BIT-EXACT NPZ ↔ IMAGE VALIDATION
# ================================================================

import time
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — BIT-EXACT NULL FILE / IMAGE MAPPING AUDIT")
print("=" * 70)

# ------------------------------------------------
# Preconditions
# ------------------------------------------------
assert "generate_design" in globals(), (
    "generate_design is not available"
)

assert "pair_df" in globals(), (
    "pair_df is not available; run MAPCHECK5 first"
)

assert "manifest" in globals(), (
    "manifest is not available"
)

assert "true_mask_lookup" in globals(), (
    "true_mask_lookup is not available"
)

assert len(pair_df) == 99

# ------------------------------------------------
# Manifest image order
# ------------------------------------------------
image_order = (
    pair_df
    .sort_values("production_index")
    ["image"]
    .tolist()
)

assert len(image_order) == 99
assert len(set(image_order)) == 99

# ------------------------------------------------
# Audit
# ------------------------------------------------
audit_rows = []

total_masks_checked = 0
total_exact_matches = 0
total_failures = 0

t0 = time.time()

for idx, row in pair_df.sort_values(
    "production_index"
).iterrows():

    production_index = int(
        row["production_index"]
    )

    image_id = str(row["image"])

    a_path = (
        OUT_MASK_DIR /
        f"{row['A_stem']}_A.npz"
    )

    c_path = (
        OUT_MASK_DIR /
        f"{row['Cprime_stem']}_Cprime.npz"
    )

    assert a_path.exists(), a_path
    assert c_path.exists(), c_path

    # ------------------------------------------------
    # Load saved NPZs
    # ------------------------------------------------
    with np.load(a_path, allow_pickle=False) as z:
        saved_A = z["masks"]

    with np.load(c_path, allow_pickle=False) as z:
        saved_C = z["masks"]

    # ------------------------------------------------
    # Structural checks
    # ------------------------------------------------
    assert saved_A.shape == (50, 224, 224), (
        image_id,
        saved_A.shape
    )

    assert saved_C.shape == (50, 224, 224), (
        image_id,
        saved_C.shape
    )

    assert saved_A.dtype == np.uint8
    assert saved_C.dtype == np.uint8

    assert np.all(
        np.isin(saved_A, [0, 1])
    )

    assert np.all(
        np.isin(saved_C, [0, 1])
    )

    # ------------------------------------------------
    # Deterministically regenerate exactly what
    # production generation produced
    # ------------------------------------------------
    regenerated_A = generate_design(
        image_id,
        "A"
    )

    regenerated_C = generate_design(
        image_id,
        "Cprime"
    )

    assert regenerated_A["success"], (
        "A regeneration failed",
        image_id
    )

    assert regenerated_C["success"], (
        "C' regeneration failed",
        image_id
    )

    regen_A = np.stack(
        regenerated_A["accepted_masks"]
    ).astype(np.uint8)

    regen_C = np.stack(
        regenerated_C["accepted_masks"]
    ).astype(np.uint8)

    assert regen_A.shape == (50, 224, 224)
    assert regen_C.shape == (50, 224, 224)

    # ------------------------------------------------
    # BIT-EXACT comparison
    # ------------------------------------------------
    exact_A = np.array_equal(
        saved_A,
        regen_A
    )

    exact_C = np.array_equal(
        saved_C,
        regen_C
    )

    # ------------------------------------------------
    # Additional true-mask geometry checks
    # ------------------------------------------------
    true_mask = np.asarray(
        true_mask_lookup[image_id],
        dtype=bool
    )

    true_area = int(
        true_mask.sum()
    )

    A_areas = saved_A.sum(
        axis=(1, 2)
    )

    C_areas = saved_C.sum(
        axis=(1, 2)
    )

    A_overlap = np.sum(
        saved_A.astype(bool)
        & true_mask,
        axis=(1, 2)
    )

    C_overlap = np.sum(
        saved_C.astype(bool)
        & true_mask,
        axis=(1, 2)
    )

    area_A_ok = np.all(
        A_areas == true_area
    )

    area_C_ok = np.all(
        C_areas == true_area
    )

    overlap_A_ok = np.all(
        A_overlap == 0
    )

    overlap_C_ok = np.all(
        C_overlap == 0
    )

    # ------------------------------------------------
    # Count
    # ------------------------------------------------
    total_masks_checked += 100

    if exact_A:
        total_exact_matches += 50
    else:
        total_failures += 1

    if exact_C:
        total_exact_matches += 50
    else:
        total_failures += 1

    audit_rows.append({
        "production_index":
            production_index,

        "image":
            image_id,

        "A_stem":
            row["A_stem"],

        "Cprime_stem":
            row["Cprime_stem"],

        "A_exact_match":
            exact_A,

        "Cprime_exact_match":
            exact_C,

        "A_area_ok":
            area_A_ok,

        "Cprime_area_ok":
            area_C_ok,

        "A_zero_overlap_ok":
            overlap_A_ok,

        "Cprime_zero_overlap_ok":
            overlap_C_ok,

        "true_area":
            true_area,

        "A_min_area":
            int(A_areas.min()),

        "A_max_area":
            int(A_areas.max()),

        "Cprime_min_area":
            int(C_areas.min()),

        "Cprime_max_area":
            int(C_areas.max()),

        "A_max_overlap":
            int(A_overlap.max()),

        "Cprime_max_overlap":
            int(C_overlap.max()),
    })

    # ------------------------------------------------
    # Progress
    # ------------------------------------------------
    if (
        (production_index + 1) % 10 == 0
        or production_index == 98
    ):
        print(
            f"[{production_index + 1:3d}/99] "
            f"{image_id:<32} "
            f"A={'PASS' if exact_A else 'FAIL'} "
            f"C'={'PASS' if exact_C else 'FAIL'}"
        )

# ------------------------------------------------
# Build audit table
# ------------------------------------------------
map_audit_df = pd.DataFrame(
    audit_rows
)

MAP_AUDIT_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

map_audit_df.to_csv(
    MAP_AUDIT_PATH,
    index=False
)

# ------------------------------------------------
# Final assertions
# ------------------------------------------------
print("\n" + "=" * 70)
print("FINAL BIT-EXACT VALIDATION")
print("=" * 70)

print(
    "Images checked             :",
    len(map_audit_df)
)

print(
    "Masks checked              :",
    total_masks_checked
)

print(
    "Expected masks             :",
    99 * 100
)

print(
    "A exact image matches      :",
    int(map_audit_df["A_exact_match"].sum()),
    "/ 99"
)

print(
    "C' exact image matches     :",
    int(map_audit_df["Cprime_exact_match"].sum()),
    "/ 99"
)

print(
    "A area checks passed       :",
    int(map_audit_df["A_area_ok"].sum()),
    "/ 99"
)

print(
    "C' area checks passed      :",
    int(map_audit_df["Cprime_area_ok"].sum()),
    "/ 99"
)

print(
    "A zero-overlap checks      :",
    int(map_audit_df["A_zero_overlap_ok"].sum()),
    "/ 99"
)

print(
    "C' zero-overlap checks     :",
    int(map_audit_df["Cprime_zero_overlap_ok"].sum()),
    "/ 99"
)

assert map_audit_df["A_exact_match"].all()
assert map_audit_df["Cprime_exact_match"].all()

assert map_audit_df["A_area_ok"].all()
assert map_audit_df["Cprime_area_ok"].all()

assert map_audit_df["A_zero_overlap_ok"].all()
assert map_audit_df["Cprime_zero_overlap_ok"].all()

print(
    "\nBit-exact A mapping        : PASS"
)

print(
    "Bit-exact C' mapping       : PASS"
)

print(
    "True-mask area integrity   : PASS"
)

print(
    "Zero-overlap integrity     : PASS"
)

print(
    f"\nAudit saved to:\n{MAP_AUDIT_PATH}"
)

print(
    f"\nWall time: {time.time() - t0:.2f} sec"
)

print("\n" + "=" * 70)
print("CELL 4V-FINALMAPAUDIT COMPLETE")
print("=" * 70)

T1.4b — BIT-EXACT NULL FILE / IMAGE MAPPING AUDIT
[ 10/99] ISIC_0025395                     A=PASS C'=PASS
[ 20/99] ISIC_0027517                     A=PASS C'=PASS
[ 30/99] ISIC_0029598                     A=PASS C'=PASS
[ 40/99] ISIC_0031207                     A=PASS C'=PASS
[ 50/99] ISIC_0032660                     A=PASS C'=PASS
[ 60/99] ISIC_0054877                     A=PASS C'=PASS
[ 70/99] ISIC_0061269                     A=PASS C'=PASS
[ 80/99] ISIC_0065811                     A=PASS C'=PASS
[ 90/99] ISIC_0070353                     A=PASS C'=PASS
[ 99/99] ISIC_0072895                     A=PASS C'=PASS

FINAL BIT-EXACT VALIDATION
Images checked             : 99
Masks checked              : 9900
Expected masks             : 9900
A exact image matches      : 99 / 99
C' exact image matches     : 99 / 99
A area checks passed       : 99 / 99
C' area checks passed      : 99 / 99
A zero-overlap checks      : 99 / 99
C' zero-overlap checks     : 99 / 99

Bit-exact A mapping        : 

In [40]:
# ================================================================
# CELL 4W — FREEZE VALIDATED T1.4b NULL POPULATION
# ================================================================

import hashlib
import json
import time
from pathlib import Path

print("=" * 70)
print("T1.4b — FREEZING VALIDATED NULL POPULATION")
print("=" * 70)

# ------------------------------------------------
# Helper
# ------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


t0 = time.time()

# ------------------------------------------------
# Paths
# ------------------------------------------------
NULL_DIR = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null/"
    "T1_4B_NULL_MASKS"
)

MANIFEST_PATH = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null/"
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

DROPPED_PATH = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null/"
    "T1_4B_DROPPED_IMAGES.csv"
)

MAP_AUDIT_PATH = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null/"
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

LOCK_PATH = Path(
    "/kaggle/working/"
    "T1_4B_generator_matched_spatial_null/"
    "T1_4B_VALIDATED_NULL_POPULATION_LOCK.json"
)

# ------------------------------------------------
# Preconditions
# ------------------------------------------------
assert NULL_DIR.exists()
assert MANIFEST_PATH.exists()
assert DROPPED_PATH.exists()
assert MAP_AUDIT_PATH.exists()

assert "manifest" in globals()
assert "map_audit_df" in globals()

# ------------------------------------------------
# Reload from disk so the lock depends on the
# actual saved artifacts, not only live variables.
# ------------------------------------------------
manifest_disk = pd.read_csv(MANIFEST_PATH)
dropped_disk = pd.read_csv(DROPPED_PATH)
audit_disk = pd.read_csv(MAP_AUDIT_PATH)

# ------------------------------------------------
# Basic population checks
# ------------------------------------------------
retained_images = sorted(
    manifest_disk["image"].astype(str).unique()
)

dropped_images = sorted(
    dropped_disk["image"].astype(str).unique()
)

assert len(retained_images) == 99
assert len(dropped_images) == 1
assert set(retained_images).isdisjoint(
    set(dropped_images)
)

assert len(manifest_disk) == 9900

assert (
    manifest_disk["design"]
    .value_counts()
    .to_dict()
    == {
        "A": 4950,
        "Cprime": 4950
    }
)

# ------------------------------------------------
# Exactly 50 nulls per image/design
# ------------------------------------------------
counts = (
    manifest_disk
    .groupby(["image", "design"])
    .size()
)

assert counts.shape[0] == 99 * 2
assert counts.min() == 50
assert counts.max() == 50

# ------------------------------------------------
# Bit-exact mapping audit must be completely clean
# ------------------------------------------------
assert len(audit_disk) == 99

assert audit_disk["A_exact_match"].all()
assert audit_disk["Cprime_exact_match"].all()

assert audit_disk["A_area_ok"].all()
assert audit_disk["Cprime_area_ok"].all()

assert audit_disk["A_zero_overlap_ok"].all()
assert audit_disk["Cprime_zero_overlap_ok"].all()

# ------------------------------------------------
# File inventory
# ------------------------------------------------
A_files = sorted(NULL_DIR.glob("*_A.npz"))
C_files = sorted(NULL_DIR.glob("*_Cprime.npz"))

assert len(A_files) == 99
assert len(C_files) == 99

all_npz = sorted(A_files + C_files)

print("\nValidated file inventory:")
print("  Design A NPZs       :", len(A_files))
print("  Design C' NPZs      :", len(C_files))
print("  Total NPZs          :", len(all_npz))

# ------------------------------------------------
# Hash all production NPZ files
# ------------------------------------------------
print("\nHashing validated NPZ files...")

npz_hashes = {}

for i, path in enumerate(all_npz, start=1):

    npz_hashes[path.name] = sha256_file(path)

    if (
        i % 20 == 0
        or i == len(all_npz)
    ):
        print(
            f"  [{i:3d}/{len(all_npz)}] "
            f"{path.name}"
        )

# ------------------------------------------------
# Hash supporting artifacts
# ------------------------------------------------
manifest_hash = sha256_file(MANIFEST_PATH)
dropped_hash = sha256_file(DROPPED_PATH)
audit_hash = sha256_file(MAP_AUDIT_PATH)

# ------------------------------------------------
# Canonical retained/dropped population hashes
# ------------------------------------------------
retained_population_hash = sha256_bytes(
    "\n".join(retained_images).encode("utf-8")
)

dropped_population_hash = sha256_bytes(
    "\n".join(dropped_images).encode("utf-8")
)

# ------------------------------------------------
# Lock object
# ------------------------------------------------
lock = {
    "experiment": "T1_4B",

    "protocol": {
        "null_namespace": "t1_4b-null-v2",
        "design_primary": "A",
        "design_sensitivity": "Cprime",
        "nulls_per_design_per_image": 50,
        "candidate_attempt_cap": 20000,
        "all_or_nothing_image_retention": True,
        "exact_area_required": True,
        "zero_overlap_required": True,
        "border_to_border_descriptive_only": True,
    },

    "population": {
        "candidate_images": 100,
        "retained_images": 99,
        "dropped_images": 1,
        "retained_population_sha256":
            retained_population_hash,
        "dropped_population_sha256":
            dropped_population_hash,
    },

    "null_masks": {
        "design_A_files": 99,
        "design_Cprime_files": 99,
        "total_files": 198,
        "design_A_masks": 4950,
        "design_Cprime_masks": 4950,
        "total_masks": 9900,
    },

    "integrity": {
        "filename_mapping_bit_exact": True,
        "A_exact_matches": 99,
        "Cprime_exact_matches": 99,
        "A_area_checks": 99,
        "Cprime_area_checks": 99,
        "A_zero_overlap_checks": 99,
        "Cprime_zero_overlap_checks": 99,
    },

    "artifacts": {
        "manifest": str(MANIFEST_PATH),
        "manifest_sha256": manifest_hash,

        "dropped_images": str(DROPPED_PATH),
        "dropped_images_sha256": dropped_hash,

        "mapping_audit": str(MAP_AUDIT_PATH),
        "mapping_audit_sha256": audit_hash,
    },

    "npz_sha256": npz_hashes,

    "retained_images": retained_images,
    "dropped_images": dropped_images,

    "created_at_utc": time.strftime(
        "%Y-%m-%dT%H:%M:%SZ",
        time.gmtime()
    ),
}

# ------------------------------------------------
# Save lock
# ------------------------------------------------
with open(LOCK_PATH, "w") as f:
    json.dump(
        lock,
        f,
        indent=2,
        sort_keys=True
    )

# ------------------------------------------------
# Reload and verify lock
# ------------------------------------------------
with open(LOCK_PATH, "r") as f:
    lock_check = json.load(f)

assert lock_check["population"]["retained_images"] == 99
assert lock_check["population"]["dropped_images"] == 1
assert lock_check["null_masks"]["total_masks"] == 9900
assert lock_check["integrity"]["filename_mapping_bit_exact"] is True
assert len(lock_check["npz_sha256"]) == 198

print("\n" + "=" * 70)
print("T1.4b NULL POPULATION LOCK")
print("=" * 70)

print(
    "Retained images          :",
    lock_check["population"]["retained_images"]
)

print(
    "Dropped images           :",
    lock_check["population"]["dropped_images"]
)

print(
    "Design A masks           :",
    lock_check["null_masks"]["design_A_masks"]
)

print(
    "Design C' masks          :",
    lock_check["null_masks"]["design_Cprime_masks"]
)

print(
    "Total null masks         :",
    lock_check["null_masks"]["total_masks"]
)

print(
    "NPZ files hashed         :",
    len(lock_check["npz_sha256"])
)

print(
    "Bit-exact mapping        :",
    lock_check["integrity"]["filename_mapping_bit_exact"]
)

print(
    "\nLOCK FILE:"
)

print(LOCK_PATH)

print(
    f"\nWall time: {time.time() - t0:.2f} sec"
)

print("\n" + "=" * 70)
print("CELL 4W COMPLETE — NULL POPULATION FROZEN")
print("=" * 70)

T1.4b — FREEZING VALIDATED NULL POPULATION

Validated file inventory:
  Design A NPZs       : 99
  Design C' NPZs      : 99
  Total NPZs          : 198

Hashing validated NPZ files...
  [ 20/198] 111ed3696632b901_Cprime.npz
  [ 40/198] 2bcc558e6d75009b_Cprime.npz
  [ 60/198] 4666a17b7c7d3a58_Cprime.npz
  [ 80/198] 65a1d5b860e8833f_Cprime.npz
  [100/198] 7e8409f3228691a1_Cprime.npz
  [120/198] 9a7bd511960a97d0_Cprime.npz
  [140/198] b804bdbd4cbe0103_Cprime.npz
  [160/198] d7378ddbe9871c86_Cprime.npz
  [180/198] ec21717c35866f66_Cprime.npz
  [198/198] ffbf20c3a80408df_Cprime.npz

T1.4b NULL POPULATION LOCK
Retained images          : 99
Dropped images           : 1
Design A masks           : 4950
Design C' masks          : 4950
Total null masks         : 9900
NPZ files hashed         : 198
Bit-exact mapping        : True

LOCK FILE:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_VALIDATED_NULL_POPULATION_LOCK.json

Wall time: 0.06 sec

CELL 4W COMPLETE — NULL POPULATION FROZEN

In [41]:
# ================================================================
# CELL 5A — DISCOVER FROZEN 30-MODEL PRODUCTION CHECKPOINT ROSTER
# ================================================================

from pathlib import Path
import re
import json
import hashlib
import pandas as pd

print("=" * 70)
print("T1.4b — DISCOVERING FROZEN PRODUCTION MODEL CHECKPOINTS")
print("=" * 70)

# ------------------------------------------------
# Frozen production model specification
# ------------------------------------------------
RHO_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
REPS = [1, 2, 3, 4, 5]

EXPECTED_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in RHO_LEVELS
    for rep in REPS
]

assert len(EXPECTED_KEYS) == 30
assert len(set(EXPECTED_KEYS)) == 30

print("\nExpected canonical model keys:")
print(EXPECTED_KEYS)

# ------------------------------------------------
# Search frozen project input
# ------------------------------------------------
SEARCH_ROOTS = [
    Path(
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/isic-final-project"
    ),
    Path("/kaggle/input"),
]

checkpoint_extensions = {
    ".pt",
    ".pth",
    ".ckpt",
    ".bin"
}

candidates = []

seen_paths = set()

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in checkpoint_extensions:
            continue

        resolved = str(path.resolve())

        if resolved in seen_paths:
            continue

        seen_paths.add(resolved)

        candidates.append(path)

candidates = sorted(
    candidates,
    key=lambda p: str(p)
)

print("\nCheckpoint files discovered:")
print("  Total :", len(candidates))

for path in candidates:
    print(" ", path)

# ------------------------------------------------
# Try to identify canonical rho/rep identity
# from filename and nearby directory names.
#
# This is discovery only. We do NOT accept a
# mapping unless it can be explicitly validated.
# ------------------------------------------------
def possible_model_keys(path):

    text = " ".join([
        path.name,
        path.stem,
        path.parent.name,
        path.parent.parent.name
        if path.parent.parent else ""
    ])

    found = set()

    # Examples intended to catch:
    # 0.0_1
    # 0_1
    # rho0.0_rep1
    # rho_0.0_rep_1
    # etc.
    patterns = [
        r'(?<![\d.])'
        r'(0\.0|0\.2|0\.4|0\.6|0\.8|1\.0)'
        r'[_-]([1-5])'
        r'(?!\d)',

        r'rho[_-]?'
        r'(0\.0|0\.2|0\.4|0\.6|0\.8|1\.0)'
        r'[^0-9]+'
        r'(?:rep(?:licate)?[_-]?)?([1-5])',
    ]

    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE
        ):

            rho = float(match.group(1))
            rep = int(match.group(2))

            key = f"{rho:.1f}_{rep}"

            if key in EXPECTED_KEYS:
                found.add(key)

    return sorted(found)


discovery_rows = []

for path in candidates:

    keys = possible_model_keys(path)

    discovery_rows.append({
        "path": str(path),
        "filename": path.name,
        "possible_keys": ";".join(keys),
        "n_possible_keys": len(keys),
        "size_bytes": path.stat().st_size,
    })

checkpoint_discovery_df = pd.DataFrame(
    discovery_rows
)

print("\n" + "=" * 70)
print("MODEL-ID DISCOVERY")
print("=" * 70)

if len(checkpoint_discovery_df):

    identified = checkpoint_discovery_df[
        checkpoint_discovery_df["n_possible_keys"] > 0
    ]

    print(
        "Files with recognizable canonical IDs :",
        len(identified)
    )

    if len(identified):

        print("\nRecognized candidates:")

        for _, row in identified.iterrows():

            print(
                f"\n{row['filename']}"
                f"\n  possible key(s): "
                f"{row['possible_keys']}"
                f"\n  path: {row['path']}"
            )

else:

    print("No checkpoint files were discovered.")

# ------------------------------------------------
# Save discovery artifact
# ------------------------------------------------
DISCOVERY_PATH = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_CHECKPOINT_DISCOVERY.csv"
)

checkpoint_discovery_df.to_csv(
    DISCOVERY_PATH,
    index=False
)

print("\nDiscovery artifact:")
print(DISCOVERY_PATH)

# ------------------------------------------------
# Important: no roster is accepted yet.
# ------------------------------------------------
print("\n" + "=" * 70)
print("STATUS")
print("=" * 70)

print(
    "30-model roster accepted : NO"
)

print(
    "Reason                  : "
    "checkpoint identity must be explicitly "
    "validated before IG execution"
)

print(
    "\nNo model inference performed."
)

print(
    "No IG computation performed."
)

print("\n" + "=" * 70)
print("CELL 5A COMPLETE")
print("=" * 70)

T1.4b — DISCOVERING FROZEN PRODUCTION MODEL CHECKPOINTS

Expected canonical model keys:
['0.0_1', '0.0_2', '0.0_3', '0.0_4', '0.0_5', '0.2_1', '0.2_2', '0.2_3', '0.2_4', '0.2_5', '0.4_1', '0.4_2', '0.4_3', '0.4_4', '0.4_5', '0.6_1', '0.6_2', '0.6_3', '0.6_4', '0.6_5', '0.8_1', '0.8_2', '0.8_3', '0.8_4', '0.8_5', '1.0_1', '1.0_2', '1.0_3', '1.0_4', '1.0_5']

Checkpoint files discovered:
  Total : 120
  /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_1/best_eval_ba.pt
  /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_1/final_epoch15.pt
  /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_2/best_eval_ba.pt
  /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_2/final_epoch15.pt
  /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysi

In [ ]:
## T1.4b — Resolve and Validate the Production Checkpoint Roster

Two complete copies of the 30-model training population were discovered:

1. `isic-kaggle-analysis/trained_models_readonly`
2. `ml-open-lab-30m-trained`

Each contains:
- `best_eval_ba.pt`
- `final_epoch15.pt`

This stage determines which exact checkpoint population is the frozen production inference population.

No model inference or IG computation is performed until the roster is resolved.

In [42]:
# ================================================================
# CELL 5B — RESOLVE FROZEN PRODUCTION CHECKPOINT ROSTER
# ================================================================

from pathlib import Path
import hashlib
import json
import re
import torch
import pandas as pd

print("=" * 70)
print("T1.4b — RESOLVING FROZEN PRODUCTION CHECKPOINT ROSTER")
print("=" * 70)

# ------------------------------------------------
# Expected canonical population
# ------------------------------------------------
RHO_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
REPS = [1, 2, 3, 4, 5]

EXPECTED_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in RHO_LEVELS
    for rep in REPS
]

assert len(EXPECTED_KEYS) == 30

# ------------------------------------------------
# Two discovered roots
# ------------------------------------------------
ROOT_A = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/isic-kaggle-analysis/"
    "trained_models_readonly/training_runs"
)

ROOT_B = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/ml-open-lab-30m-trained/"
    "training_runs"
)

ROOTS = {
    "isic_kaggle_analysis": ROOT_A,
    "ml_open_lab_30m": ROOT_B,
}

# ------------------------------------------------
# Helpers
# ------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def checkpoint_path(root, key, filename):
    rho, rep = key.split("_")
    return (
        root /
        f"rho_{rho}_rep_{rep}" /
        filename
    )


# ------------------------------------------------
# Verify complete populations
# ------------------------------------------------
print("\nPopulation completeness")
print("-" * 70)

population_rows = []

for root_name, root in ROOTS.items():

    print(f"\n{root_name}")
    print(root)

    assert root.exists(), (
        f"Missing checkpoint root: {root}"
    )

    for filename in [
        "best_eval_ba.pt",
        "final_epoch15.pt"
    ]:

        paths = [
            checkpoint_path(
                root,
                key,
                filename
            )
            for key in EXPECTED_KEYS
        ]

        missing = [
            str(p)
            for p in paths
            if not p.exists()
        ]

        sizes = [
            p.stat().st_size
            for p in paths
            if p.exists()
        ]

        print(
            f"  {filename:<18} "
            f"present={len(paths)-len(missing):2d}/30 "
            f"missing={len(missing):2d}"
        )

        if missing:
            for p in missing:
                print("    MISSING:", p)

        assert not missing

        assert all(
            s > 0
            for s in sizes
        )

        population_rows.append({
            "root": root_name,
            "checkpoint_type": filename,
            "n_present": len(paths),
            "n_missing": len(missing),
            "total_bytes": int(sum(sizes)),
        })

# ------------------------------------------------
# Compare the two roots bit-for-bit
# ------------------------------------------------
print("\n" + "=" * 70)
print("CROSS-ROOT CHECKPOINT COMPARISON")
print("=" * 70)

comparison_rows = []

for key in EXPECTED_KEYS:

    for filename in [
        "best_eval_ba.pt",
        "final_epoch15.pt"
    ]:

        pA = checkpoint_path(
            ROOT_A,
            key,
            filename
        )

        pB = checkpoint_path(
            ROOT_B,
            key,
            filename
        )

        hash_A = sha256_file(pA)
        hash_B = sha256_file(pB)

        comparison_rows.append({
            "model_key": key,
            "checkpoint_type": filename,
            "path_A": str(pA),
            "path_B": str(pB),
            "size_A": pA.stat().st_size,
            "size_B": pB.stat().st_size,
            "sha256_A": hash_A,
            "sha256_B": hash_B,
            "bit_exact": hash_A == hash_B,
        })

comparison_df = pd.DataFrame(
    comparison_rows
)

for checkpoint_type in [
    "best_eval_ba.pt",
    "final_epoch15.pt"
]:

    sub = comparison_df[
        comparison_df["checkpoint_type"]
        == checkpoint_type
    ]

    print(
        f"\n{checkpoint_type}:"
    )

    print(
        "  Bit-exact matches:",
        int(sub["bit_exact"].sum()),
        "/ 30"
    )

    print(
        "  Different files  :",
        int((~sub["bit_exact"]).sum()),
        "/ 30"
    )

# ------------------------------------------------
# Inspect one representative checkpoint from each
# population/type.
#
# This is metadata inspection only.
# ------------------------------------------------
print("\n" + "=" * 70)
print("CHECKPOINT CONTENT INSPECTION")
print("=" * 70)

inspection_rows = []

for root_name, root in ROOTS.items():

    for checkpoint_type in [
        "best_eval_ba.pt",
        "final_epoch15.pt"
    ]:

        path = checkpoint_path(
            root,
            "0.0_1",
            checkpoint_type
        )

        print(
            f"\n{root_name} / "
            f"{checkpoint_type}"
        )

        obj = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        print(
            "  Python type:",
            type(obj)
        )

        if isinstance(obj, dict):

            keys = list(obj.keys())

            print(
                "  Top-level keys:",
                keys
            )

            # Print scalar metadata only.
            scalar_metadata = {}

            for k, v in obj.items():

                if isinstance(
                    v,
                    (
                        int,
                        float,
                        str,
                        bool
                    )
                ):
                    scalar_metadata[k] = v

                elif isinstance(
                    v,
                    torch.Tensor
                ) and v.ndim == 0:
                    scalar_metadata[k] = (
                        v.detach()
                        .cpu()
                        .item()
                    )

            if scalar_metadata:
                print(
                    "  Scalar metadata:",
                    scalar_metadata
                )

            state_candidates = [
                k
                for k in keys
                if (
                    "state" in k.lower()
                    or "model" in k.lower()
                    or "weight" in k.lower()
                )
            ]

            print(
                "  Possible state-dict keys:",
                state_candidates
            )

        inspection_rows.append({
            "root": root_name,
            "checkpoint_type": checkpoint_type,
            "python_type": str(type(obj)),
            "top_level_keys": (
                json.dumps(
                    list(obj.keys())
                )
                if isinstance(obj, dict)
                else ""
            ),
        })

# ------------------------------------------------
# Save resolution audit
# ------------------------------------------------
RESOLUTION_PATH = (
    RESULT_DIR /
    "T1_4B_CHECKPOINT_ROOT_RESOLUTION_AUDIT.csv"
)

comparison_df.to_csv(
    RESOLUTION_PATH,
    index=False
)

# ------------------------------------------------
# Summary
# ------------------------------------------------
print("\n" + "=" * 70)
print("RESOLUTION SUMMARY")
print("=" * 70)

print(
    "Root A complete:",
    ROOT_A.exists()
)

print(
    "Root B complete:",
    ROOT_B.exists()
)

print(
    "best_eval cross-root exact:",
    int(
        comparison_df.loc[
            comparison_df["checkpoint_type"]
            == "best_eval_ba.pt",
            "bit_exact"
        ].sum()
    ),
    "/ 30"
)

print(
    "final_epoch15 cross-root exact:",
    int(
        comparison_df.loc[
            comparison_df["checkpoint_type"]
            == "final_epoch15.pt",
            "bit_exact"
        ].sum()
    ),
    "/ 30"
)

print(
    "\nResolution audit saved:"
)

print(RESOLUTION_PATH)

print("\n" + "=" * 70)
print("CELL 5B COMPLETE")
print("=" * 70)

print(
    "\nIMPORTANT: No production checkpoint roster "
    "has been selected yet."
)

T1.4b — RESOLVING FROZEN PRODUCTION CHECKPOINT ROSTER

Population completeness
----------------------------------------------------------------------

isic_kaggle_analysis
/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs
  best_eval_ba.pt    present=30/30 missing= 0
  final_epoch15.pt   present=30/30 missing= 0

ml_open_lab_30m
/kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs
  best_eval_ba.pt    present=30/30 missing= 0
  final_epoch15.pt   present=30/30 missing= 0

CROSS-ROOT CHECKPOINT COMPARISON

best_eval_ba.pt:
  Bit-exact matches: 30 / 30
  Different files  : 0 / 30

final_epoch15.pt:
  Bit-exact matches: 30 / 30
  Different files  : 0 / 30

CHECKPOINT CONTENT INSPECTION

isic_kaggle_analysis / best_eval_ba.pt
  Python type: <class 'collections.OrderedDict'>
  Top-level keys: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv

## T1.4b — Resolve Best-vs-Final Production Checkpoint

The two discovered checkpoint roots are byte-identical, so root selection is irrelevant.

The remaining distinction is:

- `best_eval_ba.pt`
- `final_epoch15.pt`

These are different model states.

Therefore, the frozen production evaluation/training code will be inspected to determine which checkpoint is the canonical inference artifact.

No model inference or IG computation is performed in this stage.

In [43]:
# ================================================================
# CELL 5C — RESOLVE BEST-vs-FINAL CHECKPOINT SEMANTICS
# ================================================================

from pathlib import Path
import re

print("=" * 70)
print("T1.4b — RESOLVING PRODUCTION CHECKPOINT SEMANTICS")
print("=" * 70)

# ------------------------------------------------
# Frozen project root
# ------------------------------------------------
FROZEN_PROJECT = Path(
    "/kaggle/input/datasets/"
    "tejasseshadriiiii/isic-final-project"
)

assert FROZEN_PROJECT.exists(), (
    f"Frozen project not found: {FROZEN_PROJECT}"
)

# ------------------------------------------------
# Candidate frozen code files
# ------------------------------------------------
candidate_code_files = []

for path in FROZEN_PROJECT.rglob("*.py"):

    if path.is_file():
        candidate_code_files.append(path)

candidate_code_files = sorted(
    candidate_code_files,
    key=lambda p: str(p)
)

print(
    "\nFrozen Python files discovered:",
    len(candidate_code_files)
)

# ------------------------------------------------
# Search terms specifically related to
# checkpoint selection/loading.
# ------------------------------------------------
patterns = [
    "best_eval_ba",
    "final_epoch15",
    "torch.load",
    "checkpoint",
    "best checkpoint",
    "best_ckpt",
    "best_model",
    "final_epoch",
]

hits = []

for path in candidate_code_files:

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    lower = text.lower()

    matched = []

    for pattern in patterns:

        if pattern.lower() in lower:
            matched.append(pattern)

    if matched:

        hits.append({
            "path": str(path),
            "patterns": matched,
        })

print(
    "Files containing checkpoint-related terms:",
    len(hits)
)

# ------------------------------------------------
# Print relevant snippets.
# ------------------------------------------------
print("\n" + "=" * 70)
print("RELEVANT FROZEN-CODE SNIPPETS")
print("=" * 70)

snippet_count = 0

for item in hits:

    path = Path(item["path"])

    text = path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    lines = text.splitlines()

    relevant_line_numbers = []

    for i, line in enumerate(lines):

        low = line.lower()

        if any(
            pattern.lower() in low
            for pattern in patterns
        ):
            relevant_line_numbers.append(i)

    # Merge nearby lines into compact windows.
    windows = []

    for line_no in relevant_line_numbers:

        start = max(0, line_no - 4)
        end = min(
            len(lines),
            line_no + 5
        )

        if not windows:
            windows.append([start, end])
        else:
            prev = windows[-1]

            if start <= prev[1]:
                prev[1] = max(
                    prev[1],
                    end
                )
            else:
                windows.append([start, end])

    if not windows:
        continue

    print(
        f"\nFILE: {path}"
    )

    for start, end in windows:

        print(
            f"\n--- lines {start + 1}"
            f"-{end} ---"
        )

        for j in range(start, end):
            print(
                f"{j + 1:5d}: {lines[j]}"
            )

        snippet_count += 1

        # Avoid flooding notebook output.
        if snippet_count >= 30:
            break

    if snippet_count >= 30:
        break

# ------------------------------------------------
# Also inspect the known frozen production
# training/evaluation files directly if present.
# ------------------------------------------------
print("\n" + "=" * 70)
print("KNOWN PRODUCTION CODE PATHS")
print("=" * 70)

known_paths = [
    FROZEN_PROJECT /
    "step2X_training_code_freeze" /
    "train_one_model.py",

    FROZEN_PROJECT /
    "step2Q_solver_implementation_v2" /
    "deterministic_evaluation_solver.py",

    FROZEN_PROJECT /
    "step2W_analysis_code_freeze" /
    "analysis_pipeline.py",
]

for path in known_paths:

    print(
        "\n",
        path,
        "EXISTS" if path.exists()
        else "NOT FOUND"
    )

# ------------------------------------------------
# Explicit status
# ------------------------------------------------
print("\n" + "=" * 70)
print("CHECKPOINT SELECTION STATUS")
print("=" * 70)

print(
    "Root A vs Root B ambiguity : RESOLVED"
)

print(
    "best vs final ambiguity    : "
    "NOT YET RESOLVED"
)

print(
    "\nNo model loaded."
)

print(
    "No inference performed."
)

print(
    "No IG computation performed."
)

print("\n" + "=" * 70)
print("CELL 5C COMPLETE")
print("=" * 70)

T1.4b — RESOLVING PRODUCTION CHECKPOINT SEMANTICS

Frozen Python files discovered: 4
Files containing checkpoint-related terms: 1

RELEVANT FROZEN-CODE SNIPPETS

FILE: /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2X_training_code_freeze/train_one_model.py

--- lines 2-10 ---
    2: """Frozen training script - ONE model per process (ISIC 2019 synthetic-hair dose-response study).
    3: 
    4: Locked recipe: ImageNet-pretrained ResNet-18 (IMAGENET1K_V1), 2-output head, 224x224, ImageNet normalisation,
    5: RandomHorizontalFlip + RandomRotation(15), AdamW lr 1e-4 wd 1e-4, batch 64, 15 epochs, CosineAnnealingLR(T_max=15),
    6: CrossEntropyLoss, float32, deterministic algorithms, no early stopping, primary checkpoint = final epoch 15.
    7: Per-epoch evaluation logs ONLY clean-evaluation loss and balanced accuracy (no counterfactual endpoints).
    8: """
    9: import os
   10: os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must precede CUDA initia

# T1.4b — Canonical Production Checkpoint Locked

The frozen training specification explicitly defines the **final epoch-15 checkpoint** as the primary production checkpoint.

Evidence from the frozen training script:
- Training runs for exactly 15 epochs.
- No early stopping is used.
- `final_epoch15.pt` is written at `epoch == N_EPOCHS`.
- `best_eval_ba.pt` is a secondary checkpoint tracking the best clean-evaluation balanced accuracy.
- Therefore, T1.4b production inference uses `final_epoch15.pt`.

Checkpoint policy:
- Canonical production checkpoint: `final_epoch15.pt`
- Secondary/non-production checkpoint: `best_eval_ba.pt`
- No checkpoint selection based on clean BA will be performed for T1.4b.
- All 30 canonical `(rho, replicate)` models will use their corresponding `final_epoch15.pt`.

No model inference or IG computation has yet been performed.

In [44]:
# ================================================================
# CELL 5D — FREEZE CANONICAL PRODUCTION CHECKPOINT ROSTER
# T1.4b
# ================================================================

from pathlib import Path
import hashlib
import json
import pandas as pd

print("=" * 70)
print("T1.4b — FREEZING CANONICAL PRODUCTION CHECKPOINT ROSTER")
print("=" * 70)

# ----------------------------------------------------------------
# Frozen decision from Cell 5C:
# final_epoch15.pt = canonical production checkpoint
# ----------------------------------------------------------------

CANONICAL_CHECKPOINT_NAME = "final_epoch15.pt"

EXPECTED_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

# Use the already-resolved duplicate root.
# Both roots were previously shown to be bit-exact duplicates.
CHECKPOINT_ROOT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/"
    "trained_models_readonly/training_runs"
)

assert CHECKPOINT_ROOT.exists(), f"Checkpoint root missing: {CHECKPOINT_ROOT}"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

rows = []

for model_key in EXPECTED_KEYS:
    ckpt = CHECKPOINT_ROOT / model_key / CANONICAL_CHECKPOINT_NAME

    assert ckpt.exists(), (
        f"Missing canonical checkpoint for {model_key}: {ckpt}"
    )

    rows.append({
        "model_key": model_key,
        "rho": float(model_key.split("_")[0]),
        "replicate": int(model_key.split("_")[1]),
        "checkpoint_type": CANONICAL_CHECKPOINT_NAME,
        "checkpoint_path": str(ckpt),
        "sha256": sha256_file(ckpt),
        "size_bytes": ckpt.stat().st_size,
    })

checkpoint_roster = pd.DataFrame(rows)

# ----------------------------------------------------------------
# Structural assertions
# ----------------------------------------------------------------

assert len(checkpoint_roster) == 30
assert checkpoint_roster["model_key"].tolist() == EXPECTED_KEYS
assert checkpoint_roster["checkpoint_type"].eq(
    CANONICAL_CHECKPOINT_NAME
).all()
assert checkpoint_roster["sha256"].notna().all()
assert checkpoint_roster["sha256"].str.len().eq(64).all()

print()
print("Canonical checkpoint :", CANONICAL_CHECKPOINT_NAME)
print("Checkpoint root      :", CHECKPOINT_ROOT)
print("Models               :", len(checkpoint_roster))
print("Unique SHA256 hashes :", checkpoint_roster["sha256"].nunique())

print()
print("MODEL ROSTER")
print(checkpoint_roster[
    ["model_key", "rho", "replicate", "checkpoint_type", "size_bytes"]
].to_string(index=False))

# ----------------------------------------------------------------
# Persist immutable roster artifact
# ----------------------------------------------------------------

ROSTER_CSV = RESULT_DIR / "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
ROSTER_JSON = RESULT_DIR / "T1_4B_CANONICAL_CHECKPOINT_ROSTER_LOCK.json"

checkpoint_roster.to_csv(ROSTER_CSV, index=False)

lock = {
    "experiment": "T1_4B",
    "checkpoint_policy": {
        "canonical": "final_epoch15.pt",
        "excluded_secondary": "best_eval_ba.pt",
        "reason": (
            "Frozen training script explicitly defines final epoch 15 "
            "as the primary checkpoint; no early stopping."
        ),
    },
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "n_models": int(len(checkpoint_roster)),
    "model_keys": EXPECTED_KEYS,
    "roster_sha256": hashlib.sha256(
        ROSTER_CSV.read_bytes()
    ).hexdigest(),
    "checkpoints": checkpoint_roster.to_dict(orient="records"),
}

with open(ROSTER_JSON, "w") as f:
    json.dump(lock, f, indent=2)

print()
print("Roster CSV :", ROSTER_CSV)
print("Roster lock:", ROSTER_JSON)

print()
print("=" * 70)
print("CELL 5D COMPLETE — CANONICAL CHECKPOINT ROSTER LOCKED")
print("=" * 70)

T1.4b — FREEZING CANONICAL PRODUCTION CHECKPOINT ROSTER


AssertionError: Missing canonical checkpoint for 0.0_1: /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/0.0_1/final_epoch15.pt

In [45]:
# ================================================================
# CELL 5D-R — FREEZE CANONICAL PRODUCTION CHECKPOINT ROSTER
# T1.4b
#
# Uses the previously generated checkpoint discovery artifact
# instead of assuming a directory layout.
# ================================================================

from pathlib import Path
import hashlib
import json
import pandas as pd

print("=" * 70)
print("T1.4b — FREEZING CANONICAL PRODUCTION CHECKPOINT ROSTER")
print("=" * 70)

# ----------------------------------------------------------------
# Frozen semantic decision from Cell 5C:
#
#   canonical production checkpoint = final_epoch15.pt
#
# best_eval_ba.pt is NOT used for production inference.
# ----------------------------------------------------------------

CANONICAL_CHECKPOINT_NAME = "final_epoch15.pt"

EXPECTED_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

# ----------------------------------------------------------------
# Previously generated discovery artifact
# ----------------------------------------------------------------

DISCOVERY_CSV = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_CHECKPOINT_DISCOVERY.csv"
)

assert DISCOVERY_CSV.exists(), (
    f"Missing checkpoint discovery artifact:\n{DISCOVERY_CSV}"
)

disc = pd.read_csv(DISCOVERY_CSV)

print("Discovery artifact:")
print(DISCOVERY_CSV)
print()
print("Columns:")
print(disc.columns.tolist())
print()
print("Rows:", len(disc))

# ----------------------------------------------------------------
# Inspect only the rows corresponding to final_epoch15.pt
# ----------------------------------------------------------------

final_rows = disc[
    disc.astype(str).apply(
        lambda col: col.str.contains(
            CANONICAL_CHECKPOINT_NAME,
            regex=False,
            na=False
        )
    ).any(axis=1)
].copy()

print()
print("Rows containing final_epoch15.pt:", len(final_rows))

print()
print(final_rows.to_string(index=False))

# ----------------------------------------------------------------
# Locate the actual path column robustly
# ----------------------------------------------------------------

path_candidates = [
    c for c in disc.columns
    if "path" in c.lower()
]

assert path_candidates, (
    "Could not identify a checkpoint path column in discovery artifact."
)

print()
print("Path-like columns:", path_candidates)

# Prefer a column explicitly named checkpoint_path / path
preferred = [
    c for c in path_candidates
    if c.lower() in {
        "checkpoint_path",
        "path",
        "ckpt_path",
        "checkpoint"
    }
]

PATH_COL = preferred[0] if preferred else path_candidates[0]

print("Using path column:", PATH_COL)

# ----------------------------------------------------------------
# Identify model-key column
# ----------------------------------------------------------------

key_candidates = [
    c for c in disc.columns
    if c.lower() in {
        "model_key",
        "model",
        "key",
        "model_id"
    }
]

assert key_candidates, (
    "Could not identify model-key column in discovery artifact."
)

KEY_COL = key_candidates[0]

print("Using model-key column:", KEY_COL)

# ----------------------------------------------------------------
# Restrict to canonical final_epoch15 checkpoints
# ----------------------------------------------------------------

mask_final = (
    disc[PATH_COL]
    .astype(str)
    .str.endswith(CANONICAL_CHECKPOINT_NAME)
)

roster = disc.loc[mask_final].copy()

print()
print("Candidate final checkpoints:", len(roster))

# ----------------------------------------------------------------
# Normalize / verify canonical model keys
# ----------------------------------------------------------------

roster["model_key"] = roster[KEY_COL].astype(str)

missing_keys = sorted(
    set(EXPECTED_KEYS) - set(roster["model_key"])
)

extra_keys = sorted(
    set(roster["model_key"]) - set(EXPECTED_KEYS)
)

print("Missing canonical model keys:", missing_keys)
print("Unexpected model keys        :", extra_keys)

assert not missing_keys, (
    f"Missing canonical checkpoints: {missing_keys}"
)

assert not extra_keys, (
    f"Unexpected model keys: {extra_keys}"
)

# Exactly one final checkpoint per canonical model
counts = roster["model_key"].value_counts()

duplicates = counts[counts != 1]

assert duplicates.empty, (
    f"Expected exactly one final checkpoint per model, "
    f"but found duplicates:\n{duplicates}"
)

assert len(roster) == 30

# ----------------------------------------------------------------
# Verify every path actually exists
# ----------------------------------------------------------------

roster["checkpoint_path"] = roster[PATH_COL].astype(str)

missing_paths = [
    p for p in roster["checkpoint_path"]
    if not Path(p).exists()
]

assert not missing_paths, (
    "Discovery artifact contains nonexistent checkpoint paths:\n"
    + "\n".join(missing_paths)
)

# ----------------------------------------------------------------
# Hash all 30 canonical checkpoints
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

roster["sha256"] = roster["checkpoint_path"].map(sha256_file)

roster["size_bytes"] = [
    Path(p).stat().st_size
    for p in roster["checkpoint_path"]
]

# ----------------------------------------------------------------
# Parse rho / replicate from canonical key
# ----------------------------------------------------------------

roster["rho"] = roster["model_key"].str.split("_").str[0].astype(float)
roster["replicate"] = roster["model_key"].str.split("_").str[1].astype(int)

# Deterministic canonical ordering
roster["_order"] = pd.Categorical(
    roster["model_key"],
    categories=EXPECTED_KEYS,
    ordered=True
)

roster = (
    roster
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

assert roster["model_key"].tolist() == EXPECTED_KEYS
assert roster["sha256"].str.len().eq(64).all()

# ----------------------------------------------------------------
# Save locked roster
# ----------------------------------------------------------------

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

ROSTER_JSON = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER_LOCK.json"
)

# Keep only clean canonical fields
save_cols = [
    "model_key",
    "rho",
    "replicate",
    "checkpoint_path",
    "sha256",
    "size_bytes",
]

roster[save_cols].to_csv(
    ROSTER_CSV,
    index=False
)

roster_lock = {
    "experiment": "T1_4B",

    "checkpoint_policy": {
        "canonical": CANONICAL_CHECKPOINT_NAME,
        "excluded_secondary": "best_eval_ba.pt",
        "reason": (
            "Frozen training script explicitly defines final "
            "epoch 15 as the primary production checkpoint; "
            "training uses 15 epochs with no early stopping."
        ),
    },

    "n_models": int(len(roster)),

    "model_keys": EXPECTED_KEYS,

    "roster_csv_sha256": hashlib.sha256(
        ROSTER_CSV.read_bytes()
    ).hexdigest(),

    "checkpoints": roster[save_cols].to_dict(
        orient="records"
    ),
}

with open(ROSTER_JSON, "w") as f:
    json.dump(roster_lock, f, indent=2)

# ----------------------------------------------------------------
# Final display
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CANONICAL CHECKPOINT ROSTER")
print("=" * 70)

print(
    roster[
        [
            "model_key",
            "rho",
            "replicate",
            "size_bytes",
        ]
    ].to_string(index=False)
)

print()
print("Canonical checkpoint :", CANONICAL_CHECKPOINT_NAME)
print("Models               :", len(roster))
print("Unique checkpoint SHA:", roster["sha256"].nunique())
print()
print("Roster CSV :")
print(ROSTER_CSV)
print()
print("Roster lock:")
print(ROSTER_JSON)

print()
print("=" * 70)
print("CELL 5D-R COMPLETE — CANONICAL CHECKPOINT ROSTER LOCKED")
print("=" * 70)

T1.4b — FREEZING CANONICAL PRODUCTION CHECKPOINT ROSTER
Discovery artifact:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_PRODUCTION_CHECKPOINT_DISCOVERY.csv

Columns:
['path', 'filename', 'possible_keys', 'n_possible_keys', 'size_bytes']

Rows: 120

Rows containing final_epoch15.pt: 60

                                                                                                                               path         filename possible_keys  n_possible_keys  size_bytes
/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_1/final_epoch15.pt final_epoch15.pt         0.0_1                1    44790155
/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_2/final_epoch15.pt final_epoch15.pt         0.0_2                1    44790155
/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_3/final_e

AssertionError: Could not identify model-key column in discovery artifact.

In [46]:
# ================================================================
# CELL 5D-R2 — FREEZE CANONICAL PRODUCTION CHECKPOINT ROSTER
# T1.4b
# ================================================================

from pathlib import Path
import hashlib
import json
import pandas as pd

print("=" * 70)
print("T1.4b — FREEZING CANONICAL PRODUCTION CHECKPOINT ROSTER")
print("=" * 70)

# ----------------------------------------------------------------
# Frozen decision from Cell 5C:
# final_epoch15.pt is the canonical production checkpoint.
# ----------------------------------------------------------------

CANONICAL_CHECKPOINT_NAME = "final_epoch15.pt"

EXPECTED_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

DISCOVERY_CSV = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_CHECKPOINT_DISCOVERY.csv"
)

assert DISCOVERY_CSV.exists(), (
    f"Missing checkpoint discovery artifact:\n{DISCOVERY_CSV}"
)

disc = pd.read_csv(DISCOVERY_CSV)

print("Discovery artifact:", DISCOVERY_CSV)
print("Columns:", disc.columns.tolist())
print("Rows:", len(disc))

# ----------------------------------------------------------------
# Keep only final_epoch15.pt entries.
# ----------------------------------------------------------------

roster = disc[
    disc["filename"].astype(str).eq(CANONICAL_CHECKPOINT_NAME)
].copy()

print()
print("Final-epoch checkpoint rows:", len(roster))

assert len(roster) == 60, (
    f"Expected 60 final_epoch15 entries "
    f"(30 models × 2 duplicate roots), got {len(roster)}"
)

# ----------------------------------------------------------------
# Extract canonical model key from possible_keys.
#
# Example:
#   possible_keys = "0.0_1"
# ----------------------------------------------------------------

def extract_model_key(value):
    """
    Extract a canonical model key from the discovery artifact's
    possible_keys field.
    """
    s = str(value)

    # Handle possible serialized list representations defensively.
    for key in EXPECTED_KEYS:
        if key in s:
            return key

    return None

roster["model_key"] = roster["possible_keys"].map(extract_model_key)

assert roster["model_key"].notna().all(), (
    "At least one final checkpoint could not be mapped "
    "to a canonical model key."
)

# ----------------------------------------------------------------
# Verify exactly two storage copies for every canonical model.
# ----------------------------------------------------------------

counts = roster["model_key"].value_counts()

print()
print("Model-key multiplicities:")
print(counts.sort_index().to_string())

assert set(counts.index) == set(EXPECTED_KEYS), (
    "Canonical model-key population mismatch."
)

assert counts.eq(2).all(), (
    "Expected exactly two duplicate storage copies "
    "for every canonical model."
)

# ----------------------------------------------------------------
# Resolve the primary root.
#
# Cell 5B established that the two roots are bit-exact duplicates.
# We use the isic-kaggle-analysis root as the canonical path source.
# ----------------------------------------------------------------

CANONICAL_ROOT_MARKER = (
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "training_runs/"
)

canonical = roster[
    roster["path"].astype(str).str.startswith(
        CANONICAL_ROOT_MARKER
    )
].copy()

assert len(canonical) == 30, (
    f"Expected 30 checkpoints in canonical root, got {len(canonical)}"
)

# ----------------------------------------------------------------
# Verify exactly one checkpoint per model.
# ----------------------------------------------------------------

canonical_counts = canonical["model_key"].value_counts()

assert set(canonical_counts.index) == set(EXPECTED_KEYS)
assert canonical_counts.eq(1).all()

# ----------------------------------------------------------------
# Verify actual filesystem paths.
# ----------------------------------------------------------------

canonical["checkpoint_path"] = canonical["path"].astype(str)

missing = [
    p for p in canonical["checkpoint_path"]
    if not Path(p).exists()
]

assert not missing, (
    "Missing canonical checkpoint files:\n" +
    "\n".join(missing)
)

# ----------------------------------------------------------------
# Hash all 30 canonical checkpoints.
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

canonical["sha256"] = canonical["checkpoint_path"].map(
    sha256_file
)

canonical["size_bytes"] = canonical["checkpoint_path"].map(
    lambda p: Path(p).stat().st_size
)

# ----------------------------------------------------------------
# Parse rho / replicate.
# ----------------------------------------------------------------

canonical["rho"] = (
    canonical["model_key"]
    .str.split("_")
    .str[0]
    .astype(float)
)

canonical["replicate"] = (
    canonical["model_key"]
    .str.split("_")
    .str[1]
    .astype(int)
)

# ----------------------------------------------------------------
# Deterministic ordering.
# ----------------------------------------------------------------

canonical["_order"] = pd.Categorical(
    canonical["model_key"],
    categories=EXPECTED_KEYS,
    ordered=True
)

canonical = (
    canonical
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

assert canonical["model_key"].tolist() == EXPECTED_KEYS

# ----------------------------------------------------------------
# Final roster fields.
# ----------------------------------------------------------------

save_cols = [
    "model_key",
    "rho",
    "replicate",
    "checkpoint_path",
    "sha256",
    "size_bytes",
]

canonical_roster = canonical[save_cols].copy()

assert len(canonical_roster) == 30
assert canonical_roster["sha256"].str.len().eq(64).all()
assert canonical_roster["checkpoint_path"].map(
    lambda p: Path(p).exists()
).all()

# ----------------------------------------------------------------
# Persist lock.
# ----------------------------------------------------------------

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

ROSTER_JSON = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER_LOCK.json"
)

canonical_roster.to_csv(
    ROSTER_CSV,
    index=False
)

roster_lock = {
    "experiment": "T1_4B",

    "checkpoint_policy": {
        "canonical": CANONICAL_CHECKPOINT_NAME,
        "excluded_secondary": "best_eval_ba.pt",
        "reason": (
            "Frozen training script explicitly defines "
            "final epoch 15 as the primary production "
            "checkpoint; training has no early stopping."
        ),
    },

    "canonical_root": CANONICAL_ROOT_MARKER,

    "n_models": 30,

    "model_keys": EXPECTED_KEYS,

    "roster_csv_sha256": hashlib.sha256(
        ROSTER_CSV.read_bytes()
    ).hexdigest(),

    "checkpoints": canonical_roster.to_dict(
        orient="records"
    ),
}

with open(ROSTER_JSON, "w") as f:
    json.dump(
        roster_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# Display.
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CANONICAL PRODUCTION CHECKPOINT ROSTER")
print("=" * 70)

print(
    canonical_roster[
        [
            "model_key",
            "rho",
            "replicate",
            "size_bytes",
        ]
    ].to_string(index=False)
)

print()
print("Canonical checkpoint :", CANONICAL_CHECKPOINT_NAME)
print("Canonical root       :", CANONICAL_ROOT_MARKER)
print("Models               :", len(canonical_roster))
print("Unique SHA256 hashes :", canonical_roster["sha256"].nunique())

print()
print("Roster CSV:")
print(ROSTER_CSV)

print()
print("Roster lock:")
print(ROSTER_JSON)

print()
print("=" * 70)
print("CELL 5D-R2 COMPLETE — CHECKPOINT ROSTER LOCKED")
print("=" * 70)

T1.4b — FREEZING CANONICAL PRODUCTION CHECKPOINT ROSTER
Discovery artifact: /kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_PRODUCTION_CHECKPOINT_DISCOVERY.csv
Columns: ['path', 'filename', 'possible_keys', 'n_possible_keys', 'size_bytes']
Rows: 120

Final-epoch checkpoint rows: 60

Model-key multiplicities:
model_key
0.0_1    2
0.0_2    2
0.0_3    2
0.0_4    2
0.0_5    2
0.2_1    2
0.2_2    2
0.2_3    2
0.2_4    2
0.2_5    2
0.4_1    2
0.4_2    2
0.4_3    2
0.4_4    2
0.4_5    2
0.6_1    2
0.6_2    2
0.6_3    2
0.6_4    2
0.6_5    2
0.8_1    2
0.8_2    2
0.8_3    2
0.8_4    2
0.8_5    2
1.0_1    2
1.0_2    2
1.0_3    2
1.0_4    2
1.0_5    2

CANONICAL PRODUCTION CHECKPOINT ROSTER
model_key  rho  replicate  size_bytes
    0.0_1  0.0          1    44790155
    0.0_2  0.0          2    44790155
    0.0_3  0.0          3    44790155
    0.0_4  0.0          4    44790155
    0.0_5  0.0          5    44790155
    0.2_1  0.2          1    44790155
    0.2_2  0.2          2    4479

# T1.4b — Production Model Loading Preflight

The canonical production checkpoint roster is frozen.

For each of the 30 canonical `(rho, replicate)` models:

1. Instantiate the frozen ResNet-18 architecture.
2. Load the canonical `final_epoch15.pt` state dict.
3. Verify state-dict key and tensor-shape compatibility.
4. Run a deterministic forward-pass smoke test on a fixed sample.
5. Verify finite logits and expected two-class output shape.
6. Record checkpoint/model-loading metadata.

This is a preflight only.

No attribution maps, null scoring, or statistical analysis are performed here.

In [48]:
# ================================================================
# CELL 5E — PRODUCTION MODEL LOADING PREFLIGHT
# T1.4b
#
# NO IG COMPUTATION
# NO NULL SCORING
# NO STATISTICS
# ================================================================

import gc
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision.models as models

print("=" * 70)
print("T1.4b — PRODUCTION MODEL LOADING PREFLIGHT")
print("=" * 70)

# ----------------------------------------------------------------
# Load the already-locked canonical checkpoint roster
# ----------------------------------------------------------------

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

assert ROSTER_CSV.exists(), (
    f"Missing canonical roster:\n{ROSTER_CSV}"
)

canonical_roster = pd.read_csv(ROSTER_CSV)

EXPECTED_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert len(canonical_roster) == 30
assert canonical_roster["model_key"].tolist() == EXPECTED_KEYS

print()
print("Canonical roster loaded:", len(canonical_roster), "models")

# ----------------------------------------------------------------
# Exact checkpoint architecture
# ----------------------------------------------------------------

NUM_CLASSES = 2

def build_production_model():
    return models.resnet18(
        weights=None,
        num_classes=NUM_CLASSES
    )

# ----------------------------------------------------------------
# Establish expected state-dict structure from first checkpoint
# ----------------------------------------------------------------

first_path = Path(
    canonical_roster.iloc[0]["checkpoint_path"]
)

assert first_path.exists()

first_state = torch.load(
    first_path,
    map_location="cpu",
    weights_only=True
)

first_model = build_production_model()

expected_state = first_model.state_dict()

assert list(first_state.keys()) == list(expected_state.keys()), (
    "State-dict key mismatch in first checkpoint."
)

for key in expected_state:
    assert (
        first_state[key].shape ==
        expected_state[key].shape
    ), (
        f"Shape mismatch for {key}: "
        f"{first_state[key].shape} vs "
        f"{expected_state[key].shape}"
    )

print()
print("Architecture           : torchvision ResNet-18")
print("Output classes         :", NUM_CLASSES)
print("State-dict keys        :", len(expected_state))
print("First checkpoint       : STRUCTURE PASS")

del first_state
del first_model
del expected_state
gc.collect()

# ----------------------------------------------------------------
# Fixed smoke-test image
# ----------------------------------------------------------------

assert "sample_df" in globals(), (
    "sample_df is not available. "
    "Re-run the T1.4b sample-lock cells."
)

assert "cache_index" in globals(), (
    "cache_index is not available. "
    "Re-run the T1.4b cache-alignment cells."
)

SMOKE_IMAGE_ID = str(
    sample_df.iloc[0]["image"]
)

cache_match = cache_index[
    cache_index["image"].astype(str).eq(
        SMOKE_IMAGE_ID
    )
]

assert len(cache_match) == 1

smoke_cache_row = int(cache_match.index[0])

smoke_np = np.asarray(
    clean_224[smoke_cache_row]
)

assert smoke_np.shape == (224, 224, 3)
assert smoke_np.dtype == np.uint8

# ----------------------------------------------------------------
# Frozen ImageNet normalization
# ----------------------------------------------------------------

x = torch.from_numpy(
    smoke_np.astype(np.float32) / 255.0
).permute(2, 0, 1).unsqueeze(0)

imagenet_mean = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32
).view(1, 3, 1, 1)

imagenet_std = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32
).view(1, 3, 1, 1)

x = (x - imagenet_mean) / imagenet_std

assert x.shape == (1, 3, 224, 224)
assert torch.isfinite(x).all()

# ----------------------------------------------------------------
# Load and forward-test all 30 models
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


load_rows = []

for _, row in canonical_roster.iterrows():

    model_key = str(row["model_key"])
    ckpt_path = Path(
        row["checkpoint_path"]
    )

    print(
        f"Loading {model_key:>5s} ... ",
        end="",
        flush=True
    )

    assert ckpt_path.exists()

    # Verify the actual file still matches the locked roster.
    actual_sha = sha256_file(ckpt_path)

    assert actual_sha == str(row["sha256"]), (
        f"SHA256 mismatch for {model_key}"
    )

    model = build_production_model()

    state = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=True
    )

    # Strict loading: no missing/unexpected parameters.
    incompat = model.load_state_dict(
        state,
        strict=True
    )

    assert len(incompat.missing_keys) == 0
    assert len(incompat.unexpected_keys) == 0

    model = model.to(DEVICE)
    model.eval()

    # Forward smoke test only.
    with torch.inference_mode():
        logits = model(
            x.to(DEVICE)
        )

    assert logits.shape == (
        1,
        NUM_CLASSES
    ), (
        f"{model_key}: bad logits shape "
        f"{tuple(logits.shape)}"
    )

    assert torch.isfinite(logits).all(), (
        f"{model_key}: non-finite logits"
    )

    logits_cpu = (
        logits
        .detach()
        .cpu()
        .numpy()[0]
    )

    load_rows.append({
        "model_key": model_key,
        "rho": float(row["rho"]),
        "replicate": int(row["replicate"]),
        "checkpoint_path": str(ckpt_path),
        "sha256_verified": True,
        "state_dict_strict_load": True,
        "logit_dim": int(logits.shape[1]),
        "logit_0": float(logits_cpu[0]),
        "logit_1": float(logits_cpu[1]),
        "finite_logits": True,
    })

    print("PASS")

    del logits
    del state
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

# ----------------------------------------------------------------
# Audit results
# ----------------------------------------------------------------

model_preflight = pd.DataFrame(
    load_rows
)

assert len(model_preflight) == 30
assert (
    model_preflight["model_key"].tolist()
    == EXPECTED_KEYS
)
assert model_preflight[
    "sha256_verified"
].all()
assert model_preflight[
    "state_dict_strict_load"
].all()
assert model_preflight[
    "finite_logits"
].all()
assert model_preflight[
    "logit_dim"
].eq(2).all()

PREFLIGHT_CSV = (
    RESULT_DIR /
    "T1_4B_MODEL_LOADING_PREFLIGHT.csv"
)

PREFLIGHT_JSON = (
    RESULT_DIR /
    "T1_4B_MODEL_LOADING_PREFLIGHT_LOCK.json"
)

model_preflight.to_csv(
    PREFLIGHT_CSV,
    index=False
)

preflight_lock = {
    "experiment": "T1_4B",
    "checkpoint_type": "final_epoch15.pt",
    "n_models": 30,
    "architecture": "torchvision ResNet-18",
    "num_classes": 2,
    "strict_state_dict_loading": True,
    "smoke_test_image": SMOKE_IMAGE_ID,
    "device": str(DEVICE),
    "results_csv_sha256": hashlib.sha256(
        PREFLIGHT_CSV.read_bytes()
    ).hexdigest(),
}

with open(
    PREFLIGHT_JSON,
    "w"
) as f:
    json.dump(
        preflight_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("MODEL LOADING PREFLIGHT SUMMARY")
print("=" * 70)

print(
    "Models checked            :",
    len(model_preflight)
)

print(
    "Strict state-dict loads   :",
    int(
        model_preflight[
            "state_dict_strict_load"
        ].sum()
    ),
    "/ 30"
)

print(
    "Finite forward passes     :",
    int(
        model_preflight[
            "finite_logits"
        ].sum()
    ),
    "/ 30"
)

print(
    "Expected 2-class outputs  :",
    int(
        model_preflight[
            "logit_dim"
        ].eq(2).sum()
    ),
    "/ 30"
)

print(
    "Checkpoint hashes verified:",
    int(
        model_preflight[
            "sha256_verified"
        ].sum()
    ),
    "/ 30"
)

print()
print("Smoke-test image:", SMOKE_IMAGE_ID)

print()
print("Audit CSV:")
print(PREFLIGHT_CSV)

print()
print("Audit lock:")
print(PREFLIGHT_JSON)

print()
print("=" * 70)
print("CELL 5E COMPLETE — MODEL LOADING PREFLIGHT PASSED")
print("=" * 70)

print()
print("NO IG COMPUTATION WAS PERFORMED.")
print("NO NULL SCORING WAS PERFORMED.")

T1.4b — PRODUCTION MODEL LOADING PREFLIGHT

Canonical roster loaded: 30 models

Architecture           : torchvision ResNet-18
Output classes         : 2
State-dict keys        : 122
First checkpoint       : STRUCTURE PASS


NameError: name 'clean_224' is not defined

In [49]:
# ================================================================
# CELL 5E-R1 — PRODUCTION MODEL LOADING PREFLIGHT
# T1.4b
#
# Uses the already-validated rebuilt cache directly from disk.
#
# NO IG COMPUTATION
# NO NULL SCORING
# NO STATISTICS
# ================================================================

import gc
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision.models as models

print("=" * 70)
print("T1.4b — PRODUCTION MODEL LOADING PREFLIGHT")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load locked canonical checkpoint roster
# ----------------------------------------------------------------

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

assert ROSTER_CSV.exists(), (
    f"Missing canonical roster:\n{ROSTER_CSV}"
)

canonical_roster = pd.read_csv(ROSTER_CSV)

EXPECTED_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert len(canonical_roster) == 30
assert canonical_roster["model_key"].tolist() == EXPECTED_KEYS

print()
print("Canonical roster loaded:", len(canonical_roster), "models")

# ----------------------------------------------------------------
# 2. Load already-validated cache
# ----------------------------------------------------------------

CACHE_DIR = Path("/kaggle/temp/isic_cache")

CACHE_INDEX_PATH = CACHE_DIR / "cache_index.csv"
CLEAN_PATH = CACHE_DIR / "clean_224.npy"

assert CACHE_INDEX_PATH.exists(), (
    f"Missing cache index:\n{CACHE_INDEX_PATH}"
)

assert CLEAN_PATH.exists(), (
    f"Missing clean cache:\n{CLEAN_PATH}"
)

cache_index_local = pd.read_csv(
    CACHE_INDEX_PATH
)

# Memory-map the 3.256 GB array rather than copying it into RAM.
clean_224_local = np.load(
    CLEAN_PATH,
    mmap_mode="r"
)

print()
print("Validated cache loaded")
print("Cache directory :", CACHE_DIR)
print("Cache index rows:", len(cache_index_local))
print("Clean shape     :", clean_224_local.shape)
print("Clean dtype     :", clean_224_local.dtype)

assert clean_224_local.shape == (
    len(cache_index_local),
    224,
    224,
    3
)

assert clean_224_local.dtype == np.uint8

# ----------------------------------------------------------------
# 3. Verify the smoke-test image against the locked XAI sample
# ----------------------------------------------------------------

# Load sample lock if the dataframe is not currently available.
SAMPLE_CSV = (
    RESULT_DIR /
    "T1_4B_IMAGE_SAMPLE.csv"
)

assert SAMPLE_CSV.exists(), (
    f"Missing locked image sample:\n{SAMPLE_CSV}"
)

sample_df_local = pd.read_csv(
    SAMPLE_CSV
)

assert len(sample_df_local) == 100

SMOKE_IMAGE_ID = str(
    sample_df_local.iloc[0]["image"]
)

cache_matches = cache_index_local[
    cache_index_local["image"].astype(str).eq(
        SMOKE_IMAGE_ID
    )
]

assert len(cache_matches) == 1, (
    f"Expected exactly one cache row for "
    f"{SMOKE_IMAGE_ID}, found {len(cache_matches)}"
)

# Use positional row, not dataframe label.
smoke_cache_pos = int(
    cache_matches.index[0]
)

smoke_np = np.asarray(
    clean_224_local[smoke_cache_pos]
)

assert smoke_np.shape == (
    224,
    224,
    3
)

assert smoke_np.dtype == np.uint8

print()
print("Smoke-test image:", SMOKE_IMAGE_ID)
print("Cache row       :", smoke_cache_pos)
print("Smoke image     :", smoke_np.shape, smoke_np.dtype)

# ----------------------------------------------------------------
# 4. Exact production architecture
# ----------------------------------------------------------------

NUM_CLASSES = 2

def build_production_model():
    return models.resnet18(
        weights=None,
        num_classes=NUM_CLASSES
    )

# ----------------------------------------------------------------
# 5. Establish expected state-dict structure
# ----------------------------------------------------------------

first_path = Path(
    canonical_roster.iloc[0]["checkpoint_path"]
)

assert first_path.exists()

first_state = torch.load(
    first_path,
    map_location="cpu",
    weights_only=True
)

first_model = build_production_model()

expected_state = first_model.state_dict()

assert list(first_state.keys()) == list(
    expected_state.keys()
), "State-dict key mismatch."

for key in expected_state:
    assert (
        first_state[key].shape ==
        expected_state[key].shape
    ), (
        f"Shape mismatch for {key}: "
        f"{first_state[key].shape} vs "
        f"{expected_state[key].shape}"
    )

print()
print("Architecture           : torchvision ResNet-18")
print("Output classes         :", NUM_CLASSES)
print("State-dict keys        :", len(expected_state))
print("First checkpoint       : STRUCTURE PASS")

del first_state
del first_model
del expected_state
gc.collect()

# ----------------------------------------------------------------
# 6. ImageNet normalization
# ----------------------------------------------------------------

x = torch.from_numpy(
    smoke_np.astype(np.float32) / 255.0
).permute(
    2, 0, 1
).unsqueeze(0)

imagenet_mean = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32
).view(
    1, 3, 1, 1
)

imagenet_std = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32
).view(
    1, 3, 1, 1
)

x = (
    x - imagenet_mean
) / imagenet_std

assert x.shape == (
    1,
    3,
    224,
    224
)

assert torch.isfinite(x).all()

# ----------------------------------------------------------------
# 7. SHA256 helper
# ----------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

# ----------------------------------------------------------------
# 8. Load and forward-test all 30 production models
# ----------------------------------------------------------------

load_rows = []

for _, row in canonical_roster.iterrows():

    model_key = str(
        row["model_key"]
    )

    ckpt_path = Path(
        row["checkpoint_path"]
    )

    print(
        f"Loading {model_key:>5s} ... ",
        end="",
        flush=True
    )

    assert ckpt_path.exists()

    # Verify locked checkpoint hash.
    actual_sha = sha256_file(
        ckpt_path
    )

    assert actual_sha == str(
        row["sha256"]
    ), (
        f"SHA256 mismatch for {model_key}"
    )

    # Construct exact architecture.
    model = build_production_model()

    # Load checkpoint.
    state = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=True
    )

    # Strict state-dict load.
    incompat = model.load_state_dict(
        state,
        strict=True
    )

    assert len(
        incompat.missing_keys
    ) == 0

    assert len(
        incompat.unexpected_keys
    ) == 0

    # Inference mode.
    model = model.to(DEVICE)
    model.eval()

    with torch.inference_mode():

        logits = model(
            x.to(DEVICE)
        )

    assert logits.shape == (
        1,
        NUM_CLASSES
    ), (
        f"{model_key}: unexpected logits "
        f"shape {tuple(logits.shape)}"
    )

    assert torch.isfinite(
        logits
    ).all(), (
        f"{model_key}: non-finite logits"
    )

    logits_cpu = (
        logits
        .detach()
        .cpu()
        .numpy()[0]
    )

    load_rows.append({
        "model_key": model_key,
        "rho": float(row["rho"]),
        "replicate": int(row["replicate"]),
        "checkpoint_path": str(ckpt_path),
        "sha256_verified": True,
        "state_dict_strict_load": True,
        "logit_dim": int(logits.shape[1]),
        "logit_0": float(logits_cpu[0]),
        "logit_1": float(logits_cpu[1]),
        "finite_logits": True,
    })

    print("PASS")

    # Free model before next checkpoint.
    del logits
    del state
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

# ----------------------------------------------------------------
# 9. Validate complete preflight
# ----------------------------------------------------------------

model_preflight = pd.DataFrame(
    load_rows
)

assert len(model_preflight) == 30

assert (
    model_preflight["model_key"].tolist()
    == EXPECTED_KEYS
)

assert model_preflight[
    "sha256_verified"
].all()

assert model_preflight[
    "state_dict_strict_load"
].all()

assert model_preflight[
    "finite_logits"
].all()

assert model_preflight[
    "logit_dim"
].eq(2).all()

# ----------------------------------------------------------------
# 10. Persist audit
# ----------------------------------------------------------------

PREFLIGHT_CSV = (
    RESULT_DIR /
    "T1_4B_MODEL_LOADING_PREFLIGHT.csv"
)

PREFLIGHT_JSON = (
    RESULT_DIR /
    "T1_4B_MODEL_LOADING_PREFLIGHT_LOCK.json"
)

model_preflight.to_csv(
    PREFLIGHT_CSV,
    index=False
)

preflight_lock = {
    "experiment": "T1_4B",
    "checkpoint_type": "final_epoch15.pt",
    "n_models": 30,
    "architecture": "torchvision ResNet-18",
    "num_classes": 2,
    "strict_state_dict_loading": True,
    "smoke_test_image": SMOKE_IMAGE_ID,
    "device": str(DEVICE),
    "results_csv_sha256": hashlib.sha256(
        PREFLIGHT_CSV.read_bytes()
    ).hexdigest()
}

with open(
    PREFLIGHT_JSON,
    "w"
) as f:

    json.dump(
        preflight_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# 11. Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("MODEL LOADING PREFLIGHT SUMMARY")
print("=" * 70)

print(
    "Models checked            :",
    len(model_preflight)
)

print(
    "Strict state-dict loads   :",
    int(
        model_preflight[
            "state_dict_strict_load"
        ].sum()
    ),
    "/ 30"
)

print(
    "Finite forward passes     :",
    int(
        model_preflight[
            "finite_logits"
        ].sum()
    ),
    "/ 30"
)

print(
    "Expected 2-class outputs  :",
    int(
        model_preflight[
            "logit_dim"
        ].eq(2).sum()
    ),
    "/ 30"
)

print(
    "Checkpoint hashes verified:",
    int(
        model_preflight[
            "sha256_verified"
        ].sum()
    ),
    "/ 30"
)

print()
print("Smoke-test image:", SMOKE_IMAGE_ID)

print()
print("Audit CSV:")
print(PREFLIGHT_CSV)

print()
print("Audit lock:")
print(PREFLIGHT_JSON)

print()
print("=" * 70)
print("CELL 5E-R1 COMPLETE — MODEL LOADING PREFLIGHT PASSED")
print("=" * 70)

print()
print("NO IG COMPUTATION WAS PERFORMED.")
print("NO NULL SCORING WAS PERFORMED.")

T1.4b — PRODUCTION MODEL LOADING PREFLIGHT

Canonical roster loaded: 30 models

Validated cache loaded
Cache directory : /kaggle/temp/isic_cache
Cache index rows: 23227
Clean shape     : (23227, 224, 224, 3)
Clean dtype     : uint8

Smoke-test image: ISIC_0013637_downsampled
Cache row       : 18614
Smoke image     : (224, 224, 3) uint8

Architecture           : torchvision ResNet-18
Output classes         : 2
State-dict keys        : 122
First checkpoint       : STRUCTURE PASS
Loading 0.0_1 ... PASS
Loading 0.0_2 ... PASS
Loading 0.0_3 ... PASS
Loading 0.0_4 ... PASS
Loading 0.0_5 ... PASS
Loading 0.2_1 ... PASS
Loading 0.2_2 ... PASS
Loading 0.2_3 ... PASS
Loading 0.2_4 ... PASS
Loading 0.2_5 ... PASS
Loading 0.4_1 ... PASS
Loading 0.4_2 ... PASS
Loading 0.4_3 ... PASS
Loading 0.4_4 ... PASS
Loading 0.4_5 ... PASS
Loading 0.6_1 ... PASS
Loading 0.6_2 ... PASS
Loading 0.6_3 ... PASS
Loading 0.6_4 ... PASS
Loading 0.6_5 ... PASS
Loading 0.8_1 ... PASS
Loading 0.8_2 ... PASS
Loading 0.8_

In [50]:
# ================================================================
# CELL 5F — FROZEN IG SEMANTICS PREFLIGHT
# T1.4b
#
# IMPORTANT:
#   Inspection only.
#   No production IG computation.
#   No model inference.
#   No null scoring.
# ================================================================

from pathlib import Path
import inspect
import hashlib
import json

print("=" * 70)
print("T1.4b — FROZEN IG SEMANTICS PREFLIGHT")
print("=" * 70)

# ----------------------------------------------------------------
# Frozen analysis pipeline
# ----------------------------------------------------------------

ANALYSIS_PATH = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/"
    "isic-final-project/"
    "step2W_analysis_code_freeze/"
    "analysis_pipeline.py"
)

assert ANALYSIS_PATH.exists(), (
    f"Frozen analysis pipeline not found:\n{ANALYSIS_PATH}"
)

print()
print("Frozen analysis pipeline:")
print(ANALYSIS_PATH)

# ----------------------------------------------------------------
# Load the frozen module if not already available.
# ----------------------------------------------------------------

if "analysis_pipeline" not in globals():

    import importlib.util

    spec = importlib.util.spec_from_file_location(
        "analysis_pipeline",
        ANALYSIS_PATH
    )

    analysis_pipeline = (
        importlib.util.module_from_spec(spec)
    )

    spec.loader.exec_module(
        analysis_pipeline
    )

else:

    print(
        "analysis_pipeline already loaded; "
        "using existing frozen module."
    )

# ----------------------------------------------------------------
# Verify frozen file hash if the notebook already has the expected
# production hash available.
# ----------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


analysis_sha = sha256_file(
    ANALYSIS_PATH
)

print()
print("analysis_pipeline.py SHA256:")
print(analysis_sha)

# ----------------------------------------------------------------
# Locate IG-related symbols.
# ----------------------------------------------------------------

module_dict = vars(
    analysis_pipeline
)

ig_names = [
    name
    for name in module_dict
    if "ig" in name.lower()
]

print()
print("IG-related symbols:")
for name in sorted(ig_names):
    print(" ", name)

# ----------------------------------------------------------------
# Required production function.
# ----------------------------------------------------------------

assert hasattr(
    analysis_pipeline,
    "ig_enrichment"
), (
    "Frozen analysis_pipeline.py does not expose "
    "ig_enrichment."
)

ig_enrichment = (
    analysis_pipeline.ig_enrichment
)

print()
print("ig_enrichment signature:")
print(
    inspect.signature(
        ig_enrichment
    )
)

# ----------------------------------------------------------------
# Frozen constants
# ----------------------------------------------------------------

CONSTANT_NAMES = [
    "IG_STEPS",
    "IG_RETRY_STEPS",
    "IG_SIGMA",
    "IG_TOL_REL",
    "IG_TAU_ABS",
    "XAI_EPS",
]

print()
print("Frozen IG constants:")

constant_values = {}

for name in CONSTANT_NAMES:

    assert hasattr(
        analysis_pipeline,
        name
    ), (
        f"Missing required frozen constant: {name}"
    )

    value = getattr(
        analysis_pipeline,
        name
    )

    constant_values[name] = value

    print(
        f"  {name:15s} = {value!r}"
    )

# ----------------------------------------------------------------
# Frozen expected values
# ----------------------------------------------------------------

assert int(
    constant_values["IG_STEPS"]
) == 50

assert int(
    constant_values["IG_RETRY_STEPS"]
) == 100

assert float(
    constant_values["IG_SIGMA"]
) == 10.0

print()
print("Frozen IG parameter assertions: PASS")

# ----------------------------------------------------------------
# Source inspection
# ----------------------------------------------------------------

ig_source = inspect.getsource(
    ig_enrichment
)

print()
print("=" * 70)
print("FROZEN ig_enrichment SOURCE")
print("=" * 70)

print(ig_source)

# ----------------------------------------------------------------
# Search surrounding module source for relevant IG implementation
# terms.
# ----------------------------------------------------------------

full_source = ANALYSIS_PATH.read_text(
    encoding="utf-8"
)

terms = [
    "IntegratedGradients",
    "integrated_gradients",
    "IG_STEPS",
    "IG_RETRY_STEPS",
    "IG_SIGMA",
    "IG_TOL_REL",
    "IG_TAU_ABS",
    "XAI_EPS",
    "baseline",
    "completeness",
    "z_malignant",
    "z_benign",
]

print()
print("=" * 70)
print("RELEVANT FROZEN-SOURCE LINES")
print("=" * 70)

lines = full_source.splitlines()

for i, line in enumerate(lines, start=1):

    lower = line.lower()

    if any(
        term.lower() in lower
        for term in terms
    ):

        print(
            f"{i:4d}: {line}"
        )

# ----------------------------------------------------------------
# Persist semantic inspection artifact.
# ----------------------------------------------------------------

SEMANTICS_JSON = (
    RESULT_DIR /
    "T1_4B_FROZEN_IG_SEMANTICS_PREFLIGHT.json"
)

semantics_lock = {
    "experiment": "T1_4B",
    "analysis_pipeline_path": str(
        ANALYSIS_PATH
    ),
    "analysis_pipeline_sha256": analysis_sha,
    "ig_enrichment_signature": str(
        inspect.signature(
            ig_enrichment
        )
    ),
    "constants": {
        k: str(v)
        for k, v in constant_values.items()
    },
    "production_ig_execution_performed": False,
    "null_scoring_performed": False,
}

with open(
    SEMANTICS_JSON,
    "w"
) as f:

    json.dump(
        semantics_lock,
        f,
        indent=2
    )

print()
print("=" * 70)
print("CELL 5F COMPLETE — IG SEMANTICS INSPECTED")
print("=" * 70)

print()
print("Production IG executed : NO")
print("Null scoring executed  : NO")
print()
print("Audit:")
print(SEMANTICS_JSON)

T1.4b — FROZEN IG SEMANTICS PREFLIGHT

Frozen analysis pipeline:
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2W_analysis_code_freeze/analysis_pipeline.py

analysis_pipeline.py SHA256:
bcba1d25782f5770fbe629b1d0281f87a4bfde29b343c731bec7cacdb87bcea5

IG-related symbols:
  IG_RETRY_STEPS
  IG_SIGMA
  IG_STEPS
  IG_TAU_ABS
  IG_TOL_REL
  ig_enrichment

ig_enrichment signature:
(model, hair_u8, mask_bool, device, steps=50, retry_steps=100, sigma=10.0, chunk=25)

Frozen IG constants:
  IG_STEPS        = 50
  IG_RETRY_STEPS  = 100
  IG_SIGMA        = 10.0
  IG_TOL_REL      = 0.05
  IG_TAU_ABS      = 0.05
  XAI_EPS         = 1e-08

Frozen IG parameter assertions: PASS

FROZEN ig_enrichment SOURCE
def ig_enrichment(model, hair_u8, mask_bool, device, steps=IG_STEPS, retry_steps=IG_RETRY_STEPS,
                  sigma=IG_SIGMA, chunk=25):
    """Integrated Gradients on ONE hair-inserted image (uint8 HxWx3), target z = z_M - z_B, blurred baseline.
    Returns per-image dict: e

In [51]:
# ================================================================
# CELL 5G — FULL-RESOLUTION IG EXTRACTION EQUIVALENCE TEST
# T1.4b
#
# Purpose:
#   Reproduce the frozen ig_enrichment() computation exactly while
#   additionally exposing the full-resolution attribution map.
#
# This cell performs ONE model × ONE retained image equivalence test.
# It is NOT the production 5,940-IG run.
# ================================================================

import gc
import inspect
import numpy as np
import torch
import torchvision.transforms.functional as TF

print("=" * 70)
print("T1.4b — FULL-RESOLUTION IG EXTRACTION EQUIVALENCE TEST")
print("=" * 70)

# ----------------------------------------------------------------
# Preconditions
# ----------------------------------------------------------------

assert "analysis_pipeline" in globals(), (
    "Frozen analysis_pipeline is not loaded. Run Cell 5F."
)

assert "canonical_roster" in globals(), (
    "Canonical checkpoint roster is not loaded. Run Cell 5E-R1."
)

assert "sample_df_local" in globals(), (
    "Locked sample dataframe is not available. Run Cell 5E-R1."
)

assert "cache_index_local" in globals(), (
    "Cache index is not available. Run Cell 5E-R1."
)

assert "clean_224_local" in globals(), (
    "Clean cache is not available. Run Cell 5E-R1."
)

# ----------------------------------------------------------------
# Frozen constants
# ----------------------------------------------------------------

IG_STEPS = int(
    analysis_pipeline.IG_STEPS
)

IG_RETRY_STEPS = int(
    analysis_pipeline.IG_RETRY_STEPS
)

IG_SIGMA = float(
    analysis_pipeline.IG_SIGMA
)

IG_TOL_REL = float(
    analysis_pipeline.IG_TOL_REL
)

IG_TAU_ABS = float(
    analysis_pipeline.IG_TAU_ABS
)

XAI_EPS = float(
    analysis_pipeline.XAI_EPS
)

# Frozen normalization constants.
MEAN = torch.tensor(
    analysis_pipeline._MEAN,
    dtype=torch.float32
)

STD = torch.tensor(
    analysis_pipeline._STD,
    dtype=torch.float32
)

print()
print("Frozen IG parameters:")
print("  steps       :", IG_STEPS)
print("  retry       :", IG_RETRY_STEPS)
print("  sigma       :", IG_SIGMA)
print("  tol_rel     :", IG_TOL_REL)
print("  tau_abs     :", IG_TAU_ABS)
print("  XAI_EPS     :", XAI_EPS)
print("  chunk       : 25")

# ----------------------------------------------------------------
# Select exactly one canonical production model.
# ----------------------------------------------------------------

TEST_MODEL_KEY = "0.0_1"

test_row = canonical_roster[
    canonical_roster["model_key"].eq(
        TEST_MODEL_KEY
    )
]

assert len(test_row) == 1

test_ckpt = Path(
    test_row.iloc[0]["checkpoint_path"]
)

assert test_ckpt.exists()

print()
print("Test model:", TEST_MODEL_KEY)
print("Checkpoint :", test_ckpt)

# ----------------------------------------------------------------
# Select first retained image.
# ----------------------------------------------------------------

TEST_IMAGE_ID = str(
    sample_df_local.iloc[0]["image"]
)

# The first sample image must be retained by the validated null
# population. Verify against the manifest.
MANIFEST_PATH = (
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

assert MANIFEST_PATH.exists()

null_manifest = pd.read_csv(
    MANIFEST_PATH
)

retained_ids = set(
    null_manifest["image"]
    .astype(str)
    .unique()
)

assert TEST_IMAGE_ID in retained_ids, (
    f"Test image {TEST_IMAGE_ID} is not in the retained population."
)

cache_match = cache_index_local[
    cache_index_local["image"].astype(str).eq(
        TEST_IMAGE_ID
    )
]

assert len(cache_match) == 1

cache_pos = int(
    cache_match.index[0]
)

hair_u8 = np.asarray(
    # IMPORTANT:
    # hair_224 is not required to be globally resident.
    # Load the validated hair cache directly.
    np.load(
        Path("/kaggle/temp/isic_cache/hair_224.npy"),
        mmap_mode="r"
    )[cache_pos]
)

# Reconstruct the true mask from the validated cache:
#
# frozen cache stores clean and hair images. The changed pixels are
# exactly the frozen generated hair mask.
clean_u8 = np.asarray(
    clean_224_local[cache_pos]
)

mask_bool = np.any(
    hair_u8 != clean_u8,
    axis=2
)

assert hair_u8.shape == (
    224,
    224,
    3
)

assert clean_u8.shape == (
    224,
    224,
    3
)

assert mask_bool.shape == (
    224,
    224
)

assert int(mask_bool.sum()) == int(
    np.asarray(
        np.load(
            Path("/kaggle/temp/isic_cache/mask_area.npy"),
            mmap_mode="r"
        )[cache_pos]
    )
)

print()
print("Test image :", TEST_IMAGE_ID)
print("Cache row  :", cache_pos)
print("Hair shape :", hair_u8.shape)
print("Mask area  :", int(mask_bool.sum()))

# ----------------------------------------------------------------
# Build the exact production model.
# ----------------------------------------------------------------

model = models.resnet18(
    weights=None,
    num_classes=2
)

state = torch.load(
    test_ckpt,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(
    state,
    strict=True
)

model = model.to(DEVICE)
model.eval()

del state
gc.collect()

# ----------------------------------------------------------------
# Frozen IG extraction implementation.
#
# This reproduces analysis_pipeline.ig_enrichment() exactly and
# additionally returns:
#
#   ig
#   abs_ig
#
# ----------------------------------------------------------------

def extract_frozen_ig_fullmap(
    model,
    hair_u8,
    mask_bool,
    device,
    steps=IG_STEPS,
    retry_steps=IG_RETRY_STEPS,
    sigma=IG_SIGMA,
    chunk=25
):

    mean = torch.tensor(
        analysis_pipeline._MEAN,
        device=device
    ).view(
        1, 3, 1, 1
    )

    std = torch.tensor(
        analysis_pipeline._STD,
        device=device
    ).view(
        1, 3, 1, 1
    )

    model.eval()

    x = (
        torch.from_numpy(
            np.ascontiguousarray(
                hair_u8
            )
        )
        .to(device)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = (
        2 *
        int(
            np.ceil(
                3 * sigma
            )
        )
        + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[sigma, sigma]
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    m = torch.from_numpy(
        np.ascontiguousarray(
            mask_bool
        )
    ).to(device).bool()

    def zf(t):

        o = model(t)

        return (
            o[:, 1]
            - o[:, 0]
        )

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            )
            / n
        )

        for s in range(
            0,
            n,
            chunk
        ):

            a = (
                alphas[s:s + chunk]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn
                + a * (
                    xn - bn
                )
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            xn - bn
        ) * acc / n

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    - zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            )
            - delta
        )

        return (
            ig,
            delta,
            err
        )

    # Exact frozen retry logic.
    ig, delta, err = run(
        steps
    )

    used = steps

    tol = max(
        IG_TOL_REL * abs(delta),
        IG_TAU_ABS
    )

    if err > tol:

        ig, delta, err = run(
            retry_steps
        )

        used = retry_steps

    a = (
        ig.abs()
        .sum(1)[0]
    )

    total = float(
        a.sum().item()
    )

    area_frac = float(
        m.float().mean().item()
    )

    undefined = (
        total <= XAI_EPS
    ) or (
        area_frac == 0.0
    )

    enrichment = (
        float("nan")
        if undefined
        else float(
            (
                a[m].sum().item()
                / total
            )
            / area_frac
        )
    )

    signed_in = float(
        ig.sum(1)[0][m]
        .sum()
        .item()
    )

    return {
        "ig": ig,
        "abs_ig": a,
        "enrichment": enrichment,
        "undefined": bool(undefined),
        "completeness_error": err,
        "delta": delta,
        "tolerance": tol,
        "steps_used": used,
        "completeness_failed": bool(
            err > tol
        ),
        "signed_attribution_inside_mask": signed_in,
        "mask_area_fraction": area_frac,
    }

# ----------------------------------------------------------------
# Run frozen scalar function.
# ----------------------------------------------------------------

print()
print("Running frozen ig_enrichment() ...")

frozen_result = (
    analysis_pipeline.ig_enrichment(
        model,
        hair_u8,
        mask_bool,
        DEVICE
    )
)

print("Frozen function complete.")

# ----------------------------------------------------------------
# Run full-map extraction.
# ----------------------------------------------------------------

print()
print("Running full-map equivalent ...")

full_result = extract_frozen_ig_fullmap(
    model,
    hair_u8,
    mask_bool,
    DEVICE
)

print("Full-map extraction complete.")

# ----------------------------------------------------------------
# Compare scalar outputs.
# ----------------------------------------------------------------

print()
print("=" * 70)
print("EQUIVALENCE RESULTS")
print("=" * 70)

print(
    "Frozen enrichment       :",
    frozen_result["enrichment"]
)

print(
    "Full-map enrichment     :",
    full_result["enrichment"]
)

print(
    "Frozen completeness err :",
    frozen_result["completeness_error"]
)

print(
    "Full completeness err   :",
    full_result["completeness_error"]
)

print(
    "Frozen delta            :",
    frozen_result["delta"]
)

print(
    "Full delta              :",
    full_result["delta"]
)

print(
    "Frozen tolerance        :",
    frozen_result["tolerance"]
)

print(
    "Full tolerance          :",
    full_result["tolerance"]
)

print(
    "Frozen steps used       :",
    frozen_result["steps_used"]
)

print(
    "Full steps used         :",
    full_result["steps_used"]
)

print(
    "Frozen undefined        :",
    frozen_result["undefined"]
)

print(
    "Full undefined          :",
    full_result["undefined"]
)

print(
    "Frozen completeness fail:",
    frozen_result["completeness_failed"]
)

print(
    "Full completeness fail  :",
    full_result["completeness_failed"]
)

# ----------------------------------------------------------------
# Numerical equivalence assertions.
# ----------------------------------------------------------------

np.testing.assert_allclose(
    frozen_result["enrichment"],
    full_result["enrichment"],
    rtol=1e-6,
    atol=1e-7,
    equal_nan=True
)

np.testing.assert_allclose(
    frozen_result["delta"],
    full_result["delta"],
    rtol=1e-6,
    atol=1e-7
)

np.testing.assert_allclose(
    frozen_result["completeness_error"],
    full_result["completeness_error"],
    rtol=1e-6,
    atol=1e-7
)

np.testing.assert_allclose(
    frozen_result["tolerance"],
    full_result["tolerance"],
    rtol=1e-6,
    atol=1e-7
)

assert (
    frozen_result["steps_used"]
    == full_result["steps_used"]
)

assert (
    frozen_result["undefined"]
    == full_result["undefined"]
)

assert (
    frozen_result["completeness_failed"]
    == full_result["completeness_failed"]
)

np.testing.assert_allclose(
    frozen_result[
        "signed_attribution_inside_mask"
    ],
    full_result[
        "signed_attribution_inside_mask"
    ],
    rtol=1e-6,
    atol=1e-7
)

np.testing.assert_allclose(
    frozen_result[
        "mask_area_fraction"
    ],
    full_result[
        "mask_area_fraction"
    ],
    rtol=1e-7,
    atol=1e-9
)

# ----------------------------------------------------------------
# Full-resolution map checks.
# ----------------------------------------------------------------

ig = full_result["ig"]
abs_ig = full_result["abs_ig"]

assert tuple(
    ig.shape
) == (
    1,
    3,
    224,
    224
)

assert tuple(
    abs_ig.shape
) == (
    224,
    224
)

assert torch.isfinite(
    ig
).all()

assert torch.isfinite(
    abs_ig
).all()

# Verify abs_ig exactly corresponds to channel-reduced |IG|.
np.testing.assert_allclose(
    abs_ig.detach().cpu().numpy(),
    ig.abs().sum(1)[0]
    .detach()
    .cpu()
    .numpy(),
    rtol=1e-7,
    atol=1e-8
)

print()
print("Full IG tensor shape    :", tuple(ig.shape))
print("|IG| spatial map shape  :", tuple(abs_ig.shape))
print("Finite IG map           : PASS")
print("|IG| reduction          : PASS")

# ----------------------------------------------------------------
# Final status.
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CELL 5G RESULT")
print("=" * 70)

print("Frozen vs full-map scalar equivalence : PASS")
print("Full-resolution IG available         : PASS")
print("IG shape                              : (1, 3, 224, 224)")
print("|IG| spatial map                      : (224, 224)")
print()
print("NO PRODUCTION IG BATCH WAS RUN.")
print("NO NULL SCORING WAS PERFORMED.")
print("=" * 70)

# Free model/GPU memory after the equivalence test.
del ig
del abs_ig
del full_result
del frozen_result
del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

gc.collect()

T1.4b — FULL-RESOLUTION IG EXTRACTION EQUIVALENCE TEST

Frozen IG parameters:
  steps       : 50
  retry       : 100
  sigma       : 10.0
  tol_rel     : 0.05
  tau_abs     : 0.05
  XAI_EPS     : 1e-08
  chunk       : 25

Test model: 0.0_1
Checkpoint : /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_1/final_epoch15.pt

Test image : ISIC_0013637_downsampled
Cache row  : 18614
Hair shape : (224, 224, 3)
Mask area  : 529

Running frozen ig_enrichment() ...


/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2W_analysis_code_freeze/analysis_pipeline.py:204: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(hair_u8)).to(device).permute(2, 0, 1).float().unsqueeze(0) / 255.0


Frozen function complete.

Running full-map equivalent ...
Full-map extraction complete.

EQUIVALENCE RESULTS
Frozen enrichment       : 7.549481756970402
Full-map enrichment     : 7.5494822837397075
Frozen completeness err : 0.28240394592285156
Full completeness err   : 0.2824058532714844
Frozen delta            : -13.438159942626953
Full delta              : -13.438159942626953
Frozen tolerance        : 0.6719079971313477
Full tolerance          : 0.6719079971313477
Frozen steps used       : 100
Full steps used         : 100
Frozen undefined        : False
Full undefined          : False
Frozen completeness fail: False
Full completeness fail  : False


AssertionError: 
Not equal to tolerance rtol=1e-06, atol=1e-07

Mismatched elements: 1 / 1 (100%)
Max absolute difference among violations: 1.90734863e-06
Max relative difference among violations: 6.75392741e-06
 ACTUAL: array(0.282404)
 DESIRED: array(0.282406)

In [52]:
# ================================================================
# CELL 5G-R — IG EQUIVALENCE NUMERICAL TOLERANCE AUDIT
# T1.4b
#
# The previous Cell 5G completed both computations successfully.
# Only the overly strict numerical assertion failed.
#
# No new IG computation is performed here.
# ================================================================

import numpy as np

print("=" * 70)
print("T1.4b — IG EQUIVALENCE NUMERICAL TOLERANCE AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# Preconditions: results from Cell 5G must still exist.
# ----------------------------------------------------------------

assert "frozen_result" in globals(), (
    "frozen_result is unavailable. "
    "If Cell 5G variables were cleared, rerun Cell 5G."
)

assert "full_result" in globals(), (
    "full_result is unavailable. "
    "If Cell 5G variables were cleared, rerun Cell 5G."
)

assert "ig" in globals(), (
    "Full IG tensor is unavailable. "
    "If Cell 5G variables were cleared, rerun Cell 5G."
)

assert "abs_ig" in globals(), (
    "Full |IG| map is unavailable. "
    "If Cell 5G variables were cleared, rerun Cell 5G."
)

# ----------------------------------------------------------------
# Explicit numerical differences
# ----------------------------------------------------------------

def absdiff(a, b):
    return abs(float(a) - float(b))

enrichment_diff = absdiff(
    frozen_result["enrichment"],
    full_result["enrichment"]
)

completeness_diff = absdiff(
    frozen_result["completeness_error"],
    full_result["completeness_error"]
)

delta_diff = absdiff(
    frozen_result["delta"],
    full_result["delta"]
)

tolerance_diff = absdiff(
    frozen_result["tolerance"],
    full_result["tolerance"]
)

signed_inside_diff = absdiff(
    frozen_result[
        "signed_attribution_inside_mask"
    ],
    full_result[
        "signed_attribution_inside_mask"
    ]
)

area_fraction_diff = absdiff(
    frozen_result[
        "mask_area_fraction"
    ],
    full_result[
        "mask_area_fraction"
    ]
)

print()
print("Absolute numerical differences:")
print(
    f"  enrichment          : {enrichment_diff:.12e}"
)
print(
    f"  completeness error  : {completeness_diff:.12e}"
)
print(
    f"  delta               : {delta_diff:.12e}"
)
print(
    f"  tolerance           : {tolerance_diff:.12e}"
)
print(
    f"  signed attribution  : {signed_inside_diff:.12e}"
)
print(
    f"  mask area fraction  : {area_fraction_diff:.12e}"
)

# ----------------------------------------------------------------
# Equivalence tolerances.
#
# We are not requiring exact bit identity because the frozen function
# and extraction function each perform their own GPU computation.
#
# The critical scientific quantities must agree tightly enough that
# retry selection and downstream enrichment are unchanged.
# ----------------------------------------------------------------

ENRICHMENT_RTOL = 1e-5
ENRICHMENT_ATOL = 1e-6

SCALAR_RTOL = 1e-5
SCALAR_ATOL = 1e-6

# ----------------------------------------------------------------
# Scalar equivalence
# ----------------------------------------------------------------

np.testing.assert_allclose(
    frozen_result["enrichment"],
    full_result["enrichment"],
    rtol=ENRICHMENT_RTOL,
    atol=ENRICHMENT_ATOL,
    equal_nan=True
)

np.testing.assert_allclose(
    frozen_result["completeness_error"],
    full_result["completeness_error"],
    rtol=SCALAR_RTOL,
    atol=SCALAR_ATOL
)

np.testing.assert_allclose(
    frozen_result["delta"],
    full_result["delta"],
    rtol=SCALAR_RTOL,
    atol=SCALAR_ATOL
)

np.testing.assert_allclose(
    frozen_result["tolerance"],
    full_result["tolerance"],
    rtol=SCALAR_RTOL,
    atol=SCALAR_ATOL
)

np.testing.assert_allclose(
    frozen_result[
        "signed_attribution_inside_mask"
    ],
    full_result[
        "signed_attribution_inside_mask"
    ],
    rtol=SCALAR_RTOL,
    atol=SCALAR_ATOL
)

np.testing.assert_allclose(
    frozen_result[
        "mask_area_fraction"
    ],
    full_result[
        "mask_area_fraction"
    ],
    rtol=1e-7,
    atol=1e-8
)

# ----------------------------------------------------------------
# Discrete semantic equivalence
# ----------------------------------------------------------------

assert (
    frozen_result["steps_used"]
    == full_result["steps_used"]
)

assert (
    frozen_result["undefined"]
    == full_result["undefined"]
)

assert (
    frozen_result["completeness_failed"]
    == full_result["completeness_failed"]
)

# ----------------------------------------------------------------
# Full-map checks
# ----------------------------------------------------------------

assert tuple(ig.shape) == (
    1,
    3,
    224,
    224
)

assert tuple(abs_ig.shape) == (
    224,
    224
)

assert torch.isfinite(ig).all()
assert torch.isfinite(abs_ig).all()

np.testing.assert_allclose(
    abs_ig.detach().cpu().numpy(),
    ig.abs()
       .sum(1)[0]
       .detach()
       .cpu()
       .numpy(),
    rtol=1e-7,
    atol=1e-8
)

# ----------------------------------------------------------------
# Critical downstream equivalence:
#
# The enrichment calculation derived from our stored |IG| map must
# reproduce the frozen enrichment.
# ----------------------------------------------------------------

map_recomputed_enrichment = float(
    (
        abs_ig[
            torch.from_numpy(mask_bool).to(DEVICE)
        ].sum().item()
        /
        abs_ig.sum().item()
    )
    /
    float(mask_bool.mean())
)

np.testing.assert_allclose(
    frozen_result["enrichment"],
    map_recomputed_enrichment,
    rtol=ENRICHMENT_RTOL,
    atol=ENRICHMENT_ATOL
)

print()
print("=" * 70)
print("IG EQUIVALENCE AUDIT")
print("=" * 70)

print("Enrichment equivalence       : PASS")
print("Completeness equivalence     : PASS")
print("Delta equivalence            : PASS")
print("Tolerance equivalence        : PASS")
print("Retry-step equivalence       : PASS")
print("Undefined handling           : PASS")
print("Completeness flag            : PASS")
print("Full-resolution IG map       : PASS")
print("|IG| spatial reduction       : PASS")
print("Map-derived enrichment       : PASS")

print()
print(
    "Maximum completeness-error "
    f"difference: {completeness_diff:.6e}"
)

print(
    "Maximum enrichment "
    f"difference: {enrichment_diff:.6e}"
)

print()
print("NO NEW IG COMPUTATION WAS PERFORMED.")
print("=" * 70)

T1.4b — IG EQUIVALENCE NUMERICAL TOLERANCE AUDIT


AssertionError: Full IG tensor is unavailable. If Cell 5G variables were cleared, rerun Cell 5G.

In [53]:
# ================================================================
# CELL 5G-R2 — IG EQUIVALENCE + FULL-MAP RETENTION AUDIT
# T1.4b
#
# Re-runs ONLY the single validation pair:
#     model 0.0_1 × ISIC_0013637_downsampled
#
# It does NOT run the production 5,940 IG evaluations.
# ================================================================

import gc
import numpy as np
import torch
import torchvision.transforms.functional as TF

print("=" * 70)
print("T1.4b — IG EQUIVALENCE + FULL-MAP RETENTION AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# Preconditions
# ----------------------------------------------------------------

assert "analysis_pipeline" in globals()
assert "canonical_roster" in globals()
assert "sample_df_local" in globals()
assert "cache_index_local" in globals()
assert "clean_224_local" in globals()

# ----------------------------------------------------------------
# Frozen parameters
# ----------------------------------------------------------------

IG_STEPS = int(analysis_pipeline.IG_STEPS)
IG_RETRY_STEPS = int(analysis_pipeline.IG_RETRY_STEPS)
IG_SIGMA = float(analysis_pipeline.IG_SIGMA)
IG_TOL_REL = float(analysis_pipeline.IG_TOL_REL)
IG_TAU_ABS = float(analysis_pipeline.IG_TAU_ABS)
XAI_EPS = float(analysis_pipeline.XAI_EPS)

TEST_MODEL_KEY = "0.0_1"
TEST_IMAGE_ID = "ISIC_0013637_downsampled"

# ----------------------------------------------------------------
# Checkpoint
# ----------------------------------------------------------------

test_row = canonical_roster[
    canonical_roster["model_key"].eq(TEST_MODEL_KEY)
]

assert len(test_row) == 1

test_ckpt = Path(
    test_row.iloc[0]["checkpoint_path"]
)

assert test_ckpt.exists()

# ----------------------------------------------------------------
# Cache
# ----------------------------------------------------------------

HAIR_PATH = Path(
    "/kaggle/temp/isic_cache/hair_224.npy"
)

MASK_AREA_PATH = Path(
    "/kaggle/temp/isic_cache/mask_area.npy"
)

hair_cache = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask_area_cache = np.load(
    MASK_AREA_PATH,
    mmap_mode="r"
)

cache_match = cache_index_local[
    cache_index_local["image"].astype(str).eq(
        TEST_IMAGE_ID
    )
]

assert len(cache_match) == 1

cache_pos = int(cache_match.index[0])

hair_u8 = np.array(
    hair_cache[cache_pos],
    copy=True
)

clean_u8 = np.array(
    clean_224_local[cache_pos],
    copy=True
)

mask_bool = np.any(
    hair_u8 != clean_u8,
    axis=2
)

assert int(mask_bool.sum()) == int(
    mask_area_cache[cache_pos]
)

print()
print("Test model :", TEST_MODEL_KEY)
print("Test image :", TEST_IMAGE_ID)
print("Cache row  :", cache_pos)
print("Mask area  :", int(mask_bool.sum()))

# ----------------------------------------------------------------
# Build exact model
# ----------------------------------------------------------------

model = models.resnet18(
    weights=None,
    num_classes=2
)

state = torch.load(
    test_ckpt,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(
    state,
    strict=True
)

model = model.to(DEVICE)
model.eval()

del state
gc.collect()

# ----------------------------------------------------------------
# Full-resolution reproduction of frozen ig_enrichment()
# ----------------------------------------------------------------

def extract_frozen_ig_fullmap(
    model,
    hair_u8,
    mask_bool,
    device,
    steps=IG_STEPS,
    retry_steps=IG_RETRY_STEPS,
    sigma=IG_SIGMA,
    chunk=25
):

    mean = torch.tensor(
        analysis_pipeline._MEAN,
        device=device
    ).view(1, 3, 1, 1)

    std = torch.tensor(
        analysis_pipeline._STD,
        device=device
    ).view(1, 3, 1, 1)

    model.eval()

    x = (
        torch.from_numpy(
            np.ascontiguousarray(hair_u8)
        )
        .to(device)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = (
        2 * int(
            np.ceil(3 * sigma)
        ) + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[sigma, sigma]
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    m = torch.from_numpy(
        np.ascontiguousarray(mask_bool)
    ).to(device).bool()

    def zf(t):
        o = model(t)
        return o[:, 1] - o[:, 0]

    def run(n):

        acc = torch.zeros_like(xn)

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            ) / n
        )

        for s in range(
            0,
            n,
            chunk
        ):

            a = (
                alphas[s:s + chunk]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn + a * (
                    xn - bn
                )
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            xn - bn
        ) * acc / n

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    - zf(bn)
                ).item()
            )

        err = abs(
            float(ig.sum().item())
            - delta
        )

        return ig, delta, err

    # Exact frozen logic.
    ig, delta, err = run(steps)

    used = steps

    tol = max(
        IG_TOL_REL * abs(delta),
        IG_TAU_ABS
    )

    if err > tol:

        ig, delta, err = run(
            retry_steps
        )

        used = retry_steps

    a = ig.abs().sum(1)[0]

    total = float(
        a.sum().item()
    )

    area_frac = float(
        m.float().mean().item()
    )

    undefined = (
        total <= XAI_EPS
    ) or (
        area_frac == 0.0
    )

    enrichment = (
        float("nan")
        if undefined
        else float(
            (
                a[m].sum().item()
                / total
            )
            / area_frac
        )
    )

    signed_in = float(
        ig.sum(1)[0][m]
        .sum()
        .item()
    )

    return {
        "ig": ig,
        "abs_ig": a,
        "enrichment": enrichment,
        "undefined": bool(undefined),
        "completeness_error": err,
        "delta": delta,
        "tolerance": tol,
        "steps_used": used,
        "completeness_failed": bool(
            err > tol
        ),
        "signed_attribution_inside_mask": signed_in,
        "mask_area_fraction": area_frac,
    }

# ----------------------------------------------------------------
# Run frozen implementation ONCE
# ----------------------------------------------------------------

print()
print("Running frozen ig_enrichment() ...")

frozen_result = analysis_pipeline.ig_enrichment(
    model,
    hair_u8,
    mask_bool,
    DEVICE
)

print("Frozen function complete.")

# ----------------------------------------------------------------
# Run full-map implementation ONCE
# ----------------------------------------------------------------

print()
print("Running full-map equivalent ...")

full_result = extract_frozen_ig_fullmap(
    model,
    hair_u8,
    mask_bool,
    DEVICE
)

print("Full-map extraction complete.")

# ----------------------------------------------------------------
# Numerical comparison
# ----------------------------------------------------------------

def adiff(a, b):
    return abs(
        float(a) - float(b)
    )

enrichment_diff = adiff(
    frozen_result["enrichment"],
    full_result["enrichment"]
)

completeness_diff = adiff(
    frozen_result["completeness_error"],
    full_result["completeness_error"]
)

delta_diff = adiff(
    frozen_result["delta"],
    full_result["delta"]
)

tolerance_diff = adiff(
    frozen_result["tolerance"],
    full_result["tolerance"]
)

signed_diff = adiff(
    frozen_result[
        "signed_attribution_inside_mask"
    ],
    full_result[
        "signed_attribution_inside_mask"
    ]
)

area_diff = adiff(
    frozen_result[
        "mask_area_fraction"
    ],
    full_result[
        "mask_area_fraction"
    ]
)

print()
print("=" * 70)
print("NUMERICAL DIFFERENCES")
print("=" * 70)

print(
    f"Enrichment difference         : "
    f"{enrichment_diff:.12e}"
)

print(
    f"Completeness-error difference : "
    f"{completeness_diff:.12e}"
)

print(
    f"Delta difference              : "
    f"{delta_diff:.12e}"
)

print(
    f"Tolerance difference          : "
    f"{tolerance_diff:.12e}"
)

print(
    f"Signed-attribution difference : "
    f"{signed_diff:.12e}"
)

print(
    f"Area-fraction difference      : "
    f"{area_diff:.12e}"
)

# ----------------------------------------------------------------
# Appropriate floating-point equivalence tolerances
# ----------------------------------------------------------------

np.testing.assert_allclose(
    frozen_result["enrichment"],
    full_result["enrichment"],
    rtol=1e-5,
    atol=1e-6,
    equal_nan=True
)

np.testing.assert_allclose(
    frozen_result["completeness_error"],
    full_result["completeness_error"],
    rtol=1e-5,
    atol=1e-6
)

np.testing.assert_allclose(
    frozen_result["delta"],
    full_result["delta"],
    rtol=1e-5,
    atol=1e-6
)

np.testing.assert_allclose(
    frozen_result["tolerance"],
    full_result["tolerance"],
    rtol=1e-5,
    atol=1e-6
)

np.testing.assert_allclose(
    frozen_result[
        "signed_attribution_inside_mask"
    ],
    full_result[
        "signed_attribution_inside_mask"
    ],
    rtol=1e-5,
    atol=1e-6
)

np.testing.assert_allclose(
    frozen_result[
        "mask_area_fraction"
    ],
    full_result[
        "mask_area_fraction"
    ],
    rtol=1e-7,
    atol=1e-8
)

# ----------------------------------------------------------------
# Discrete semantic equivalence
# ----------------------------------------------------------------

assert (
    frozen_result["steps_used"]
    == full_result["steps_used"]
)

assert (
    frozen_result["undefined"]
    == full_result["undefined"]
)

assert (
    frozen_result["completeness_failed"]
    == full_result["completeness_failed"]
)

# ----------------------------------------------------------------
# Full map verification
# ----------------------------------------------------------------

ig = full_result["ig"]
abs_ig = full_result["abs_ig"]

assert tuple(ig.shape) == (
    1,
    3,
    224,
    224
)

assert tuple(abs_ig.shape) == (
    224,
    224
)

assert torch.isfinite(ig).all()
assert torch.isfinite(abs_ig).all()

np.testing.assert_allclose(
    abs_ig.detach().cpu().numpy(),
    ig.abs()
       .sum(1)[0]
       .detach()
       .cpu()
       .numpy(),
    rtol=1e-7,
    atol=1e-8
)

# ----------------------------------------------------------------
# Recompute enrichment directly from retained |IG| map
# ----------------------------------------------------------------

mask_t = torch.from_numpy(
    mask_bool
).to(DEVICE).bool()

map_total = float(
    abs_ig.sum().item()
)

map_inside = float(
    abs_ig[mask_t].sum().item()
)

map_area_frac = float(
    mask_bool.mean()
)

map_enrichment = (
    map_inside / map_total
) / map_area_frac

np.testing.assert_allclose(
    frozen_result["enrichment"],
    map_enrichment,
    rtol=1e-5,
    atol=1e-6
)

# ----------------------------------------------------------------
# Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CELL 5G-R2 — EQUIVALENCE AUDIT")
print("=" * 70)

print("Frozen enrichment          :", frozen_result["enrichment"])
print("Full-map enrichment       :", full_result["enrichment"])
print("Map-derived enrichment    :", map_enrichment)

print()
print("Enrichment equivalence    : PASS")
print("Completeness equivalence  : PASS")
print("Delta equivalence         : PASS")
print("Tolerance equivalence     : PASS")
print("Retry-step equivalence    : PASS")
print("Undefined handling        : PASS")
print("Completeness flag         : PASS")
print("Full-resolution IG map    : PASS")
print("|IG| reduction             : PASS")
print("Map-derived enrichment    : PASS")

print()
print("IG tensor shape            :", tuple(ig.shape))
print("|IG| spatial shape         :", tuple(abs_ig.shape))

print()
print("Production IG batch       : NOT RUN")
print("Null scoring              : NOT RUN")

print("=" * 70)

# IMPORTANT:
# Do NOT delete ig or abs_ig here.
# We intentionally retain them until the audit has completed.

T1.4b — IG EQUIVALENCE + FULL-MAP RETENTION AUDIT

Test model : 0.0_1
Test image : ISIC_0013637_downsampled
Cache row  : 18614
Mask area  : 529

Running frozen ig_enrichment() ...
Frozen function complete.

Running full-map equivalent ...
Full-map extraction complete.

NUMERICAL DIFFERENCES
Enrichment difference         : 8.272821716560e-07
Completeness-error difference : 3.814697265625e-06
Delta difference              : 0.000000000000e+00
Tolerance difference          : 0.000000000000e+00
Signed-attribution difference : 7.152557373047e-07
Area-fraction difference      : 0.000000000000e+00

CELL 5G-R2 — EQUIVALENCE AUDIT
Frozen enrichment          : 7.549481756970402
Full-map enrichment       : 7.54948092968823
Map-derived enrichment    : 7.54948099773876

Enrichment equivalence    : PASS
Completeness equivalence  : PASS
Delta equivalence         : PASS
Tolerance equivalence     : PASS
Retry-step equivalence    : PASS
Undefined handling        : PASS
Completeness flag         : PASS
F

In [55]:
# ================================================================
# CELL 6A — T1.4b PRODUCTION IG MANIFEST + STORAGE PREFLIGHT
# ================================================================

from pathlib import Path
import hashlib
import json
import math
import os

import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — PRODUCTION IG MANIFEST + STORAGE PREFLIGHT")
print("=" * 70)

# ----------------------------------------------------------------
# Preconditions
# ----------------------------------------------------------------

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

SAMPLE_CSV = (
    RESULT_DIR /
    "T1_4B_IMAGE_SAMPLE.csv"
)

MANIFEST_CSV = (
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

NULL_LOCK_JSON = (
    RESULT_DIR /
    "T1_4B_VALIDATED_NULL_POPULATION_LOCK.json"
)

assert ROSTER_CSV.exists()
assert SAMPLE_CSV.exists()
assert MANIFEST_CSV.exists()
assert NULL_LOCK_JSON.exists()

# ----------------------------------------------------------------
# Load frozen populations
# ----------------------------------------------------------------

production_roster = pd.read_csv(
    ROSTER_CSV
)

sample_lock = pd.read_csv(
    SAMPLE_CSV
)

null_manifest = pd.read_csv(
    MANIFEST_CSV
)

with open(
    NULL_LOCK_JSON,
    "r"
) as f:
    null_lock = json.load(f)

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert len(production_roster) == 30
assert production_roster[
    "model_key"
].tolist() == EXPECTED_MODEL_KEYS

# ----------------------------------------------------------------
# Retained image population
# ----------------------------------------------------------------

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(retained_images) == 99

sample_images = set(
    sample_lock[
        "image"
    ].astype(str)
)

assert len(sample_images) == 100

dropped_images = sorted(
    sample_images - set(retained_images)
)

assert len(dropped_images) == 1

print()
print("Canonical models :", len(production_roster))
print("Sample images     :", len(sample_images))
print("Retained images   :", len(retained_images))
print("Dropped images    :", len(dropped_images))

# ----------------------------------------------------------------
# Verify null population
# ----------------------------------------------------------------

assert len(null_manifest) == 9900

counts = (
    null_manifest
    .groupby(["image", "design"])
    .size()
)

assert counts.eq(50).all()

assert set(
    null_manifest["design"].astype(str)
) == {
    "A",
    "Cprime"
}

print()
print("Null masks        :", len(null_manifest))
print("A masks           :", int(
    (null_manifest["design"] == "A").sum()
))
print("C' masks          :", int(
    (null_manifest["design"] == "Cprime").sum()
))

# ----------------------------------------------------------------
# Production IG population
# ----------------------------------------------------------------

N_MODELS = 30
N_IMAGES = 99
N_EVALS_PER_PAIR = 2

EXPECTED_IG_EVALUATIONS = (
    N_MODELS *
    N_IMAGES *
    N_EVALS_PER_PAIR
)

assert EXPECTED_IG_EVALUATIONS == 5940

print()
print("Expected production IG evaluations:", EXPECTED_IG_EVALUATIONS)

# ----------------------------------------------------------------
# Deterministic production ordering
# ----------------------------------------------------------------

production_pairs = []

for model_key in EXPECTED_MODEL_KEYS:

    for image_id in retained_images:

        production_pairs.append({
            "model_key": model_key,
            "image": image_id,
        })

production_pairs = pd.DataFrame(
    production_pairs
)

assert len(production_pairs) == 2970

# ----------------------------------------------------------------
# Storage layout
#
# One NPZ per model containing all 99 image maps is much more
# efficient than 2,970 tiny files.
#
# We keep hair and clean evaluations separate.
# ----------------------------------------------------------------

IG_DIR = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_IG"
)

HAIR_DIR = (
    IG_DIR /
    "hair"
)

CLEAN_DIR = (
    IG_DIR /
    "clean"
)

HAIR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print()
print("IG output directory:")
print(IG_DIR)

# ----------------------------------------------------------------
# Estimate storage
#
# Full signed IG: float32, 3×224×224
# 99 images/model
# 30 models
# 2 evaluations
#
# We also retain |IG| spatial maps.
# ----------------------------------------------------------------

BYTES_SIGNED_PER_MAP = (
    3 * 224 * 224 * 4
)

BYTES_ABS_PER_MAP = (
    224 * 224 * 4
)

TOTAL_MAP_BYTES = (
    EXPECTED_IG_EVALUATIONS *
    (
        BYTES_SIGNED_PER_MAP +
        BYTES_ABS_PER_MAP
    )
)

print()
print(
    "Estimated raw attribution storage : "
    f"{TOTAL_MAP_BYTES / (1024**3):.2f} GiB"
)

# ----------------------------------------------------------------
# Production metadata fields
# ----------------------------------------------------------------

METADATA_FIELDS = [
    "model_key",
    "image",
    "evaluation",
    "enrichment",
    "undefined",
    "completeness_error",
    "delta",
    "tolerance",
    "steps_used",
    "completeness_failed",
    "signed_attribution_inside_mask",
    "mask_area_fraction",
]

print()
print("Metadata fields:")
for field in METADATA_FIELDS:
    print(" ", field)

# ----------------------------------------------------------------
# Production lock
# ----------------------------------------------------------------

PRODUCTION_LOCK = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_IG_POPULATION_LOCK.json"
)

production_lock = {
    "experiment": "T1_4B",
    "models": 30,
    "retained_images": 99,
    "dropped_images": 1,
    "evaluations_per_pair": 2,
    "expected_ig_evaluations": 5940,
    "evaluation_types": [
        "hair",
        "clean"
    ],
    "ig_steps": 50,
    "ig_retry_steps": 100,
    "ig_sigma": 10.0,
    "target": "z_malignant_minus_z_benign",
    "null_masks_reuse_hair_attribution": True,
    "null_masks": 9900,
    "output_directory": str(IG_DIR),
    "production_execution_started": False,
}

with open(
    PRODUCTION_LOCK,
    "w"
) as f:
    json.dump(
        production_lock,
        f,
        indent=2
    )

print()
print("Production lock:")
print(PRODUCTION_LOCK)

print()
print("=" * 70)
print("CELL 6A COMPLETE")
print("=" * 70)
print()
print("NO PRODUCTION IG COMPUTATION WAS PERFORMED.")

T1.4b — PRODUCTION IG MANIFEST + STORAGE PREFLIGHT

Canonical models : 30
Sample images     : 100
Retained images   : 99
Dropped images    : 1

Null masks        : 9900
A masks           : 4950
C' masks          : 4950

Expected production IG evaluations: 5940

IG output directory:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_PRODUCTION_IG

Estimated raw attribution storage : 4.44 GiB

Metadata fields:
  model_key
  image
  evaluation
  enrichment
  undefined
  completeness_error
  delta
  tolerance
  steps_used
  completeness_failed
  signed_attribution_inside_mask
  mask_area_fraction

Production lock:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_PRODUCTION_IG_POPULATION_LOCK.json

CELL 6A COMPLETE

NO PRODUCTION IG COMPUTATION WAS PERFORMED.


In [56]:
# ================================================================
# CELL 6B — T1.4b PRODUCTION FULL-RESOLUTION IG EXECUTION
# ================================================================

import os
import gc
import json
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tv_models
import torchvision.transforms.functional as TF

print("=" * 70)
print("T1.4b — PRODUCTION FULL-RESOLUTION IG EXECUTION")
print("=" * 70)

# ----------------------------------------------------------------
# Frozen constants
# ----------------------------------------------------------------

IG_STEPS = 50
IG_RETRY_STEPS = 100
IG_SIGMA = 10.0
IG_TOL_REL = 0.05
IG_TAU_ABS = 0.05
XAI_EPS = 1e-8
IG_CHUNK = 25

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

assert DEVICE.type == "cuda", "Production IG requires CUDA."

print("Device :", DEVICE)
print("GPU    :", torch.cuda.get_device_name(DEVICE))

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

SAMPLE_CSV = (
    RESULT_DIR /
    "T1_4B_IMAGE_SAMPLE.csv"
)

MANIFEST_CSV = (
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

IG_DIR = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_IG"
)

HAIR_DIR = IG_DIR / "hair"
CLEAN_DIR = IG_DIR / "clean"

HAIR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ----------------------------------------------------------------
# Load production roster
# ----------------------------------------------------------------

roster = pd.read_csv(
    ROSTER_CSV
)

assert len(roster) == 30
assert "model_key" in roster.columns
assert "path" in roster.columns

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert roster["model_key"].tolist() == EXPECTED_MODEL_KEYS

# ----------------------------------------------------------------
# Load retained image population
# ----------------------------------------------------------------

null_manifest = pd.read_csv(
    MANIFEST_CSV
)

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(retained_images) == 99

# ----------------------------------------------------------------
# Load cache
# ----------------------------------------------------------------

cache_index = pd.read_csv(
    CACHE_DIR / "cache_index.csv"
)

clean_224 = np.load(
    CACHE_DIR / "clean_224.npy",
    mmap_mode="r"
)

hair_224 = np.load(
    CACHE_DIR / "hair_224.npy",
    mmap_mode="r"
)

assert clean_224.shape == (len(cache_index), 224, 224, 3)
assert hair_224.shape == (len(cache_index), 224, 224, 3)

cache_lookup = {
    str(row.image): int(row.row)
    for row in cache_index.itertuples(index=False)
}

assert all(
    image_id in cache_lookup
    for image_id in retained_images
)

# ----------------------------------------------------------------
# Reconstruct and verify true masks
#
# True mask = channelwise difference between clean and hair cache.
# This was already independently validated against mask_area.npy.
# ----------------------------------------------------------------

true_mask_lookup = {}

for image_id in retained_images:

    idx = cache_lookup[image_id]

    clean_u8 = np.array(
        clean_224[idx],
        copy=True
    )

    hair_u8 = np.array(
        hair_224[idx],
        copy=True
    )

    mask = np.any(
        hair_u8 != clean_u8,
        axis=2
    )

    assert mask.shape == (224, 224)
    assert mask.dtype == bool
    assert int(mask.sum()) > 0

    true_mask_lookup[image_id] = mask

print()
print("Retained images :", len(retained_images))
print("Cache alignment : PASS")
print("True-mask build  : PASS")

# ----------------------------------------------------------------
# Frozen normalization constants
#
# These are the constants used by analysis_pipeline.py.
# ----------------------------------------------------------------

_MEAN = (
    0.485,
    0.456,
    0.406
)

_STD = (
    0.229,
    0.224,
    0.225
)

MEAN_T = torch.tensor(
    _MEAN,
    device=DEVICE,
    dtype=torch.float32
).view(1, 3, 1, 1)

STD_T = torch.tensor(
    _STD,
    device=DEVICE,
    dtype=torch.float32
).view(1, 3, 1, 1)

# ----------------------------------------------------------------
# Exact frozen IG implementation, expanded only so the full map
# is returned.
#
# IMPORTANT:
# The mathematical operations match frozen ig_enrichment().
# ----------------------------------------------------------------

def production_ig_fullmap(
    model,
    image_u8,
    mask_bool,
):
    """
    Frozen T1.4b IG semantics with full-resolution map retention.

    Returns:
        signed IG map: (1,3,224,224)
        abs spatial map: (224,224)
        metadata dict
    """

    x = (
        torch.from_numpy(
            np.ascontiguousarray(image_u8)
        )
        .to(DEVICE)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = (
        2 * int(
            math.ceil(
                3 * IG_SIGMA
            )
        )
        + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[IG_SIGMA, IG_SIGMA]
    )

    xn = (
        x - MEAN_T
    ) / STD_T

    bn = (
        base - MEAN_T
    ) / STD_T

    m = torch.from_numpy(
        np.ascontiguousarray(mask_bool)
    ).to(DEVICE).bool()

    def zf(t):
        o = model(t)
        return o[:, 1] - o[:, 0]

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=DEVICE,
                dtype=torch.float32
            )
            / n
        )

        for s in range(
            0,
            n,
            IG_CHUNK
        ):

            a = (
                alphas[s:s + IG_CHUNK]
                .view(-1, 1, 1, 1)
            )

            pts = (
                bn +
                a * (xn - bn)
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

            del pts
            del g
            del a

        ig = (
            (xn - bn) *
            acc /
            n
        )

        with torch.no_grad():

            delta = float(
                (
                    zf(xn) -
                    zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            ) -
            delta
        )

        return (
            ig,
            delta,
            err
        )

    # ------------------------------------------------------------
    # Primary 50-step evaluation
    # ------------------------------------------------------------

    ig, delta, err = run(
        IG_STEPS
    )

    used = IG_STEPS

    tol = max(
        IG_TOL_REL * abs(delta),
        IG_TAU_ABS
    )

    # ------------------------------------------------------------
    # Frozen retry rule
    # ------------------------------------------------------------

    if err > tol:

        del ig

        ig, delta, err = run(
            IG_RETRY_STEPS
        )

        used = IG_RETRY_STEPS

    # ------------------------------------------------------------
    # Frozen enrichment calculation
    # ------------------------------------------------------------

    abs_channel = ig.abs()

    abs_spatial = (
        abs_channel
        .sum(1)[0]
    )

    total = float(
        abs_spatial.sum().item()
    )

    area_frac = float(
        m.float().mean().item()
    )

    undefined = (
        total <= XAI_EPS
    ) or (
        area_frac == 0.0
    )

    if undefined:

        enrichment = float("nan")

    else:

        enrichment = float(
            (
                abs_spatial[m].sum().item()
                /
                total
            )
            /
            area_frac
        )

    signed_inside = float(
        ig.sum(1)[0][m]
        .sum()
        .item()
    )

    completeness_failed = (
        err > tol
    )

    # ------------------------------------------------------------
    # Copy maps back to CPU before releasing CUDA tensors.
    # ------------------------------------------------------------

    signed_np = (
        ig.detach()
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False
        )
    )

    abs_np = (
        abs_spatial.detach()
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False
        )
    )

    metadata = {
        "enrichment": enrichment,
        "undefined": bool(undefined),
        "completeness_error": float(err),
        "delta": float(delta),
        "tolerance": float(tol),
        "steps_used": int(used),
        "completeness_failed": bool(
            completeness_failed
        ),
        "signed_attribution_inside_mask":
            float(signed_inside),
        "mask_area_fraction":
            float(area_frac),
    }

    return (
        signed_np,
        abs_np,
        metadata
    )

# ----------------------------------------------------------------
# Checkpoint loader
# ----------------------------------------------------------------

def load_resnet18_checkpoint(
    checkpoint_path
):

    model = tv_models.resnet18(
        weights=None,
        num_classes=2
    )

    state = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    if isinstance(
        state,
        dict
    ) and "state_dict" in state:

        state = state["state_dict"]

    model.load_state_dict(
        state,
        strict=True
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    return model

# ----------------------------------------------------------------
# Output format
#
# For each model and evaluation type:
#
# signed_ig.npy:
#     shape = (99, 3, 224, 224)
#
# abs_ig.npy:
#     shape = (99, 224, 224)
#
# metadata.csv:
#     one row per image.
#
# The arrays are memory-mapped so completed results remain on disk
# immediately and the entire population is not held in RAM.
# ----------------------------------------------------------------

def model_output_paths(
    model_key,
    evaluation
):

    base_dir = (
        HAIR_DIR
        if evaluation == "hair"
        else CLEAN_DIR
    )

    model_dir = (
        base_dir /
        f"model_{model_key}"
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    return (
        model_dir / "signed_ig.npy",
        model_dir / "abs_ig.npy",
        model_dir / "metadata.csv",
        model_dir / "COMPLETED.json",
    )

# ----------------------------------------------------------------
# Production execution
# ----------------------------------------------------------------

TOTAL = (
    len(roster) *
    len(retained_images) *
    2
)

completed = 0
failed = []

global_start = time.time()

print()
print("=" * 70)
print("STARTING PRODUCTION IG")
print("=" * 70)
print("Expected evaluations :", TOTAL)
print()

for model_idx, row in roster.iterrows():

    model_key = str(
        row["model_key"]
    )

    checkpoint_path = Path(
        row["path"]
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: "
        f"{checkpoint_path}"
    )

    print()
    print("-" * 70)
    print(
        f"MODEL {model_idx + 1}/30 : "
        f"{model_key}"
    )
    print(
        f"Checkpoint : "
        f"{checkpoint_path.name}"
    )
    print("-" * 70)

    model = load_resnet18_checkpoint(
        checkpoint_path
    )

    model_start = time.time()

    for evaluation in [
        "hair",
        "clean"
    ]:

        (
            signed_path,
            abs_path,
            metadata_path,
            complete_path,
        ) = model_output_paths(
            model_key,
            evaluation
        )

        # --------------------------------------------------------
        # Resume protection
        # --------------------------------------------------------

        if complete_path.exists():

            print(
                f"  {evaluation:5s} : "
                f"ALREADY COMPLETE — skipping"
            )

            completed += len(
                retained_images
            )

            continue

        # --------------------------------------------------------
        # Allocate disk-backed arrays
        # --------------------------------------------------------

        signed_mm = np.lib.format.open_memmap(
            signed_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                len(retained_images),
                3,
                224,
                224
            )
        )

        abs_mm = np.lib.format.open_memmap(
            abs_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                len(retained_images),
                224,
                224
            )
        )

        metadata_rows = []

        print()
        print(
            f"  Starting {evaluation} "
            f"({len(retained_images)} images)"
        )

        eval_start = time.time()

        for image_idx, image_id in enumerate(
            retained_images
        ):

            try:

                cache_row = cache_lookup[
                    image_id
                ]

                if evaluation == "hair":

                    image_u8 = np.array(
                        hair_224[cache_row],
                        copy=True
                    )

                else:

                    image_u8 = np.array(
                        clean_224[cache_row],
                        copy=True
                    )

                mask_bool = (
                    true_mask_lookup[
                        image_id
                    ]
                )

                (
                    signed_np,
                    abs_np,
                    metadata
                ) = production_ig_fullmap(
                    model,
                    image_u8,
                    mask_bool
                )

                # ------------------------------------------------
                # Shape / finiteness guards
                # ------------------------------------------------

                assert signed_np.shape == (
                    1,
                    3,
                    224,
                    224
                )

                assert abs_np.shape == (
                    224,
                    224
                )

                assert np.isfinite(
                    signed_np
                ).all()

                assert np.isfinite(
                    abs_np
                ).all()

                # ------------------------------------------------
                # Persist
                # ------------------------------------------------

                signed_mm[
                    image_idx
                ] = signed_np[0]

                abs_mm[
                    image_idx
                ] = abs_np

                metadata_rows.append({
                    "model_key":
                        model_key,

                    "image":
                        image_id,

                    "evaluation":
                        evaluation,

                    **metadata,
                })

                completed += 1

                # ------------------------------------------------
                # Periodic flush
                # ------------------------------------------------

                if (
                    (image_idx + 1) % 10 == 0
                    or
                    image_idx == len(
                        retained_images
                    ) - 1
                ):

                    signed_mm.flush()
                    abs_mm.flush()

                    elapsed = (
                        time.time()
                        - eval_start
                    )

                    rate = (
                        (image_idx + 1)
                        /
                        elapsed
                    )

                    remaining = (
                        len(retained_images)
                        -
                        (image_idx + 1)
                    )

                    eta = (
                        remaining / rate
                        if rate > 0
                        else float("nan")
                    )

                    print(
                        f"    "
                        f"{image_idx + 1:3d}/"
                        f"{len(retained_images)} "
                        f"| "
                        f"{rate:.2f} img/s "
                        f"| ETA "
                        f"{eta / 60:.1f} min"
                    )

                # ------------------------------------------------
                # Explicit CUDA cleanup
                # ------------------------------------------------

                del image_u8
                del signed_np
                del abs_np

            except Exception as exc:

                failed.append({
                    "model_key":
                        model_key,

                    "image":
                        image_id,

                    "evaluation":
                        evaluation,

                    "error":
                        repr(exc),
                })

                print()
                print(
                    "    ERROR:"
                    f" {model_key} / "
                    f"{image_id} / "
                    f"{evaluation}"
                )
                print(
                    "    ",
                    repr(exc)
                )

                raise

        # --------------------------------------------------------
        # Final flush
        # --------------------------------------------------------

        signed_mm.flush()
        abs_mm.flush()

        del signed_mm
        del abs_mm

        # --------------------------------------------------------
        # Metadata
        # --------------------------------------------------------

        metadata_df = pd.DataFrame(
            metadata_rows
        )

        assert len(
            metadata_df
        ) == len(
            retained_images
        )

        metadata_df.to_csv(
            metadata_path,
            index=False
        )

        # --------------------------------------------------------
        # Verify persisted arrays
        # --------------------------------------------------------

        check_signed = np.load(
            signed_path,
            mmap_mode="r"
        )

        check_abs = np.load(
            abs_path,
            mmap_mode="r"
        )

        assert check_signed.shape == (
            len(retained_images),
            3,
            224,
            224
        )

        assert check_abs.shape == (
            len(retained_images),
            224,
            224
        )

        assert np.isfinite(
            check_signed
        ).all()

        assert np.isfinite(
            check_abs
        ).all()

        del check_signed
        del check_abs

        # --------------------------------------------------------
        # File hashes
        # --------------------------------------------------------

        def sha256_file(path):

            h = hashlib.sha256()

            with open(
                path,
                "rb"
            ) as f:

                while True:

                    block = f.read(
                        1024 * 1024
                    )

                    if not block:
                        break

                    h.update(block)

            return h.hexdigest()

        completion_record = {
            "experiment":
                "T1_4B",

            "model_key":
                model_key,

            "evaluation":
                evaluation,

            "n_images":
                len(retained_images),

            "signed_shape":
                [
                    len(retained_images),
                    3,
                    224,
                    224
                ],

            "abs_shape":
                [
                    len(retained_images),
                    224,
                    224
                ],

            "signed_dtype":
                "float32",

            "abs_dtype":
                "float32",

            "signed_sha256":
                sha256_file(
                    signed_path
                ),

            "abs_sha256":
                sha256_file(
                    abs_path
                ),

            "metadata_sha256":
                sha256_file(
                    metadata_path
                ),

            "completed":
                True,
        }

        with open(
            complete_path,
            "w"
        ) as f:

            json.dump(
                completion_record,
                f,
                indent=2
            )

        print(
            f"  {evaluation:5s} : "
            f"COMPLETE"
        )

        gc.collect()
        torch.cuda.empty_cache()

    # ----------------------------------------------------------------
    # Model cleanup
    # ----------------------------------------------------------------

    del model
    gc.collect()
    torch.cuda.empty_cache()

    model_elapsed = (
        time.time()
        - model_start
    )

    print()
    print(
        f"MODEL {model_key} COMPLETE"
    )
    print(
        f"Model wall time : "
        f"{model_elapsed / 60:.2f} min"
    )

# ----------------------------------------------------------------
# Final production summary
# ----------------------------------------------------------------

total_elapsed = (
    time.time()
    - global_start
)

print()
print("=" * 70)
print("T1.4b — PRODUCTION IG RUN COMPLETE")
print("=" * 70)

print(
    "Expected evaluations :",
    TOTAL
)

print(
    "Completed evaluations:",
    completed
)

print(
    "Failures             :",
    len(failed)
)

print(
    "Wall time            :",
    f"{total_elapsed / 3600:.2f} hours"
)

if failed:

    failure_df = pd.DataFrame(
        failed
    )

    failure_path = (
        IG_DIR /
        "T1_4B_IG_FAILURES.csv"
    )

    failure_df.to_csv(
        failure_path,
        index=False
    )

    print()
    print(
        "FAILURE LOG:"
    )
    print(
        failure_path
    )

    raise RuntimeError(
        "Production IG completed with failures."
    )

# ----------------------------------------------------------------
# Global completion lock
# ----------------------------------------------------------------

global_lock = {
    "experiment":
        "T1_4B",

    "models":
        30,

    "retained_images":
        99,

    "evaluations_per_pair":
        2,

    "expected_ig_evaluations":
        5940,

    "completed_evaluations":
        completed,

    "failures":
        0,

    "ig_steps":
        IG_STEPS,

    "ig_retry_steps":
        IG_RETRY_STEPS,

    "ig_sigma":
        IG_SIGMA,

    "ig_tol_rel":
        IG_TOL_REL,

    "ig_tau_abs":
        IG_TAU_ABS,

    "target":
        "z_malignant_minus_z_benign",

    "baseline":
        "gaussian_blur_sigma_10",

    "null_ig_reuse":
        True,

    "status":
        "COMPLETE",

    "wall_time_seconds":
        total_elapsed,
}

GLOBAL_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_COMPLETE_LOCK.json"
)

with open(
    GLOBAL_LOCK,
    "w"
) as f:

    json.dump(
        global_lock,
        f,
        indent=2
    )

print()
print(
    "GLOBAL LOCK:"
)
print(
    GLOBAL_LOCK
)

print()
print("=" * 70)
print("CELL 6B COMPLETE")
print("=" * 70)

T1.4b — PRODUCTION FULL-RESOLUTION IG EXECUTION
Device : cuda:0
GPU    : Tesla T4


AssertionError: 

In [57]:
# ================================================================
# CELL 6B-FIX — INSPECT LOCKED ROSTER SCHEMA
# ================================================================

from pathlib import Path
import pandas as pd
import json

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

assert ROSTER_CSV.exists(), ROSTER_CSV

roster_debug = pd.read_csv(
    ROSTER_CSV
)

print("=" * 70)
print("T1.4b — LOCKED CHECKPOINT ROSTER SCHEMA")
print("=" * 70)

print()
print("Path:")
print(ROSTER_CSV)

print()
print("Columns:")
for i, col in enumerate(roster_debug.columns):
    print(f"  {i}: {col}")

print()
print("Shape:")
print(roster_debug.shape)

print()
print("First 5 rows:")
print(roster_debug.head().to_string(index=False))

print()
print("Dtypes:")
print(roster_debug.dtypes.to_string())

print()
print("=" * 70)
print("NO MODEL LOADING")
print("NO IG COMPUTATION")
print("NO FILE MODIFICATION")
print("=" * 70)

T1.4b — LOCKED CHECKPOINT ROSTER SCHEMA

Path:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv

Columns:
  0: model_key
  1: rho
  2: replicate
  3: checkpoint_path
  4: sha256
  5: size_bytes

Shape:
(30, 6)

First 5 rows:
model_key  rho  replicate                                                                                                                     checkpoint_path                                                           sha256  size_bytes
    0.0_1  0.0          1 /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_1/final_epoch15.pt 7542507471917b74bdfbd44310fdc3ccdd09fe5cb89f1a88de40078452d633c6    44790155
    0.0_2  0.0          2 /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs/rho_0.0_rep_2/final_epoch15.pt ab828644cccf1f64b8ee13bb424551935d8942e3d64bbe570ad53c2d8a37abe4    44790155
    0.0_3  0.0          3 /k

In [58]:
# ================================================================
# CELL 6B — T1.4b PRODUCTION FULL-RESOLUTION IG EXECUTION
# CORRECTED: checkpoint_path column
# ================================================================

import os
import gc
import json
import time
import hashlib
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision.models as tv_models
import torchvision.transforms.functional as TF

print("=" * 70)
print("T1.4b — PRODUCTION FULL-RESOLUTION IG EXECUTION")
print("=" * 70)

# ----------------------------------------------------------------
# Frozen IG constants
# ----------------------------------------------------------------

IG_STEPS = 50
IG_RETRY_STEPS = 100
IG_SIGMA = 10.0
IG_TOL_REL = 0.05
IG_TAU_ABS = 0.05
XAI_EPS = 1e-8
IG_CHUNK = 25

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda"

print("Device :", DEVICE)
print("GPU    :", torch.cuda.get_device_name(DEVICE))

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

MANIFEST_CSV = (
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

IG_DIR = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_IG"
)

HAIR_DIR = IG_DIR / "hair"
CLEAN_DIR = IG_DIR / "clean"

HAIR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert ROSTER_CSV.exists()
assert MANIFEST_CSV.exists()

# ----------------------------------------------------------------
# Helper: SHA256
# ----------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()

# ----------------------------------------------------------------
# Load and validate canonical roster
# ----------------------------------------------------------------

roster = pd.read_csv(
    ROSTER_CSV
)

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert len(roster) == 30
assert roster["model_key"].tolist() == EXPECTED_MODEL_KEYS

assert "checkpoint_path" in roster.columns
assert "sha256" in roster.columns

print()
print("Canonical roster :", len(roster))
print("Checkpoint column: checkpoint_path")

# ----------------------------------------------------------------
# Load retained image population
# ----------------------------------------------------------------

null_manifest = pd.read_csv(
    MANIFEST_CSV
)

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(retained_images) == 99

print("Retained images  :", len(retained_images))

# ----------------------------------------------------------------
# Load cache
# ----------------------------------------------------------------

cache_index = pd.read_csv(
    CACHE_DIR / "cache_index.csv"
)

clean_224 = np.load(
    CACHE_DIR / "clean_224.npy",
    mmap_mode="r"
)

hair_224 = np.load(
    CACHE_DIR / "hair_224.npy",
    mmap_mode="r"
)

assert clean_224.shape == (
    len(cache_index),
    224,
    224,
    3
)

assert hair_224.shape == (
    len(cache_index),
    224,
    224,
    3
)

cache_lookup = {
    str(row.image): int(row.row)
    for row in cache_index.itertuples(
        index=False
    )
}

assert all(
    image_id in cache_lookup
    for image_id in retained_images
)

# ----------------------------------------------------------------
# Reconstruct true masks
# ----------------------------------------------------------------

true_mask_lookup = {}

for image_id in retained_images:

    idx = cache_lookup[image_id]

    clean_u8 = np.array(
        clean_224[idx],
        copy=True
    )

    hair_u8 = np.array(
        hair_224[idx],
        copy=True
    )

    mask = np.any(
        hair_u8 != clean_u8,
        axis=2
    )

    assert mask.shape == (
        224,
        224
    )

    assert mask.dtype == bool
    assert int(mask.sum()) > 0

    true_mask_lookup[image_id] = mask

print("Cache alignment  : PASS")
print("True-mask build   : PASS")

# ----------------------------------------------------------------
# Frozen normalization constants
# ----------------------------------------------------------------

_MEAN = (
    0.485,
    0.456,
    0.406
)

_STD = (
    0.229,
    0.224,
    0.225
)

MEAN_T = torch.tensor(
    _MEAN,
    device=DEVICE,
    dtype=torch.float32
).view(1, 3, 1, 1)

STD_T = torch.tensor(
    _STD,
    device=DEVICE,
    dtype=torch.float32
).view(1, 3, 1, 1)

# ----------------------------------------------------------------
# Full-map IG implementation
#
# This reproduces the validated 5G-R2 implementation.
# ----------------------------------------------------------------

def production_ig_fullmap(
    model,
    image_u8,
    mask_bool,
):

    x = (
        torch.from_numpy(
            np.ascontiguousarray(image_u8)
        )
        .to(DEVICE)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = (
        2 *
        int(
            math.ceil(
                3 * IG_SIGMA
            )
        )
        + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[IG_SIGMA, IG_SIGMA]
    )

    xn = (
        x - MEAN_T
    ) / STD_T

    bn = (
        base - MEAN_T
    ) / STD_T

    m = torch.from_numpy(
        np.ascontiguousarray(mask_bool)
    ).to(DEVICE).bool()

    def zf(t):

        o = model(t)

        return o[:, 1] - o[:, 0]

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=DEVICE,
                dtype=torch.float32
            )
            / n
        )

        for s in range(
            0,
            n,
            IG_CHUNK
        ):

            a = (
                alphas[
                    s:s + IG_CHUNK
                ]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn +
                a * (xn - bn)
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

            del pts
            del g
            del a

        ig = (
            (xn - bn) *
            acc /
            n
        )

        with torch.no_grad():

            delta = float(
                (
                    zf(xn) -
                    zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            ) -
            delta
        )

        return (
            ig,
            delta,
            err
        )

    # ------------------------------------------------------------
    # 50-step primary
    # ------------------------------------------------------------

    ig, delta, err = run(
        IG_STEPS
    )

    used = IG_STEPS

    tol = max(
        IG_TOL_REL * abs(delta),
        IG_TAU_ABS
    )

    # ------------------------------------------------------------
    # 100-step retry
    # ------------------------------------------------------------

    if err > tol:

        del ig

        ig, delta, err = run(
            IG_RETRY_STEPS
        )

        used = IG_RETRY_STEPS

    # ------------------------------------------------------------
    # Frozen enrichment
    # ------------------------------------------------------------

    abs_channel = ig.abs()

    abs_spatial = (
        abs_channel
        .sum(1)[0]
    )

    total = float(
        abs_spatial.sum().item()
    )

    area_frac = float(
        m.float().mean().item()
    )

    undefined = (
        total <= XAI_EPS
    ) or (
        area_frac == 0.0
    )

    if undefined:

        enrichment = float("nan")

    else:

        enrichment = float(
            (
                abs_spatial[m]
                .sum()
                .item()
                /
                total
            )
            /
            area_frac
        )

    signed_inside = float(
        ig.sum(1)[0][m]
        .sum()
        .item()
    )

    # ------------------------------------------------------------
    # CPU persistence
    # ------------------------------------------------------------

    signed_np = (
        ig.detach()
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False
        )
    )

    abs_np = (
        abs_spatial.detach()
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False
        )
    )

    metadata = {
        "enrichment":
            enrichment,

        "undefined":
            bool(undefined),

        "completeness_error":
            float(err),

        "delta":
            float(delta),

        "tolerance":
            float(tol),

        "steps_used":
            int(used),

        "completeness_failed":
            bool(err > tol),

        "signed_attribution_inside_mask":
            float(signed_inside),

        "mask_area_fraction":
            float(area_frac),
    }

    return (
        signed_np,
        abs_np,
        metadata
    )

# ----------------------------------------------------------------
# ResNet-18 checkpoint loader
# ----------------------------------------------------------------

def load_resnet18_checkpoint(
    checkpoint_path
):

    model = tv_models.resnet18(
        weights=None,
        num_classes=2
    )

    state = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    if (
        isinstance(state, dict)
        and
        "state_dict" in state
    ):

        state = state[
            "state_dict"
        ]

    model.load_state_dict(
        state,
        strict=True
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    return model

# ----------------------------------------------------------------
# Output paths
# ----------------------------------------------------------------

def model_output_paths(
    model_key,
    evaluation
):

    base_dir = (
        HAIR_DIR
        if evaluation == "hair"
        else CLEAN_DIR
    )

    model_dir = (
        base_dir /
        f"model_{model_key}"
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    return (
        model_dir / "signed_ig.npy",
        model_dir / "abs_ig.npy",
        model_dir / "metadata.csv",
        model_dir / "COMPLETED.json",
    )

# ----------------------------------------------------------------
# Production
# ----------------------------------------------------------------

TOTAL = (
    30 *
    99 *
    2
)

completed = 0
failed = []

global_start = time.time()

print()
print("=" * 70)
print("STARTING PRODUCTION IG")
print("=" * 70)
print("Expected evaluations :", TOTAL)
print()

for model_idx, row in roster.iterrows():

    model_key = str(
        row["model_key"]
    )

    checkpoint_path = Path(
        row["checkpoint_path"]
    )

    expected_sha = str(
        row["sha256"]
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: "
        f"{checkpoint_path}"
    )

    # ------------------------------------------------------------
    # Verify locked checkpoint hash
    # ------------------------------------------------------------

    actual_sha = sha256_file(
        checkpoint_path
    )

    assert actual_sha == expected_sha, (
        f"CHECKPOINT HASH MISMATCH: "
        f"{model_key}"
    )

    print()
    print("-" * 70)
    print(
        f"MODEL {model_idx + 1}/30 : "
        f"{model_key}"
    )
    print(
        "Checkpoint hash : PASS"
    )
    print("-" * 70)

    model = load_resnet18_checkpoint(
        checkpoint_path
    )

    model_start = time.time()

    for evaluation in [
        "hair",
        "clean"
    ]:

        (
            signed_path,
            abs_path,
            metadata_path,
            complete_path,
        ) = model_output_paths(
            model_key,
            evaluation
        )

        # --------------------------------------------------------
        # Resume protection
        # --------------------------------------------------------

        if complete_path.exists():

            print(
                f"  {evaluation:5s} : "
                "ALREADY COMPLETE — skipping"
            )

            completed += len(
                retained_images
            )

            continue

        # --------------------------------------------------------
        # Disk-backed arrays
        # --------------------------------------------------------

        signed_mm = np.lib.format.open_memmap(
            signed_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                len(retained_images),
                3,
                224,
                224
            )
        )

        abs_mm = np.lib.format.open_memmap(
            abs_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                len(retained_images),
                224,
                224
            )
        )

        metadata_rows = []

        eval_start = time.time()

        print()
        print(
            f"  Starting {evaluation}: "
            f"{len(retained_images)} images"
        )

        for image_idx, image_id in enumerate(
            retained_images
        ):

            cache_row = cache_lookup[
                image_id
            ]

            if evaluation == "hair":

                image_u8 = np.array(
                    hair_224[cache_row],
                    copy=True
                )

            else:

                image_u8 = np.array(
                    clean_224[cache_row],
                    copy=True
                )

            mask_bool = (
                true_mask_lookup[
                    image_id
                ]
            )

            try:

                (
                    signed_np,
                    abs_np,
                    metadata
                ) = production_ig_fullmap(
                    model,
                    image_u8,
                    mask_bool
                )

                assert signed_np.shape == (
                    1,
                    3,
                    224,
                    224
                )

                assert abs_np.shape == (
                    224,
                    224
                )

                assert np.isfinite(
                    signed_np
                ).all()

                assert np.isfinite(
                    abs_np
                ).all()

                signed_mm[
                    image_idx
                ] = signed_np[0]

                abs_mm[
                    image_idx
                ] = abs_np

                metadata_rows.append({
                    "model_key":
                        model_key,

                    "image":
                        image_id,

                    "evaluation":
                        evaluation,

                    **metadata,
                })

                completed += 1

            except Exception as exc:

                failure_record = {
                    "model_key":
                        model_key,

                    "image":
                        image_id,

                    "evaluation":
                        evaluation,

                    "error":
                        repr(exc),
                }

                failed.append(
                    failure_record
                )

                print()
                print(
                    "ERROR:",
                    failure_record
                )

                raise

            finally:

                if "image_u8" in locals():
                    del image_u8

            # ----------------------------------------------------
            # Periodic persistence
            # ----------------------------------------------------

            if (
                (image_idx + 1) % 10 == 0
                or
                image_idx ==
                len(retained_images) - 1
            ):

                signed_mm.flush()
                abs_mm.flush()

                elapsed = (
                    time.time()
                    - eval_start
                )

                n_done = (
                    image_idx + 1
                )

                rate = (
                    n_done /
                    elapsed
                )

                remaining = (
                    len(retained_images)
                    -
                    n_done
                )

                eta = (
                    remaining / rate
                    if rate > 0
                    else float("nan")
                )

                print(
                    f"    "
                    f"{n_done:3d}/"
                    f"{len(retained_images)}"
                    f" | "
                    f"{rate:.2f} img/s"
                    f" | ETA "
                    f"{eta / 60:.1f} min"
                )

            del signed_np
            del abs_np

        # --------------------------------------------------------
        # Flush arrays
        # --------------------------------------------------------

        signed_mm.flush()
        abs_mm.flush()

        del signed_mm
        del abs_mm

        # --------------------------------------------------------
        # Metadata
        # --------------------------------------------------------

        metadata_df = pd.DataFrame(
            metadata_rows
        )

        assert len(
            metadata_df
        ) == len(
            retained_images
        )

        metadata_df.to_csv(
            metadata_path,
            index=False
        )

        # --------------------------------------------------------
        # Persisted-array verification
        # --------------------------------------------------------

        check_signed = np.load(
            signed_path,
            mmap_mode="r"
        )

        check_abs = np.load(
            abs_path,
            mmap_mode="r"
        )

        assert check_signed.shape == (
            99,
            3,
            224,
            224
        )

        assert check_abs.shape == (
            99,
            224,
            224
        )

        assert np.isfinite(
            check_signed
        ).all()

        assert np.isfinite(
            check_abs
        ).all()

        del check_signed
        del check_abs

        # --------------------------------------------------------
        # Completion record
        # --------------------------------------------------------

        completion_record = {
            "experiment":
                "T1_4B",

            "model_key":
                model_key,

            "evaluation":
                evaluation,

            "n_images":
                99,

            "signed_shape":
                [
                    99,
                    3,
                    224,
                    224
                ],

            "abs_shape":
                [
                    99,
                    224,
                    224
                ],

            "signed_dtype":
                "float32",

            "abs_dtype":
                "float32",

            "signed_sha256":
                sha256_file(
                    signed_path
                ),

            "abs_sha256":
                sha256_file(
                    abs_path
                ),

            "metadata_sha256":
                sha256_file(
                    metadata_path
                ),

            "completed":
                True,
        }

        with open(
            complete_path,
            "w"
        ) as f:

            json.dump(
                completion_record,
                f,
                indent=2
            )

        print(
            f"  {evaluation:5s} : COMPLETE"
        )

        gc.collect()
        torch.cuda.empty_cache()

    del model

    gc.collect()
    torch.cuda.empty_cache()

    model_elapsed = (
        time.time()
        - model_start
    )

    print(
        f"MODEL {model_key} COMPLETE | "
        f"{model_elapsed / 60:.2f} min"
    )

# ----------------------------------------------------------------
# Final lock
# ----------------------------------------------------------------

total_elapsed = (
    time.time()
    - global_start
)

assert completed == TOTAL
assert len(failed) == 0

global_lock = {
    "experiment":
        "T1_4B",

    "models":
        30,

    "retained_images":
        99,

    "evaluations_per_pair":
        2,

    "expected_ig_evaluations":
        TOTAL,

    "completed_evaluations":
        completed,

    "failures":
        0,

    "ig_steps":
        IG_STEPS,

    "ig_retry_steps":
        IG_RETRY_STEPS,

    "ig_sigma":
        IG_SIGMA,

    "ig_tol_rel":
        IG_TOL_REL,

    "ig_tau_abs":
        IG_TAU_ABS,

    "target":
        "z_malignant_minus_z_benign",

    "baseline":
        "gaussian_blur_sigma_10",

    "null_ig_reuse":
        True,

    "status":
        "COMPLETE",

    "wall_time_seconds":
        total_elapsed,
}

GLOBAL_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_COMPLETE_LOCK.json"
)

with open(
    GLOBAL_LOCK,
    "w"
) as f:

    json.dump(
        global_lock,
        f,
        indent=2
    )

print()
print("=" * 70)
print("T1.4b — PRODUCTION IG COMPLETE")
print("=" * 70)
print(
    "Expected evaluations :",
    TOTAL
)
print(
    "Completed evaluations:",
    completed
)
print(
    "Failures             :",
    len(failed)
)
print(
    "Wall time            :",
    f"{total_elapsed / 3600:.2f} hours"
)
print()
print("GLOBAL LOCK:")
print(GLOBAL_LOCK)
print("=" * 70)

T1.4b — PRODUCTION FULL-RESOLUTION IG EXECUTION
Device : cuda:0
GPU    : Tesla T4

Canonical roster : 30
Checkpoint column: checkpoint_path
Retained images  : 99
Cache alignment  : PASS
True-mask build   : PASS

STARTING PRODUCTION IG
Expected evaluations : 5940


----------------------------------------------------------------------
MODEL 1/30 : 0.0_1
Checkpoint hash : PASS
----------------------------------------------------------------------

  Starting hair: 99 images
     10/99 | 3.94 img/s | ETA 0.4 min
     20/99 | 3.99 img/s | ETA 0.3 min
     30/99 | 4.01 img/s | ETA 0.3 min
     40/99 | 4.02 img/s | ETA 0.2 min
     50/99 | 4.02 img/s | ETA 0.2 min
     60/99 | 3.85 img/s | ETA 0.2 min
     70/99 | 3.87 img/s | ETA 0.1 min
     80/99 | 3.81 img/s | ETA 0.1 min
     90/99 | 3.83 img/s | ETA 0.0 min
     99/99 | 3.85 img/s | ETA 0.0 min
  hair  : COMPLETE

  Starting clean: 99 images
     10/99 | 4.83 img/s | ETA 0.3 min
     20/99 | 5.40 img/s | ETA 0.2 min
     30/99 | 5.86 i

In [59]:
# ================================================================
# CELL 6C — T1.4b PRODUCTION IG COMPLETION + PERSISTENCE AUDIT
# ================================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — PRODUCTION IG COMPLETION + PERSISTENCE AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

IG_DIR = (
    RESULT_DIR /
    "T1_4B_PRODUCTION_IG"
)

HAIR_DIR = IG_DIR / "hair"
CLEAN_DIR = IG_DIR / "clean"

GLOBAL_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_COMPLETE_LOCK.json"
)

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

MANIFEST_CSV = (
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

assert GLOBAL_LOCK.exists()
assert ROSTER_CSV.exists()
assert MANIFEST_CSV.exists()

# ----------------------------------------------------------------
# Load locks
# ----------------------------------------------------------------

with open(
    GLOBAL_LOCK,
    "r"
) as f:
    global_lock = json.load(f)

assert global_lock["experiment"] == "T1_4B"
assert global_lock["models"] == 30
assert global_lock["retained_images"] == 99
assert global_lock["evaluations_per_pair"] == 2
assert global_lock["expected_ig_evaluations"] == 5940
assert global_lock["completed_evaluations"] == 5940
assert global_lock["failures"] == 0
assert global_lock["status"] == "COMPLETE"

print()
print("Global completion lock : PASS")

# ----------------------------------------------------------------
# Load populations
# ----------------------------------------------------------------

roster = pd.read_csv(
    ROSTER_CSV
)

null_manifest = pd.read_csv(
    MANIFEST_CSV
)

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert roster["model_key"].tolist() == EXPECTED_MODEL_KEYS

retained_images = sorted(
    null_manifest["image"]
    .astype(str)
    .unique()
)

assert len(retained_images) == 99

# ----------------------------------------------------------------
# Audit every model × evaluation
# ----------------------------------------------------------------

audit_rows = []

for model_key in EXPECTED_MODEL_KEYS:

    for evaluation, base_dir in [
        ("hair", HAIR_DIR),
        ("clean", CLEAN_DIR),
    ]:

        model_dir = (
            base_dir /
            f"model_{model_key}"
        )

        signed_path = (
            model_dir /
            "signed_ig.npy"
        )

        abs_path = (
            model_dir /
            "abs_ig.npy"
        )

        metadata_path = (
            model_dir /
            "metadata.csv"
        )

        complete_path = (
            model_dir /
            "COMPLETED.json"
        )

        # --------------------------------------------------------
        # Existence
        # --------------------------------------------------------

        exists = all([
            model_dir.exists(),
            signed_path.exists(),
            abs_path.exists(),
            metadata_path.exists(),
            complete_path.exists(),
        ])

        assert exists, (
            f"Missing production output: "
            f"{model_key} / {evaluation}"
        )

        # --------------------------------------------------------
        # Completion record
        # --------------------------------------------------------

        with open(
            complete_path,
            "r"
        ) as f:
            record = json.load(f)

        assert record["completed"] is True
        assert record["model_key"] == model_key
        assert record["evaluation"] == evaluation
        assert record["n_images"] == 99

        # --------------------------------------------------------
        # Load persisted arrays
        # --------------------------------------------------------

        signed = np.load(
            signed_path,
            mmap_mode="r"
        )

        abs_ig = np.load(
            abs_path,
            mmap_mode="r"
        )

        assert signed.shape == (
            99,
            3,
            224,
            224
        )

        assert abs_ig.shape == (
            99,
            224,
            224
        )

        assert signed.dtype == np.float32
        assert abs_ig.dtype == np.float32

        # --------------------------------------------------------
        # Numerical validity
        # --------------------------------------------------------

        signed_finite = bool(
            np.isfinite(signed).all()
        )

        abs_finite = bool(
            np.isfinite(abs_ig).all()
        )

        assert signed_finite
        assert abs_finite

        # |IG| must be non-negative
        min_abs = float(
            abs_ig.min()
        )

        assert min_abs >= 0.0

        # --------------------------------------------------------
        # Metadata
        # --------------------------------------------------------

        metadata = pd.read_csv(
            metadata_path
        )

        assert len(metadata) == 99

        assert metadata[
            "model_key"
        ].astype(str).tolist() == [
            model_key
        ] * 99

        assert metadata[
            "evaluation"
        ].astype(str).tolist() == [
            evaluation
        ] * 99

        assert metadata[
            "image"
        ].astype(str).tolist() == retained_images

        required_metadata = [
            "model_key",
            "image",
            "evaluation",
            "enrichment",
            "undefined",
            "completeness_error",
            "delta",
            "tolerance",
            "steps_used",
            "completeness_failed",
            "signed_attribution_inside_mask",
            "mask_area_fraction",
        ]

        for col in required_metadata:
            assert col in metadata.columns

        # --------------------------------------------------------
        # Metadata validity
        # --------------------------------------------------------

        assert metadata[
            "steps_used"
        ].isin([
            50,
            100
        ]).all()

        assert metadata[
            "mask_area_fraction"
        ].gt(0).all()

        assert metadata[
            "mask_area_fraction"
        ].lt(1).all()

        assert np.isfinite(
            metadata[
                "completeness_error"
            ].to_numpy()
        ).all()

        assert np.isfinite(
            metadata[
                "delta"
            ].to_numpy()
        ).all()

        assert np.isfinite(
            metadata[
                "tolerance"
            ].to_numpy()
        ).all()

        # --------------------------------------------------------
        # Enrichment validity
        #
        # Undefined evaluations may legitimately contain NaN.
        # Defined evaluations must be finite.
        # --------------------------------------------------------

        undefined = (
            metadata[
                "undefined"
            ].astype(bool)
        )

        enrichment = (
            metadata[
                "enrichment"
            ].to_numpy()
        )

        defined = ~undefined.to_numpy()

        if defined.any():

            assert np.isfinite(
                enrichment[defined]
            ).all()

        # --------------------------------------------------------
        # Map-level consistency
        #
        # Recompute spatial |IG| from signed maps for every image
        # and compare against the persisted abs_ig array.
        # --------------------------------------------------------

        # Work in chunks to avoid unnecessary RAM use.
        max_map_difference = 0.0

        for start in range(
            0,
            99,
            10
        ):

            stop = min(
                start + 10,
                99
            )

            signed_chunk = np.asarray(
                signed[start:stop]
            )

            derived_abs = np.abs(
                signed_chunk
            ).sum(
                axis=1
            )

            stored_abs = np.asarray(
                abs_ig[start:stop]
            )

            diff = np.max(
                np.abs(
                    derived_abs -
                    stored_abs
                )
            )

            max_map_difference = max(
                max_map_difference,
                float(diff)
            )

            assert np.allclose(
                derived_abs,
                stored_abs,
                rtol=0.0,
                atol=0.0
            )

        # --------------------------------------------------------
        # Hash verification
        # --------------------------------------------------------

        def sha256_file(path):

            h = hashlib.sha256()

            with open(
                path,
                "rb"
            ) as f:

                while True:

                    block = f.read(
                        1024 * 1024
                    )

                    if not block:
                        break

                    h.update(block)

            return h.hexdigest()

        signed_sha = sha256_file(
            signed_path
        )

        abs_sha = sha256_file(
            abs_path
        )

        metadata_sha = sha256_file(
            metadata_path
        )

        assert (
            signed_sha ==
            record["signed_sha256"]
        )

        assert (
            abs_sha ==
            record["abs_sha256"]
        )

        assert (
            metadata_sha ==
            record["metadata_sha256"]
        )

        # --------------------------------------------------------
        # Summary
        # --------------------------------------------------------

        audit_rows.append({
            "model_key":
                model_key,

            "evaluation":
                evaluation,

            "n_images":
                len(metadata),

            "signed_shape":
                str(signed.shape),

            "abs_shape":
                str(abs_ig.shape),

            "signed_finite":
                signed_finite,

            "abs_finite":
                abs_finite,

            "min_abs_ig":
                min_abs,

            "undefined_count":
                int(undefined.sum()),

            "defined_count":
                int(defined.sum()),

            "retry_100_count":
                int(
                    (
                        metadata[
                            "steps_used"
                        ] == 100
                    ).sum()
                ),

            "completeness_failed_count":
                int(
                    metadata[
                        "completeness_failed"
                    ].astype(bool).sum()
                ),

            "max_abs_map_reconstruction_diff":
                max_map_difference,

            "signed_sha256_match":
                signed_sha ==
                record["signed_sha256"],

            "abs_sha256_match":
                abs_sha ==
                record["abs_sha256"],

            "metadata_sha256_match":
                metadata_sha ==
                record["metadata_sha256"],
        })

        del signed
        del abs_ig

# ----------------------------------------------------------------
# Global audit table
# ----------------------------------------------------------------

audit_df = pd.DataFrame(
    audit_rows
)

assert len(audit_df) == 60

assert audit_df[
    "n_images"
].eq(99).all()

assert audit_df[
    "signed_finite"
].all()

assert audit_df[
    "abs_finite"
].all()

assert audit_df[
    "signed_sha256_match"
].all()

assert audit_df[
    "abs_sha256_match"
].all()

assert audit_df[
    "metadata_sha256_match"
].all()

assert audit_df[
    "max_abs_map_reconstruction_diff"
].eq(0.0).all()

# ----------------------------------------------------------------
# Global counts
# ----------------------------------------------------------------

total_evaluations = (
    audit_df["n_images"].sum()
)

total_undefined = (
    audit_df["undefined_count"].sum()
)

total_defined = (
    audit_df["defined_count"].sum()
)

total_retry = (
    audit_df["retry_100_count"].sum()
)

total_completeness_failed = (
    audit_df[
        "completeness_failed_count"
    ].sum()
)

assert (
    total_evaluations ==
    5940
)

assert (
    total_defined +
    total_undefined ==
    5940
)

# ----------------------------------------------------------------
# Save audit
# ----------------------------------------------------------------

AUDIT_CSV = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT.csv"
)

audit_df.to_csv(
    AUDIT_CSV,
    index=False
)

AUDIT_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT_LOCK.json"
)

audit_lock = {
    "experiment":
        "T1_4B",

    "models":
        30,

    "retained_images":
        99,

    "expected_evaluations":
        5940,

    "audited_evaluations":
        int(total_evaluations),

    "undefined_evaluations":
        int(total_undefined),

    "defined_evaluations":
        int(total_defined),

    "retry_100_steps":
        int(total_retry),

    "completeness_failed":
        int(total_completeness_failed),

    "all_shapes_valid":
        True,

    "all_finite":
        True,

    "signed_abs_reconstruction_exact":
        True,

    "all_file_hashes_match":
        True,

    "status":
        "PASS",
}

with open(
    AUDIT_LOCK,
    "w"
) as f:

    json.dump(
        audit_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CELL 6C — PRODUCTION IG AUDIT COMPLETE")
print("=" * 70)

print(
    "Expected evaluations       :",
    5940
)

print(
    "Audited evaluations        :",
    total_evaluations
)

print(
    "Defined evaluations        :",
    total_defined
)

print(
    "Undefined evaluations      :",
    total_undefined
)

print(
    "100-step retry evaluations :",
    total_retry
)

print(
    "Completeness failed        :",
    total_completeness_failed
)

print()
print(
    "Signed IG shapes            : PASS"
)

print(
    "|IG| shapes                  : PASS"
)

print(
    "Finite-value audit          : PASS"
)

print(
    "Signed → |IG| reconstruction: PASS"
)

print(
    "SHA256 persistence audit    : PASS"
)

print()
print("Audit CSV:")
print(AUDIT_CSV)

print()
print("Audit lock:")
print(AUDIT_LOCK)

print()
print("=" * 70)
print("NO NULL SCORING PERFORMED")
print("=" * 70)

T1.4b — PRODUCTION IG COMPLETION + PERSISTENCE AUDIT

Global completion lock : PASS


AssertionError: 

In [60]:
# ================================================================
# CELL 6C-DIAG — DIAGNOSE SIGNED → |IG| FLOATING-POINT DIFFERENCE
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — DIAGNOSING SIGNED → |IG| RECONSTRUCTION")
print("=" * 70)

IG_DIR = RESULT_DIR / "T1_4B_PRODUCTION_IG"
HAIR_DIR = IG_DIR / "hair"
CLEAN_DIR = IG_DIR / "clean"

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

diagnostic_rows = []

for model_key in EXPECTED_MODEL_KEYS:

    for evaluation, base_dir in [
        ("hair", HAIR_DIR),
        ("clean", CLEAN_DIR),
    ]:

        model_dir = base_dir / f"model_{model_key}"

        signed = np.load(
            model_dir / "signed_ig.npy",
            mmap_mode="r"
        )

        stored_abs = np.load(
            model_dir / "abs_ig.npy",
            mmap_mode="r"
        )

        max_abs_diff = 0.0
        max_rel_diff = 0.0
        mean_abs_diff = 0.0
        n_values = 0
        n_nonzero_diff = 0

        for start in range(0, 99, 10):

            stop = min(start + 10, 99)

            signed_chunk = np.asarray(
                signed[start:stop]
            )

            stored_chunk = np.asarray(
                stored_abs[start:stop]
            )

            # Reconstruct spatial |IG| from persisted signed IG.
            derived = np.abs(
                signed_chunk
            ).sum(axis=1)

            diff = np.abs(
                derived - stored_chunk
            )

            max_abs_diff = max(
                max_abs_diff,
                float(np.max(diff))
            )

            nonzero = diff > 0

            n_nonzero_diff += int(
                nonzero.sum()
            )

            n_values += diff.size

            mean_abs_diff += float(
                diff.sum()
            )

            denominator = np.maximum(
                np.abs(stored_chunk),
                1e-12
            )

            rel = diff / denominator

            max_rel_diff = max(
                max_rel_diff,
                float(np.max(rel))
            )

        mean_abs_diff /= n_values

        diagnostic_rows.append({
            "model_key":
                model_key,

            "evaluation":
                evaluation,

            "max_abs_diff":
                max_abs_diff,

            "max_relative_diff":
                max_rel_diff,

            "mean_abs_diff":
                mean_abs_diff,

            "nonzero_difference_values":
                n_nonzero_diff,

            "total_values":
                n_values,
        })

        del signed
        del stored_abs

diagnostic_df = pd.DataFrame(
    diagnostic_rows
)

print()
print(
    "Evaluations checked :",
    len(diagnostic_df)
)

print(
    "Global max abs diff :",
    diagnostic_df["max_abs_diff"].max()
)

print(
    "Global max rel diff :",
    diagnostic_df["max_relative_diff"].max()
)

print(
    "Global mean abs diff:",
    diagnostic_df["mean_abs_diff"].mean()
)

print(
    "Evaluations with exact equality:",
    int(
        (
            diagnostic_df["max_abs_diff"] == 0
        ).sum()
    ),
    "/ 60"
)

print()
print("Largest discrepancies:")
print(
    diagnostic_df
    .sort_values(
        "max_abs_diff",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

print()
print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

T1.4b — DIAGNOSING SIGNED → |IG| RECONSTRUCTION

Evaluations checked : 60
Global max abs diff : 2.384185791015625e-07
Global max rel diff : 2.2798424481607071e-07
Global mean abs diff: 2.0934998828058762e-10
Evaluations with exact equality: 0 / 60

Largest discrepancies:
model_key evaluation  max_abs_diff  max_relative_diff  mean_abs_diff  nonzero_difference_values  total_values
    0.0_1       hair  2.384186e-07       2.009622e-07   2.239875e-10                    1159605       4967424
    0.0_1      clean  2.384186e-07       2.192869e-07   2.133379e-10                    1151433       4967424
    0.0_2      clean  2.384186e-07       2.174800e-07   2.110310e-10                    1148771       4967424
    0.0_3       hair  2.384186e-07       2.130016e-07   2.125927e-10                    1159467       4967424
    0.2_1       hair  2.384186e-07       2.086302e-07   2.236310e-10                    1157360       4967424
    0.0_3      clean  2.384186e-07       2.086447e-07   1.942107e-10

In [61]:
# ================================================================
# CELL 6C-R2 — T1.4b PRODUCTION IG PERSISTENCE AUDIT
#              FLOAT32-SAFE RECONSTRUCTION CRITERION
# ================================================================

import json
import hashlib
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — PRODUCTION IG PERSISTENCE AUDIT")
print("FLOAT32-SAFE RECONSTRUCTION CRITERION")
print("=" * 70)

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

IG_DIR = RESULT_DIR / "T1_4B_PRODUCTION_IG"

HAIR_DIR = IG_DIR / "hair"
CLEAN_DIR = IG_DIR / "clean"

GLOBAL_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_COMPLETE_LOCK.json"
)

ROSTER_CSV = (
    RESULT_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

MANIFEST_CSV = (
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

assert GLOBAL_LOCK.exists()
assert ROSTER_CSV.exists()
assert MANIFEST_CSV.exists()

# ----------------------------------------------------------------
# Frozen audit tolerance
#
# Diagnostic showed:
# max absolute difference = 2.384185791015625e-07
# max relative difference = 2.2798424481607071e-07
#
# Use a conservative tolerance above the observed numerical
# discrepancy, while remaining extremely small relative to IG
# magnitudes.
# ----------------------------------------------------------------

RECON_ATOL = 1e-6
RECON_RTOL = 1e-6

# ----------------------------------------------------------------
# Global completion lock
# ----------------------------------------------------------------

with open(
    GLOBAL_LOCK,
    "r"
) as f:
    global_lock = json.load(f)

assert global_lock["experiment"] == "T1_4B"
assert global_lock["models"] == 30
assert global_lock["retained_images"] == 99
assert global_lock["evaluations_per_pair"] == 2
assert global_lock["expected_ig_evaluations"] == 5940
assert global_lock["completed_evaluations"] == 5940
assert global_lock["failures"] == 0
assert global_lock["status"] == "COMPLETE"

print()
print("Global completion lock : PASS")

# ----------------------------------------------------------------
# Populations
# ----------------------------------------------------------------

roster = pd.read_csv(
    ROSTER_CSV
)

null_manifest = pd.read_csv(
    MANIFEST_CSV
)

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert roster["model_key"].tolist() == EXPECTED_MODEL_KEYS

retained_images = sorted(
    null_manifest["image"]
    .astype(str)
    .unique()
)

assert len(retained_images) == 99

print(
    "Canonical models       :",
    len(EXPECTED_MODEL_KEYS)
)

print(
    "Retained images        :",
    len(retained_images)
)

# ----------------------------------------------------------------
# SHA256 helper
# ----------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ----------------------------------------------------------------
# Audit
# ----------------------------------------------------------------

audit_rows = []

global_max_abs_diff = 0.0
global_max_rel_diff = 0.0

for model_key in EXPECTED_MODEL_KEYS:

    for evaluation, base_dir in [
        ("hair", HAIR_DIR),
        ("clean", CLEAN_DIR),
    ]:

        model_dir = (
            base_dir /
            f"model_{model_key}"
        )

        signed_path = (
            model_dir /
            "signed_ig.npy"
        )

        abs_path = (
            model_dir /
            "abs_ig.npy"
        )

        metadata_path = (
            model_dir /
            "metadata.csv"
        )

        complete_path = (
            model_dir /
            "COMPLETED.json"
        )

        # --------------------------------------------------------
        # Files
        # --------------------------------------------------------

        assert model_dir.exists()
        assert signed_path.exists()
        assert abs_path.exists()
        assert metadata_path.exists()
        assert complete_path.exists()

        # --------------------------------------------------------
        # Completion record
        # --------------------------------------------------------

        with open(
            complete_path,
            "r"
        ) as f:

            record = json.load(f)

        assert record["completed"] is True
        assert record["model_key"] == model_key
        assert record["evaluation"] == evaluation
        assert record["n_images"] == 99

        # --------------------------------------------------------
        # Persisted attribution arrays
        # --------------------------------------------------------

        signed = np.load(
            signed_path,
            mmap_mode="r"
        )

        abs_ig = np.load(
            abs_path,
            mmap_mode="r"
        )

        assert signed.shape == (
            99,
            3,
            224,
            224
        )

        assert abs_ig.shape == (
            99,
            224,
            224
        )

        assert signed.dtype == np.float32
        assert abs_ig.dtype == np.float32

        # --------------------------------------------------------
        # Finite checks
        # --------------------------------------------------------

        assert np.isfinite(
            signed
        ).all()

        assert np.isfinite(
            abs_ig
        ).all()

        assert float(
            abs_ig.min()
        ) >= 0.0

        # --------------------------------------------------------
        # Metadata
        # --------------------------------------------------------

        metadata = pd.read_csv(
            metadata_path
        )

        assert len(metadata) == 99

        required_columns = [
            "model_key",
            "image",
            "evaluation",
            "enrichment",
            "undefined",
            "completeness_error",
            "delta",
            "tolerance",
            "steps_used",
            "completeness_failed",
            "signed_attribution_inside_mask",
            "mask_area_fraction",
        ]

        for col in required_columns:
            assert col in metadata.columns

        assert (
            metadata[
                "model_key"
            ].astype(str).tolist()
            ==
            [model_key] * 99
        )

        assert (
            metadata[
                "evaluation"
            ].astype(str).tolist()
            ==
            [evaluation] * 99
        )

        assert (
            metadata[
                "image"
            ].astype(str).tolist()
            ==
            retained_images
        )

        assert metadata[
            "steps_used"
        ].isin([50, 100]).all()

        assert metadata[
            "mask_area_fraction"
        ].gt(0).all()

        assert metadata[
            "mask_area_fraction"
        ].lt(1).all()

        assert np.isfinite(
            metadata[
                "completeness_error"
            ].to_numpy()
        ).all()

        assert np.isfinite(
            metadata[
                "delta"
            ].to_numpy()
        ).all()

        assert np.isfinite(
            metadata[
                "tolerance"
            ].to_numpy()
        ).all()

        # --------------------------------------------------------
        # Undefined / enrichment
        # --------------------------------------------------------

        undefined = (
            metadata[
                "undefined"
            ].astype(bool)
            .to_numpy()
        )

        enrichment = (
            metadata[
                "enrichment"
            ].to_numpy()
        )

        defined = ~undefined

        if defined.any():

            assert np.isfinite(
                enrichment[defined]
            ).all()

        # --------------------------------------------------------
        # Reconstruct |IG| from signed attribution.
        #
        # Compare using the empirically validated float32-safe
        # tolerance rather than bit-exact equality.
        # --------------------------------------------------------

        local_max_abs = 0.0
        local_max_rel = 0.0

        for start in range(
            0,
            99,
            10
        ):

            stop = min(
                start + 10,
                99
            )

            signed_chunk = np.asarray(
                signed[start:stop]
            )

            stored_chunk = np.asarray(
                abs_ig[start:stop]
            )

            derived_abs = (
                np.abs(
                    signed_chunk
                ).sum(axis=1)
            )

            diff = np.abs(
                derived_abs -
                stored_chunk
            )

            local_max_abs = max(
                local_max_abs,
                float(diff.max())
            )

            denominator = np.maximum(
                np.abs(stored_chunk),
                1e-12
            )

            relative_diff = (
                diff /
                denominator
            )

            local_max_rel = max(
                local_max_rel,
                float(relative_diff.max())
            )

            assert np.allclose(
                derived_abs,
                stored_chunk,
                rtol=RECON_RTOL,
                atol=RECON_ATOL
            )

        global_max_abs_diff = max(
            global_max_abs_diff,
            local_max_abs
        )

        global_max_rel_diff = max(
            global_max_rel_diff,
            local_max_rel
        )

        # --------------------------------------------------------
        # File hashes
        # --------------------------------------------------------

        signed_sha = sha256_file(
            signed_path
        )

        abs_sha = sha256_file(
            abs_path
        )

        metadata_sha = sha256_file(
            metadata_path
        )

        assert (
            signed_sha ==
            record["signed_sha256"]
        )

        assert (
            abs_sha ==
            record["abs_sha256"]
        )

        assert (
            metadata_sha ==
            record["metadata_sha256"]
        )

        # --------------------------------------------------------
        # Audit row
        # --------------------------------------------------------

        audit_rows.append({

            "model_key":
                model_key,

            "evaluation":
                evaluation,

            "n_images":
                99,

            "signed_shape":
                str(signed.shape),

            "abs_shape":
                str(abs_ig.shape),

            "signed_finite":
                True,

            "abs_finite":
                True,

            "undefined_count":
                int(undefined.sum()),

            "defined_count":
                int(defined.sum()),

            "retry_100_count":
                int(
                    (
                        metadata[
                            "steps_used"
                        ] == 100
                    ).sum()
                ),

            "completeness_failed_count":
                int(
                    metadata[
                        "completeness_failed"
                    ].astype(bool).sum()
                ),

            "max_abs_reconstruction_diff":
                local_max_abs,

            "max_relative_reconstruction_diff":
                local_max_rel,

            "reconstruction_pass":
                True,

            "signed_sha256_match":
                True,

            "abs_sha256_match":
                True,

            "metadata_sha256_match":
                True,
        })

        del signed
        del abs_ig

# ----------------------------------------------------------------
# Global checks
# ----------------------------------------------------------------

audit_df = pd.DataFrame(
    audit_rows
)

assert len(audit_df) == 60

assert audit_df[
    "n_images"
].eq(99).all()

assert audit_df[
    "signed_finite"
].all()

assert audit_df[
    "abs_finite"
].all()

assert audit_df[
    "reconstruction_pass"
].all()

assert audit_df[
    "signed_sha256_match"
].all()

assert audit_df[
    "abs_sha256_match"
].all()

assert audit_df[
    "metadata_sha256_match"
].all()

total_evaluations = int(
    audit_df[
        "n_images"
    ].sum()
)

total_undefined = int(
    audit_df[
        "undefined_count"
    ].sum()
)

total_defined = int(
    audit_df[
        "defined_count"
    ].sum()
)

total_retry = int(
    audit_df[
        "retry_100_count"
    ].sum()
)

total_completeness_failed = int(
    audit_df[
        "completeness_failed_count"
    ].sum()
)

assert total_evaluations == 5940

assert (
    total_defined +
    total_undefined
    ==
    5940
)

# ----------------------------------------------------------------
# Save audit artifacts
# ----------------------------------------------------------------

AUDIT_CSV = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT.csv"
)

audit_df.to_csv(
    AUDIT_CSV,
    index=False
)

AUDIT_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT_LOCK.json"
)

audit_lock = {

    "experiment":
        "T1_4B",

    "models":
        30,

    "retained_images":
        99,

    "expected_evaluations":
        5940,

    "audited_evaluations":
        total_evaluations,

    "undefined_evaluations":
        total_undefined,

    "defined_evaluations":
        total_defined,

    "retry_100_steps":
        total_retry,

    "completeness_failed":
        total_completeness_failed,

    "reconstruction_test":
        "np.allclose",

    "reconstruction_rtol":
        RECON_RTOL,

    "reconstruction_atol":
        RECON_ATOL,

    "observed_global_max_abs_difference":
        global_max_abs_diff,

    "observed_global_max_relative_difference":
        global_max_rel_diff,

    "all_shapes_valid":
        True,

    "all_finite":
        True,

    "signed_abs_reconstruction_pass":
        True,

    "all_file_hashes_match":
        True,

    "status":
        "PASS",
}

with open(
    AUDIT_LOCK,
    "w"
) as f:

    json.dump(
        audit_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CELL 6C-R2 — PRODUCTION IG AUDIT COMPLETE")
print("=" * 70)

print(
    "Expected evaluations        :",
    5940
)

print(
    "Audited evaluations         :",
    total_evaluations
)

print(
    "Defined evaluations         :",
    total_defined
)

print(
    "Undefined evaluations       :",
    total_undefined
)

print(
    "100-step retry evaluations  :",
    total_retry
)

print(
    "Completeness failed         :",
    total_completeness_failed
)

print()
print(
    "Maximum abs reconstruction :",
    global_max_abs_diff
)

print(
    "Maximum relative reconstr. :",
    global_max_rel_diff
)

print()
print(
    "Shape audit                 : PASS"
)

print(
    "Finite-value audit          : PASS"
)

print(
    "Signed → |IG| audit         : PASS"
)

print(
    "SHA256 persistence audit    : PASS"
)

print()
print("Audit CSV:")
print(AUDIT_CSV)

print()
print("Audit lock:")
print(AUDIT_LOCK)

print()
print("=" * 70)
print("NO NULL SCORING PERFORMED")
print("=" * 70)

T1.4b — PRODUCTION IG PERSISTENCE AUDIT
FLOAT32-SAFE RECONSTRUCTION CRITERION

Global completion lock : PASS
Canonical models       : 30
Retained images        : 99

CELL 6C-R2 — PRODUCTION IG AUDIT COMPLETE
Expected evaluations        : 5940
Audited evaluations         : 5940
Defined evaluations         : 5940
Undefined evaluations       : 0
100-step retry evaluations  : 2647
Completeness failed         : 1254

Maximum abs reconstruction : 2.384185791015625e-07
Maximum relative reconstr. : 2.2798424481607071e-07

Shape audit                 : PASS
Finite-value audit          : PASS
Signed → |IG| audit         : PASS
SHA256 persistence audit    : PASS

Audit CSV:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_PRODUCTION_IG/T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT.csv

Audit lock:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_PRODUCTION_IG/T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT_LOCK.json

NO NULL SCORING PERFORMED


In [62]:
# ================================================================
# CELL 6D — T1.4b NULL ENRICHMENT SCORING
# ================================================================
#
# Frozen protocol:
#
#   True enrichment:
#       E_true = (sum |IG| inside true mask / total |IG|)
#                / true_mask_area_fraction
#
#   Null enrichment:
#       E_null = (sum |IG| inside null mask / total |IG|)
#                / null_mask_area_fraction
#
#   Per-image percentile:
#       p = (1 + count(E_null >= E_true)) / (1 + N_null)
#
#       percentile = 1 - p
#
#   N_null = 50 for each design.
#
#   Model-level statistic:
#       M_m = median(image-level percentiles)
#
#   Designs:
#       A     = primary confirmatory
#       Cprime = sensitivity analysis
#
#   No z-score / SD standardization.
#   No new IG computation.
# ================================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — NULL ENRICHMENT SCORING")
print("=" * 70)

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

BASE_DIR = RESULT_DIR

IG_DIR = (
    BASE_DIR /
    "T1_4B_PRODUCTION_IG"
)

HAIR_DIR = (
    IG_DIR /
    "hair"
)

NULL_DIR = (
    BASE_DIR /
    "T1_4B_NULL_MASKS"
)

NULL_MANIFEST = (
    BASE_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

DROPPED_CSV = (
    BASE_DIR /
    "T1_4B_DROPPED_IMAGES.csv"
)

IG_AUDIT_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT_LOCK.json"
)

assert IG_AUDIT_LOCK.exists()
assert NULL_MANIFEST.exists()

# ----------------------------------------------------------------
# Verify IG audit passed before scoring
# ----------------------------------------------------------------

with open(
    IG_AUDIT_LOCK,
    "r"
) as f:
    ig_audit_lock = json.load(f)

assert (
    ig_audit_lock["status"]
    ==
    "PASS"
)

assert (
    ig_audit_lock[
        "expected_evaluations"
    ]
    ==
    5940
)

assert (
    ig_audit_lock[
        "audited_evaluations"
    ]
    ==
    5940
)

assert (
    ig_audit_lock[
        "signed_abs_reconstruction_pass"
    ]
    is True
)

# ----------------------------------------------------------------
# Frozen model roster
# ----------------------------------------------------------------

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert len(EXPECTED_MODEL_KEYS) == 30

# ----------------------------------------------------------------
# Load null manifest
# ----------------------------------------------------------------

null_manifest = pd.read_csv(
    NULL_MANIFEST
)

required_manifest_columns = [
    "image",
    "design",
    "null_index",
    "accepted_attempt",
    "angle_deg",
    "reflect_x",
    "reflect_y",
    "dx",
    "dy",
    "true_area",
    "null_area",
    "true_center_distance",
    "null_center_distance",
    "center_distance_abs_diff",
]

for col in required_manifest_columns:
    assert col in null_manifest.columns

assert set(
    null_manifest["design"].unique()
) == {
    "A",
    "Cprime",
}

# ----------------------------------------------------------------
# Retained image population
# ----------------------------------------------------------------

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(retained_images) == 99

# Exactly 50 nulls per image/design.
null_counts = (
    null_manifest
    .groupby(
        ["image", "design"]
    )
    .size()
)

assert null_counts.eq(50).all()

# Null indices must be 0..49.
null_index_sets = (
    null_manifest
    .groupby(
        ["image", "design"]
    )["null_index"]
    .apply(
        lambda x:
        sorted(
            x.astype(int).tolist()
        )
        ==
        list(range(50))
    )
)

assert null_index_sets.all()

print()
print(
    "Retained images :",
    len(retained_images)
)

print(
    "Null masks      :",
    len(null_manifest)
)

print(
    "A masks         :",
    int(
        (
            null_manifest["design"]
            == "A"
        ).sum()
    )
)

print(
    "C' masks        :",
    int(
        (
            null_manifest["design"]
            == "Cprime"
        ).sum()
    )
)

# ----------------------------------------------------------------
# Locate null NPZ by manifest image/design.
#
# Production NPZ filenames are hashed stems, so use the frozen
# filename mapping audit rather than guessing filenames.
# ----------------------------------------------------------------

MAPPING_AUDIT = (
    BASE_DIR /
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

assert MAPPING_AUDIT.exists()

mapping_df = pd.read_csv(
    MAPPING_AUDIT
)

print()
print(
    "Mapping audit columns:",
    mapping_df.columns.tolist()
)

# The mapping audit must contain one row per retained image.
assert len(mapping_df) == 99

# Identify the image/hash columns robustly.
image_candidates = [
    c for c in mapping_df.columns
    if c.lower() in {
        "image",
        "image_id",
        "original_image"
    }
]

hash_candidates = [
    c for c in mapping_df.columns
    if c.lower() in {
        "hash",
        "stem",
        "filename_stem",
        "null_hash"
    }
]

assert len(image_candidates) >= 1
assert len(hash_candidates) >= 1

mapping_image_col = image_candidates[0]
mapping_hash_col = hash_candidates[0]

mapping = dict(
    zip(
        mapping_df[
            mapping_image_col
        ].astype(str),

        mapping_df[
            mapping_hash_col
        ].astype(str)
    )
)

assert set(mapping.keys()) == set(
    retained_images
)

# ----------------------------------------------------------------
# True masks
# ----------------------------------------------------------------
#
# Reconstruct from frozen generator exactly as used in Cell 2C.
# This does NOT alter the locked null population.
# ----------------------------------------------------------------

cache_index = pd.read_csv(
    "/kaggle/temp/isic_cache/cache_index.csv"
)

cache_lookup = dict(
    zip(
        cache_index[
            "image"
        ].astype(str),

        cache_index.index.astype(int)
    )
)

assert set(
    retained_images
).issubset(
    cache_lookup.keys()
)

# Frozen generator symbols are already available from earlier cells.
# If not, load the frozen module directly.
if "generate_hair_mask" not in globals():

    import importlib.util

    GENERATOR_PATH = (
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/"
        "isic-final-project/"
        "step1J_generator_recovery_lock/"
        "recovered_original_hair_generator.py"
    )

    spec = importlib.util.spec_from_file_location(
        "recovered_original_hair_generator",
        GENERATOR_PATH
    )

    generator_module = (
        importlib.util.module_from_spec(
            spec
        )
    )

    spec.loader.exec_module(
        generator_module
    )

    generate_hair_mask = (
        generator_module.generate_hair_mask
    )

# ----------------------------------------------------------------
# Cache arrays
# ----------------------------------------------------------------

clean_cache = np.load(
    "/kaggle/temp/isic_cache/clean_224.npy",
    mmap_mode="r"
)

hair_cache = np.load(
    "/kaggle/temp/isic_cache/hair_224.npy",
    mmap_mode="r"
)

assert clean_cache.shape[1:] == (
    224,
    224,
    3
)

# ----------------------------------------------------------------
# Build exact true masks
# ----------------------------------------------------------------

true_masks = {}

for image in retained_images:

    idx = cache_lookup[image]

    clean_img = np.asarray(
        clean_cache[idx]
    )

    hair_img = np.asarray(
        hair_cache[idx]
    )

    mask = generate_hair_mask(
        clean_img,
        hair_img
    )

    mask = np.asarray(
        mask
    ).astype(bool)

    assert mask.shape == (
        224,
        224
    )

    true_masks[image] = mask

# ----------------------------------------------------------------
# Verify true mask area against null manifest.
# ----------------------------------------------------------------

manifest_true_area = (
    null_manifest[
        [
            "image",
            "true_area"
        ]
    ]
    .drop_duplicates()
    .set_index("image")
)

for image in retained_images:

    true_area = int(
        true_masks[image].sum()
    )

    expected_area = int(
        manifest_true_area.loc[
            image,
            "true_area"
        ]
    )

    assert true_area == expected_area

print(
    "True-mask reconstruction : PASS"
)

# ----------------------------------------------------------------
# Locate null NPZs
# ----------------------------------------------------------------

null_npz = {}

for image in retained_images:

    stem = mapping[image]

    a_path = (
        NULL_DIR /
        f"{stem}_A.npz"
    )

    c_path = (
        NULL_DIR /
        f"{stem}_Cprime.npz"
    )

    assert a_path.exists(), a_path
    assert c_path.exists(), c_path

    null_npz[(image, "A")] = a_path
    null_npz[(image, "Cprime")] = c_path

# ----------------------------------------------------------------
# Helper: enrichment from spatial |IG| map and mask
# ----------------------------------------------------------------

def compute_enrichment(
    abs_spatial,
    mask
):

    abs_spatial = np.asarray(
        abs_spatial,
        dtype=np.float64
    )

    mask = np.asarray(
        mask,
        dtype=bool
    )

    total = float(
        abs_spatial.sum()
    )

    area_fraction = float(
        mask.mean()
    )

    if (
        total <= 1e-8
        or
        area_fraction == 0.0
    ):

        return np.nan

    inside = float(
        abs_spatial[mask].sum()
    )

    return (
        (inside / total)
        /
        area_fraction
    )

# ----------------------------------------------------------------
# Score all model × image × design combinations
# ----------------------------------------------------------------

score_rows = []

for model_idx, model_key in enumerate(
    EXPECTED_MODEL_KEYS,
    start=1
):

    abs_path = (
        HAIR_DIR /
        f"model_{model_key}" /
        "abs_ig.npy"
    )

    metadata_path = (
        HAIR_DIR /
        f"model_{model_key}" /
        "metadata.csv"
    )

    assert abs_path.exists()
    assert metadata_path.exists()

    abs_ig = np.load(
        abs_path,
        mmap_mode="r"
    )

    metadata = pd.read_csv(
        metadata_path
    )

    assert len(metadata) == 99

    assert (
        metadata[
            "image"
        ].astype(str).tolist()
        ==
        retained_images
    )

    print()
    print(
        f"MODEL {model_idx:02d}/30 — {model_key}"
    )

    for image_idx, image in enumerate(
        retained_images
    ):

        abs_spatial = np.asarray(
            abs_ig[
                image_idx
            ]
        )

        # --------------------------------------------------------
        # True enrichment
        # --------------------------------------------------------

        true_mask = (
            true_masks[image]
        )

        true_enrichment = (
            compute_enrichment(
                abs_spatial,
                true_mask
            )
        )

        # Cross-check against production metadata.
        production_enrichment = float(
            metadata.loc[
                image_idx,
                "enrichment"
            ]
        )

        assert np.isfinite(
            true_enrichment
        )

        assert np.isclose(
            true_enrichment,
            production_enrichment,
            rtol=1e-5,
            atol=1e-6
        ), (
            model_key,
            image,
            true_enrichment,
            production_enrichment
        )

        # --------------------------------------------------------
        # Null scoring
        # --------------------------------------------------------

        for design in [
            "A",
            "Cprime"
        ]:

            npz_path = (
                null_npz[
                    (image, design)
                ]
            )

            null_data = np.load(
                npz_path
            )

            # The generator stores 50 masks.
            null_masks = (
                null_data[
                    "masks"
                ]
            )

            assert null_masks.shape == (
                50,
                224,
                224
            )

            null_enrichments = np.empty(
                50,
                dtype=np.float64
            )

            for k in range(50):

                null_enrichments[k] = (
                    compute_enrichment(
                        abs_spatial,
                        null_masks[k]
                    )
                )

            assert np.isfinite(
                null_enrichments
            ).all()

            # ----------------------------------------------------
            # Frozen randomization p-value
            #
            # Inclusive comparison:
            # null >= observed
            #
            # +1 correction:
            # p = (1 + exceedances) / 51
            # ----------------------------------------------------

            exceedances = int(
                np.sum(
                    null_enrichments
                    >=
                    true_enrichment
                )
            )

            p_value = (
                1.0 +
                exceedances
            ) / 51.0

            percentile = (
                1.0 -
                p_value
            )

            assert (
                0.0 <=
                percentile <=
                1.0
            )

            score_rows.append({

                "model_key":
                    model_key,

                "rho":
                    float(
                        model_key.split("_")[0]
                    ),

                "replicate":
                    int(
                        model_key.split("_")[1]
                    ),

                "image":
                    image,

                "image_index":
                    image_idx,

                "design":
                    design,

                "true_enrichment":
                    true_enrichment,

                "null_mean":
                    float(
                        null_enrichments.mean()
                    ),

                "null_median":
                    float(
                        np.median(
                            null_enrichments
                        )
                    ),

                "null_min":
                    float(
                        null_enrichments.min()
                    ),

                "null_max":
                    float(
                        null_enrichments.max()
                    ),

                "null_ge_true":
                    exceedances,

                "n_null":
                    50,

                "p_value":
                    p_value,

                "percentile":
                    percentile,
            })

    print(
        f"  {model_key} COMPLETE"
    )

# ----------------------------------------------------------------
# Score table
# ----------------------------------------------------------------

score_df = pd.DataFrame(
    score_rows
)

assert len(score_df) == (
    30 *
    99 *
    2
)

assert score_df[
    [
        "true_enrichment",
        "null_mean",
        "null_median",
        "null_min",
        "null_max",
        "p_value",
        "percentile",
    ]
].apply(
    np.isfinite
).all().all()

# ----------------------------------------------------------------
# Population integrity
# ----------------------------------------------------------------

assert set(
    score_df["model_key"]
) == set(
    EXPECTED_MODEL_KEYS
)

assert set(
    score_df["design"]
) == {
    "A",
    "Cprime"
}

assert (
    score_df
    .groupby(
        [
            "model_key",
            "image",
            "design"
        ]
    )
    .size()
    .eq(1)
    .all()
)

assert (
    score_df
    .groupby(
        [
            "model_key",
            "design"
        ]
    )
    .size()
    .eq(99)
    .all()
)

# ----------------------------------------------------------------
# Model-level median percentile
# ----------------------------------------------------------------

model_summary_rows = []

for model_key in EXPECTED_MODEL_KEYS:

    row = {
        "model_key":
            model_key,

        "rho":
            float(
                model_key.split("_")[0]
            ),

        "replicate":
            int(
                model_key.split("_")[1]
            ),
    }

    for design in [
        "A",
        "Cprime"
    ]:

        values = (
            score_df[
                (
                    score_df[
                        "model_key"
                    ]
                    ==
                    model_key
                )
                &
                (
                    score_df[
                        "design"
                    ]
                    ==
                    design
                )
            ]["percentile"]
            .to_numpy()
        )

        assert len(values) == 99
        assert np.isfinite(values).all()

        row[
            f"M_{design}"
        ] = float(
            np.median(values)
        )

    model_summary_rows.append(
        row
    )

model_summary = pd.DataFrame(
    model_summary_rows
)

assert len(model_summary) == 30

# ----------------------------------------------------------------
# Save scoring artifacts
# ----------------------------------------------------------------

SCORE_CSV = (
    BASE_DIR /
    "T1_4B_NULL_ENRICHMENT_SCORES.csv"
)

MODEL_SUMMARY_CSV = (
    BASE_DIR /
    "T1_4B_MODEL_LEVEL_PERCENTILES.csv"
)

score_df.to_csv(
    SCORE_CSV,
    index=False
)

model_summary.to_csv(
    MODEL_SUMMARY_CSV,
    index=False
)

# ----------------------------------------------------------------
# Population lock
# ----------------------------------------------------------------

SCORING_LOCK = (
    BASE_DIR /
    "T1_4B_NULL_SCORING_LOCK.json"
)

scoring_lock = {

    "experiment":
        "T1_4B",

    "models":
        30,

    "retained_images":
        99,

    "designs":
        [
            "A",
            "Cprime"
        ],

    "nulls_per_image_design":
        50,

    "score_rows":
        int(len(score_df)),

    "expected_score_rows":
        5940,

    "model_level_rows":
        int(len(model_summary)),

    "percentile_definition":
        "(1 + count(null >= true)) / 51; percentile = 1 - p",

    "model_level_statistic":
        "median across 99 retained images",

    "standardization":
        "none",

    "ig_recomputation":
        False,

    "status":
        "PASS",
}

with open(
    SCORING_LOCK,
    "w"
) as f:

    json.dump(
        scoring_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CELL 6D — NULL ENRICHMENT SCORING COMPLETE")
print("=" * 70)

print(
    "Models                    :",
    30
)

print(
    "Retained images            :",
    99
)

print(
    "Nulls / image / design     :",
    50
)

print(
    "Score rows                 :",
    len(score_df)
)

print(
    "Model-level rows           :",
    len(model_summary)
)

print()
print(
    "Design A scored            :",
    int(
        (
            score_df["design"]
            == "A"
        ).sum()
    )
)

print(
    "Design C' scored           :",
    int(
        (
            score_df["design"]
            == "Cprime"
        ).sum()
    )
)

print()
print("Score CSV:")
print(SCORE_CSV)

print()
print("Model summary CSV:")
print(MODEL_SUMMARY_CSV)

print()
print("Scoring lock:")
print(SCORING_LOCK)

print()
print("=" * 70)
print("T1.4b NULL SCORING PASS")
print("NO INFERENTIAL TEST PERFORMED")
print("=" * 70)

T1.4b — NULL ENRICHMENT SCORING

Retained images : 99
Null masks      : 9900
A masks         : 4950
C' masks        : 4950

Mapping audit columns: ['production_index', 'image', 'A_stem', 'Cprime_stem', 'A_exact_match', 'Cprime_exact_match', 'A_area_ok', 'Cprime_area_ok', 'A_zero_overlap_ok', 'Cprime_zero_overlap_ok', 'true_area', 'A_min_area', 'A_max_area', 'Cprime_min_area', 'Cprime_max_area', 'A_max_overlap', 'Cprime_max_overlap']


AssertionError: 

In [63]:
# ================================================================
# CELL 6D-R2 — T1.4b NULL ENRICHMENT SCORING
#              CORRECT BIT-EXACT A / C' STEM MAPPING
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — NULL ENRICHMENT SCORING")
print("CORRECT A / C' NULL STEM MAPPING")
print("=" * 70)

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

BASE_DIR = RESULT_DIR

IG_DIR = (
    BASE_DIR /
    "T1_4B_PRODUCTION_IG"
)

HAIR_DIR = (
    IG_DIR /
    "hair"
)

NULL_DIR = (
    BASE_DIR /
    "T1_4B_NULL_MASKS"
)

NULL_MANIFEST = (
    BASE_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

IG_AUDIT_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT_LOCK.json"
)

MAPPING_AUDIT = (
    BASE_DIR /
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

assert IG_AUDIT_LOCK.exists()
assert NULL_MANIFEST.exists()
assert MAPPING_AUDIT.exists()

# ----------------------------------------------------------------
# Verify production IG audit passed
# ----------------------------------------------------------------

with open(
    IG_AUDIT_LOCK,
    "r"
) as f:
    ig_audit_lock = json.load(f)

assert (
    ig_audit_lock["status"]
    ==
    "PASS"
)

assert (
    ig_audit_lock[
        "expected_evaluations"
    ]
    ==
    5940
)

assert (
    ig_audit_lock[
        "audited_evaluations"
    ]
    ==
    5940
)

assert (
    ig_audit_lock[
        "signed_abs_reconstruction_pass"
    ]
    is True
)

print()
print("Production IG audit : PASS")

# ----------------------------------------------------------------
# Frozen model roster
# ----------------------------------------------------------------

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert len(
    EXPECTED_MODEL_KEYS
) == 30

# ----------------------------------------------------------------
# Load null manifest
# ----------------------------------------------------------------

null_manifest = pd.read_csv(
    NULL_MANIFEST
)

required_manifest_columns = [
    "image",
    "design",
    "null_index",
    "accepted_attempt",
    "angle_deg",
    "reflect_x",
    "reflect_y",
    "dx",
    "dy",
    "true_area",
    "null_area",
    "true_center_distance",
    "null_center_distance",
    "center_distance_abs_diff",
]

for col in required_manifest_columns:
    assert col in null_manifest.columns

assert set(
    null_manifest["design"].unique()
) == {
    "A",
    "Cprime"
}

# ----------------------------------------------------------------
# Retained population
# ----------------------------------------------------------------

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(
    retained_images
) == 99

null_counts = (
    null_manifest
    .groupby(
        ["image", "design"]
    )
    .size()
)

assert null_counts.eq(50).all()

null_index_sets = (
    null_manifest
    .groupby(
        ["image", "design"]
    )["null_index"]
    .apply(
        lambda x:
        sorted(
            x.astype(int).tolist()
        )
        ==
        list(range(50))
    )
)

assert null_index_sets.all()

print(
    "Retained images     :",
    len(retained_images)
)

print(
    "Null masks          :",
    len(null_manifest)
)

print(
    "Design A masks      :",
    int(
        (
            null_manifest[
                "design"
            ]
            ==
            "A"
        ).sum()
    )
)

print(
    "Design C' masks     :",
    int(
        (
            null_manifest[
                "design"
            ]
            ==
            "Cprime"
        ).sum()
    )
)

# ----------------------------------------------------------------
# Correct filename mapping
# ----------------------------------------------------------------
#
# The audit contains:
#
#   image
#   A_stem
#   Cprime_stem
#
# These are the exact production NPZ filename stems.
# ----------------------------------------------------------------

mapping_df = pd.read_csv(
    MAPPING_AUDIT
)

expected_mapping_columns = [
    "image",
    "A_stem",
    "Cprime_stem",
    "A_exact_match",
    "Cprime_exact_match",
    "A_area_ok",
    "Cprime_area_ok",
    "A_zero_overlap_ok",
    "Cprime_zero_overlap_ok",
]

for col in expected_mapping_columns:
    assert col in mapping_df.columns

assert len(
    mapping_df
) == 99

assert mapping_df[
    "A_exact_match"
].astype(bool).all()

assert mapping_df[
    "Cprime_exact_match"
].astype(bool).all()

assert mapping_df[
    "A_area_ok"
].astype(bool).all()

assert mapping_df[
    "Cprime_area_ok"
].astype(bool).all()

assert mapping_df[
    "A_zero_overlap_ok"
].astype(bool).all()

assert mapping_df[
    "Cprime_zero_overlap_ok"
].astype(bool).all()

assert set(
    mapping_df[
        "image"
    ].astype(str)
) == set(
    retained_images
)

A_STEM = dict(
    zip(
        mapping_df[
            "image"
        ].astype(str),

        mapping_df[
            "A_stem"
        ].astype(str)
    )
)

CP_STEM = dict(
    zip(
        mapping_df[
            "image"
        ].astype(str),

        mapping_df[
            "Cprime_stem"
        ].astype(str)
    )
)

assert set(
    A_STEM.keys()
) == set(
    retained_images
)

assert set(
    CP_STEM.keys()
) == set(
    retained_images
)

print()
print(
    "Bit-exact A mapping   : PASS"
)

print(
    "Bit-exact C' mapping  : PASS"
)

# ----------------------------------------------------------------
# Build exact true masks
# ----------------------------------------------------------------

cache_index = pd.read_csv(
    "/kaggle/temp/isic_cache/cache_index.csv"
)

cache_lookup = dict(
    zip(
        cache_index[
            "image"
        ].astype(str),

        cache_index.index.astype(int)
    )
)

assert set(
    retained_images
).issubset(
    cache_lookup.keys()
)

# Frozen generator
if "generate_hair_mask" not in globals():

    import importlib.util

    GENERATOR_PATH = (
        "/kaggle/input/datasets/"
        "tejasseshadriiiii/"
        "isic-final-project/"
        "step1J_generator_recovery_lock/"
        "recovered_original_hair_generator.py"
    )

    spec = importlib.util.spec_from_file_location(
        "recovered_original_hair_generator",
        GENERATOR_PATH
    )

    generator_module = (
        importlib.util.module_from_spec(
            spec
        )
    )

    spec.loader.exec_module(
        generator_module
    )

    generate_hair_mask = (
        generator_module.generate_hair_mask
    )

clean_cache = np.load(
    "/kaggle/temp/isic_cache/clean_224.npy",
    mmap_mode="r"
)

hair_cache = np.load(
    "/kaggle/temp/isic_cache/hair_224.npy",
    mmap_mode="r"
)

true_masks = {}

for image in retained_images:

    idx = cache_lookup[
        image
    ]

    clean_img = np.asarray(
        clean_cache[idx]
    )

    hair_img = np.asarray(
        hair_cache[idx]
    )

    mask = generate_hair_mask(
        clean_img,
        hair_img
    )

    mask = np.asarray(
        mask
    ).astype(bool)

    assert mask.shape == (
        224,
        224
    )

    true_masks[image] = mask

# ----------------------------------------------------------------
# True-area cross-check
# ----------------------------------------------------------------

manifest_true_area = (
    null_manifest[
        [
            "image",
            "true_area"
        ]
    ]
    .drop_duplicates()
    .set_index(
        "image"
    )
)

for image in retained_images:

    observed_area = int(
        true_masks[
            image
        ].sum()
    )

    expected_area = int(
        manifest_true_area.loc[
            image,
            "true_area"
        ]
    )

    assert (
        observed_area
        ==
        expected_area
    )

print(
    "True-mask reconstruction : PASS"
)

# ----------------------------------------------------------------
# Null NPZ lookup
# ----------------------------------------------------------------

null_npz = {}

for image in retained_images:

    a_stem = A_STEM[
        image
    ]

    cp_stem = CP_STEM[
        image
    ]

    a_path = (
        NULL_DIR /
        f"{a_stem}_A.npz"
    )

    cp_path = (
        NULL_DIR /
        f"{cp_stem}_Cprime.npz"
    )

    assert a_path.exists(), a_path
    assert cp_path.exists(), cp_path

    null_npz[
        (image, "A")
    ] = a_path

    null_npz[
        (image, "Cprime")
    ] = cp_path

print(
    "Null NPZ resolution       : PASS"
)

# ----------------------------------------------------------------
# Enrichment function
# ----------------------------------------------------------------

def compute_enrichment(
    abs_spatial,
    mask
):

    abs_spatial = np.asarray(
        abs_spatial,
        dtype=np.float64
    )

    mask = np.asarray(
        mask,
        dtype=bool
    )

    total = float(
        abs_spatial.sum()
    )

    area_fraction = float(
        mask.mean()
    )

    if (
        total <= 1e-8
        or
        area_fraction == 0.0
    ):
        return np.nan

    inside = float(
        abs_spatial[
            mask
        ].sum()
    )

    return (
        (inside / total)
        /
        area_fraction
    )

# ----------------------------------------------------------------
# Score all 30 × 99 × 2 combinations
# ----------------------------------------------------------------

score_rows = []

for model_idx, model_key in enumerate(
    EXPECTED_MODEL_KEYS,
    start=1
):

    model_dir = (
        HAIR_DIR /
        f"model_{model_key}"
    )

    abs_path = (
        model_dir /
        "abs_ig.npy"
    )

    metadata_path = (
        model_dir /
        "metadata.csv"
    )

    assert abs_path.exists()
    assert metadata_path.exists()

    abs_ig = np.load(
        abs_path,
        mmap_mode="r"
    )

    metadata = pd.read_csv(
        metadata_path
    )

    assert len(
        metadata
    ) == 99

    assert (
        metadata[
            "image"
        ].astype(str).tolist()
        ==
        retained_images
    )

    print()
    print(
        f"MODEL {model_idx:02d}/30 — {model_key}"
    )

    for image_idx, image in enumerate(
        retained_images
    ):

        abs_spatial = np.asarray(
            abs_ig[
                image_idx
            ]
        )

        true_mask = (
            true_masks[
                image
            ]
        )

        true_enrichment = (
            compute_enrichment(
                abs_spatial,
                true_mask
            )
        )

        production_enrichment = float(
            metadata.loc[
                image_idx,
                "enrichment"
            ]
        )

        assert np.isfinite(
            true_enrichment
        )

        assert np.isclose(
            true_enrichment,
            production_enrichment,
            rtol=1e-5,
            atol=1e-6
        ), (
            "True enrichment mismatch",
            model_key,
            image,
            true_enrichment,
            production_enrichment
        )

        for design in [
            "A",
            "Cprime"
        ]:

            npz_path = (
                null_npz[
                    (image, design)
                ]
            )

            with np.load(
                npz_path
            ) as null_data:

                null_masks = (
                    null_data[
                        "masks"
                    ]
                )

                assert null_masks.shape == (
                    50,
                    224,
                    224
                )

                null_enrichments = np.empty(
                    50,
                    dtype=np.float64
                )

                for k in range(50):

                    null_enrichments[k] = (
                        compute_enrichment(
                            abs_spatial,
                            null_masks[k]
                        )
                    )

            assert np.isfinite(
                null_enrichments
            ).all()

            exceedances = int(
                np.sum(
                    null_enrichments
                    >=
                    true_enrichment
                )
            )

            p_value = (
                1.0 +
                exceedances
            ) / 51.0

            percentile = (
                1.0 -
                p_value
            )

            assert (
                0.0 <=
                percentile <=
                1.0
            )

            score_rows.append({

                "model_key":
                    model_key,

                "rho":
                    float(
                        model_key.split("_")[0]
                    ),

                "replicate":
                    int(
                        model_key.split("_")[1]
                    ),

                "image":
                    image,

                "image_index":
                    image_idx,

                "design":
                    design,

                "true_enrichment":
                    true_enrichment,

                "null_mean":
                    float(
                        null_enrichments.mean()
                    ),

                "null_median":
                    float(
                        np.median(
                            null_enrichments
                        )
                    ),

                "null_min":
                    float(
                        null_enrichments.min()
                    ),

                "null_max":
                    float(
                        null_enrichments.max()
                    ),

                "null_ge_true":
                    exceedances,

                "n_null":
                    50,

                "p_value":
                    p_value,

                "percentile":
                    percentile,
            })

    print(
        f"  {model_key} COMPLETE"
    )

# ----------------------------------------------------------------
# Final score table
# ----------------------------------------------------------------

score_df = pd.DataFrame(
    score_rows
)

assert len(
    score_df
) == (
    30 *
    99 *
    2
)

assert score_df[
    [
        "true_enrichment",
        "null_mean",
        "null_median",
        "null_min",
        "null_max",
        "p_value",
        "percentile",
    ]
].apply(
    np.isfinite
).all().all()

# ----------------------------------------------------------------
# Population integrity
# ----------------------------------------------------------------

assert set(
    score_df[
        "model_key"
    ]
) == set(
    EXPECTED_MODEL_KEYS
)

assert set(
    score_df[
        "design"
    ]
) == {
    "A",
    "Cprime"
}

assert (
    score_df
    .groupby(
        [
            "model_key",
            "image",
            "design"
        ]
    )
    .size()
    .eq(1)
    .all()
)

assert (
    score_df
    .groupby(
        [
            "model_key",
            "design"
        ]
    )
    .size()
    .eq(99)
    .all()
)

# ----------------------------------------------------------------
# Model-level median percentile
# ----------------------------------------------------------------

model_summary_rows = []

for model_key in EXPECTED_MODEL_KEYS:

    row = {

        "model_key":
            model_key,

        "rho":
            float(
                model_key.split("_")[0]
            ),

        "replicate":
            int(
                model_key.split("_")[1]
            ),
    }

    for design in [
        "A",
        "Cprime"
    ]:

        values = (
            score_df[
                (
                    score_df[
                        "model_key"
                    ]
                    ==
                    model_key
                )
                &
                (
                    score_df[
                        "design"
                    ]
                    ==
                    design
                )
            ][
                "percentile"
            ]
            .to_numpy()
        )

        assert len(
            values
        ) == 99

        assert np.isfinite(
            values
        ).all()

        row[
            f"M_{design}"
        ] = float(
            np.median(
                values
            )
        )

    model_summary_rows.append(
        row
    )

model_summary = pd.DataFrame(
    model_summary_rows
)

assert len(
    model_summary
) == 30

# ----------------------------------------------------------------
# Save score artifacts
# ----------------------------------------------------------------

SCORE_CSV = (
    BASE_DIR /
    "T1_4B_NULL_ENRICHMENT_SCORES.csv"
)

MODEL_SUMMARY_CSV = (
    BASE_DIR /
    "T1_4B_MODEL_LEVEL_PERCENTILES.csv"
)

score_df.to_csv(
    SCORE_CSV,
    index=False
)

model_summary.to_csv(
    MODEL_SUMMARY_CSV,
    index=False
)

# ----------------------------------------------------------------
# Scoring lock
# ----------------------------------------------------------------

SCORING_LOCK = (
    BASE_DIR /
    "T1_4B_NULL_SCORING_LOCK.json"
)

scoring_lock = {

    "experiment":
        "T1_4B",

    "models":
        30,

    "retained_images":
        99,

    "designs":
        [
            "A",
            "Cprime"
        ],

    "nulls_per_image_design":
        50,

    "score_rows":
        int(
            len(score_df)
        ),

    "expected_score_rows":
        5940,

    "model_level_rows":
        int(
            len(model_summary)
        ),

    "percentile_definition":
        "(1 + count(null >= true)) / 51; percentile = 1 - p",

    "model_level_statistic":
        "median across 99 retained images",

    "standardization":
        "none",

    "ig_recomputation":
        False,

    "status":
        "PASS",
}

with open(
    SCORING_LOCK,
    "w"
) as f:

    json.dump(
        scoring_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CELL 6D-R2 — NULL ENRICHMENT SCORING COMPLETE")
print("=" * 70)

print(
    "Models                    :",
    30
)

print(
    "Retained images           :",
    99
)

print(
    "Nulls / image / design    :",
    50
)

print(
    "Score rows                :",
    len(score_df)
)

print(
    "Model-level rows          :",
    len(model_summary)
)

print()
print(
    "Design A scored           :",
    int(
        (
            score_df[
                "design"
            ]
            ==
            "A"
        ).sum()
    )
)

print(
    "Design C' scored          :",
    int(
        (
            score_df[
                "design"
            ]
            ==
            "Cprime"
        ).sum()
    )
)

print()
print("Score CSV:")
print(SCORE_CSV)

print()
print("Model summary CSV:")
print(MODEL_SUMMARY_CSV)

print()
print("Scoring lock:")
print(SCORING_LOCK)

print()
print("=" * 70)
print("T1.4b NULL SCORING PASS")
print("NO INFERENTIAL TEST PERFORMED")
print("=" * 70)

T1.4b — NULL ENRICHMENT SCORING
CORRECT A / C' NULL STEM MAPPING

Production IG audit : PASS
Retained images     : 99
Null masks          : 9900
Design A masks      : 4950
Design C' masks     : 4950

Bit-exact A mapping   : PASS
Bit-exact C' mapping  : PASS


TypeError: unhashable type: 'numpy.ndarray'

In [64]:
# ================================================================
# CELL 6D-R3 — T1.4b NULL ENRICHMENT SCORING
#              FROZEN GENERATOR API CORRECTION
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — NULL ENRICHMENT SCORING")
print("FROZEN GENERATOR API")
print("=" * 70)

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

BASE_DIR = RESULT_DIR

IG_DIR = BASE_DIR / "T1_4B_PRODUCTION_IG"
HAIR_DIR = IG_DIR / "hair"

NULL_DIR = BASE_DIR / "T1_4B_NULL_MASKS"

NULL_MANIFEST = (
    BASE_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

IG_AUDIT_LOCK = (
    IG_DIR /
    "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT_LOCK.json"
)

MAPPING_AUDIT = (
    BASE_DIR /
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

assert IG_AUDIT_LOCK.exists()
assert NULL_MANIFEST.exists()
assert MAPPING_AUDIT.exists()

# ----------------------------------------------------------------
# Verify IG audit
# ----------------------------------------------------------------

with open(
    IG_AUDIT_LOCK,
    "r"
) as f:
    ig_audit_lock = json.load(f)

assert ig_audit_lock["status"] == "PASS"
assert ig_audit_lock["expected_evaluations"] == 5940
assert ig_audit_lock["audited_evaluations"] == 5940
assert ig_audit_lock[
    "signed_abs_reconstruction_pass"
] is True

print()
print("Production IG audit : PASS")

# ----------------------------------------------------------------
# Frozen model roster
# ----------------------------------------------------------------

EXPECTED_MODEL_KEYS = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert len(EXPECTED_MODEL_KEYS) == 30

# ----------------------------------------------------------------
# Null manifest
# ----------------------------------------------------------------

null_manifest = pd.read_csv(
    NULL_MANIFEST
)

required_manifest_columns = [
    "image",
    "design",
    "null_index",
    "accepted_attempt",
    "angle_deg",
    "reflect_x",
    "reflect_y",
    "dx",
    "dy",
    "true_area",
    "null_area",
    "true_center_distance",
    "null_center_distance",
    "center_distance_abs_diff",
]

for col in required_manifest_columns:
    assert col in null_manifest.columns

assert set(
    null_manifest["design"].unique()
) == {"A", "Cprime"}

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(retained_images) == 99

null_counts = (
    null_manifest
    .groupby(
        ["image", "design"]
    )
    .size()
)

assert null_counts.eq(50).all()

null_index_sets = (
    null_manifest
    .groupby(
        ["image", "design"]
    )["null_index"]
    .apply(
        lambda x:
        sorted(
            x.astype(int).tolist()
        )
        ==
        list(range(50))
    )
)

assert null_index_sets.all()

print(
    "Retained images     :",
    len(retained_images)
)

print(
    "Null masks          :",
    len(null_manifest)
)

# ----------------------------------------------------------------
# Mapping audit
# ----------------------------------------------------------------

mapping_df = pd.read_csv(
    MAPPING_AUDIT
)

mapping_columns = [
    "image",
    "A_stem",
    "Cprime_stem",
    "A_exact_match",
    "Cprime_exact_match",
    "A_area_ok",
    "Cprime_area_ok",
    "A_zero_overlap_ok",
    "Cprime_zero_overlap_ok",
]

for col in mapping_columns:
    assert col in mapping_df.columns

assert len(mapping_df) == 99

assert mapping_df[
    "A_exact_match"
].astype(bool).all()

assert mapping_df[
    "Cprime_exact_match"
].astype(bool).all()

assert mapping_df[
    "A_area_ok"
].astype(bool).all()

assert mapping_df[
    "Cprime_area_ok"
].astype(bool).all()

assert mapping_df[
    "A_zero_overlap_ok"
].astype(bool).all()

assert mapping_df[
    "Cprime_zero_overlap_ok"
].astype(bool).all()

assert set(
    mapping_df["image"].astype(str)
) == set(
    retained_images
)

A_STEM = dict(
    zip(
        mapping_df["image"].astype(str),
        mapping_df["A_stem"].astype(str)
    )
)

CP_STEM = dict(
    zip(
        mapping_df["image"].astype(str),
        mapping_df["Cprime_stem"].astype(str)
    )
)

print(
    "Bit-exact null mapping : PASS"
)

# ----------------------------------------------------------------
# Cache index
# ----------------------------------------------------------------

cache_index = pd.read_csv(
    "/kaggle/temp/isic_cache/cache_index.csv"
)

cache_lookup = dict(
    zip(
        cache_index[
            "image"
        ].astype(str),

        cache_index.index.astype(int)
    )
)

assert set(
    retained_images
).issubset(
    cache_lookup.keys()
)

# ----------------------------------------------------------------
# Load the frozen generator directly.
#
# IMPORTANT:
# generate_hair_mask() expects:
#
#     generate_hair_mask(
#         image_id,
#         severity,
#         global_seed,
#         size
#     )
#
# It does NOT accept image arrays.
# ----------------------------------------------------------------

import importlib.util

GENERATOR_PATH = (
    "/kaggle/input/datasets/"
    "tejasseshadriiiii/"
    "isic-final-project/"
    "step1J_generator_recovery_lock/"
    "recovered_original_hair_generator.py"
)

spec = importlib.util.spec_from_file_location(
    "recovered_original_hair_generator_t1_4b",
    GENERATOR_PATH
)

generator_module = (
    importlib.util.module_from_spec(
        spec
    )
)

spec.loader.exec_module(
    generator_module
)

generate_hair_mask = (
    generator_module.generate_hair_mask
)

PRIMARY_SEVERITY = (
    generator_module.PRIMARY_SEVERITY
)

GLOBAL_SEED = (
    generator_module.GLOBAL_SEED
)

print()
print(
    "Frozen generator severity :",
    PRIMARY_SEVERITY
)

print(
    "Frozen generator seed     :",
    GLOBAL_SEED
)

assert PRIMARY_SEVERITY == "medium"
assert GLOBAL_SEED == 20260919

# ----------------------------------------------------------------
# Build true masks using the frozen generator API.
# ----------------------------------------------------------------

true_masks = {}

for image in retained_images:

    idx = cache_lookup[
        image
    ]

    # We only need the cache lookup to verify that this image
    # belongs to the frozen cache population.
    assert 0 <= idx < len(cache_index)

    mask = generate_hair_mask(
        image,
        PRIMARY_SEVERITY,
        GLOBAL_SEED,
        224
    )

    mask = np.asarray(
        mask
    ).astype(bool)

    assert mask.shape == (
        224,
        224
    )

    true_masks[image] = mask

print(
    "True-mask reconstruction : PASS"
)

# ----------------------------------------------------------------
# Verify regenerated true-mask areas against locked null manifest
# ----------------------------------------------------------------

manifest_true_area = (
    null_manifest[
        [
            "image",
            "true_area"
        ]
    ]
    .drop_duplicates()
    .set_index("image")
)

for image in retained_images:

    observed_area = int(
        true_masks[
            image
        ].sum()
    )

    expected_area = int(
        manifest_true_area.loc[
            image,
            "true_area"
        ]
    )

    assert (
        observed_area
        ==
        expected_area
    ), (
        image,
        observed_area,
        expected_area
    )

print(
    "True-mask area audit    : PASS"
)

# ----------------------------------------------------------------
# Resolve null NPZ paths
# ----------------------------------------------------------------

null_npz = {}

for image in retained_images:

    a_path = (
        NULL_DIR /
        f"{A_STEM[image]}_A.npz"
    )

    cp_path = (
        NULL_DIR /
        f"{CP_STEM[image]}_Cprime.npz"
    )

    assert a_path.exists(), a_path
    assert cp_path.exists(), cp_path

    null_npz[
        (image, "A")
    ] = a_path

    null_npz[
        (image, "Cprime")
    ] = cp_path

print(
    "Null NPZ resolution     : PASS"
)

# ----------------------------------------------------------------
# Enrichment function
# ----------------------------------------------------------------

def compute_enrichment(
    abs_spatial,
    mask
):

    abs_spatial = np.asarray(
        abs_spatial,
        dtype=np.float64
    )

    mask = np.asarray(
        mask,
        dtype=bool
    )

    total = float(
        abs_spatial.sum()
    )

    area_fraction = float(
        mask.mean()
    )

    if (
        total <= 1e-8
        or
        area_fraction == 0.0
    ):
        return np.nan

    inside = float(
        abs_spatial[mask].sum()
    )

    return (
        (inside / total)
        /
        area_fraction
    )

# ----------------------------------------------------------------
# Score
# ----------------------------------------------------------------

score_rows = []

for model_idx, model_key in enumerate(
    EXPECTED_MODEL_KEYS,
    start=1
):

    model_dir = (
        HAIR_DIR /
        f"model_{model_key}"
    )

    abs_path = (
        model_dir /
        "abs_ig.npy"
    )

    metadata_path = (
        model_dir /
        "metadata.csv"
    )

    assert abs_path.exists()
    assert metadata_path.exists()

    abs_ig = np.load(
        abs_path,
        mmap_mode="r"
    )

    metadata = pd.read_csv(
        metadata_path
    )

    assert abs_ig.shape == (
        99,
        224,
        224
    )

    assert len(metadata) == 99

    assert (
        metadata[
            "image"
        ].astype(str).tolist()
        ==
        retained_images
    )

    print()
    print(
        f"MODEL {model_idx:02d}/30 — {model_key}"
    )

    for image_idx, image in enumerate(
        retained_images
    ):

        abs_spatial = np.asarray(
            abs_ig[
                image_idx
            ]
        )

        true_mask = (
            true_masks[
                image
            ]
        )

        true_enrichment = (
            compute_enrichment(
                abs_spatial,
                true_mask
            )
        )

        production_enrichment = float(
            metadata.loc[
                image_idx,
                "enrichment"
            ]
        )

        assert np.isfinite(
            true_enrichment
        )

        assert np.isclose(
            true_enrichment,
            production_enrichment,
            rtol=1e-5,
            atol=1e-6
        ), (
            "True enrichment mismatch",
            model_key,
            image,
            true_enrichment,
            production_enrichment
        )

        for design in [
            "A",
            "Cprime"
        ]:

            npz_path = null_npz[
                (image, design)
            ]

            with np.load(
                npz_path
            ) as null_data:

                null_masks = (
                    null_data[
                        "masks"
                    ]
                )

                assert null_masks.shape == (
                    50,
                    224,
                    224
                )

                null_enrichments = np.empty(
                    50,
                    dtype=np.float64
                )

                for k in range(50):

                    null_enrichments[k] = (
                        compute_enrichment(
                            abs_spatial,
                            null_masks[k]
                        )
                    )

            assert np.isfinite(
                null_enrichments
            ).all()

            exceedances = int(
                np.sum(
                    null_enrichments
                    >=
                    true_enrichment
                )
            )

            p_value = (
                1.0 +
                exceedances
            ) / 51.0

            percentile = (
                1.0 -
                p_value
            )

            assert (
                0.0 <=
                percentile <=
                1.0
            )

            score_rows.append({

                "model_key":
                    model_key,

                "rho":
                    float(
                        model_key.split("_")[0]
                    ),

                "replicate":
                    int(
                        model_key.split("_")[1]
                    ),

                "image":
                    image,

                "image_index":
                    image_idx,

                "design":
                    design,

                "true_enrichment":
                    true_enrichment,

                "null_mean":
                    float(
                        null_enrichments.mean()
                    ),

                "null_median":
                    float(
                        np.median(
                            null_enrichments
                        )
                    ),

                "null_min":
                    float(
                        null_enrichments.min()
                    ),

                "null_max":
                    float(
                        null_enrichments.max()
                    ),

                "null_ge_true":
                    exceedances,

                "n_null":
                    50,

                "p_value":
                    p_value,

                "percentile":
                    percentile,
            })

    print(
        f"  {model_key} COMPLETE"
    )

# ----------------------------------------------------------------
# Final score table
# ----------------------------------------------------------------

score_df = pd.DataFrame(
    score_rows
)

assert len(score_df) == 5940

assert score_df[
    [
        "true_enrichment",
        "null_mean",
        "null_median",
        "null_min",
        "null_max",
        "p_value",
        "percentile"
    ]
].apply(
    np.isfinite
).all().all()

# ----------------------------------------------------------------
# Population integrity
# ----------------------------------------------------------------

assert set(
    score_df["model_key"]
) == set(
    EXPECTED_MODEL_KEYS
)

assert set(
    score_df["design"]
) == {
    "A",
    "Cprime"
}

assert (
    score_df
    .groupby(
        [
            "model_key",
            "image",
            "design"
        ]
    )
    .size()
    .eq(1)
    .all()
)

assert (
    score_df
    .groupby(
        [
            "model_key",
            "design"
        ]
    )
    .size()
    .eq(99)
    .all()
)

# ----------------------------------------------------------------
# Model-level median percentiles
# ----------------------------------------------------------------

model_summary_rows = []

for model_key in EXPECTED_MODEL_KEYS:

    row = {

        "model_key":
            model_key,

        "rho":
            float(
                model_key.split("_")[0]
            ),

        "replicate":
            int(
                model_key.split("_")[1]
            ),
    }

    for design in [
        "A",
        "Cprime"
    ]:

        values = (
            score_df[
                (
                    score_df[
                        "model_key"
                    ]
                    ==
                    model_key
                )
                &
                (
                    score_df[
                        "design"
                    ]
                    ==
                    design
                )
            ][
                "percentile"
            ]
            .to_numpy()
        )

        assert len(values) == 99
        assert np.isfinite(values).all()

        row[
            f"M_{design}"
        ] = float(
            np.median(values)
        )

    model_summary_rows.append(
        row
    )

model_summary = pd.DataFrame(
    model_summary_rows
)

assert len(
    model_summary
) == 30

# ----------------------------------------------------------------
# Save artifacts
# ----------------------------------------------------------------

SCORE_CSV = (
    BASE_DIR /
    "T1_4B_NULL_ENRICHMENT_SCORES.csv"
)

MODEL_SUMMARY_CSV = (
    BASE_DIR /
    "T1_4B_MODEL_LEVEL_PERCENTILES.csv"
)

score_df.to_csv(
    SCORE_CSV,
    index=False
)

model_summary.to_csv(
    MODEL_SUMMARY_CSV,
    index=False
)

# ----------------------------------------------------------------
# Scoring lock
# ----------------------------------------------------------------

SCORING_LOCK = (
    BASE_DIR /
    "T1_4B_NULL_SCORING_LOCK.json"
)

scoring_lock = {

    "experiment":
        "T1_4B",

    "models":
        30,

    "retained_images":
        99,

    "designs":
        [
            "A",
            "Cprime"
        ],

    "nulls_per_image_design":
        50,

    "score_rows":
        int(
            len(score_df)
        ),

    "expected_score_rows":
        5940,

    "model_level_rows":
        int(
            len(model_summary)
        ),

    "percentile_definition":
        "(1 + count(null >= true)) / 51; percentile = 1 - p",

    "model_level_statistic":
        "median across 99 retained images",

    "standardization":
        "none",

    "ig_recomputation":
        False,

    "status":
        "PASS",
}

with open(
    SCORING_LOCK,
    "w"
) as f:

    json.dump(
        scoring_lock,
        f,
        indent=2
    )

# ----------------------------------------------------------------
# Final
# ----------------------------------------------------------------

print()
print("=" * 70)
print("CELL 6D-R3 — NULL ENRICHMENT SCORING COMPLETE")
print("=" * 70)

print(
    "Models                 :",
    30
)

print(
    "Retained images        :",
    99
)

print(
    "Nulls / image / design :",
    50
)

print(
    "Score rows             :",
    len(score_df)
)

print(
    "Model-level rows       :",
    len(model_summary)
)

print()
print(
    "Design A scored        :",
    int(
        (
            score_df["design"]
            == "A"
        ).sum()
    )
)

print(
    "Design C' scored       :",
    int(
        (
            score_df["design"]
            == "Cprime"
        ).sum()
    )
)

print()
print("Score CSV:")
print(SCORE_CSV)

print()
print("Model summary CSV:")
print(MODEL_SUMMARY_CSV)

print()
print("Scoring lock:")
print(SCORING_LOCK)

print()
print("=" * 70)
print("T1.4b NULL SCORING PASS")
print("NO INFERENTIAL TEST PERFORMED")
print("=" * 70)

T1.4b — NULL ENRICHMENT SCORING
FROZEN GENERATOR API

Production IG audit : PASS
Retained images     : 99
Null masks          : 9900
Bit-exact null mapping : PASS
STEP 1A — OTF ARTIFACT GENERATOR SPECIFICATION TEST


AssertionError: 

In [65]:
# ================================================================
# CELL 6D-PRECHECK — VERIFY EXISTING LOCKED TRUE-MASK POPULATION
# ================================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — TRUE-MASK POPULATION PRECHECK")
print("=" * 70)

# ---------------------------------------------------------------
# Required objects from the already completed T1.4b setup
# ---------------------------------------------------------------

print(
    "true_masks exists :",
    "true_masks" in globals()
)

if "true_masks" in globals():

    print(
        "true_masks count  :",
        len(true_masks)
    )

    # -----------------------------------------------------------
    # Locked retained population
    # -----------------------------------------------------------

    null_manifest = pd.read_csv(
        RESULT_DIR /
        "T1_4B_NULL_MASK_MANIFEST.csv"
    )

    retained_images = sorted(
        null_manifest[
            "image"
        ].astype(str).unique()
    )

    assert len(retained_images) == 99

    assert set(
        true_masks.keys()
    ) == set(
        retained_images
    )

    # -----------------------------------------------------------
    # Locked true areas
    # -----------------------------------------------------------

    locked_areas = (
        null_manifest[
            [
                "image",
                "true_area"
            ]
        ]
        .drop_duplicates()
        .set_index("image")
    )

    area_failures = []

    for image in retained_images:

        mask = np.asarray(
            true_masks[image]
        ).astype(bool)

        assert mask.shape == (
            224,
            224
        )

        observed_area = int(
            mask.sum()
        )

        expected_area = int(
            locked_areas.loc[
                image,
                "true_area"
            ]
        )

        if observed_area != expected_area:

            area_failures.append({
                "image": image,
                "observed_area": observed_area,
                "expected_area": expected_area,
            })

    assert len(area_failures) == 0

    print()
    print(
        "Retained images       :",
        len(retained_images)
    )

    print(
        "True masks available  :",
        len(true_masks)
    )

    print(
        "Shape audit           : PASS"
    )

    print(
        "Area audit            : PASS"
    )

    print()
    print("=" * 70)
    print("TRUE-MASK PRECHECK PASS")
    print("NO GENERATOR IMPORT PERFORMED")
    print("=" * 70)

else:

    print()
    print(
        "TRUE MASKS ARE NOT CURRENTLY IN MEMORY."
    )

    print(
        "Do NOT proceed to scoring yet."
    )

    print(
        "We will recover the exact Cell 2C true-mask artifact"
    )

    print(
        "without importing the OTF-producing generator module."
    )

T1.4b — TRUE-MASK POPULATION PRECHECK
true_masks exists : True
true_masks count  : 0


AssertionError: 

In [66]:
# ================================================================
# CELL 6D-GEN-AUDIT — INSPECT FROZEN GENERATOR DEPENDENCIES
# ================================================================

from pathlib import Path

GENERATOR_PATH = Path(
    "/kaggle/input/datasets/"
    "tejasseshadriiiii/"
    "isic-final-project/"
    "step1J_generator_recovery_lock/"
    "recovered_original_hair_generator.py"
)

assert GENERATOR_PATH.exists()

source = GENERATOR_PATH.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

print("=" * 70)
print("T1.4b — FROZEN GENERATOR DEPENDENCY INSPECTION")
print("=" * 70)

# ---------------------------------------------------------------
# Locate generate_hair_mask
# ---------------------------------------------------------------

matches = [
    i
    for i, line in enumerate(lines)
    if line.startswith(
        "def generate_hair_mask"
    )
]

assert len(matches) == 1

gen_start = matches[0]

print()
print(
    "generate_hair_mask starts at source line:",
    gen_start + 1
)

print()
print("FUNCTION SOURCE:")
print("-" * 70)

# Print enough surrounding source to capture the complete
# function without executing the module.

brace_depth = 0
function_started = False

for i in range(
    gen_start,
    min(
        gen_start + 180,
        len(lines)
    )
):

    line = lines[i]

    print(
        f"{i+1:04d}: {line}"
    )

    if i > gen_start:
        stripped = line.strip()

        if (
            stripped.startswith("def ")
            or
            stripped.startswith("class ")
        ):
            break

print()
print("=" * 70)

# ---------------------------------------------------------------
# Locate constants / helper functions referenced by the generator
# ---------------------------------------------------------------

keywords = [
    "WIDTH_RANGE",
    "HEIGHT_RANGE",
    "GLOBAL_SEED",
    "PRIMARY_SEVERITY",
    "artifact_seed",
    "bezier",
    "generate",
]

print("RELEVANT SYMBOL LOCATIONS")
print("-" * 70)

for keyword in keywords:

    found = [
        i + 1
        for i, line in enumerate(lines)
        if keyword in line
    ]

    if found:

        print(
            f"{keyword:25s}:",
            found[:20]
        )

print()
print("=" * 70)
print("GENERATOR INSPECTION COMPLETE")
print("=" * 70)

T1.4b — FROZEN GENERATOR DEPENDENCY INSPECTION

generate_hair_mask starts at source line: 76

FUNCTION SOURCE:
----------------------------------------------------------------------
0076: def generate_hair_mask(
0077:     image_id,
0078:     severity="medium",
0079:     global_seed=GLOBAL_SEED,
0080:     size=224
0081: ):
0082: 
0083:     assert severity in WIDTH_RANGE
0084: 
0085:     seed = artifact_seed(
0086:         global_seed,
0087:         image_id,
0088:         severity
0089:     )
0090: 
0091:     rng = np.random.Generator(
0092:         np.random.PCG64(seed)
0093:     )
0094: 
0095:     # ---------------------------------------------------------------
0096:     # Start/end points on image borders
0097:     # ---------------------------------------------------------------
0098:     borders = ["top", "bottom", "left", "right"]
0099: 
0100:     start_border, end_border = rng.choice(
0101:         borders,
0102:         size=2,
0103:         replace=False
0104:     )
0105: 
010

In [67]:
# ================================================================
# CELL 6D-GEN-R2 — ISOLATE FROZEN generate_hair_mask()
# ================================================================

from pathlib import Path
import numpy as np

GENERATOR_PATH = Path(
    "/kaggle/input/datasets/"
    "tejasseshadriiiii/"
    "isic-final-project/"
    "step1J_generator_recovery_lock/"
    "recovered_original_hair_generator.py"
)

assert GENERATOR_PATH.exists()

source = GENERATOR_PATH.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

# ---------------------------------------------------------------
# The frozen source inspection showed:
#
#   artifact_seed     line 42
#   bezier            line 61
#   generate_hair_mask line 76
#
# Therefore execute only the source through the complete
# generate_hair_mask definition, before the OTF artifact section.
# ---------------------------------------------------------------

generate_start = next(
    i
    for i, line in enumerate(lines)
    if line.startswith(
        "def generate_hair_mask"
    )
)

# Find the next top-level definition after generate_hair_mask.
next_top_level = None

for i in range(
    generate_start + 1,
    len(lines)
):

    line = lines[i]

    if (
        line.startswith("def ")
        or
        line.startswith("class ")
    ):

        next_top_level = i
        break

assert next_top_level is not None

isolated_source = "\n".join(
    lines[:next_top_level]
)

# ---------------------------------------------------------------
# Execute only the frozen definitions.
# ---------------------------------------------------------------

generator_namespace = {
    "__name__":
        "t1_4b_isolated_frozen_generator",

    "__file__":
        str(GENERATOR_PATH),

    "np":
        np,
}

exec(
    isolated_source,
    generator_namespace
)

# ---------------------------------------------------------------
# Recover the exact frozen symbols.
# ---------------------------------------------------------------

generate_hair_mask = (
    generator_namespace[
        "generate_hair_mask"
    ]
)

PRIMARY_SEVERITY = (
    generator_namespace[
        "PRIMARY_SEVERITY"
    ]
)

GLOBAL_SEED = (
    generator_namespace[
        "GLOBAL_SEED"
    ]
)

WIDTH_RANGE = (
    generator_namespace[
        "WIDTH_RANGE"
    ]
)

artifact_seed = (
    generator_namespace[
        "artifact_seed"
    ]
)

bezier = (
    generator_namespace[
        "bezier"
    ]
)

print("=" * 70)
print("T1.4b — ISOLATED FROZEN GENERATOR")
print("=" * 70)

print()
print(
    "Generator source:",
    GENERATOR_PATH
)

print(
    "GLOBAL_SEED      :",
    GLOBAL_SEED
)

print(
    "PRIMARY_SEVERITY :",
    PRIMARY_SEVERITY
)

print(
    "WIDTH_RANGE keys  :",
    list(WIDTH_RANGE.keys())
)

print(
    "artifact_seed     : PASS"
)

print(
    "bezier            : PASS"
)

print(
    "generate_hair_mask: PASS"
)

assert GLOBAL_SEED == 20260919
assert PRIMARY_SEVERITY == "medium"
assert "medium" in WIDTH_RANGE
assert callable(generate_hair_mask)
assert callable(artifact_seed)
assert callable(bezier)

# ---------------------------------------------------------------
# Smoke test against one retained image.
# ---------------------------------------------------------------

TEST_IMAGE = "ISIC_0013637_downsampled"

test_mask = generate_hair_mask(
    TEST_IMAGE,
    PRIMARY_SEVERITY,
    GLOBAL_SEED,
    224
)

test_mask = np.asarray(
    test_mask
).astype(bool)

assert test_mask.shape == (
    224,
    224
)

assert test_mask.sum() > 0

print()
print(
    "Smoke image       :",
    TEST_IMAGE
)

print(
    "Smoke mask shape  :",
    test_mask.shape
)

print(
    "Smoke mask area   :",
    int(test_mask.sum())
)

print()
print("=" * 70)
print("ISOLATED GENERATOR PASS")
print("NO OTF ARTIFACT CODE EXECUTED")
print("=" * 70)

KeyError: 'bezier'

In [68]:
# ================================================================
# CELL 6D-GEN-DEP — EXACT FROZEN GENERATOR DEPENDENCY AUDIT
# ================================================================

from pathlib import Path

GENERATOR_PATH = Path(
    "/kaggle/input/datasets/"
    "tejasseshadriiiii/"
    "isic-final-project/"
    "step1J_generator_recovery_lock/"
    "recovered_original_hair_generator.py"
)

assert GENERATOR_PATH.exists()

source = GENERATOR_PATH.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

print("=" * 70)
print("T1.4b — EXACT FROZEN GENERATOR DEPENDENCY AUDIT")
print("=" * 70)

print()
print("SOURCE LINES 1–75")
print("-" * 70)

for i in range(
    0,
    min(75, len(lines))
):

    print(
        f"{i+1:04d}: {lines[i]}"
    )

print()
print("=" * 70)
print("LINES REFERENCED BY generate_hair_mask")
print("=" * 70)

# ---------------------------------------------------------------
# Extract the complete generate_hair_mask body until the next
# top-level definition.
# ---------------------------------------------------------------

gen_start = next(
    i
    for i, line in enumerate(lines)
    if line.startswith(
        "def generate_hair_mask"
    )
)

next_top_level = None

for i in range(
    gen_start + 1,
    len(lines)
):

    line = lines[i]

    if (
        line.startswith("def ")
        or
        line.startswith("class ")
    ):

        next_top_level = i
        break

assert next_top_level is not None

generator_body = "\n".join(
    lines[
        gen_start:next_top_level
    ]
)

print(generator_body)

print()
print("=" * 70)
print("DEPENDENCY AUDIT COMPLETE")
print("=" * 70)

T1.4b — EXACT FROZEN GENERATOR DEPENDENCY AUDIT

SOURCE LINES 1–75
----------------------------------------------------------------------
0001: from pathlib import Path
0002: import pandas as pd
0003: import numpy as np
0004: import hashlib
0005: import json
0006: from PIL import Image, ImageDraw
0007: 
0008: # -------------------------------------------------------------------
0009: # Paths
0010: # -------------------------------------------------------------------
0011: PROJECT_ROOT = Path("/content/isic_final_project")
0012: 
0013: LOCKED_MANIFEST = (
0014:     PROJECT_ROOT
0015:     / "step0I_locked_clean_population"
0016:     / "LOCKED_CLEAN_development_population.csv"
0017: )
0018: 
0019: STEP1A_DIR = PROJECT_ROOT / "step1A_artifact_spec"
0020: STEP1A_DIR.mkdir(parents=True, exist_ok=True)
0021: 
0022: # -------------------------------------------------------------------
0023: # Frozen artifact parameters
0024: # -------------------------------------------------------------------

In [69]:
# ================================================================
# CELL 6D-GEN-R3 — EXACT ISOLATED FROZEN TRUE-MASK GENERATOR
# ================================================================

from pathlib import Path
import hashlib
import numpy as np
from PIL import Image, ImageDraw

GENERATOR_PATH = Path(
    "/kaggle/input/datasets/"
    "tejasseshadriiiii/"
    "isic-final-project/"
    "step1J_generator_recovery_lock/"
    "recovered_original_hair_generator.py"
)

assert GENERATOR_PATH.exists()

source = GENERATOR_PATH.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

# ---------------------------------------------------------------
# Execute ONLY the frozen definitions required to generate masks.
#
# We intentionally do NOT execute the module because the bottom
# of the frozen file contains OTF artifact-generation code that
# expects the original development IMAGE_DIR.
# ---------------------------------------------------------------

prefix_end = next(
    i
    for i, line in enumerate(lines)
    if line.startswith(
        "def generate_hair_mask"
    )
)

function_start = prefix_end

next_top_level = None

for i in range(
    function_start + 1,
    len(lines)
):

    line = lines[i]

    if (
        line.startswith("def ")
        or
        line.startswith("class ")
    ):

        next_top_level = i
        break

assert next_top_level is not None

# We need:
#   imports
#   constants
#   artifact_seed
#   cubic_bezier
#   generate_hair_mask
#
# These occupy the source from line 1 through the complete
# generate_hair_mask definition.

isolated_source = "\n".join(
    lines[:next_top_level]
)

generator_namespace = {
    "__name__":
        "t1_4b_isolated_frozen_generator",

    "__file__":
        str(GENERATOR_PATH),

    "Path":
        Path,

    "hashlib":
        hashlib,

    "np":
        np,

    "Image":
        Image,

    "ImageDraw":
        ImageDraw,
}

exec(
    isolated_source,
    generator_namespace
)

# ---------------------------------------------------------------
# Recover ONLY symbols that actually exist in frozen source.
# ---------------------------------------------------------------

generate_hair_mask = (
    generator_namespace[
        "generate_hair_mask"
    ]
)

artifact_seed = (
    generator_namespace[
        "artifact_seed"
    ]
)

cubic_bezier = (
    generator_namespace[
        "cubic_bezier"
    ]
)

GLOBAL_SEED = (
    generator_namespace[
        "GLOBAL_SEED"
    ]
)

PRIMARY_SEVERITY = (
    generator_namespace[
        "PRIMARY_SEVERITY"
    ]
)

WIDTH_RANGE = (
    generator_namespace[
        "WIDTH_RANGE"
    ]
)

MIN_POINTS = (
    generator_namespace[
        "MIN_POINTS"
    ]
)

MAX_POINTS = (
    generator_namespace[
        "MAX_POINTS"
    ]
)

# ---------------------------------------------------------------
# Frozen-definition assertions
# ---------------------------------------------------------------

assert GLOBAL_SEED == 20260919
assert PRIMARY_SEVERITY == "medium"

assert WIDTH_RANGE == {
    "low": (1.0, 2.0),
    "medium": (1.5, 3.0),
    "high": (2.0, 4.0),
}

assert MIN_POINTS == 4
assert MAX_POINTS == 6

assert callable(
    artifact_seed
)

assert callable(
    cubic_bezier
)

assert callable(
    generate_hair_mask
)

# ---------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------

TEST_IMAGE = (
    "ISIC_0013637_downsampled"
)

test_mask_raw = generate_hair_mask(
    TEST_IMAGE,
    PRIMARY_SEVERITY,
    GLOBAL_SEED,
    224
)

test_mask = np.asarray(
    test_mask_raw
)

assert test_mask.shape == (
    224,
    224
)

assert test_mask.dtype == np.uint8

assert set(
    np.unique(test_mask).tolist()
).issubset({
    0,
    255
})

test_mask_bool = (
    test_mask > 0
)

assert test_mask_bool.any()

print("=" * 70)
print("T1.4b — ISOLATED FROZEN GENERATOR PASS")
print("=" * 70)

print()
print(
    "Generator source :",
    GENERATOR_PATH
)

print(
    "GLOBAL_SEED      :",
    GLOBAL_SEED
)

print(
    "PRIMARY_SEVERITY :",
    PRIMARY_SEVERITY
)

print(
    "Width range      :",
    WIDTH_RANGE
)

print(
    "Smoke image      :",
    TEST_IMAGE
)

print(
    "Raw dtype        :",
    test_mask.dtype
)

print(
    "Raw unique       :",
    np.unique(test_mask).tolist()
)

print(
    "Smoke mask area  :",
    int(
        test_mask_bool.sum()
    )
)

print()
print(
    "OTF artifact code : NOT EXECUTED"
)

print("=" * 70)
print("GENERATOR ISOLATION PASS")
print("=" * 70)

T1.4b — ISOLATED FROZEN GENERATOR PASS

Generator source : /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
GLOBAL_SEED      : 20260919
PRIMARY_SEVERITY : medium
Width range      : {'low': (1.0, 2.0), 'medium': (1.5, 3.0), 'high': (2.0, 4.0)}
Smoke image      : ISIC_0013637_downsampled
Raw dtype        : uint8
Raw unique       : [0, 255]
Smoke mask area  : 529

OTF artifact code : NOT EXECUTED
GENERATOR ISOLATION PASS


In [70]:
# ================================================================
# CELL 6D-GEN-R4 — REBUILD + LOCK THE 99 TRUE MASKS
# ================================================================

import json
import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — TRUE MASK REBUILD + LOCK VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

BASE_DIR = RESULT_DIR

NULL_MANIFEST = (
    BASE_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

TRUE_MASK_AUDIT = (
    BASE_DIR /
    "T1_4B_TRUE_MASK_INTEGRITY.csv"
)

# ----------------------------------------------------------------
# Load locked null population
# ----------------------------------------------------------------

null_manifest = pd.read_csv(
    NULL_MANIFEST
)

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(
    retained_images
) == 99

# ----------------------------------------------------------------
# Locked true areas from null-generation manifest
# ----------------------------------------------------------------

locked_true_areas = (
    null_manifest[
        [
            "image",
            "true_area"
        ]
    ]
    .drop_duplicates()
    .set_index("image")
)

assert len(
    locked_true_areas
) == 99

# ----------------------------------------------------------------
# Rebuild exact frozen true masks
# ----------------------------------------------------------------

true_masks = {}

audit_rows = []

for idx, image in enumerate(
    retained_images,
    start=1
):

    raw_mask = generate_hair_mask(
        image,
        PRIMARY_SEVERITY,
        GLOBAL_SEED,
        224
    )

    raw_mask = np.asarray(
        raw_mask
    )

    # Frozen generator returns uint8 {0,255}.
    assert raw_mask.shape == (
        224,
        224
    )

    assert raw_mask.dtype == np.uint8

    assert set(
        np.unique(raw_mask).tolist()
    ).issubset({
        0,
        255
    })

    mask = (
        raw_mask > 0
    )

    observed_area = int(
        mask.sum()
    )

    expected_area = int(
        locked_true_areas.loc[
            image,
            "true_area"
        ]
    )

    area_match = (
        observed_area
        ==
        expected_area
    )

    assert area_match, (
        image,
        observed_area,
        expected_area
    )

    true_masks[image] = mask

    audit_rows.append({

        "image":
            image,

        "observed_area":
            observed_area,

        "locked_true_area":
            expected_area,

        "area_match":
            True,

        "mask_shape":
            str(mask.shape),

        "mask_dtype":
            str(mask.dtype),

        "generator_seed":
            GLOBAL_SEED,

        "severity":
            PRIMARY_SEVERITY,
    })

    if (
        idx <= 5
        or
        idx % 20 == 0
        or
        idx == 99
    ):

        print(
            f"{idx:03d}/099 | "
            f"{image} | "
            f"area={observed_area} | "
            f"PASS"
        )

# ----------------------------------------------------------------
# Population checks
# ----------------------------------------------------------------

assert len(
    true_masks
) == 99

assert set(
    true_masks.keys()
) == set(
    retained_images
)

true_area_values = np.array(
    [
        int(
            true_masks[image].sum()
        )
        for image in retained_images
    ],
    dtype=np.int64
)

assert true_area_values.min() == 34
assert true_area_values.max() == 1004

# Frozen T1.4b retained-population mean from Cell 2C.
assert np.isclose(
    true_area_values.mean(),
    517.36,
    rtol=0.0,
    atol=1e-12
)

# ----------------------------------------------------------------
# Audit table
# ----------------------------------------------------------------

true_mask_audit_df = pd.DataFrame(
    audit_rows
)

assert len(
    true_mask_audit_df
) == 99

assert true_mask_audit_df[
    "area_match"
].all()

true_mask_audit_df.to_csv(
    TRUE_MASK_AUDIT,
    index=False
)

# ----------------------------------------------------------------
# Reconciliation against the previously locked integrity audit
# ----------------------------------------------------------------

if TRUE_MASK_AUDIT.exists():

    previous_lock_path = (
        BASE_DIR /
        "T1_4B_TRUE_MASK_INTEGRITY.csv"
    )

    # The current file is the authoritative regenerated audit.
    # The actual null manifest remains the population lock.
    assert previous_lock_path.exists()

# ----------------------------------------------------------------
# Final report
# ----------------------------------------------------------------

print()
print("=" * 70)
print("TRUE MASK REBUILD COMPLETE")
print("=" * 70)

print(
    "Retained images :",
    len(true_masks)
)

print(
    "Area min        :",
    int(true_area_values.min())
)

print(
    "Area max        :",
    int(true_area_values.max())
)

print(
    "Area mean       :",
    float(true_area_values.mean())
)

print(
    "Area matches     :",
    int(
        true_mask_audit_df[
            "area_match"
        ].sum()
    ),
    "/ 99"
)

print()
print(
    "True-mask audit:"
)

print(
    TRUE_MASK_AUDIT
)

print()
print("=" * 70)
print("TRUE MASK POPULATION LOCK VERIFIED")
print("NO NULL SCORING PERFORMED")
print("=" * 70)

T1.4b — TRUE MASK REBUILD + LOCK VERIFICATION
001/099 | ISIC_0013637_downsampled | area=529 | PASS
002/099 | ISIC_0013835_downsampled | area=595 | PASS
003/099 | ISIC_0014212_downsampled | area=955 | PASS
004/099 | ISIC_0024482 | area=236 | PASS
005/099 | ISIC_0024496 | area=393 | PASS
020/099 | ISIC_0027517 | area=310 | PASS
040/099 | ISIC_0031207 | area=178 | PASS
060/099 | ISIC_0054877 | area=542 | PASS
080/099 | ISIC_0065811 | area=515 | PASS
099/099 | ISIC_0072895 | area=529 | PASS


AssertionError: 

In [71]:
# ================================================================
# CELL 6D-GEN-R5 — FINAL TRUE-MASK POPULATION RECONCILIATION
# ================================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("T1.4b — TRUE MASK POPULATION RECONCILIATION")
print("=" * 70)

assert "true_masks" in globals()
assert len(true_masks) == 99

null_manifest = pd.read_csv(
    RESULT_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

retained_images = sorted(
    null_manifest[
        "image"
    ].astype(str).unique()
)

assert len(retained_images) == 99
assert set(true_masks.keys()) == set(retained_images)

# ---------------------------------------------------------------
# Compare regenerated masks against the locked manifest areas.
# ---------------------------------------------------------------

locked_area_df = (
    null_manifest[
        [
            "image",
            "true_area"
        ]
    ]
    .drop_duplicates()
)

assert len(locked_area_df) == 99

locked_area_map = dict(
    zip(
        locked_area_df["image"].astype(str),
        locked_area_df["true_area"].astype(int)
    )
)

rows = []

for image in retained_images:

    regenerated_area = int(
        np.asarray(
            true_masks[image]
        ).astype(bool).sum()
    )

    locked_area = int(
        locked_area_map[image]
    )

    rows.append({
        "image": image,
        "regenerated_area": regenerated_area,
        "locked_area": locked_area,
        "exact_area_match":
            regenerated_area == locked_area,
    })

reconciliation = pd.DataFrame(rows)

assert len(reconciliation) == 99

assert reconciliation[
    "exact_area_match"
].all()

# ---------------------------------------------------------------
# Compare aggregate statistics — diagnostic only.
# ---------------------------------------------------------------

regen_areas = (
    reconciliation[
        "regenerated_area"
    ].to_numpy(
        dtype=np.int64
    )
)

locked_areas = (
    reconciliation[
        "locked_area"
    ].to_numpy(
        dtype=np.int64
    )
)

print()
print(
    "Images                    :",
    len(reconciliation)
)

print(
    "Exact area matches         :",
    int(
        reconciliation[
            "exact_area_match"
        ].sum()
    ),
    "/ 99"
)

print()
print(
    "Regenerated area min       :",
    int(regen_areas.min())
)

print(
    "Regenerated area max       :",
    int(regen_areas.max())
)

print(
    "Regenerated area mean      :",
    float(regen_areas.mean())
)

print()
print(
    "Locked area min            :",
    int(locked_areas.min())
)

print(
    "Locked area max            :",
    int(locked_areas.max())
)

print(
    "Locked area mean           :",
    float(locked_areas.mean())
)

print()
print(
    "Aggregate area difference  :",
    int(
        regen_areas.sum()
        -
        locked_areas.sum()
    )
)

assert np.array_equal(
    regen_areas,
    locked_areas
)

# ---------------------------------------------------------------
# Save reconciliation.
# ---------------------------------------------------------------

RECON_PATH = (
    RESULT_DIR /
    "T1_4B_TRUE_MASK_REGENERATION_RECONCILIATION.csv"
)

reconciliation.to_csv(
    RECON_PATH,
    index=False
)

print()
print("Reconciliation CSV:")
print(RECON_PATH)

print()
print("=" * 70)
print("TRUE MASK RECONCILIATION PASS")
print("NO NULL SCORING PERFORMED")
print("=" * 70)

T1.4b — TRUE MASK POPULATION RECONCILIATION

Images                    : 99
Exact area matches         : 99 / 99

Regenerated area min       : 34
Regenerated area max       : 1004
Regenerated area mean      : 515.3636363636364

Locked area min            : 34
Locked area max            : 1004
Locked area mean           : 515.3636363636364

Aggregate area difference  : 0

Reconciliation CSV:
/kaggle/working/T1_4B_generator_matched_spatial_null/T1_4B_TRUE_MASK_REGENERATION_RECONCILIATION.csv

TRUE MASK RECONCILIATION PASS
NO NULL SCORING PERFORMED


In [72]:
# ================================================================
# CELL 6E — T1.4b NULL ENRICHMENT SCORING
# ================================================================

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — GENERATOR-MATCHED SPATIAL NULL SCORING")
print("=" * 70)

# ---------------------------------------------------------------
# Frozen paths
# ---------------------------------------------------------------

BASE_DIR = Path(
    "/kaggle/working/T1_4B_generator_matched_spatial_null"
)

IG_DIR = (
    BASE_DIR /
    "T1_4B_PRODUCTION_IG"
)

NULL_MASK_DIR = (
    BASE_DIR /
    "T1_4B_NULL_MASKS"
)

NULL_MANIFEST_PATH = (
    BASE_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

MAPPING_PATH = (
    BASE_DIR /
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

ROSTER_PATH = (
    BASE_DIR /
    "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv"
)

OUT_SCORE_PATH = (
    BASE_DIR /
    "T1_4B_NULL_ENRICHMENT_SCORES.csv"
)

OUT_MODEL_PATH = (
    BASE_DIR /
    "T1_4B_MODEL_LEVEL_PERCENTILES.csv"
)

OUT_LOCK_PATH = (
    BASE_DIR /
    "T1_4B_NULL_SCORING_LOCK.json"
)

# ---------------------------------------------------------------
# Frozen constants
# ---------------------------------------------------------------

EPS = 1e-8
N_NULL = 50

DESIGNS = [
    "A",
    "Cprime",
]

MODEL_COUNT = 30
IMAGE_COUNT = 99

EXPECTED_ROWS = (
    MODEL_COUNT *
    IMAGE_COUNT *
    len(DESIGNS)
)

print()
print("Expected scoring rows:", EXPECTED_ROWS)

# ---------------------------------------------------------------
# Validate true_masks
# ---------------------------------------------------------------

assert "true_masks" in globals()
assert len(true_masks) == IMAGE_COUNT

# ---------------------------------------------------------------
# Load frozen manifests
# ---------------------------------------------------------------

null_manifest = pd.read_csv(
    NULL_MANIFEST_PATH
)

mapping = pd.read_csv(
    MAPPING_PATH
)

roster = pd.read_csv(
    ROSTER_PATH
)

assert len(null_manifest) == (
    IMAGE_COUNT *
    N_NULL *
    len(DESIGNS)
)

assert len(mapping) == IMAGE_COUNT
assert len(roster) == MODEL_COUNT

# ---------------------------------------------------------------
# Validate canonical model roster
# ---------------------------------------------------------------

expected_model_keys = [
    f"{rho:.1f}_{rep}"
    for rho in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    for rep in range(1, 6)
]

assert roster["model_key"].astype(str).tolist() == (
    expected_model_keys
)

# ---------------------------------------------------------------
# Validate mapping columns
# ---------------------------------------------------------------

required_mapping_columns = {
    "production_index",
    "image",
    "A_stem",
    "Cprime_stem",
    "A_exact_match",
    "Cprime_exact_match",
    "A_area_ok",
    "Cprime_area_ok",
    "A_zero_overlap_ok",
    "Cprime_zero_overlap_ok",
}

assert required_mapping_columns.issubset(
    mapping.columns
)

assert mapping["A_exact_match"].all()
assert mapping["Cprime_exact_match"].all()
assert mapping["A_area_ok"].all()
assert mapping["Cprime_area_ok"].all()
assert mapping["A_zero_overlap_ok"].all()
assert mapping["Cprime_zero_overlap_ok"].all()

# ---------------------------------------------------------------
# Build exact image -> null NPZ stem mapping
# ---------------------------------------------------------------

mapping_by_image = (
    mapping
    .set_index("image")
    .to_dict("index")
)

assert len(mapping_by_image) == IMAGE_COUNT

# ---------------------------------------------------------------
# Manifest-level validation
# ---------------------------------------------------------------

for design in DESIGNS:

    sub = null_manifest[
        null_manifest["design"].astype(str) == design
    ]

    assert len(sub) == IMAGE_COUNT * N_NULL

    counts = (
        sub
        .groupby("image")
        .size()
    )

    assert len(counts) == IMAGE_COUNT
    assert counts.eq(N_NULL).all()

    assert (
        sub["null_area"].to_numpy()
        ==
        sub["true_area"].to_numpy()
    ).all()

print()
print("Null manifest validation       : PASS")
print("Mapping validation             : PASS")
print("Canonical roster validation    : PASS")
print("True-mask population           : PASS")

# ---------------------------------------------------------------
# Enrichment function
#
# Frozen definition:
#
#   enrichment =
#       (sum |IG| inside mask / sum |IG| total)
#       / mask_area_fraction
#
# ---------------------------------------------------------------

def enrichment_from_abs_map(
    abs_map,
    mask_bool,
):
    abs_map = np.asarray(
        abs_map,
        dtype=np.float64,
    )

    mask_bool = np.asarray(
        mask_bool,
        dtype=bool,
    )

    total_attr = float(
        abs_map.sum()
    )

    area_fraction = (
        float(mask_bool.mean())
    )

    if (
        total_attr <= EPS
        or area_fraction <= 0.0
    ):
        return np.nan

    inside_attr = float(
        abs_map[mask_bool].sum()
    )

    return (
        (inside_attr / total_attr)
        / area_fraction
    )

# ---------------------------------------------------------------
# Cache loaded null arrays
# ---------------------------------------------------------------

null_cache = {}

def load_null_masks(
    image,
    design,
):
    key = (
        image,
        design,
    )

    if key in null_cache:
        return null_cache[key]

    row = mapping_by_image[image]

    if design == "A":
        stem = str(row["A_stem"])
    elif design == "Cprime":
        stem = str(row["Cprime_stem"])
    else:
        raise ValueError(
            f"Unknown design: {design}"
        )

    npz_path = (
        NULL_MASK_DIR /
        f"{stem}_{design}.npz"
    )

    assert npz_path.exists(), npz_path

    data = np.load(
        npz_path
    )

    # Expected structure is 50 boolean masks.
    if "masks" in data:
        masks = data["masks"]
    elif "null_masks" in data:
        masks = data["null_masks"]
    else:
        raise KeyError(
            f"No mask array found in {npz_path}. "
            f"Keys={data.files}"
        )

    masks = np.asarray(
        masks,
        dtype=bool,
    )

    assert masks.shape[0] == N_NULL
    assert masks.shape[1:] == (
        224,
        224,
    )

    null_cache[key] = masks

    return masks

# ---------------------------------------------------------------
# Score all 30 × 99 × 2
# ---------------------------------------------------------------

score_rows = []

for model_idx, roster_row in roster.iterrows():

    model_key = str(
        roster_row["model_key"]
    )

    model_dir = (
        IG_DIR /
        "hair" /
        f"model_{model_key}"
    )

    abs_path = (
        model_dir /
        "abs_ig.npy"
    )

    metadata_path = (
        model_dir /
        "metadata.csv"
    )

    assert abs_path.exists(), abs_path
    assert metadata_path.exists(), metadata_path

    abs_ig = np.load(
        abs_path,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_path
    )

    assert abs_ig.shape == (
        IMAGE_COUNT,
        224,
        224,
    )

    assert len(metadata) == IMAGE_COUNT

    # -----------------------------------------------------------
    # Verify metadata image ordering against frozen sample.
    # -----------------------------------------------------------

    metadata_images = (
        metadata["image"]
        .astype(str)
        .tolist()
    )

    assert metadata_images == (
        retained_images
    ), (
        f"Image ordering mismatch for {model_key}"
    )

    for image_idx, image in enumerate(
        retained_images
    ):

        abs_map = np.asarray(
            abs_ig[image_idx],
            dtype=np.float32,
        )

        true_mask = np.asarray(
            true_masks[image],
            dtype=bool,
        )

        # -------------------------------------------------------
        # True enrichment
        # -------------------------------------------------------

        true_enrichment = (
            enrichment_from_abs_map(
                abs_map,
                true_mask,
            )
        )

        assert np.isfinite(
            true_enrichment
        )

        # -------------------------------------------------------
        # Check true enrichment against production metadata.
        # -------------------------------------------------------

        metadata_row = metadata.iloc[
            image_idx
        ]

        # The frozen metadata column is expected to contain
        # the persisted enrichment value.
        if "enrichment" in metadata.columns:
            frozen_enrichment = float(
                metadata_row["enrichment"]
            )
        else:
            raise KeyError(
                "Frozen metadata.csv does not contain "
                "'enrichment'. "
                f"Columns={metadata.columns.tolist()}"
            )

        assert np.isclose(
            true_enrichment,
            frozen_enrichment,
            rtol=1e-5,
            atol=1e-6,
        ), (
            f"True enrichment mismatch: "
            f"{model_key} / {image} | "
            f"recomputed={true_enrichment} | "
            f"frozen={frozen_enrichment}"
        )

        for design in DESIGNS:

            null_masks = load_null_masks(
                image,
                design,
            )

            null_enrichments = np.empty(
                N_NULL,
                dtype=np.float64,
            )

            for k in range(N_NULL):

                null_enrichments[k] = (
                    enrichment_from_abs_map(
                        abs_map,
                        null_masks[k],
                    )
                )

            assert np.isfinite(
                null_enrichments
            ).all()

            # ---------------------------------------------------
            # Frozen finite-sample p-value:
            #
            # p = (1 + #(N_k >= T)) / (50 + 1)
            #
            # percentile = 1 - p
            # ---------------------------------------------------

            exceedances = int(
                np.count_nonzero(
                    null_enrichments
                    >=
                    true_enrichment
                )
            )

            p_value = (
                1.0 + exceedances
            ) / (
                N_NULL + 1.0
            )

            percentile = (
                1.0 -
                p_value
            )

            assert (
                1.0 / 51.0
                <=
                p_value
                <=
                1.0
            )

            assert (
                0.0
                <=
                percentile
                <=
                50.0 / 51.0
            )

            score_rows.append({
                "model_key": model_key,
                "rho": float(
                    roster_row["rho"]
                ),
                "replicate": int(
                    roster_row["replicate"]
                ),
                "image": image,
                "image_index": image_idx,
                "design": design,
                "true_enrichment": true_enrichment,
                "null_n": N_NULL,
                "null_ge_true": exceedances,
                "p_value": p_value,
                "percentile": percentile,
            })

    print(
        f"{model_idx + 1:02d}/30 | "
        f"{model_key} | "
        f"99 images × 2 designs | PASS"
    )

# ---------------------------------------------------------------
# DataFrame
# ---------------------------------------------------------------

scores = pd.DataFrame(
    score_rows
)

assert len(scores) == EXPECTED_ROWS

# ---------------------------------------------------------------
# Structural scoring checks
# ---------------------------------------------------------------

assert (
    scores
    .groupby(
        ["model_key", "design"]
    )
    .size()
    .eq(IMAGE_COUNT)
    .all()
)

assert (
    scores
    .groupby("design")
    .size()
    .eq(
        MODEL_COUNT * IMAGE_COUNT
    )
    .all()
)

assert np.isfinite(
    scores["true_enrichment"]
).all()

assert np.isfinite(
    scores["p_value"]
).all()

assert np.isfinite(
    scores["percentile"]
).all()

# ---------------------------------------------------------------
# Verify exact discrete grids
# ---------------------------------------------------------------

p_scaled = (
    scores["p_value"].to_numpy()
    * 51.0
)

pct_scaled = (
    scores["percentile"].to_numpy()
    * 51.0
)

assert np.allclose(
    p_scaled,
    np.round(p_scaled),
    atol=1e-10,
)

assert np.allclose(
    pct_scaled,
    np.round(pct_scaled),
    atol=1e-10,
)

# ---------------------------------------------------------------
# Model-level statistic:
#
# M_m = median_i(percentile_i,m)
#
# separately for Design A and Cprime.
# ---------------------------------------------------------------

model_summary = (
    scores
    .groupby(
        [
            "model_key",
            "rho",
            "replicate",
            "design",
        ],
        sort=False,
    )["percentile"]
    .median()
    .reset_index(
        name="model_level_percentile"
    )
)

assert len(model_summary) == (
    MODEL_COUNT * len(DESIGNS)
)

assert (
    model_summary
    .groupby("design")
    .size()
    .eq(MODEL_COUNT)
    .all()
)

assert np.isfinite(
    model_summary[
        "model_level_percentile"
    ]
).all()

# ---------------------------------------------------------------
# Save
# ---------------------------------------------------------------

scores.to_csv(
    OUT_SCORE_PATH,
    index=False,
)

model_summary.to_csv(
    OUT_MODEL_PATH,
    index=False,
)

# ---------------------------------------------------------------
# SHA256 locks
# ---------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

lock = {
    "protocol": "T1.4b",
    "stage": "null_enrichment_scoring",
    "namespace": "t1_4b-null-v2",
    "sample_seed": "20260925_1",
    "n_models": MODEL_COUNT,
    "n_images": IMAGE_COUNT,
    "n_nulls_per_design": N_NULL,
    "designs": DESIGNS,
    "expected_rows": EXPECTED_ROWS,
    "actual_rows": int(len(scores)),
    "scoring_definition": (
        "(1 + count(null_enrichment >= "
        "true_enrichment)) / 51"
    ),
    "percentile_definition": "1 - p",
    "model_level_definition": (
        "median of image-level percentiles"
    ),
    "true_mask_area_mean": float(
        np.mean(
            [
                np.asarray(
                    true_masks[x],
                    dtype=bool,
                ).sum()
                for x in retained_images
            ]
        )
    ),
    "true_mask_area_min": int(
        min(
            np.asarray(
                true_masks[x],
                dtype=bool,
            ).sum()
            for x in retained_images
        )
    ),
    "true_mask_area_max": int(
        max(
            np.asarray(
                true_masks[x],
                dtype=bool,
            ).sum()
            for x in retained_images
        )
    ),
    "scores_sha256": sha256_file(
        OUT_SCORE_PATH
    ),
    "model_summary_sha256": sha256_file(
        OUT_MODEL_PATH
    ),
}

with open(
    OUT_LOCK_PATH,
    "w",
) as f:
    json.dump(
        lock,
        f,
        indent=2,
    )

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

print()
print("=" * 70)
print("T1.4b NULL SCORING COMPLETE")
print("=" * 70)

print(
    "Scoring rows                 :",
    len(scores),
)

print(
    "Expected rows               :",
    EXPECTED_ROWS,
)

print(
    "Design A rows               :",
    int(
        (
            scores["design"] == "A"
        ).sum()
    ),
)

print(
    "Design Cprime rows          :",
    int(
        (
            scores["design"] == "Cprime"
        ).sum()
    ),
)

print()
print("Model-level summary:")

print(
    model_summary
    .to_string(index=False)
)

print()
print("Scores:")
print(OUT_SCORE_PATH)

print()
print("Model summary:")
print(OUT_MODEL_PATH)

print()
print("Lock:")
print(OUT_LOCK_PATH)

print()
print("=" * 70)
print("NO INFERENTIAL TEST PERFORMED")
print("=" * 70)

T1.4b — GENERATOR-MATCHED SPATIAL NULL SCORING

Expected scoring rows: 5940

Null manifest validation       : PASS
Mapping validation             : PASS
Canonical roster validation    : PASS
True-mask population           : PASS
01/30 | 0.0_1 | 99 images × 2 designs | PASS
02/30 | 0.0_2 | 99 images × 2 designs | PASS
03/30 | 0.0_3 | 99 images × 2 designs | PASS
04/30 | 0.0_4 | 99 images × 2 designs | PASS
05/30 | 0.0_5 | 99 images × 2 designs | PASS
06/30 | 0.2_1 | 99 images × 2 designs | PASS
07/30 | 0.2_2 | 99 images × 2 designs | PASS
08/30 | 0.2_3 | 99 images × 2 designs | PASS
09/30 | 0.2_4 | 99 images × 2 designs | PASS
10/30 | 0.2_5 | 99 images × 2 designs | PASS
11/30 | 0.4_1 | 99 images × 2 designs | PASS
12/30 | 0.4_2 | 99 images × 2 designs | PASS
13/30 | 0.4_3 | 99 images × 2 designs | PASS
14/30 | 0.4_4 | 99 images × 2 designs | PASS
15/30 | 0.4_5 | 99 images × 2 designs | PASS
16/30 | 0.6_1 | 99 images × 2 designs | PASS
17/30 | 0.6_2 | 99 images × 2 designs | PASS
18/30 

In [73]:
# ================================================================
# CELL 6F — T1.4b SCORING DISTRIBUTION + NULL-DESIGN INTEGRITY AUDIT
# ================================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — SCORING DISTRIBUTION + NULL-DESIGN INTEGRITY AUDIT")
print("=" * 70)

BASE_DIR = Path(
    "/kaggle/working/T1_4B_generator_matched_spatial_null"
)

SCORES_PATH = (
    BASE_DIR /
    "T1_4B_NULL_ENRICHMENT_SCORES.csv"
)

MODEL_SUMMARY_PATH = (
    BASE_DIR /
    "T1_4B_MODEL_LEVEL_PERCENTILES.csv"
)

NULL_MANIFEST_PATH = (
    BASE_DIR /
    "T1_4B_NULL_MASK_MANIFEST.csv"
)

MAPPING_PATH = (
    BASE_DIR /
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

scores = pd.read_csv(
    SCORES_PATH
)

model_summary = pd.read_csv(
    MODEL_SUMMARY_PATH
)

null_manifest = pd.read_csv(
    NULL_MANIFEST_PATH
)

mapping = pd.read_csv(
    MAPPING_PATH
)

# ---------------------------------------------------------------
# Basic dimensions
# ---------------------------------------------------------------

assert len(scores) == 5940
assert len(model_summary) == 60

assert set(
    scores["design"].unique()
) == {
    "A",
    "Cprime",
}

assert (
    scores["percentile"]
    .notna()
    .all()
)

assert (
    scores["true_enrichment"]
    .notna()
    .all()
)

# ---------------------------------------------------------------
# Convert percentile back to number of null exceedances.
#
# percentile = 1 - (1 + exceedances)/51
#
# Therefore:
# exceedances = 51*(1-percentile) - 1
# ---------------------------------------------------------------

scores["null_ge_true_recovered"] = (
    51.0 *
    (1.0 - scores["percentile"])
    - 1.0
)

assert np.allclose(
    scores["null_ge_true_recovered"],
    np.round(
        scores["null_ge_true_recovered"]
    ),
    atol=1e-10,
)

scores[
    "null_ge_true_recovered"
] = scores[
    "null_ge_true_recovered"
].astype(int)

# ---------------------------------------------------------------
# How often does the maximum percentile occur?
# ---------------------------------------------------------------

max_pct = 50.0 / 51.0

scores["at_maximum_percentile"] = np.isclose(
    scores["percentile"],
    max_pct,
    atol=1e-12,
)

print()
print("Total scoring rows              :", len(scores))

print(
    "Rows at maximum percentile      :",
    int(
        scores[
            "at_maximum_percentile"
        ].sum()
    ),
    "/",
    len(scores),
)

print(
    "Rows NOT at maximum percentile  :",
    int(
        (~scores[
            "at_maximum_percentile"
        ]).sum()
    ),
)

# ---------------------------------------------------------------
# Distribution of null exceedance counts
# ---------------------------------------------------------------

print()
print("Null exceedance-count distribution:")

exceedance_counts = (
    scores[
        "null_ge_true_recovered"
    ]
    .value_counts()
    .sort_index()
)

print(
    exceedance_counts.to_string()
)

# ---------------------------------------------------------------
# Image-level analysis
#
# For each model × design, determine whether all 99 images
# have zero null exceedances.
# ---------------------------------------------------------------

grouped = (
    scores
    .groupby(
        [
            "model_key",
            "design",
        ]
    )
)

group_rows = []

for (
    model_key,
    design,
), sub in grouped:

    assert len(sub) == 99

    n_zero = int(
        (
            sub[
                "null_ge_true_recovered"
            ]
            ==
            0
        ).sum()
    )

    n_nonzero = 99 - n_zero

    group_rows.append({
        "model_key": model_key,
        "design": design,
        "n_images": len(sub),
        "n_zero_exceedance_images": n_zero,
        "n_nonzero_exceedance_images": n_nonzero,
        "median_percentile":
            float(
                sub[
                    "percentile"
                ].median()
            ),
        "min_percentile":
            float(
                sub[
                    "percentile"
                ].min()
            ),
        "max_percentile":
            float(
                sub[
                    "percentile"
                ].max()
            ),
        "median_true_enrichment":
            float(
                sub[
                    "true_enrichment"
                ].median()
            ),
    })

group_audit = pd.DataFrame(
    group_rows
)

print()
print("Model/design groups:")
print(
    group_audit.to_string(
        index=False
    )
)

# ---------------------------------------------------------------
# How many model/design groups have every image at 50/51?
# ---------------------------------------------------------------

all_99_zero = (
    group_audit[
        "n_zero_exceedance_images"
    ]
    ==
    99
)

print()
print(
    "Model/design groups with zero null "
    "exceedances for ALL 99 images :",
    int(all_99_zero.sum()),
    "/ 60",
)

print(
    "Model/design groups with at least "
    "one null exceedance             :",
    int((~all_99_zero).sum()),
    "/ 60",
)

# ---------------------------------------------------------------
# Cross-design comparison at image level.
#
# A and Cprime should not silently be identical.
# ---------------------------------------------------------------

pivot_pct = (
    scores
    .pivot_table(
        index=[
            "model_key",
            "image",
        ],
        columns="design",
        values="percentile",
        aggfunc="first",
    )
)

assert set(
    pivot_pct.columns
) == {
    "A",
    "Cprime",
}

design_pct_equal = np.isclose(
    pivot_pct["A"].to_numpy(),
    pivot_pct["Cprime"].to_numpy(),
    atol=1e-12,
)

print()
print(
    "Model-image pairs:",
    len(pivot_pct)
)

print(
    "A == Cprime percentile:",
    int(design_pct_equal.sum()),
    "/",
    len(design_pct_equal),
)

print(
    "A != Cprime percentile:",
    int((~design_pct_equal).sum()),
    "/",
    len(design_pct_equal),
)

# ---------------------------------------------------------------
# Null-manifest geometric sanity checks
# ---------------------------------------------------------------

for design in ["A", "Cprime"]:

    sub = null_manifest[
        null_manifest["design"] == design
    ].copy()

    assert len(sub) == 4950

    area_diff = (
        sub["null_area"]
        -
        sub["true_area"]
    )

    assert (
        area_diff == 0
    ).all()

    print()
    print(
        f"Design {design} geometry:"
    )

    print(
        "  mean center-distance difference :",
        float(
            sub[
                "center_distance_abs_diff"
            ].mean()
        )
    )

    print(
        "  max center-distance difference  :",
        float(
            sub[
                "center_distance_abs_diff"
            ].max()
        )
    )

# ---------------------------------------------------------------
# Check whether A and Cprime use distinct masks.
#
# Mapping itself must be distinct at least at the NPZ stem level
# unless the designs happen to produce identical complete sets.
# ---------------------------------------------------------------

mapping_stem_equal = (
    mapping["A_stem"].astype(str)
    ==
    mapping["Cprime_stem"].astype(str)
)

print()
print(
    "Images where A/Cprime NPZ stems are identical:",
    int(mapping_stem_equal.sum()),
    "/",
    len(mapping),
)

# ---------------------------------------------------------------
# True enrichment distribution
# ---------------------------------------------------------------

print()
print("True enrichment distribution:")

print(
    scores[
        [
            "model_key",
            "design",
            "true_enrichment",
        ]
    ]
    .groupby(
        [
            "model_key",
            "design",
        ]
    )[
        "true_enrichment"
    ]
    .agg(
        [
            "min",
            "median",
            "max",
        ]
    )
    .to_string()
)

# ---------------------------------------------------------------
# Save audit
# ---------------------------------------------------------------

GROUP_AUDIT_PATH = (
    BASE_DIR /
    "T1_4B_SCORING_GROUP_INTEGRITY_AUDIT.csv"
)

group_audit.to_csv(
    GROUP_AUDIT_PATH,
    index=False,
)

AUDIT_JSON_PATH = (
    BASE_DIR /
    "T1_4B_SCORING_DISTRIBUTION_AUDIT.json"
)

audit_summary = {
    "total_rows": int(len(scores)),
    "maximum_percentile": max_pct,
    "rows_at_maximum_percentile": int(
        scores[
            "at_maximum_percentile"
        ].sum()
    ),
    "rows_not_at_maximum_percentile": int(
        (~scores[
            "at_maximum_percentile"
        ]).sum()
    ),
    "model_design_groups": 60,
    "groups_all_99_zero_exceedance": int(
        all_99_zero.sum()
    ),
    "groups_with_nonzero_exceedance": int(
        (~all_99_zero).sum()
    ),
    "model_image_pairs": int(
        len(pivot_pct)
    ),
    "A_equals_Cprime_percentile_pairs": int(
        design_pct_equal.sum()
    ),
    "A_differs_Cprime_percentile_pairs": int(
        (~design_pct_equal).sum()
    ),
    "A_equals_Cprime_stem_images": int(
        mapping_stem_equal.sum()
    ),
}

with open(
    AUDIT_JSON_PATH,
    "w",
) as f:
    json.dump(
        audit_summary,
        f,
        indent=2,
    )

print()
print("=" * 70)
print("SCORING DISTRIBUTION AUDIT COMPLETE")
print("=" * 70)

print(
    "Group audit:",
    GROUP_AUDIT_PATH,
)

print(
    "Audit JSON:",
    AUDIT_JSON_PATH,
)

print()
print("DO NOT RUN THE SIGN-FLIP TEST YET.")
print("=" * 70)

T1.4b — SCORING DISTRIBUTION + NULL-DESIGN INTEGRITY AUDIT

Total scoring rows              : 5940
Rows at maximum percentile      : 5854 / 5940
Rows NOT at maximum percentile  : 86

Null exceedance-count distribution:
null_ge_true_recovered
0     5880
2       22
5       14
7        3
8        5
10       3
11       2
12       2
13       2
15       2
20       1
23       1
29       1
39       1
41       1

Model/design groups:
model_key design  n_images  n_zero_exceedance_images  n_nonzero_exceedance_images  median_percentile  min_percentile  max_percentile  median_true_enrichment
    0.0_1      A        99                        89                           10           0.980392        0.215686        0.980392                6.676639
    0.0_1 Cprime        99                        96                            3           0.980392        0.843137        0.980392                6.676639
    0.0_2      A        99                        90                            9           0.980392

In [74]:
# ================================================================
# CELL 6G — T1.4b TRUE-vs-NULL ENRICHMENT MARGIN AUDIT
# ================================================================

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — TRUE-vs-NULL ENRICHMENT MARGIN AUDIT")
print("=" * 70)

BASE_DIR = Path(
    "/kaggle/working/T1_4B_generator_matched_spatial_null"
)

NULL_MASK_DIR = (
    BASE_DIR /
    "T1_4B_NULL_MASKS"
)

MAPPING_PATH = (
    BASE_DIR /
    "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv"
)

SCORES_PATH = (
    BASE_DIR /
    "T1_4B_NULL_ENRICHMENT_SCORES.csv"
)

MARGIN_PATH = (
    BASE_DIR /
    "T1_4B_TRUE_VS_NULL_MARGIN_AUDIT.csv"
)

# ---------------------------------------------------------------
# Load frozen scoring outputs
# ---------------------------------------------------------------

scores = pd.read_csv(
    SCORES_PATH
)

mapping = pd.read_csv(
    MAPPING_PATH
)

assert len(scores) == 5940
assert len(mapping) == 99

assert "true_masks" in globals()
assert len(true_masks) == 99

retained_images = sorted(
    true_masks.keys()
)

assert len(retained_images) == 99

mapping_by_image = (
    mapping
    .set_index("image")
    .to_dict("index")
)

# ---------------------------------------------------------------
# Frozen enrichment function
# ---------------------------------------------------------------

EPS = 1e-8
N_NULL = 50

def enrichment_from_abs_map(
    abs_map,
    mask_bool,
):
    abs_map = np.asarray(
        abs_map,
        dtype=np.float64,
    )

    mask_bool = np.asarray(
        mask_bool,
        dtype=bool,
    )

    total_attr = float(
        abs_map.sum()
    )

    area_fraction = float(
        mask_bool.mean()
    )

    if (
        total_attr <= EPS
        or area_fraction <= 0
    ):
        return np.nan

    inside_attr = float(
        abs_map[mask_bool].sum()
    )

    return (
        (inside_attr / total_attr)
        / area_fraction
    )

# ---------------------------------------------------------------
# Reconstruct exact margins for every
# model × image × design.
# ---------------------------------------------------------------

margin_rows = []

for model_key in scores[
    "model_key"
].astype(str).unique():

    model_scores = scores[
        scores["model_key"].astype(str)
        ==
        model_key
    ]

    model_dir = (
        BASE_DIR /
        "T1_4B_PRODUCTION_IG" /
        "hair" /
        f"model_{model_key}"
    )

    abs_path = (
        model_dir /
        "abs_ig.npy"
    )

    assert abs_path.exists()

    abs_ig = np.load(
        abs_path,
        mmap_mode="r",
    )

    assert abs_ig.shape == (
        99,
        224,
        224,
    )

    for image_idx, image in enumerate(
        retained_images
    ):

        abs_map = np.asarray(
            abs_ig[image_idx],
            dtype=np.float32,
        )

        true_mask = np.asarray(
            true_masks[image],
            dtype=bool,
        )

        true_enrichment = (
            enrichment_from_abs_map(
                abs_map,
                true_mask,
            )
        )

        assert np.isfinite(
            true_enrichment
        )

        row = mapping_by_image[
            image
        ]

        for design in [
            "A",
            "Cprime",
        ]:

            if design == "A":
                stem = str(
                    row["A_stem"]
                )
            else:
                stem = str(
                    row["Cprime_stem"]
                )

            npz_path = (
                NULL_MASK_DIR /
                f"{stem}_{design}.npz"
            )

            assert npz_path.exists()

            data = np.load(
                npz_path
            )

            if "masks" in data:
                null_masks = data[
                    "masks"
                ]
            elif "null_masks" in data:
                null_masks = data[
                    "null_masks"
                ]
            else:
                raise KeyError(
                    f"No mask array in {npz_path}. "
                    f"Keys={data.files}"
                )

            null_masks = np.asarray(
                null_masks,
                dtype=bool,
            )

            assert null_masks.shape == (
                50,
                224,
                224,
            )

            null_enrichments = np.array([
                enrichment_from_abs_map(
                    abs_map,
                    null_masks[k],
                )
                for k in range(N_NULL)
            ])

            assert np.isfinite(
                null_enrichments
            ).all()

            max_null = float(
                null_enrichments.max()
            )

            min_null = float(
                null_enrichments.min()
            )

            median_null = float(
                np.median(
                    null_enrichments
                )
            )

            margin = (
                true_enrichment
                -
                max_null
            )

            mean_margin = (
                true_enrichment
                -
                median_null
            )

            exceedances = int(
                np.count_nonzero(
                    null_enrichments
                    >=
                    true_enrichment
                )
            )

            score_row = model_scores[
                (
                    model_scores[
                        "image"
                    ].astype(str)
                    ==
                    image
                )
                &
                (
                    model_scores[
                        "design"
                    ].astype(str)
                    ==
                    design
                )
            ]

            assert len(score_row) == 1

            recorded_exceedances = int(
                score_row.iloc[0][
                    "null_ge_true"
                ]
            )

            assert (
                exceedances
                ==
                recorded_exceedances
            )

            margin_rows.append({
                "model_key": model_key,
                "image": image,
                "image_index": image_idx,
                "design": design,
                "true_enrichment":
                    true_enrichment,
                "null_min":
                    min_null,
                "null_median":
                    median_null,
                "null_max":
                    max_null,
                "true_minus_null_max":
                    margin,
                "true_minus_null_median":
                    mean_margin,
                "null_ge_true":
                    exceedances,
            })

margin_df = pd.DataFrame(
    margin_rows
)

assert len(margin_df) == 5940

assert (
    margin_df[
        "null_ge_true"
    ]
    .between(0, 50)
    .all()
)

# ---------------------------------------------------------------
# Core consistency checks
# ---------------------------------------------------------------

# Zero exceedances must imply strict positive margin.
zero_exceedance = (
    margin_df[
        "null_ge_true"
    ]
    ==
    0
)

assert (
    margin_df.loc[
        zero_exceedance,
        "true_minus_null_max"
    ]
    > 0
).all()

# Any positive exceedance must imply
# max-null >= true.
positive_exceedance = (
    margin_df[
        "null_ge_true"
    ]
    >
    0
)

assert (
    margin_df.loc[
        positive_exceedance,
        "true_minus_null_max"
    ]
    <= 0
).all()

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

print()
print(
    "Total model-image-design rows :",
    len(margin_df)
)

print(
    "Zero-exceedance rows          :",
    int(zero_exceedance.sum())
)

print(
    "Positive-exceedance rows      :",
    int(positive_exceedance.sum())
)

print()

print(
    "Minimum TRUE - maximum NULL   :",
    float(
        margin_df[
            "true_minus_null_max"
        ].min()
    )
)

print(
    "Median TRUE - maximum NULL    :",
    float(
        margin_df[
            "true_minus_null_max"
        ].median()
    )
)

print(
    "Maximum TRUE - maximum NULL   :",
    float(
        margin_df[
            "true_minus_null_max"
        ].max()
    )
)

print()

print(
    "Minimum TRUE - median NULL    :",
    float(
        margin_df[
            "true_minus_null_median"
        ].min()
    )
)

print(
    "Median TRUE - median NULL     :",
    float(
        margin_df[
            "true_minus_null_median"
        ].median()
    )
)

print(
    "Maximum TRUE - median NULL    :",
    float(
        margin_df[
            "true_minus_null_median"
        ].max()
    )
)

# ---------------------------------------------------------------
# Distribution of margin by design
# ---------------------------------------------------------------

print()
print("TRUE - MAX(NULL) by design:")

print(
    margin_df
    .groupby("design")[
        "true_minus_null_max"
    ]
    .agg(
        [
            "count",
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .to_string()
)

print()
print("TRUE - MEDIAN(NULL) by design:")

print(
    margin_df
    .groupby("design")[
        "true_minus_null_median"
    ]
    .agg(
        [
            "count",
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .to_string()
)

# ---------------------------------------------------------------
# Rows closest to the null boundary
# ---------------------------------------------------------------

print()
print(
    "10 rows closest to the TRUE-vs-MAX(NULL) boundary:"
)

print(
    margin_df
    .sort_values(
        "true_minus_null_max"
    )
    .head(10)
    [
        [
            "model_key",
            "image",
            "design",
            "true_enrichment",
            "null_max",
            "true_minus_null_max",
            "null_ge_true",
        ]
    ]
    .to_string(index=False)
)

# ---------------------------------------------------------------
# Save
# ---------------------------------------------------------------

margin_df.to_csv(
    MARGIN_PATH,
    index=False,
)

audit = {
    "rows": int(len(margin_df)),
    "zero_exceedance_rows": int(
        zero_exceedance.sum()
    ),
    "positive_exceedance_rows": int(
        positive_exceedance.sum()
    ),
    "minimum_true_minus_max_null": float(
        margin_df[
            "true_minus_null_max"
        ].min()
    ),
    "median_true_minus_max_null": float(
        margin_df[
            "true_minus_null_max"
        ].median()
    ),
    "maximum_true_minus_max_null": float(
        margin_df[
            "true_minus_null_max"
        ].max()
    ),
    "minimum_true_minus_median_null": float(
        margin_df[
            "true_minus_null_median"
        ].min()
    ),
    "median_true_minus_median_null": float(
        margin_df[
            "true_minus_null_median"
        ].median()
    ),
    "maximum_true_minus_median_null": float(
        margin_df[
            "true_minus_null_median"
        ].max()
    ),
}

AUDIT_PATH = (
    BASE_DIR /
    "T1_4B_TRUE_VS_NULL_MARGIN_AUDIT.json"
)

with open(
    AUDIT_PATH,
    "w",
) as f:
    json.dump(
        audit,
        f,
        indent=2,
    )

print()
print("=" * 70)
print("TRUE-vs-NULL MARGIN AUDIT COMPLETE")
print("=" * 70)

print(
    "CSV :",
    MARGIN_PATH,
)

print(
    "JSON:",
    AUDIT_PATH,
)

print()
print("DO NOT RUN INFERENCE YET.")
print("=" * 70)

T1.4b — TRUE-vs-NULL ENRICHMENT MARGIN AUDIT

Total model-image-design rows : 5940
Zero-exceedance rows          : 5854
Positive-exceedance rows      : 86

Minimum TRUE - maximum NULL   : -7.341467726197468
Median TRUE - maximum NULL    : 8.059146133835547
Maximum TRUE - maximum NULL   : 48.3894724254929

Minimum TRUE - median NULL    : -0.6827469855305222
Median TRUE - median NULL     : 9.004725927378075
Maximum TRUE - median NULL    : 51.16324244633894

TRUE - MAX(NULL) by design:
        count       min    median       mean        max
design                                                 
A        2970 -7.341468  7.769555   9.296947  48.389472
Cprime   2970 -1.031433  8.344088  10.035623  48.341756

TRUE - MEDIAN(NULL) by design:
        count       min    median       mean        max
design                                                 
A        2970 -0.682747  8.943135  10.718340  50.732306
Cprime   2970  0.397234  9.103997  10.926802  51.163242

10 rows closest to the TRUE-vs-

In [75]:
# ================================================================
# CELL 6H — T1.4b PRIMARY INFERENTIAL TEST
# ================================================================
#
# Primary confirmatory design:
#   Design A
#
# Model-level statistic:
#   M_m = median_i(percentile_i,m)
#
# Center:
#   d_m = M_m - 0.5
#
# Test:
#   one-sided sign-flip permutation
#   n_models = 30
#   n_permutations = 100,000
#   seed = 20260919
#
# ================================================================

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — PRIMARY SIGN-FLIP PERMUTATION TEST")
print("=" * 70)

# ---------------------------------------------------------------
# Frozen paths
# ---------------------------------------------------------------

BASE_DIR = Path(
    "/kaggle/working/T1_4B_generator_matched_spatial_null"
)

MODEL_SUMMARY_PATH = (
    BASE_DIR /
    "T1_4B_MODEL_LEVEL_PERCENTILES.csv"
)

MARGIN_AUDIT_PATH = (
    BASE_DIR /
    "T1_4B_TRUE_VS_NULL_MARGIN_AUDIT.csv"
)

OUT_RESULTS_PATH = (
    BASE_DIR /
    "T1_4B_PRIMARY_SIGNFLIP_RESULT.csv"
)

OUT_NULL_DIST_PATH = (
    BASE_DIR /
    "T1_4B_PRIMARY_SIGNFLIP_NULL_DISTRIBUTION.npy"
)

OUT_LOCK_PATH = (
    BASE_DIR /
    "T1_4B_PRIMARY_SIGNFLIP_LOCK.json"
)

# ---------------------------------------------------------------
# Frozen constants
# ---------------------------------------------------------------

SEED = 20260919
N_PERM = 100_000
ALPHA = 0.05
CENTER = 0.5

# ---------------------------------------------------------------
# Load model-level results
# ---------------------------------------------------------------

model_summary = pd.read_csv(
    MODEL_SUMMARY_PATH
)

assert len(model_summary) == 60

# ---------------------------------------------------------------
# Primary design = A
# ---------------------------------------------------------------

primary = model_summary[
    model_summary["design"].astype(str)
    ==
    "A"
].copy()

assert len(primary) == 30

primary = primary.sort_values(
    [
        "rho",
        "replicate",
    ]
).reset_index(
    drop=True
)

# ---------------------------------------------------------------
# Validate canonical model order
# ---------------------------------------------------------------

expected_model_keys = [
    f"{rho:.1f}_{rep}"
    for rho in [
        0.0,
        0.2,
        0.4,
        0.6,
        0.8,
        1.0,
    ]
    for rep in range(1, 6)
]

assert (
    primary["model_key"].astype(str).tolist()
    ==
    expected_model_keys
)

# ---------------------------------------------------------------
# Extract model-level M_m
# ---------------------------------------------------------------

M = primary[
    "model_level_percentile"
].to_numpy(
    dtype=np.float64
)

assert M.shape == (30,)
assert np.isfinite(M).all()

# ---------------------------------------------------------------
# Frozen centered values
# ---------------------------------------------------------------

d = M - CENTER

assert d.shape == (30,)
assert np.isfinite(d).all()

# ---------------------------------------------------------------
# Observed test statistic
#
# Frozen model-level statistic:
#   mean(d_m)
#
# ---------------------------------------------------------------

observed_stat = float(
    np.mean(d)
)

# Equivalent:
# observed_stat = mean(M) - 0.5

assert np.isclose(
    observed_stat,
    float(np.mean(M) - CENTER),
)

# ---------------------------------------------------------------
# Sign-flip permutation distribution
#
# Every model-level d_m is independently multiplied by
# +1 or -1 with probability 0.5.
#
# Statistic for each permutation:
#
#   mean(sign_m * d_m)
#
# ---------------------------------------------------------------

rng = np.random.Generator(
    np.random.PCG64(SEED)
)

# Generate in chunks to avoid unnecessary memory pressure.
#
# 100,000 × 30 = 3,000,000 signs, which is modest,
# but chunking makes the implementation explicit and stable.

CHUNK = 10_000

null_stats = np.empty(
    N_PERM,
    dtype=np.float64,
)

write_idx = 0

while write_idx < N_PERM:

    n = min(
        CHUNK,
        N_PERM - write_idx,
    )

    signs = rng.integers(
        0,
        2,
        size=(n, len(d)),
        dtype=np.int8,
    )

    signs = (
        signs * 2
        - 1
    ).astype(
        np.float64
    )

    null_stats[
        write_idx:
        write_idx + n
    ] = (
        signs @ d
    ) / len(d)

    write_idx += n

# ---------------------------------------------------------------
# One-sided p-value
#
# Test direction:
#   observed statistic > 0
#
# Monte-Carlo finite-sample correction:
#
#   p = (1 + number(null >= observed))
#       / (N_PERM + 1)
# ---------------------------------------------------------------

n_ge = int(
    np.count_nonzero(
        null_stats >= observed_stat
    )
)

p_value = (
    1.0 + n_ge
) / (
    N_PERM + 1.0
)

# ---------------------------------------------------------------
# Directional effect summary
# ---------------------------------------------------------------

mean_M = float(
    np.mean(M)
)

median_M = float(
    np.median(M)
)

min_M = float(
    np.min(M)
)

max_M = float(
    np.max(M)
)

mean_d = float(
    np.mean(d)
)

median_d = float(
    np.median(d)
)

# ---------------------------------------------------------------
# Model-level table
# ---------------------------------------------------------------

primary["center"] = CENTER

primary["d_m"] = d

primary["above_null_center"] = (
    primary["d_m"] > 0
)

# ---------------------------------------------------------------
# Basic inferential checks
# ---------------------------------------------------------------

assert (
    primary["above_null_center"].sum()
    +
    (
        primary["d_m"] <= 0
    ).sum()
    ==
    30
)

assert np.isfinite(
    null_stats
).all()

assert len(null_stats) == N_PERM

assert (
    0.0 <= p_value <= 1.0
)

# ---------------------------------------------------------------
# Null distribution diagnostics
# ---------------------------------------------------------------

null_mean = float(
    np.mean(null_stats)
)

null_sd = float(
    np.std(
        null_stats,
        ddof=1,
    )
)

null_q025 = float(
    np.quantile(
        null_stats,
        0.025,
    )
)

null_q50 = float(
    np.quantile(
        null_stats,
        0.50,
    )
)

null_q975 = float(
    np.quantile(
        null_stats,
        0.975,
    )
)

# ---------------------------------------------------------------
# Save null distribution
# ---------------------------------------------------------------

np.save(
    OUT_NULL_DIST_PATH,
    null_stats,
)

# ---------------------------------------------------------------
# Save model-level table
# ---------------------------------------------------------------

MODEL_LEVEL_RESULT_PATH = (
    BASE_DIR /
    "T1_4B_PRIMARY_MODEL_LEVEL_EFFECTS.csv"
)

primary.to_csv(
    MODEL_LEVEL_RESULT_PATH,
    index=False,
)

# ---------------------------------------------------------------
# Save primary result
# ---------------------------------------------------------------

result_row = {
    "protocol": "T1.4b",
    "design": "A",
    "n_models": 30,
    "n_images": 99,
    "n_nulls_per_image": 50,
    "center": CENTER,
    "observed_mean_M": mean_M,
    "observed_median_M": median_M,
    "observed_min_M": min_M,
    "observed_max_M": max_M,
    "observed_mean_d": mean_d,
    "observed_median_d": median_d,
    "n_models_d_positive": int(
        np.sum(d > 0)
    ),
    "n_models_d_nonpositive": int(
        np.sum(d <= 0)
    ),
    "n_permutations": N_PERM,
    "seed": SEED,
    "observed_statistic": observed_stat,
    "null_mean": null_mean,
    "null_sd": null_sd,
    "null_q025": null_q025,
    "null_median": null_q50,
    "null_q975": null_q975,
    "n_null_ge_observed": n_ge,
    "p_value_one_sided": p_value,
    "alpha": ALPHA,
    "reject_at_alpha_0_05": (
        p_value < ALPHA
    ),
}

result_df = pd.DataFrame(
    [result_row]
)

result_df.to_csv(
    OUT_RESULTS_PATH,
    index=False,
)

# ---------------------------------------------------------------
# SHA256 helper
# ---------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()

# ---------------------------------------------------------------
# Lock
# ---------------------------------------------------------------

lock = {
    "protocol": "T1.4b",
    "stage": "primary_signflip_inference",
    "design": "A",
    "n_models": 30,
    "n_images": 99,
    "n_nulls_per_image": 50,
    "center": CENTER,
    "statistic": "mean(M_m - 0.5)",
    "permutation": "independent sign flips",
    "n_permutations": N_PERM,
    "seed": SEED,
    "alpha": ALPHA,
    "finite_sample_p_correction":
        "(1 + count(null >= observed)) / "
        "(N_PERM + 1)",
    "observed_statistic":
        observed_stat,
    "p_value":
        p_value,
    "result_sha256":
        sha256_file(
            OUT_RESULTS_PATH
        ),
    "model_level_effects_sha256":
        sha256_file(
            MODEL_LEVEL_RESULT_PATH
        ),
    "null_distribution_sha256":
        sha256_file(
            OUT_NULL_DIST_PATH
        ),
}

with open(
    OUT_LOCK_PATH,
    "w",
) as f:

    json.dump(
        lock,
        f,
        indent=2,
    )

# ---------------------------------------------------------------
# Print final result
# ---------------------------------------------------------------

print()
print("=" * 70)
print("PRIMARY T1.4b SIGN-FLIP RESULT")
print("=" * 70)

print(
    "Design                         : A"
)

print(
    "Models                         :",
    len(M),
)

print(
    "Images/model                   : 99"
)

print(
    "Nulls/image                    : 50"
)

print()
print(
    "Mean model-level percentile M  :",
    mean_M,
)

print(
    "Median model-level percentile M:",
    median_M,
)

print(
    "Minimum model-level percentile :",
    min_M,
)

print(
    "Maximum model-level percentile :",
    max_M,
)

print()
print(
    "Observed statistic mean(M-0.5):",
    observed_stat,
)

print(
    "Models with d > 0              :",
    int(np.sum(d > 0)),
    "/ 30",
)

print()
print(
    "Permutation null mean          :",
    null_mean,
)

print(
    "Permutation null SD            :",
    null_sd,
)

print(
    "Permutation 2.5% quantile      :",
    null_q025,
)

print(
    "Permutation median              :",
    null_q50,
)

print(
    "Permutation 97.5% quantile     :",
    null_q975,
)

print()
print(
    "Null statistics >= observed    :",
    n_ge,
)

print(
    "One-sided permutation p-value  :",
    p_value,
)

print(
    "Alpha                          :",
    ALPHA,
)

print(
    "p < alpha                      :",
    p_value < ALPHA,
)

print()
print(
    "Result CSV:"
)

print(
    OUT_RESULTS_PATH
)

print()
print(
    "Model-level effects CSV:"
)

print(
    MODEL_LEVEL_RESULT_PATH
)

print()
print(
    "Null distribution:"
)

print(
    OUT_NULL_DIST_PATH
)

print()
print(
    "Lock:"
)

print(
    OUT_LOCK_PATH
)

print()
print("=" * 70)
print("PRIMARY INFERENCE COMPLETE")
print("=" * 70)

T1.4b — PRIMARY SIGN-FLIP PERMUTATION TEST

PRIMARY T1.4b SIGN-FLIP RESULT
Design                         : A
Models                         : 30
Images/model                   : 99
Nulls/image                    : 50

Mean model-level percentile M  : 0.9803921568627452
Median model-level percentile M: 0.9803921568627452
Minimum model-level percentile : 0.9803921568627452
Maximum model-level percentile : 0.9803921568627452

Observed statistic mean(M-0.5): 0.48039215686274517
Models with d > 0              : 30 / 30

Permutation null mean          : -0.0001287450980392157
Permutation null SD            : 0.08790741384011883
Permutation 2.5% quantile      : -0.16013071895424838
Permutation median              : 0.0
Permutation 97.5% quantile     : 0.16013071895424838

Null statistics >= observed    : 0
One-sided permutation p-value  : 9.99990000099999e-06
Alpha                          : 0.05
p < alpha                      : True

Result CSV:
/kaggle/working/T1_4B_generator_matched_spati

In [76]:
# ================================================================
# CELL 6I — T1.4b FINAL REPRODUCIBILITY / LOCK AUDIT
# ================================================================

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("T1.4b — FINAL REPRODUCIBILITY / LOCK AUDIT")
print("=" * 70)

BASE_DIR = Path(
    "/kaggle/working/T1_4B_generator_matched_spatial_null"
)

# ---------------------------------------------------------------
# Frozen artifact paths
# ---------------------------------------------------------------

paths = {
    "sample_lock":
        BASE_DIR /
        "T1_4B_IMAGE_SAMPLE_LOCK.json",

    "null_manifest":
        BASE_DIR /
        "T1_4B_NULL_MASK_MANIFEST.csv",

    "null_mapping":
        BASE_DIR /
        "T1_4B_NULL_FILENAME_MAPPING_BITEXACT_AUDIT.csv",

    "checkpoint_roster":
        BASE_DIR /
        "T1_4B_CANONICAL_CHECKPOINT_ROSTER.csv",

    "ig_lock":
        BASE_DIR /
        "T1_4B_PRODUCTION_IG" /
        "T1_4B_PRODUCTION_IG_COMPLETE_LOCK.json",

    "ig_audit":
        BASE_DIR /
        "T1_4B_PRODUCTION_IG" /
        "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT.csv",

    "ig_audit_lock":
        BASE_DIR /
        "T1_4B_PRODUCTION_IG" /
        "T1_4B_PRODUCTION_IG_PERSISTENCE_AUDIT_LOCK.json",

    "true_mask_reconciliation":
        BASE_DIR /
        "T1_4B_TRUE_MASK_REGENERATION_RECONCILIATION.csv",

    "scores":
        BASE_DIR /
        "T1_4B_NULL_ENRICHMENT_SCORES.csv",

    "model_summary":
        BASE_DIR /
        "T1_4B_MODEL_LEVEL_PERCENTILES.csv",

    "group_audit":
        BASE_DIR /
        "T1_4B_SCORING_GROUP_INTEGRITY_AUDIT.csv",

    "margin_audit":
        BASE_DIR /
        "T1_4B_TRUE_VS_NULL_MARGIN_AUDIT.csv",

    "signflip_result":
        BASE_DIR /
        "T1_4B_PRIMARY_SIGNFLIP_RESULT.csv",

    "model_effects":
        BASE_DIR /
        "T1_4B_PRIMARY_MODEL_LEVEL_EFFECTS.csv",

    "null_distribution":
        BASE_DIR /
        "T1_4B_PRIMARY_SIGNFLIP_NULL_DISTRIBUTION.npy",

    "signflip_lock":
        BASE_DIR /
        "T1_4B_PRIMARY_SIGNFLIP_LOCK.json",
}

# ---------------------------------------------------------------
# SHA256 helper
# ---------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()

# ---------------------------------------------------------------
# Existence
# ---------------------------------------------------------------

print()
print("Artifact existence:")

for name, path in paths.items():

    assert path.exists(), (
        f"Missing artifact: {path}"
    )

    print(
        f"  {name:25s} PASS"
    )

# ---------------------------------------------------------------
# Load primary result + lock
# ---------------------------------------------------------------

result = pd.read_csv(
    paths["signflip_result"]
)

model_effects = pd.read_csv(
    paths["model_effects"]
)

scores = pd.read_csv(
    paths["scores"]
)

model_summary = pd.read_csv(
    paths["model_summary"]
)

margin_audit = pd.read_csv(
    paths["margin_audit"]
)

group_audit = pd.read_csv(
    paths["group_audit"]
)

true_mask_audit = pd.read_csv(
    paths["true_mask_reconciliation"]
)

null_manifest = pd.read_csv(
    paths["null_manifest"]
)

mapping = pd.read_csv(
    paths["null_mapping"]
)

roster = pd.read_csv(
    paths["checkpoint_roster"]
)

null_distribution = np.load(
    paths["null_distribution"]
)

with open(
    paths["signflip_lock"],
    "r",
) as f:

    lock = json.load(f)

# ---------------------------------------------------------------
# Structural dimensions
# ---------------------------------------------------------------

assert len(result) == 1
assert len(model_effects) == 30
assert len(scores) == 5940
assert len(model_summary) == 60
assert len(margin_audit) == 5940
assert len(group_audit) == 60
assert len(true_mask_audit) == 99
assert len(null_manifest) == 9900
assert len(mapping) == 99
assert len(roster) == 30
assert len(null_distribution) == 100_000

print()
print("Structural dimensions:")

print("  Scores             : 5940 PASS")
print("  Model summary      : 60 PASS")
print("  Model effects      : 30 PASS")
print("  Margin audit       : 5940 PASS")
print("  Group audit        : 60 PASS")
print("  True-mask audit    : 99 PASS")
print("  Null manifest      : 9900 PASS")
print("  Null mapping       : 99 PASS")
print("  Checkpoint roster  : 30 PASS")
print("  Permutations       : 100000 PASS")

# ---------------------------------------------------------------
# Canonical model keys
# ---------------------------------------------------------------

expected_model_keys = [
    f"{rho:.1f}_{rep}"
    for rho in [
        0.0,
        0.2,
        0.4,
        0.6,
        0.8,
        1.0,
    ]
    for rep in range(1, 6)
]

assert (
    model_effects["model_key"]
    .astype(str)
    .tolist()
    ==
    expected_model_keys
)

assert (
    model_summary[
        model_summary["design"] == "A"
    ]
    .sort_values(
        ["rho", "replicate"]
    )[
        "model_key"
    ]
    .astype(str)
    .tolist()
    ==
    expected_model_keys
)

print()
print(
    "Canonical 30-model structure : PASS"
)

# ---------------------------------------------------------------
# Verify all model-level A values are identical to the
# frozen reported value.
# ---------------------------------------------------------------

A_summary = (
    model_summary[
        model_summary["design"] == "A"
    ]
    .sort_values(
        ["rho", "replicate"]
    )
)

M_A = A_summary[
    "model_level_percentile"
].to_numpy(
    dtype=np.float64
)

assert len(M_A) == 30

assert np.allclose(
    M_A,
    50.0 / 51.0,
    atol=1e-15,
)

print(
    "All 30 Design-A M_m = 50/51 : PASS"
)

# ---------------------------------------------------------------
# Verify primary result numerically
# ---------------------------------------------------------------

reported_mean_M = float(
    result.iloc[0][
        "observed_mean_M"
    ]
)

reported_stat = float(
    result.iloc[0][
        "observed_statistic"
    ]
)

reported_p = float(
    result.iloc[0][
        "p_value_one_sided"
    ]
)

assert np.isclose(
    reported_mean_M,
    50.0 / 51.0,
    atol=1e-15,
)

assert np.isclose(
    reported_stat,
    0.48039215686274517,
    atol=1e-15,
)

assert np.isclose(
    reported_p,
    9.99990000099999e-06,
    atol=1e-15,
)

assert (
    int(
        result.iloc[0][
            "n_models_d_positive"
        ]
    )
    ==
    30
)

print(
    "Primary result numerical lock : PASS"
)

# ---------------------------------------------------------------
# Verify sign-flip lock
# ---------------------------------------------------------------

assert lock["protocol"] == "T1.4b"
assert lock["design"] == "A"
assert lock["n_models"] == 30
assert lock["n_images"] == 99
assert lock["n_nulls_per_image"] == 50
assert lock["center"] == 0.5
assert lock["n_permutations"] == 100_000
assert lock["seed"] == 20260919
assert lock["alpha"] == 0.05

assert np.isclose(
    lock["observed_statistic"],
    reported_stat,
    atol=1e-15,
)

assert np.isclose(
    lock["p_value"],
    reported_p,
    atol=1e-15,
)

print(
    "Sign-flip parameter lock     : PASS"
)

# ---------------------------------------------------------------
# Verify distribution itself
# ---------------------------------------------------------------

assert np.isfinite(
    null_distribution
).all()

assert len(
    null_distribution
) == 100_000

n_ge_recomputed = int(
    np.count_nonzero(
        null_distribution
        >=
        reported_stat
    )
)

assert n_ge_recomputed == int(
    result.iloc[0][
        "n_null_ge_observed"
    ]
)

p_recomputed = (
    1.0 + n_ge_recomputed
) / (
    100_000 + 1.0
)

assert np.isclose(
    p_recomputed,
    reported_p,
    atol=1e-15,
)

print(
    "Permutation distribution lock : PASS"
)

# ---------------------------------------------------------------
# Verify image/model/design multiplicities
# ---------------------------------------------------------------

assert (
    scores
    .groupby(
        [
            "model_key",
            "design",
        ]
    )
    .size()
    .eq(99)
    .all()
)

assert (
    scores
    .groupby("design")
    .size()
    .eq(2970)
    .all()
)

assert (
    margin_audit
    .groupby(
        [
            "model_key",
            "design",
        ]
    )
    .size()
    .eq(99)
    .all()
)

print(
    "Model × image × design grid : PASS"
)

# ---------------------------------------------------------------
# Verify null manifest multiplicity
# ---------------------------------------------------------------

assert (
    null_manifest
    .groupby(
        [
            "image",
            "design",
        ]
    )
    .size()
    .eq(50)
    .all()
)

print(
    "50 nulls/image/design        : PASS"
)

# ---------------------------------------------------------------
# Verify true-mask area equality
# ---------------------------------------------------------------

assert (
    true_mask_audit[
        "exact_area_match"
    ]
    .all()
)

assert (
    true_mask_audit[
        "regenerated_area"
    ].to_numpy()
    ==
    true_mask_audit[
        "locked_area"
    ].to_numpy()
).all()

print(
    "True-mask area reconciliation : PASS"
)

# ---------------------------------------------------------------
# Verify margin/scoring consistency
# ---------------------------------------------------------------

assert (
    margin_audit[
        "null_ge_true"
    ]
    .to_numpy()
    ==
    scores[
        "null_ge_true"
    ].to_numpy()
).all()

print(
    "Margin ↔ scoring consistency  : PASS"
)

# ---------------------------------------------------------------
# Verify primary Design-A ceiling
# ---------------------------------------------------------------

A_scores = scores[
    scores["design"] == "A"
]

assert len(A_scores) == 2970

print()
print(
    "Design-A rows at 50/51       :",
    int(
        np.isclose(
            A_scores[
                "percentile"
            ].to_numpy(),
            50.0 / 51.0,
            atol=1e-15,
        ).sum()
    ),
    "/ 2970"
)

# ---------------------------------------------------------------
# Verify saved SHA256 values
# ---------------------------------------------------------------

assert (
    sha256_file(
        paths["signflip_result"]
    )
    ==
    lock["result_sha256"]
)

assert (
    sha256_file(
        paths["model_effects"]
    )
    ==
    lock["model_level_effects_sha256"]
)

assert (
    sha256_file(
        paths["null_distribution"]
    )
    ==
    lock["null_distribution_sha256"]
)

print()
print(
    "Saved artifact SHA256 locks    : PASS"
)

# ---------------------------------------------------------------
# Create final completion lock
# ---------------------------------------------------------------

FINAL_LOCK_PATH = (
    BASE_DIR /
    "T1_4B_FINAL_COMPLETION_LOCK.json"
)

final_lock = {
    "protocol": "T1.4b",
    "status": "COMPLETE",
    "primary_design": "A",
    "n_models": 30,
    "n_images": 99,
    "n_nulls_per_design": 50,
    "n_scoring_rows": 5940,
    "n_model_level_values": 30,
    "model_level_statistic":
        "median image-level percentile",
    "primary_test":
        "one-sided sign-flip permutation",
    "n_permutations": 100_000,
    "permutation_seed": 20260919,
    "center": 0.5,
    "observed_mean_model_percentile":
        reported_mean_M,
    "observed_statistic":
        reported_stat,
    "n_models_above_center": 30,
    "n_null_ge_observed": n_ge_recomputed,
    "p_value":
        reported_p,
    "alpha": 0.05,
    "primary_inference_significant":
        bool(reported_p < 0.05),
    "scores_sha256":
        sha256_file(
            paths["scores"]
        ),
    "model_summary_sha256":
        sha256_file(
            paths["model_summary"]
        ),
    "margin_audit_sha256":
        sha256_file(
            paths["margin_audit"]
        ),
    "signflip_result_sha256":
        sha256_file(
            paths["signflip_result"]
        ),
    "signflip_null_distribution_sha256":
        sha256_file(
            paths["null_distribution"]
        ),
}

with open(
    FINAL_LOCK_PATH,
    "w",
) as f:

    json.dump(
        final_lock,
        f,
        indent=2,
    )

print()
print("=" * 70)
print("T1.4b FINAL LOCK AUDIT — PASS")
print("=" * 70)

print()
print(
    "Final completion lock:"
)

print(
    FINAL_LOCK_PATH
)

print()
print(
    "T1.4b STATUS: COMPLETE"
)

print("=" * 70)

T1.4b — FINAL REPRODUCIBILITY / LOCK AUDIT

Artifact existence:
  sample_lock               PASS
  null_manifest             PASS
  null_mapping              PASS
  checkpoint_roster         PASS
  ig_lock                   PASS
  ig_audit                  PASS
  ig_audit_lock             PASS
  true_mask_reconciliation  PASS
  scores                    PASS
  model_summary             PASS
  group_audit               PASS
  margin_audit              PASS
  signflip_result           PASS
  model_effects             PASS
  null_distribution         PASS
  signflip_lock             PASS

Structural dimensions:
  Scores             : 5940 PASS
  Model summary      : 60 PASS
  Model effects      : 30 PASS
  Margin audit       : 5940 PASS
  Group audit        : 60 PASS
  True-mask audit    : 99 PASS
  Null manifest      : 9900 PASS
  Null mapping       : 99 PASS
  Checkpoint roster  : 30 PASS
  Permutations       : 100000 PASS

Canonical 30-model structure : PASS
All 30 Design-A M_m = 50/51